# 06 — Promotion-Aware Forecasting and Causal Uplift

## 1. Objectives and Causal/Forecasting Separation

The first forecasting system intentionally excluded promotion information in
order to establish a clean demand-history baseline.

This notebook extends the platform with promotion information while separating
two distinct questions.

### Predictive question

> Does knowing that a store-family will be on promotion improve future sales
> forecasts?

For forecasting, `onpromotion` is useful whenever the promotion plan is known
at forecast time.

The predictive target remains:

$$
Y_{s,f,T+h},
$$

and the promotion-aware forecasting model becomes:

$$
\hat Y_{s,f,T+h}
=
F(
X_{s,f,T},
C_{T+h},
P_{s,f,T+h},
h
),
$$

where:

- $X_{s,f,T}$ contains information available by forecast origin $T$;
- $C_{T+h}$ contains future-known calendar information;
- $P_{s,f,T+h}$ indicates whether the future store-family observation is
  planned to be on promotion;
- $h$ is the forecast horizon.

### Causal question

A different question is:

> How many additional units would have been sold because of the promotion?

This requires comparing two potential outcomes:

$$
Y(1)
$$

and

$$
Y(0),
$$

where:

- $Y(1)$ is sales if the store-family is promoted;
- $Y(0)$ is sales if the same store-family is not promoted.

The individual treatment effect is:

$$
\tau_i = Y_i(1)-Y_i(0).
$$

Only one potential outcome is observed for each historical observation, so
causal promotion uplift cannot be obtained from predictive accuracy alone.

This notebook therefore treats:

1. **promotion-aware forecasting**, and
2. **causal promotion-uplift estimation**

as related but distinct modeling problems.

The previously validated 14-day XGBoost forecasting system remains frozen as
`point_forecast_v1` and serves as Model A.

Promotion-aware Model B is developed and selected exclusively from historical
rolling temporal deployments.

The August 2–15 period was the untouched final holdout for Model A, but its
labels and diagnostics have already been inspected during the previous
forecasting stage. It is therefore no longer a pristine holdout for subsequent
model development.

After Model B is frozen without using this period for feature or model
selection, August 2–15 will nevertheless be reused as a fixed out-of-time
comparison period, allowing Model A and Model B to be evaluated on exactly the
same future observations.

## 2. Promotion Data Audit

In [1]:
# ---------------------------------------------------------------------
# Inspect available promotion-related data before defining any feature
# or causal treatment.
# ---------------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np


project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent


candidate_files = []


for file_path in project_root.rglob("*.csv"):

    name_lower = file_path.name.lower()

    if any(
        token in name_lower
        for token in [
            "train",
            "test",
            "promo",
        ]
    ):

        candidate_files.append(
            {
                "file":
                    str(
                        file_path.relative_to(
                            project_root
                        )
                    ),

                "size_mb":
                    file_path.stat().st_size
                    / (1024 ** 2),
            }
        )


candidate_files = (
    pd.DataFrame(
        candidate_files
    )
    .sort_values(
        "file"
    )
)


display(
    candidate_files
    .style.hide(
        axis="index"
    )
)

file,size_mb
.venv/lib/python3.12/site-packages/numpy/random/tests/data/mt19937-testset-1.csv,0.015110
.venv/lib/python3.12/site-packages/numpy/random/tests/data/mt19937-testset-2.csv,0.015092
.venv/lib/python3.12/site-packages/numpy/random/tests/data/pcg64-testset-1.csv,0.022735
.venv/lib/python3.12/site-packages/numpy/random/tests/data/pcg64-testset-2.csv,0.022740
.venv/lib/python3.12/site-packages/numpy/random/tests/data/pcg64dxsm-testset-1.csv,0.022729
.venv/lib/python3.12/site-packages/numpy/random/tests/data/pcg64dxsm-testset-2.csv,0.022735
.venv/lib/python3.12/site-packages/numpy/random/tests/data/philox-testset-1.csv,0.022747
.venv/lib/python3.12/site-packages/numpy/random/tests/data/philox-testset-2.csv,0.022734
.venv/lib/python3.12/site-packages/numpy/random/tests/data/sfc64-testset-1.csv,0.022736
.venv/lib/python3.12/site-packages/numpy/random/tests/data/sfc64-testset-2.csv,0.022729


In [2]:
# ---------------------------------------------------------------------
# Audit the actual Favorita raw-data inventory only.
#
# The previous recursive search also inspected the Python virtual
# environment. Restricting the search to the dataset directory gives us
# the real project data contract.
# ---------------------------------------------------------------------

from pathlib import Path
import pandas as pd


favorita_raw_dir = (
    project_root
    / "data"
    / "raw"
    / "favorita"
)


assert (
    favorita_raw_dir.exists()
), (
    f"Favorita raw directory not found: {favorita_raw_dir}"
)


favorita_files = []


for file_path in favorita_raw_dir.rglob("*"):

    if not file_path.is_file():
        continue

    # Ignore notebook-generated checkpoint copies.
    if ".ipynb_checkpoints" in file_path.parts:
        continue

    favorita_files.append(
        {
            "file":
                str(
                    file_path.relative_to(
                        project_root
                    )
                ),

            "extension":
                file_path.suffix.lower(),

            "size_mb":
                file_path.stat().st_size
                / (1024 ** 2),
        }
    )


favorita_files = (
    pd.DataFrame(
        favorita_files
    )
    .sort_values(
        "file"
    )
    .reset_index(
        drop=True
    )
)


display(
    favorita_files
    .style.hide(
        axis="index"
    )
    .format(
        {
            "size_mb":
                "{:.3f}",
        }
    )
)

file,extension,size_mb
data/raw/favorita/.DS_Store,,0.012
data/raw/favorita/holidays_events/holidays_events.csv,.csv,0.021
data/raw/favorita/oil/oil.csv,.csv,0.020
data/raw/favorita/stores/stores.csv,.csv,0.001
data/raw/favorita/train/train.csv,.csv,116.158
data/raw/favorita/transactions/transactions.csv,.csv,1.481


### 2.1 Data Contract and Availability

The promotion signal is stored directly in the historical sales table.

For each `store_nbr × family × date` observation, `onpromotion` measures the
number of products within that family that are under promotion in the store on
that date.

It is therefore not simply a binary flag and does not represent the number of
promoted units sold.

For forecasting, this variable can act as a future-known promotion-intensity
signal when the promotion plan is available at forecast time.

For causal analysis, the initial treatment will instead be defined as:

$$
D_{s,f,t}
=
\mathbb{1}
\left[
\text{onpromotion}_{s,f,t} > 0
\right].
$$

This creates a binary treatment corresponding to:

- $D=1$: at least one product in the store-family is promoted;
- $D=0$: no product in the store-family is promoted.

The continuous promotion count will still be retained for descriptive and
predictive analysis. A continuous dose-response causal analysis is considered
a future extension because it requires substantially stronger overlap and
modeling assumptions.

Before any modeling decision, the historical distribution, missingness and
temporal coverage of `onpromotion` are audited below.

In [3]:
# ---------------------------------------------------------------------
# Load only the columns required for the initial promotion audit.
#
# This stage is descriptive only: no model is trained and no feature
# transformation is selected yet.
# ---------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


train_path = (
    project_root
    / "data"
    / "raw"
    / "favorita"
    / "train"
    / "train.csv"
)


assert train_path.exists(), (
    f"Training file not found: {train_path}"
)


promotion_sales = pd.read_csv(
    train_path,
    usecols=[
        "id",
        "date",
        "store_nbr",
        "family",
        "sales",
        "onpromotion",
    ],
    parse_dates=[
        "date",
    ],
)


# ---------------------------------------------------------------------
# Basic data contract.
# ---------------------------------------------------------------------

promotion_contract = pd.DataFrame(
    {
        "column":
            promotion_sales.columns,

        "dtype":
            [
                str(dtype)
                for dtype
                in promotion_sales.dtypes
            ],

        "missing_rows":
            [
                int(
                    promotion_sales[column]
                    .isna()
                    .sum()
                )
                for column
                in promotion_sales.columns
            ],

        "missing_rate":
            [
                promotion_sales[column]
                .isna()
                .mean()
                for column
                in promotion_sales.columns
            ],
    }
)


display(
    promotion_contract
    .style.hide(
        axis="index"
    )
    .format(
        {
            "missing_rate":
                "{:.4%}",
        }
    )
)


# ---------------------------------------------------------------------
# Promotion-count distribution.
# ---------------------------------------------------------------------

promotion_quantiles = (
    promotion_sales[
        "onpromotion"
    ]
    .quantile(
        [
            0.00,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999,
            1.00,
        ]
    )
    .rename(
        "onpromotion"
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "quantile",
        }
    )
)


display(
    promotion_quantiles
    .style.hide(
        axis="index"
    )
    .format(
        {
            "quantile":
                "{:.3f}",

            "onpromotion":
                "{:.3f}",
        }
    )
)


promotion_summary = pd.DataFrame(
    [
        {
            "rows":
                len(
                    promotion_sales
                ),

            "date_min":
                promotion_sales[
                    "date"
                ]
                .min(),

            "date_max":
                promotion_sales[
                    "date"
                ]
                .max(),

            "unique_stores":
                promotion_sales[
                    "store_nbr"
                ]
                .nunique(),

            "unique_families":
                promotion_sales[
                    "family"
                ]
                .nunique(),

            "onpromotion_min":
                promotion_sales[
                    "onpromotion"
                ]
                .min(),

            "onpromotion_max":
                promotion_sales[
                    "onpromotion"
                ]
                .max(),

            "onpromotion_mean":
                promotion_sales[
                    "onpromotion"
                ]
                .mean(),

            "onpromotion_median":
                promotion_sales[
                    "onpromotion"
                ]
                .median(),

            "zero_promotion_rate":
                (
                    promotion_sales[
                        "onpromotion"
                    ]
                    == 0
                )
                .mean(),

            "positive_promotion_rate":
                (
                    promotion_sales[
                        "onpromotion"
                    ]
                    > 0
                )
                .mean(),

            "missing_promotion_rate":
                promotion_sales[
                    "onpromotion"
                ]
                .isna()
                .mean(),

            "unique_promotion_counts":
                promotion_sales[
                    "onpromotion"
                ]
                .nunique(
                    dropna=True
                ),
        }
    ]
)


display(
    promotion_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "onpromotion_mean":
                "{:.3f}",

            "onpromotion_median":
                "{:.3f}",

            "zero_promotion_rate":
                "{:.2%}",

            "positive_promotion_rate":
                "{:.2%}",

            "missing_promotion_rate":
                "{:.2%}",
        }
    )
)


# ---------------------------------------------------------------------
# Verify whether the raw promotion count behaves as an integer-valued
# intensity variable.
# ---------------------------------------------------------------------

non_missing_promotion = (
    promotion_sales[
        "onpromotion"
    ]
    .dropna()
)


integer_valued = np.allclose(
    non_missing_promotion,
    np.round(
        non_missing_promotion
    ),
)


print(
    "Promotion counts are integer-valued:",
    integer_valued,
)

print(
    "Observed date range:",
    promotion_sales[
        "date"
    ].min().date(),
    "to",
    promotion_sales[
        "date"
    ].max().date(),
)

column,dtype,missing_rows,missing_rate
id,int64,0,0.0000%
date,datetime64[us],0,0.0000%
store_nbr,int64,0,0.0000%
family,str,0,0.0000%
sales,float64,0,0.0000%
onpromotion,int64,0,0.0000%


quantile,onpromotion
0.000,0.000
0.250,0.000
0.500,0.000
0.750,0.000
0.900,4.000
0.950,13.000
0.990,55.000
0.995,76.000
0.999,170.000
1.000,741.000


rows,date_min,date_max,unique_stores,unique_families,onpromotion_min,onpromotion_max,onpromotion_mean,onpromotion_median,zero_promotion_rate,positive_promotion_rate,missing_promotion_rate,unique_promotion_counts
3000888,2013-01-01 00:00:00,2017-08-15 00:00:00,54,33,0,741,2.603,0.000,79.63%,20.37%,0.00%,362


Promotion counts are integer-valued: True
Observed date range: 2013-01-01 to 2017-08-15


### 2.2 Promotion Coverage and Sparsity

Promotion exposure is highly asymmetric.

Approximately 80% of historical `store × family × date` observations have no
promoted product, while roughly 20% have at least one product under promotion.

The positive promotion-count distribution is also strongly right-skewed.
Therefore, the aggregate promotion rate alone is not sufficient to determine
whether the signal is usable.

Before modeling, three additional questions must be answered:

1. **Temporal coverage:** has promotion information been recorded consistently
   throughout the historical period?
2. **Cross-sectional coverage:** are promotions concentrated in only a small
   subset of stores, families or store-family series?
3. **Treatment overlap:** do individual series contain both promoted and
   non-promoted observations, or is promotion status nearly deterministic for
   some segments?

These questions matter for both forecasting and causal inference.

For forecasting, a signal with changing historical coverage can create
distribution shift.

For causal inference, treatment variation within comparable historical states
is essential: if some segments are effectively always treated or never treated,
their counterfactual outcomes cannot be identified from the observed data
without strong extrapolation.

In [4]:
# ---------------------------------------------------------------------
# Promotion coverage across time and store-family series.
#
# This audit remains purely descriptive. The binary treatment is created
# for analysis, but no causal effect is estimated at this stage.
# ---------------------------------------------------------------------

promotion_sales = (
    promotion_sales
    .copy()
)


promotion_sales[
    "promo_flag"
] = (
    promotion_sales[
        "onpromotion"
    ]
    > 0
).astype(
    "int8"
)


promotion_sales[
    "year"
] = (
    promotion_sales[
        "date"
    ]
    .dt.year
)


promotion_sales[
    "month"
] = (
    promotion_sales[
        "date"
    ]
    .dt.to_period(
        "M"
    )
    .astype(
        str
    )
)


# ---------------------------------------------------------------------
# First observed promotion date.
# ---------------------------------------------------------------------

positive_promotion_rows = (
    promotion_sales.loc[
        promotion_sales[
            "promo_flag"
        ]
        == 1
    ]
)


first_positive_promotion_date = (
    positive_promotion_rows[
        "date"
    ]
    .min()
)


print(
    "First positive promotion date:",
    first_positive_promotion_date.date(),
)


# ---------------------------------------------------------------------
# Annual promotion coverage.
# ---------------------------------------------------------------------

annual_promotion_coverage = (
    promotion_sales
    .groupby(
        "year",
        as_index=False,
    )
    .agg(
        rows=(
            "promo_flag",
            "size",
        ),

        promoted_rows=(
            "promo_flag",
            "sum",
        ),

        promotion_rate=(
            "promo_flag",
            "mean",
        ),

        mean_promotion_count=(
            "onpromotion",
            "mean",
        ),

        max_promotion_count=(
            "onpromotion",
            "max",
        ),
    )
)


positive_count_by_year = (
    promotion_sales.loc[
        promotion_sales[
            "promo_flag"
        ]
        == 1
    ]
    .groupby(
        "year",
        as_index=False,
    )
    .agg(
        mean_count_when_promoted=(
            "onpromotion",
            "mean",
        ),

        median_count_when_promoted=(
            "onpromotion",
            "median",
        ),
    )
)


annual_promotion_coverage = (
    annual_promotion_coverage
    .merge(
        positive_count_by_year,
        on="year",
        how="left",
        validate="one_to_one",
    )
)


display(
    annual_promotion_coverage
    .style.hide(
        axis="index"
    )
    .format(
        {
            "promotion_rate":
                "{:.2%}",

            "mean_promotion_count":
                "{:.3f}",

            "mean_count_when_promoted":
                "{:.3f}",

            "median_count_when_promoted":
                "{:.3f}",
        }
    )
)


# ---------------------------------------------------------------------
# Monthly coverage near the development period.
#
# This lets us detect abrupt changes in promotion prevalence that annual
# averages could hide.
# ---------------------------------------------------------------------

monthly_promotion_coverage = (
    promotion_sales
    .groupby(
        "month",
        as_index=False,
    )
    .agg(
        rows=(
            "promo_flag",
            "size",
        ),

        promoted_rows=(
            "promo_flag",
            "sum",
        ),

        promotion_rate=(
            "promo_flag",
            "mean",
        ),

        mean_promotion_count=(
            "onpromotion",
            "mean",
        ),

        max_promotion_count=(
            "onpromotion",
            "max",
        ),
    )
)


print(
    "Most recent 18 months:"
)


display(
    monthly_promotion_coverage
    .tail(
        18
    )
    .style.hide(
        axis="index"
    )
    .format(
        {
            "promotion_rate":
                "{:.2%}",

            "mean_promotion_count":
                "{:.3f}",
        }
    )
)


# ---------------------------------------------------------------------
# Store-family treatment variation.
#
# Causal identification is much more credible for series where both
# treatment states are historically observed.
# ---------------------------------------------------------------------

series_promotion_coverage = (
    promotion_sales
    .groupby(
        [
            "store_nbr",
            "family",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        observed_rows=(
            "promo_flag",
            "size",
        ),

        promoted_rows=(
            "promo_flag",
            "sum",
        ),

        promotion_rate=(
            "promo_flag",
            "mean",
        ),

        mean_promotion_count=(
            "onpromotion",
            "mean",
        ),

        max_promotion_count=(
            "onpromotion",
            "max",
        ),
    )
)


series_promotion_coverage[
    "untreated_rows"
] = (
    series_promotion_coverage[
        "observed_rows"
    ]
    - series_promotion_coverage[
        "promoted_rows"
    ]
)


series_promotion_coverage[
    "has_both_treatment_states"
] = (
    (
        series_promotion_coverage[
            "promoted_rows"
        ]
        > 0
    )
    &
    (
        series_promotion_coverage[
            "untreated_rows"
        ]
        > 0
    )
)


series_coverage_summary = pd.DataFrame(
    [
        {
            "series_count":
                len(
                    series_promotion_coverage
                ),

            "never_promoted_series":
                int(
                    (
                        series_promotion_coverage[
                            "promoted_rows"
                        ]
                        == 0
                    )
                    .sum()
                ),

            "always_promoted_series":
                int(
                    (
                        series_promotion_coverage[
                            "untreated_rows"
                        ]
                        == 0
                    )
                    .sum()
                ),

            "series_with_both_states":
                int(
                    series_promotion_coverage[
                        "has_both_treatment_states"
                    ]
                    .sum()
                ),

            "share_with_both_states":
                series_promotion_coverage[
                    "has_both_treatment_states"
                ]
                .mean(),

            "median_series_promotion_rate":
                series_promotion_coverage[
                    "promotion_rate"
                ]
                .median(),

            "p90_series_promotion_rate":
                series_promotion_coverage[
                    "promotion_rate"
                ]
                .quantile(
                    0.90
                ),

            "max_series_promotion_rate":
                series_promotion_coverage[
                    "promotion_rate"
                ]
                .max(),
        }
    ]
)


display(
    series_coverage_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "share_with_both_states":
                "{:.2%}",

            "median_series_promotion_rate":
                "{:.2%}",

            "p90_series_promotion_rate":
                "{:.2%}",

            "max_series_promotion_rate":
                "{:.2%}",
        }
    )
)


# ---------------------------------------------------------------------
# Quantiles of store-family promotion prevalence.
# ---------------------------------------------------------------------

series_promotion_rate_quantiles = (
    series_promotion_coverage[
        "promotion_rate"
    ]
    .quantile(
        [
            0.00,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            1.00,
        ]
    )
    .rename(
        "promotion_rate"
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "quantile",
        }
    )
)


display(
    series_promotion_rate_quantiles
    .style.hide(
        axis="index"
    )
    .format(
        {
            "quantile":
                "{:.2f}",

            "promotion_rate":
                "{:.2%}",
        }
    )
)

First positive promotion date: 2014-04-01


year,rows,promoted_rows,promotion_rate,mean_promotion_count,max_promotion_count,mean_count_when_promoted,median_count_when_promoted
2013,648648,0,0.00%,0.000,0,nan,nan
2014,648648,64979,10.02%,0.708,205,7.066,2.000
2015,648648,135302,20.86%,1.676,235,8.036,2.000
2016,650430,227981,35.05%,5.403,741,15.416,5.000
2017,404514,183067,45.26%,6.797,697,15.020,6.000


Most recent 18 months:


month,rows,promoted_rows,promotion_rate,mean_promotion_count,max_promotion_count
2016-03,55242,13481,24.40%,3.291,228
2016-04,53460,15722,29.41%,4.028,710
2016-05,55242,21994,39.81%,6.698,741
2016-06,53460,21842,40.86%,5.339,237
2016-07,55242,22599,40.91%,4.790,229
2016-08,55242,20129,36.44%,4.916,236
2016-09,53460,19170,35.86%,5.544,244
2016-10,55242,22406,40.56%,6.707,244
2016-11,53460,23720,44.37%,8.200,243
2016-12,53460,25513,47.72%,9.680,251


series_count,never_promoted_series,always_promoted_series,series_with_both_states,share_with_both_states,median_series_promotion_rate,p90_series_promotion_rate,max_series_promotion_rate
1782,182,0,1600,89.79%,15.08%,51.60%,70.78%


quantile,promotion_rate
0.00,0.00%
0.10,0.00%
0.25,1.19%
0.50,15.08%
0.75,35.69%
0.90,51.60%
0.95,61.28%
0.99,67.47%
1.00,70.78%


### 2.3 Promotion Dynamics Across Stores and Families

The promotion field exhibits a major temporal discontinuity.

No positive `onpromotion` value appears before April 1, 2014, after which
promotion prevalence rises substantially over time.

This pattern must not automatically be interpreted as evidence that Favorita
ran no promotions before that date.

The underlying item-level Favorita data historically contained unavailable
promotion-status values during early periods. Therefore, early zeros in the
family-level dataset may reflect promotion-data availability rather than a
true untreated state.

This distinction is particularly important for causal inference:

$$
D=0
$$

must represent a genuine absence of treatment, not an unknown treatment state
encoded as zero.

The transition around April 2014 is therefore audited before defining the
causal estimation population.

For predictive modeling, the issue is less severe because the selected
forecasting procedure uses recent rolling training windows, where promotion
information is substantially populated.

In [5]:
# ---------------------------------------------------------------------
# Inspect the transition from an all-zero promotion field to meaningful
# promotion coverage.
#
# A sharp synchronized start would suggest a data-availability regime
# change rather than a genuine retailer-wide switch from zero promotion.
# ---------------------------------------------------------------------

promotion_transition_daily = (
    promotion_sales
    .loc[
        promotion_sales[
            "date"
        ].between(
            pd.Timestamp(
                "2014-03-01"
            ),
            pd.Timestamp(
                "2014-05-15"
            ),
        )
    ]
    .groupby(
        "date",
        as_index=False,
    )
    .agg(
        rows=(
            "promo_flag",
            "size",
        ),

        promoted_rows=(
            "promo_flag",
            "sum",
        ),

        promotion_rate=(
            "promo_flag",
            "mean",
        ),

        promoted_stores=(
            "store_nbr",
            lambda values:
                promotion_sales.loc[
                    values.index
                ]
                .loc[
                    lambda frame:
                        frame[
                            "promo_flag"
                        ]
                        == 1,
                    "store_nbr",
                ]
                .nunique(),
        ),

        promoted_families=(
            "family",
            lambda values:
                promotion_sales.loc[
                    values.index
                ]
                .loc[
                    lambda frame:
                        frame[
                            "promo_flag"
                        ]
                        == 1,
                    "family",
                ]
                .nunique(),
        ),
    )
)


display(
    promotion_transition_daily
    .loc[
        (
            promotion_transition_daily[
                "promoted_rows"
            ]
            > 0
        )
        |
        (
            promotion_transition_daily[
                "date"
            ]
            >= pd.Timestamp(
                "2014-03-25"
            )
        )
    ]
    .head(
        45
    )
    .style.hide(
        axis="index"
    )
    .format(
        {
            "promotion_rate":
                "{:.2%}",
        }
    )
)


# ---------------------------------------------------------------------
# Monthly breadth: how many stores, families and store-family series
# actually show at least one promotion each month?
# ---------------------------------------------------------------------

promotion_monthly_breadth = (
    promotion_sales
    .loc[
        promotion_sales[
            "date"
        ]
        >= pd.Timestamp(
            "2014-01-01"
        )
    ]
    .assign(
        month=lambda frame:
            frame[
                "date"
            ]
            .dt.to_period(
                "M"
            )
            .astype(
                str
            )
    )
)


promotion_monthly_breadth = (
    promotion_monthly_breadth
    .loc[
        promotion_monthly_breadth[
            "promo_flag"
        ]
        == 1
    ]
    .groupby(
        "month",
        as_index=False,
    )
    .agg(
        promoted_rows=(
            "promo_flag",
            "size",
        ),

        promoted_stores=(
            "store_nbr",
            "nunique",
        ),

        promoted_families=(
            "family",
            "nunique",
        ),

        promoted_series=(
            "family",
            lambda values:
                promotion_sales.loc[
                    values.index,
                    [
                        "store_nbr",
                        "family",
                    ],
                ]
                .drop_duplicates()
                .shape[
                    0
                ],
        ),
    )
)


print(
    "Promotion breadth during the transition period:"
)


display(
    promotion_monthly_breadth
    .loc[
        promotion_monthly_breadth[
            "month"
        ]
        <= "2015-06"
    ]
    .style.hide(
        axis="index"
    )
)

date,rows,promoted_rows,promotion_rate,promoted_stores,promoted_families
2014-03-25 00:00:00,1782,0,0.00%,0,0
2014-03-26 00:00:00,1782,0,0.00%,0,0
2014-03-27 00:00:00,1782,0,0.00%,0,0
2014-03-28 00:00:00,1782,0,0.00%,0,0
2014-03-29 00:00:00,1782,0,0.00%,0,0
2014-03-30 00:00:00,1782,0,0.00%,0,0
2014-03-31 00:00:00,1782,0,0.00%,0,0
2014-04-01 00:00:00,1782,2,0.11%,1,2
2014-04-02 00:00:00,1782,2,0.11%,1,2
2014-04-03 00:00:00,1782,7,0.39%,1,7


Promotion breadth during the transition period:


month,promoted_rows,promoted_stores,promoted_families,promoted_series
2014-04,399,5,16,59
2014-05,1964,24,19,285
2014-06,4367,45,22,577
2014-07,8950,48,30,844
2014-08,7670,48,22,659
2014-09,9346,48,29,808
2014-10,10624,48,29,869
2014-11,9309,48,27,832
2014-12,12350,48,28,842
2015-01,10336,48,23,754


The early promotion history exhibits a clear regime transition.

Positive promotion observations begin only in April 2014 and subsequently
expand across stores and families over several months. This pattern cannot be
unambiguously separated into genuine promotion adoption versus changes in
promotion-data coverage from the aggregated dataset alone.

The causal analysis will therefore not rely on the early historical period.

Its primary estimation population will be restricted to the recent two-year
operating regime:

$$
\text{2015-08-16 through 2017-08-15}.
$$

This choice is independent of the forecasting model's two-year training-window
selection: here the motivation is data comparability, promotion-data maturity
and relevance to the current operating regime.

Observations will still be subject to later propensity-overlap diagnostics;
belonging to this recent period alone does not guarantee causal
identifiability.

### 2.4 Point-in-Time Safety

Promotion information has a different temporal status from historical sales
lags.

For a deployment at forecast origin $T$, a promotion value for target date
$T+h$ is usable only if the promotion plan for that date is known at $T$.

The promotion-aware forecasting experiment therefore assumes an operational
setting in which future promotions are planned before the forecast is issued.

Under this assumption:

$$
\text{onpromotion}_{s,f,T+h}
$$

is a **future-known covariate**, not target leakage.

This differs from variables such as realized future sales or same-day future
transactions, which would only become observable after the target date.

The experiment will preserve the forecasting protocol established for
`point_forecast_v1`:

$$
\hat Y_{s,f,T+h}
=
F(
X_{s,f,T},
C_{T+h},
P_{s,f,T+h},
h
).
$$

All historical sales-derived predictors remain restricted to information
available by origin $T$, while promotion information may refer to $T+h$
because it represents the pre-planned future promotion schedule.

The same six recent rolling validation origins and the same two-year training
policy used for Model A will be reused for Model B.

The only initial experimental change is the addition of promotion information,
allowing its incremental predictive value to be isolated.

## 3. Promotion-Aware Forecasting

The objective of this section is to measure the incremental predictive value of
future-known promotion information while keeping the previously validated
forecasting architecture fixed.

The frozen reference model is:

$$
A = \texttt{point\_forecast\_v1}.
$$

Three promotion specifications are considered:

$$
B_1
=
A
+
\mathbb{1}
[
\text{onpromotion}_{s,f,T+h}>0
],
$$

$$
B_2
=
A
+
\text{onpromotion}_{s,f,T+h},
$$

and, if justified by the strongly right-skewed promotion distribution,

$$
B_3
=
A
+
\log
\left(
1+\text{onpromotion}_{s,f,T+h}
\right).
$$

The comparison is intentionally controlled:

- the same global XGBoost architecture is used;
- the same 14-day forecast horizon is used;
- the same leakage-safe Version 1 predictors are retained;
- the same two-year rolling training policy is used;
- the same six recent validation origins are used;
- the same `candidate_27` structural hyperparameters are used;
- the same fixed boosting length is used.

The only experimental change is the representation of future promotion
information.

This allows the experiment to answer:

> How much predictive value does the planned promotion schedule add beyond the
> previously validated demand-history model?

### 3.1 Experimental Protocol

Promotion representations are compared using paired rolling temporal
backtests.

For each validation origin $V$, all candidate models receive:

- exactly the same eligible training observations;
- exactly the same validation observations;
- the same historical and target-aligned base predictors;
- the same XGBoost hyperparameters and boosting length.

Only the promotion representation changes.

This paired design prevents differences in sampling, target periods or
training-history eligibility from being mistaken for promotion-feature gains.

Model selection is based exclusively on the same six recent rolling
deployments previously used for `point_forecast_v1`.

The primary metric remains WAPE. MAE, RMSE and normalized bias remain
secondary diagnostics.

Promotion-aware models will additionally be evaluated separately on:

- promoted target observations;
- non-promoted target observations;
- promotion-intensity buckets;
- forecast horizon.

This is important because a feature may improve portfolio WAPE primarily by
improving promoted observations while leaving the much larger untreated
population nearly unchanged.

### 3.2 Forecast Feature Extension

Promotion information is joined at the forecast-target grain.

For a supervised observation indexed by:

$$
(s,f,T,h),
$$

the corresponding target date is:

$$
D=T+h.
$$

The promotion feature is therefore:

$$
P_{s,f,D}
=
\text{onpromotion}_{s,f,D}.
$$

Because the promotion schedule is assumed known at origin $T$, this feature is
point-in-time safe even though it refers to the future target date.

No historical sales feature is recomputed in this section. The previously
validated Version 1 supervised forecasting dataset is reused and augmented
only with target-date promotion information.

In [6]:
# ---------------------------------------------------------------------
# Locate reusable supervised / forecasting feature artifacts produced by
# Notebook 05.
#
# We intentionally reuse the validated feature-engineering output rather
# than rebuilding millions of historical rows in this notebook.
# ---------------------------------------------------------------------

from pathlib import Path
import pandas as pd


search_roots = [
    project_root / "data",
    project_root / "artifacts",
]


candidate_feature_artifacts = []


for search_root in search_roots:

    if not search_root.exists():
        continue

    for file_path in search_root.rglob("*"):

        if not file_path.is_file():
            continue

        if ".ipynb_checkpoints" in file_path.parts:
            continue

        if file_path.suffix.lower() not in {
            ".parquet",
            ".csv",
            ".pkl",
            ".pickle",
        }:
            continue

        name_lower = file_path.name.lower()

        if any(
            token in name_lower
            for token in [
                "supervised",
                "feature",
                "xgboost",
                "development",
            ]
        ):

            candidate_feature_artifacts.append(
                {
                    "file":
                        str(
                            file_path.relative_to(
                                project_root
                            )
                        ),

                    "extension":
                        file_path.suffix.lower(),

                    "size_mb":
                        file_path.stat().st_size
                        / (1024 ** 2),
                }
            )


candidate_feature_artifacts = (
    pd.DataFrame(
        candidate_feature_artifacts
    )
    .sort_values(
        [
            "size_mb",
            "file",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


display(
    candidate_feature_artifacts
    .style.hide(
        axis="index"
    )
    .format(
        {
            "size_mb":
                "{:.2f}",
        }
    )
)

file,extension,size_mb
data/global_xgboost_development_sales.csv,.csv,110.21


In [7]:
# ---------------------------------------------------------------------
# Inspect relevant S3 prefixes for reusable supervised feature artifacts.
# ---------------------------------------------------------------------

import boto3


S3_BUCKET = (
    "retail-demand-forecasting-platform-aws-zaamine"
)


s3_client = boto3.client(
    "s3",
    region_name="eu-west-1",
)


s3_feature_candidates = []


for prefix in [
    "features/",
    "artifacts/",
]:

    paginator = (
        s3_client
        .get_paginator(
            "list_objects_v2"
        )
    )

    for page in paginator.paginate(
        Bucket=S3_BUCKET,
        Prefix=prefix,
    ):

        for obj in page.get(
            "Contents",
            [],
        ):

            key = obj[
                "Key"
            ]

            key_lower = key.lower()

            if any(
                token in key_lower
                for token in [
                    "supervised",
                    "feature",
                    "xgboost",
                    "development",
                ]
            ):

                s3_feature_candidates.append(
                    {
                        "key":
                            key,

                        "size_mb":
                            obj[
                                "Size"
                            ]
                            / (1024 ** 2),

                        "last_modified":
                            obj[
                                "LastModified"
                            ],
                    }
                )


s3_feature_candidates = (
    pd.DataFrame(
        s3_feature_candidates
    )
)


if len(
    s3_feature_candidates
) > 0:

    s3_feature_candidates = (
        s3_feature_candidates
        .sort_values(
            [
                "size_mb",
                "key",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )


display(
    s3_feature_candidates
    .style.hide(
        axis="index"
    )
    .format(
        {
            "size_mb":
                "{:.2f}",
        }
    )
)

key,size_mb,last_modified
features/global_xgboost/origin_features/v1/development_end=2017-08-01/origin_features.parquet,97.62,2026-11-01 11:37:57+00:00
artifacts/global_xgboost/v1/xgboost_point_forecast_v1.json,94.69,2026-11-03 21:18:15+00:00
artifacts/global_xgboost/v1/model.tar.gz,26.99,2026-11-03 21:43:38+00:00
features/global_xgboost/supervised_training/v1/development_end=2017-08-01/part-00022.parquet,4.91,2026-11-01 11:38:31+00:00
features/global_xgboost/supervised_training/v1/development_end=2017-08-01/part-00020.parquet,4.87,2026-11-01 11:38:29+00:00
features/global_xgboost/supervised_training/v1/development_end=2017-08-01/part-00021.parquet,4.85,2026-11-01 11:38:30+00:00
features/global_xgboost/supervised_training/v1/development_end=2017-08-01/part-00019.parquet,4.78,2026-11-01 11:38:28+00:00
features/global_xgboost/supervised_training/v1/development_end=2017-08-01/part-00018.parquet,4.58,2026-11-01 11:38:26+00:00
features/global_xgboost/supervised_training/v1/development_end=2017-08-01/part-00017.parquet,4.40,2026-11-01 11:38:25+00:00
features/global_xgboost/supervised_training/v1/development_end=2017-08-01/part-00015.parquet,4.39,2026-11-01 11:38:23+00:00


In [8]:
# ---------------------------------------------------------------------
# Recover the frozen Model A contract and inspect the exact schema of the
# validated supervised-training artifact.
#
# Before rebuilding historical folds, we verify that all eligibility and
# metadata columns required by the original training protocol are still
# available. This avoids silently changing Model A's training population.
# ---------------------------------------------------------------------

from pathlib import Path
import json

import boto3
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.fs as pafs


S3_BUCKET = (
    "retail-demand-forecasting-platform-aws-zaamine"
)

S3_REGION = (
    "eu-west-1"
)

SUPERVISED_PREFIX = (
    "features/global_xgboost/"
    "supervised_training/v1/"
    "development_end=2017-08-01/"
)


# ---------------------------------------------------------------------
# Load the frozen point_forecast_v1 contract from Git-versioned metadata.
# ---------------------------------------------------------------------

contract_path = (
    project_root
    / "artifacts"
    / "global_xgboost"
    / "point_forecast_model_contract_v1.json"
)


assert contract_path.exists(), (
    f"Model contract not found: {contract_path}"
)


with open(
    contract_path,
    "r",
    encoding="utf-8",
) as file:

    point_forecast_contract = json.load(
        file
    )


MODEL_FEATURES_A = list(
    point_forecast_contract[
        "model_features"
    ]
)


print(
    "Frozen Model A:",
    point_forecast_contract[
        "champion_id"
    ],
)

print(
    "Training policy:",
    point_forecast_contract[
        "training_policy"
    ],
)

print(
    "n_estimators:",
    point_forecast_contract[
        "n_estimators"
    ],
)

print(
    "Base feature count:",
    len(
        MODEL_FEATURES_A
    ),
)

print()


# ---------------------------------------------------------------------
# Locate the exact parquet parts in S3.
# ---------------------------------------------------------------------

s3_client = boto3.client(
    "s3",
    region_name=S3_REGION,
)


part_keys = []


paginator = (
    s3_client
    .get_paginator(
        "list_objects_v2"
    )
)


for page in paginator.paginate(
    Bucket=S3_BUCKET,
    Prefix=SUPERVISED_PREFIX,
):

    for obj in page.get(
        "Contents",
        [],
    ):

        key = obj[
            "Key"
        ]

        if (
            Path(
                key
            ).name.startswith(
                "part-"
            )
            and key.endswith(
                ".parquet"
            )
        ):

            part_keys.append(
                key
            )


part_keys = sorted(
    part_keys
)


assert len(
    part_keys
) > 0


print(
    "Parquet parts found:",
    len(
        part_keys
    ),
)

print(
    "First part:",
    part_keys[
        0
    ],
)

print()


# ---------------------------------------------------------------------
# Read only one part for schema inspection.
# ---------------------------------------------------------------------

s3_filesystem = pafs.S3FileSystem(
    region=S3_REGION
)


sample_table = pq.read_table(
    f"{S3_BUCKET}/{part_keys[0]}",
    filesystem=s3_filesystem,
)


sample_supervised = (
    sample_table
    .to_pandas()
)


print(
    "Sample part shape:",
    sample_supervised.shape,
)

print()

print(
    "Supervised columns:"
)

for column in sample_supervised.columns:

    print(
        "-",
        column,
        ":",
        sample_supervised[
            column
        ].dtype,
    )


# ---------------------------------------------------------------------
# Confirm that the frozen Model A feature contract is fully represented.
# ---------------------------------------------------------------------

missing_model_features = [
    feature
    for feature
    in MODEL_FEATURES_A
    if feature
    not in sample_supervised.columns
]


assert (
    len(
        missing_model_features
    )
    == 0
), (
    "Frozen Model A features missing from supervised artifact: "
    f"{missing_model_features}"
)


# ---------------------------------------------------------------------
# Surface metadata columns potentially involved in training eligibility,
# fold construction or target attachment.
# ---------------------------------------------------------------------

metadata_candidates = [
    column
    for column
    in sample_supervised.columns
    if any(
        token
        in column.lower()
        for token
        in [
            "origin",
            "target",
            "observ",
            "history",
            "active",
            "start",
            "availability",
        ]
    )
]


print()
print(
    "Potential fold / eligibility metadata:"
)

for column in metadata_candidates:

    print(
        "-",
        column,
    )


print()
print(
    "SUPERVISED ARTIFACT SCHEMA AUDIT: PASSED"
)

Frozen Model A: candidate_27
Training policy: last_2_years
n_estimators: 3200
Base feature count: 31

Parquet parts found: 24
First part: features/global_xgboost/supervised_training/v1/development_end=2017-08-01/part-00000.parquet

Sample part shape: (249480, 38)

Supervised columns:
- store_nbr : int16
- family : string
- forecast_origin : datetime64[us]
- horizon : int8
- target_date : datetime64[us]
- sales_observed : bool
- history_days_available : int16
- sales_at_origin : float32
- sales_lag_1d : float32
- sales_lag_2d : float32
- sales_lag_3d : float32
- sales_lag_7d : float32
- sales_lag_14d : float32
- sales_lag_21d : float32
- sales_lag_28d : float32
- sales_lag_35d : float32
- sales_lag_364d : float32
- rolling_mean_7d : float32
- rolling_mean_14d : float32
- rolling_mean_28d : float32
- recent_level_ratio : float32
- rolling_std_28d : float32
- observed_days_28d : float32
- zero_share_28d : float32
- positive_rate_28d : float32
- positive_sales_mean_28d : float32
- days_sin

In [9]:
# ---------------------------------------------------------------------
# Load the reusable supervised Version 1 dataset and attach future-known
# promotion information at the target-date grain.
#
# Important:
# - only Parquet part files are included in the PyArrow dataset;
# - manifest.csv is deliberately excluded;
# - only the historical scope required by the six recent rolling folds is
#   loaded;
# - promotion features change predictors only, never row eligibility or
#   target values.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as pads
import pyarrow.fs as pafs


VALIDATION_ORIGINS = pd.to_datetime(
    [
        "2017-05-09",
        "2017-05-23",
        "2017-06-06",
        "2017-06-20",
        "2017-07-04",
        "2017-07-18",
    ]
)


FORECAST_HORIZON = 14

LABEL_COLUMN = "target_sales"


# ---------------------------------------------------------------------
# The earliest recent fold uses a rolling two-year training window.
# Loading anything older is unnecessary for this controlled experiment.
# ---------------------------------------------------------------------

EARLIEST_REQUIRED_ORIGIN = (
    VALIDATION_ORIGINS.min()
    - pd.DateOffset(
        years=2
    )
)


LATEST_REQUIRED_TRAINING_TARGET = (
    VALIDATION_ORIGINS.max()
)


SUPERVISED_DATASET_S3_URI = (
    "s3://"
    f"{S3_BUCKET}/"
    "features/global_xgboost/"
    "supervised_training/v1/"
    "development_end=2017-08-01/"
)


supervised_fs, supervised_base_path = (
    pafs.FileSystem.from_uri(
        SUPERVISED_DATASET_S3_URI
    )
)


# ---------------------------------------------------------------------
# Discover ONLY the physical Parquet parts.
#
# The same pattern was used in Notebook 05. The directory also contains
# manifest.csv, which must not be interpreted as a Parquet fragment.
# ---------------------------------------------------------------------

supervised_s3_files = sorted(
    [
        file_info.path

        for file_info
        in supervised_fs.get_file_info(
            pafs.FileSelector(
                supervised_base_path,
                recursive=True,
            )
        )

        if (
            file_info.type
            == pafs.FileType.File

            and file_info.path.endswith(
                ".parquet"
            )
        )
    ]
)


assert (
    len(
        supervised_s3_files
    )
    == 24
), (
    "Expected exactly 24 Version 1 supervised Parquet parts."
)


supervised_arrow_dataset = (
    pads.dataset(
        supervised_s3_files,
        filesystem=supervised_fs,
        format="parquet",
    )
)


# ---------------------------------------------------------------------
# Read exactly the frozen Model A predictors plus label and temporal
# metadata required for rolling-fold construction.
# ---------------------------------------------------------------------

ROLLING_READ_COLUMNS = list(
    dict.fromkeys(
        MODEL_FEATURES_A
        + [
            LABEL_COLUMN,
            "forecast_origin",
            "target_date",
        ]
    )
)


assert (
    len(
        ROLLING_READ_COLUMNS
    )
    == 34
)


# Both predicates are pushed down into the Parquet scan.
#
# 1. No origin older than the earliest required two-year window is needed.
# 2. No training label later than the latest recent validation origin may
#    be needed for the six development folds.
# ---------------------------------------------------------------------

recent_training_filter = (
    (
        pads.field(
            "forecast_origin"
        )
        >= pa.scalar(
            EARLIEST_REQUIRED_ORIGIN.date(),
            type=pa.date32(),
        )
    )
    &
    (
        pads.field(
            "target_date"
        )
        <= pa.scalar(
            LATEST_REQUIRED_TRAINING_TARGET.date(),
            type=pa.date32(),
        )
    )
)


supervised_recent = (
    supervised_arrow_dataset
    .to_table(
        columns=
            ROLLING_READ_COLUMNS,

        filter=
            recent_training_filter,
    )
    .to_pandas()
)


for column in [
    "forecast_origin",
    "target_date",
]:

    supervised_recent[
        column
    ] = pd.to_datetime(
        supervised_recent[
            column
        ]
    )


rows_before_promotion_join = (
    len(
        supervised_recent
    )
)


# ---------------------------------------------------------------------
# Build the unique historical promotion lookup.
#
# `onpromotion` is aligned with the TARGET date, not the forecast origin,
# because the promotion schedule is assumed known in advance.
# ---------------------------------------------------------------------

promotion_lookup = (
    promotion_sales[
        [
            "store_nbr",
            "family",
            "date",
            "onpromotion",
        ]
    ]
    .rename(
        columns={
            "date":
                "target_date",
        }
    )
    .copy()
)


promotion_lookup[
    "target_date"
] = pd.to_datetime(
    promotion_lookup[
        "target_date"
    ]
)


assert (
    promotion_lookup
    .duplicated(
        [
            "store_nbr",
            "family",
            "target_date",
        ]
    )
    .sum()
    == 0
), (
    "Promotion lookup is not unique at "
    "store_nbr × family × target_date."
)


# ---------------------------------------------------------------------
# Attach the future-known promotion signal.
# ---------------------------------------------------------------------

supervised_promo = (
    supervised_recent
    .merge(
        promotion_lookup,
        on=[
            "store_nbr",
            "family",
            "target_date",
        ],
        how="left",
        validate="many_to_one",
        sort=False,
    )
)


assert (
    len(
        supervised_promo
    )
    == rows_before_promotion_join
), (
    "Promotion join changed the supervised row population."
)


assert (
    supervised_promo[
        "onpromotion"
    ]
    .notna()
    .all()
), (
    "Some supervised target dates have no promotion information."
)


# ---------------------------------------------------------------------
# Construct the three controlled promotion representations.
# ---------------------------------------------------------------------

supervised_promo[
    "promo_flag"
] = (
    supervised_promo[
        "onpromotion"
    ]
    > 0
).astype(
    "int8"
)


supervised_promo[
    "promo_count"
] = (
    supervised_promo[
        "onpromotion"
    ]
    .astype(
        "int16"
    )
)


supervised_promo[
    "promo_log_count"
] = (
    np.log1p(
        supervised_promo[
            "onpromotion"
        ]
    )
    .astype(
        "float32"
    )
)


# ---------------------------------------------------------------------
# Verify the long multi-horizon identity.
# ---------------------------------------------------------------------

expected_target_date = (
    supervised_promo[
        "forecast_origin"
    ]
    + pd.to_timedelta(
        supervised_promo[
            "horizon"
        ],
        unit="D",
    )
)


assert (
    supervised_promo[
        "target_date"
    ]
    == expected_target_date
).all(), (
    "target_date != forecast_origin + horizon."
)


# ---------------------------------------------------------------------
# Freeze the controlled feature contracts.
# ---------------------------------------------------------------------

MODEL_FEATURES_B_FLAG = (
    MODEL_FEATURES_A
    + [
        "promo_flag",
    ]
)


MODEL_FEATURES_B_COUNT = (
    MODEL_FEATURES_A
    + [
        "promo_count",
    ]
)


MODEL_FEATURES_B_LOG = (
    MODEL_FEATURES_A
    + [
        "promo_log_count",
    ]
)


assert (
    len(
        MODEL_FEATURES_A
    )
    == 31
)

assert (
    len(
        MODEL_FEATURES_B_FLAG
    )
    == 32
)

assert (
    len(
        MODEL_FEATURES_B_COUNT
    )
    == 32
)

assert (
    len(
        MODEL_FEATURES_B_LOG
    )
    == 32
)


# ---------------------------------------------------------------------
# Compact audit.
# ---------------------------------------------------------------------

promotion_join_summary = pd.DataFrame(
    [
        {
            "rows":
                len(
                    supervised_promo
                ),

            "origin_min":
                supervised_promo[
                    "forecast_origin"
                ]
                .min(),

            "origin_max":
                supervised_promo[
                    "forecast_origin"
                ]
                .max(),

            "target_min":
                supervised_promo[
                    "target_date"
                ]
                .min(),

            "target_max":
                supervised_promo[
                    "target_date"
                ]
                .max(),

            "promotion_rate":
                supervised_promo[
                    "promo_flag"
                ]
                .mean(),

            "mean_promo_count":
                supervised_promo[
                    "promo_count"
                ]
                .mean(),

            "max_promo_count":
                supervised_promo[
                    "promo_count"
                ]
                .max(),

            "missing_promotion_rows":
                supervised_promo[
                    "promo_count"
                ]
                .isna()
                .sum(),
        }
    ]
)


display(
    promotion_join_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "promotion_rate":
                "{:.2%}",

            "mean_promo_count":
                "{:.3f}",
        }
    )
)


print()
print(
    "Parquet parts:",
    len(
        supervised_s3_files
    ),
)

print(
    "Model A features:",
    len(
        MODEL_FEATURES_A
    ),
)

print(
    "Model B flag features:",
    len(
        MODEL_FEATURES_B_FLAG
    ),
)

print(
    "Model B count features:",
    len(
        MODEL_FEATURES_B_COUNT
    ),
)

print(
    "Model B log-count features:",
    len(
        MODEL_FEATURES_B_LOG
    ),
)

print()

print(
    "PROMOTION FEATURE JOIN: PASSED"
)

rows,origin_min,origin_max,target_min,target_max,promotion_rate,mean_promo_count,max_promo_count,missing_promotion_rows
2849418,2015-05-10 00:00:00,2017-07-11 00:00:00,2015-05-11 00:00:00,2017-07-18 00:00:00,34.13%,4.814,741,0



Parquet parts: 24
Model A features: 31
Model B flag features: 32
Model B count features: 32
Model B log-count features: 32

PROMOTION FEATURE JOIN: PASSED


### 3.3 Frozen Model A baseline— No Promotion Signal vs Model B— Planned Promotion Signal

The previously validated forecasting champion remains frozen as Model A:

$$
A=\texttt{point\_forecast\_v1}.
$$

No feature, hyperparameter, training-history policy or boosting-length decision
is reconsidered in this notebook.

Model A retains:

- the 31-feature Version 1 predictor contract;
- the global direct multi-horizon XGBoost architecture;
- `candidate_27` structural hyperparameters;
- the rolling two-year training-history policy;
- 3,200 boosting trees;
- the original store-activity eligibility rule.

The July 18 development deployment is reconstructed below only as a parity
check.

Its purpose is to verify that the promotion-aware experimental pipeline
reproduces the frozen reference model before any Model B result is interpreted.

Model B extends the frozen reference architecture with future-known planned
promotion information.

Three representations are evaluated:

$$
B_{\mathrm{flag}}
=
A
+
\mathbb{1}
[
\text{onpromotion}_{s,f,T+h}>0
],
$$

$$
B_{\mathrm{count}}
=
A
+
\text{onpromotion}_{s,f,T+h},
$$

and:

$$
B_{\log}
=
A
+
\log
\left(
1+\text{onpromotion}_{s,f,T+h}
\right).
$$

The binary representation tests whether simply knowing that a promotion exists
is sufficient.

The raw count tests whether the number of promoted items carries additional
predictive information.

The logarithmic representation retains the ordering of promotion intensity
while compressing extreme counts. Because XGBoost can already learn nonlinear
thresholds, this transformation is not theoretically required; it is included
as a low-cost empirical sensitivity test.

Before the complete rolling evaluation, all three variants are tested on the
July 18 deployment together with frozen Model A.

In [10]:
# ---------------------------------------------------------------------
# Reconstruct the exact July 18 training scope and validation population.
#
# Training:
# - labels must be observable by the validation origin;
# - forecast origins must fall inside the frozen two-year window;
# - pre-activity store states are excluded.
#
# Validation:
# - historical state comes from the materialized origin-feature layer;
# - target-aligned seasonal features are recovered from the validated
#   supervised artifact;
# - realized sales are used only as offline labels;
# - target-date promotion is attached as a future-known predictor.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as pads
import pyarrow.fs as pafs


SMOKE_VALIDATION_ORIGIN = pd.Timestamp(
    "2017-07-18"
)

SMOKE_VALIDATION_END = (
    SMOKE_VALIDATION_ORIGIN
    + pd.Timedelta(
        days=FORECAST_HORIZON
    )
)


# ---------------------------------------------------------------------
# 1. Reconstruct the frozen point-in-time store-activity definition.
#
# A store becomes eligible on its first positive store-wide sales day and
# remains eligible afterward.
# ---------------------------------------------------------------------

store_daily_sales = (
    promotion_sales
    .groupby(
        [
            "store_nbr",
            "date",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        store_total_sales=(
            "sales",
            lambda values:
                values.sum(
                    min_count=1
                ),
        )
    )
    .sort_values(
        [
            "store_nbr",
            "date",
        ]
    )
    .reset_index(
        drop=True
    )
)


store_daily_sales[
    "store_positive_day"
] = (
    store_daily_sales[
        "store_total_sales"
    ]
    .fillna(
        0.0
    )
    .gt(
        0.0
    )
)


store_daily_sales[
    "store_started"
] = (
    store_daily_sales
    .groupby(
        "store_nbr",
        observed=True,
        sort=False,
    )[
        "store_positive_day"
    ]
    .cummax()
)


store_availability_lookup = (
    store_daily_sales[
        [
            "store_nbr",
            "date",
            "store_started",
        ]
    ]
    .rename(
        columns={
            "date":
                "forecast_origin",
        }
    )
)


assert (
    store_availability_lookup
    .duplicated(
        [
            "store_nbr",
            "forecast_origin",
        ]
    )
    .sum()
    == 0
)


def filter_to_started_store_scope(
    frame,
):
    """
    Retain only rows whose store had already demonstrated positive
    store-wide sales at the forecast origin.
    """

    scoped_frame = (
        frame
        .merge(
            store_availability_lookup,
            on=[
                "store_nbr",
                "forecast_origin",
            ],
            how="left",
            validate="many_to_one",
            sort=False,
        )
    )


    assert (
        scoped_frame[
            "store_started"
        ]
        .notna()
        .all()
    )


    excluded_rows = int(
        (
            ~scoped_frame[
                "store_started"
            ]
        )
        .sum()
    )


    scoped_frame = (
        scoped_frame.loc[
            scoped_frame[
                "store_started"
            ]
        ]
        .drop(
            columns=[
                "store_started",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    return (
        scoped_frame,
        excluded_rows,
    )


# ---------------------------------------------------------------------
# 2. Build the exact two-year training population.
# ---------------------------------------------------------------------

smoke_training_start = (
    SMOKE_VALIDATION_ORIGIN
    - pd.DateOffset(
        years=2
    )
)


smoke_train = (
    supervised_promo.loc[
        (
            supervised_promo[
                "target_date"
            ]
            <= SMOKE_VALIDATION_ORIGIN
        )
        &
        (
            supervised_promo[
                "forecast_origin"
            ]
            >= smoke_training_start
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_smoke_training_rows = (
    len(
        smoke_train
    )
)


(
    smoke_train,
    smoke_excluded_training_rows,
) = filter_to_started_store_scope(
    smoke_train
)


# This is the exact training population previously observed for the
# July 18 rolling two-year policy.
assert (
    len(
        smoke_train
    )
    == 2_524_764
), (
    "July 18 training population does not match frozen Notebook 05."
)


assert (
    smoke_train[
        "target_date"
    ]
    .max()
    <= SMOKE_VALIDATION_ORIGIN
)


assert (
    smoke_train[
        "forecast_origin"
    ]
    .min()
    >= smoke_training_start
)


# ---------------------------------------------------------------------
# 3. Load the origin-state rows for the July 18 historical deployment.
# ---------------------------------------------------------------------

ORIGIN_FEATURES_S3_URI = (
    "s3://"
    f"{S3_BUCKET}/"
    "features/global_xgboost/"
    "origin_features/v1/"
    "development_end=2017-08-01/"
    "origin_features.parquet"
)


(
    origin_fs,
    origin_features_path,
) = pafs.FileSystem.from_uri(
    ORIGIN_FEATURES_S3_URI
)


origin_features_dataset = (
    pads.dataset(
        origin_features_path,
        filesystem=origin_fs,
        format="parquet",
    )
)


origin_date_type = (
    origin_features_dataset
    .schema
    .field(
        "forecast_origin"
    )
    .type
)


if (
    pa.types.is_date32(
        origin_date_type
    )
    or
    pa.types.is_date64(
        origin_date_type
    )
):

    origin_scalar_value = (
        SMOKE_VALIDATION_ORIGIN.date()
    )

elif pa.types.is_timestamp(
    origin_date_type
):

    origin_scalar_value = (
        SMOKE_VALIDATION_ORIGIN
        .to_pydatetime()
    )

else:

    raise TypeError(
        "Unsupported forecast_origin Arrow type: "
        f"{origin_date_type}"
    )


origin_filter = (
    pads.field(
        "forecast_origin"
    )
    ==
    pa.scalar(
        origin_scalar_value,
        type=origin_date_type,
    )
)


smoke_origin_state = (
    origin_features_dataset
    .to_table(
        filter=origin_filter
    )
    .to_pandas()
)


smoke_origin_state[
    "forecast_origin"
] = pd.to_datetime(
    smoke_origin_state[
        "forecast_origin"
    ]
)


assert (
    len(
        smoke_origin_state
    )
    == 1_782
)


# ---------------------------------------------------------------------
# 4. Expand the origin state into the 14 forecast horizons.
# ---------------------------------------------------------------------

smoke_validation = (
    smoke_origin_state.loc[
        smoke_origin_state
        .index
        .repeat(
            FORECAST_HORIZON
        )
    ]
    .reset_index(
        drop=True
    )
)


smoke_validation[
    "horizon"
] = np.tile(
    np.arange(
        1,
        FORECAST_HORIZON + 1,
        dtype=np.int8,
    ),
    len(
        smoke_origin_state
    ),
)


smoke_validation[
    "target_date"
] = (
    smoke_validation[
        "forecast_origin"
    ]
    + pd.to_timedelta(
        smoke_validation[
            "horizon"
        ],
        unit="D",
    )
)


smoke_validation[
    "target_day_of_week"
] = (
    smoke_validation[
        "target_date"
    ]
    .dt.dayofweek
    .astype(
        "int8"
    )
)


smoke_validation[
    "target_month"
] = (
    smoke_validation[
        "target_date"
    ]
    .dt.month
    .astype(
        "int8"
    )
)


smoke_validation[
    "target_is_weekend"
] = (
    smoke_validation[
        "target_day_of_week"
    ]
    .isin(
        [
            5,
            6,
        ]
    )
    .astype(
        "int8"
    )
)


# ---------------------------------------------------------------------
# 5. Recover target-aligned historical predictors.
#
# These predictors depend only on store, family and target date, so their
# validated values can be reused from any supervised row carrying the
# same target-date key.
# ---------------------------------------------------------------------

TARGET_ALIGNED_FEATURES_V1 = [
    "target_sales_lag_364d",
    "target_sales_lag_1y_calendar",
    "target_sales_lag_728d",
    "historical_seasonal_level",
    "seasonal_level_trend_ratio",
    "seasonal_years_available",
]


target_date_type = (
    supervised_arrow_dataset
    .schema
    .field(
        "target_date"
    )
    .type
)


if (
    pa.types.is_date32(
        target_date_type
    )
    or
    pa.types.is_date64(
        target_date_type
    )
):

    validation_start_scalar = (
        pa.scalar(
            (
                SMOKE_VALIDATION_ORIGIN
                + pd.Timedelta(
                    days=1
                )
            ).date(),
            type=target_date_type,
        )
    )

    validation_end_scalar = (
        pa.scalar(
            SMOKE_VALIDATION_END.date(),
            type=target_date_type,
        )
    )

elif pa.types.is_timestamp(
    target_date_type
):

    validation_start_scalar = (
        pa.scalar(
            (
                SMOKE_VALIDATION_ORIGIN
                + pd.Timedelta(
                    days=1
                )
            ).to_pydatetime(),
            type=target_date_type,
        )
    )

    validation_end_scalar = (
        pa.scalar(
            SMOKE_VALIDATION_END
            .to_pydatetime(),
            type=target_date_type,
        )
    )

else:

    raise TypeError(
        "Unsupported target_date Arrow type: "
        f"{target_date_type}"
    )


target_feature_filter = (
    (
        pads.field(
            "target_date"
        )
        >= validation_start_scalar
    )
    &
    (
        pads.field(
            "target_date"
        )
        <= validation_end_scalar
    )
)


target_feature_rows = (
    supervised_arrow_dataset
    .to_table(
        columns=[
            "store_nbr",
            "family",
            "target_date",
        ]
        + TARGET_ALIGNED_FEATURES_V1,

        filter=
            target_feature_filter,
    )
    .to_pandas()
)


target_feature_rows[
    "target_date"
] = pd.to_datetime(
    target_feature_rows[
        "target_date"
    ]
)


# Multiple sampled training origins can point to the same target date.
# Their target-aligned predictors must be identical.
target_feature_lookup = (
    target_feature_rows
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


assert (
    target_feature_lookup
    .duplicated(
        [
            "store_nbr",
            "family",
            "target_date",
        ]
    )
    .sum()
    == 0
), (
    "Target-aligned predictors are inconsistent across supervised rows."
)


assert (
    len(
        target_feature_lookup
    )
    == (
        1_782
        * FORECAST_HORIZON
    )
), (
    "Incomplete target-aligned feature coverage for the validation window."
)


smoke_validation = (
    smoke_validation
    .merge(
        target_feature_lookup,
        on=[
            "store_nbr",
            "family",
            "target_date",
        ],
        how="left",
        validate="many_to_one",
        sort=False,
    )
)


# ---------------------------------------------------------------------
# 6. Attach offline labels and future-known promotion information.
# ---------------------------------------------------------------------

validation_target_lookup = (
    promotion_sales[
        [
            "store_nbr",
            "family",
            "date",
            "sales",
            "onpromotion",
        ]
    ]
    .rename(
        columns={
            "date":
                "target_date",

            "sales":
                "target_sales",
        }
    )
    .copy()
)


validation_target_lookup[
    "target_date"
] = pd.to_datetime(
    validation_target_lookup[
        "target_date"
    ]
)


smoke_validation = (
    smoke_validation
    .merge(
        validation_target_lookup,
        on=[
            "store_nbr",
            "family",
            "target_date",
        ],
        how="left",
        validate="many_to_one",
        sort=False,
    )
)


assert (
    smoke_validation[
        "target_sales"
    ]
    .notna()
    .all()
)


assert (
    smoke_validation[
        "onpromotion"
    ]
    .notna()
    .all()
)


smoke_validation[
    "promo_flag"
] = (
    smoke_validation[
        "onpromotion"
    ]
    > 0
).astype(
    "int8"
)


smoke_validation[
    "promo_count"
] = (
    smoke_validation[
        "onpromotion"
    ]
    .astype(
        "int16"
    )
)


smoke_validation[
    "promo_log_count"
] = (
    np.log1p(
        smoke_validation[
            "onpromotion"
        ]
    )
    .astype(
        "float32"
    )
)


# ---------------------------------------------------------------------
# 7. Construct the cross-layer feature used by Model A.
# ---------------------------------------------------------------------

seasonal_level = (
    smoke_validation[
        "historical_seasonal_level"
    ]
)


smoke_validation[
    "current_vs_seasonal_ratio"
] = np.where(
    seasonal_level.notna()
    &
    (
        seasonal_level.abs()
        > 1e-8
    ),

    (
        smoke_validation[
            "rolling_mean_28d"
        ]
        / seasonal_level
    ),

    np.nan,
).astype(
    "float32"
)


# ---------------------------------------------------------------------
# 8. Apply the frozen store-scope definition to validation.
# ---------------------------------------------------------------------

(
    smoke_validation,
    smoke_excluded_validation_rows,
) = filter_to_started_store_scope(
    smoke_validation
)


assert (
    smoke_excluded_validation_rows
    == 0
)


assert (
    len(
        smoke_validation
    )
    == 24_948
)


# ---------------------------------------------------------------------
# Compact parity audit.
# ---------------------------------------------------------------------

smoke_population_audit = pd.DataFrame(
    [
        {
            "validation_origin":
                SMOKE_VALIDATION_ORIGIN,

            "raw_training_rows":
                raw_smoke_training_rows,

            "excluded_pre_activity_training_rows":
                smoke_excluded_training_rows,

            "eligible_training_rows":
                len(
                    smoke_train
                ),

            "training_origin_min":
                smoke_train[
                    "forecast_origin"
                ]
                .min(),

            "training_origin_max":
                smoke_train[
                    "forecast_origin"
                ]
                .max(),

            "validation_rows":
                len(
                    smoke_validation
                ),

            "validation_promotion_rate":
                smoke_validation[
                    "promo_flag"
                ]
                .mean(),

            "validation_mean_promo_count":
                smoke_validation[
                    "promo_count"
                ]
                .mean(),

            "validation_max_promo_count":
                smoke_validation[
                    "promo_count"
                ]
                .max(),
        }
    ]
)


display(
    smoke_population_audit
    .style.hide(
        axis="index"
    )
    .format(
        {
            "validation_promotion_rate":
                "{:.2%}",

            "validation_mean_promo_count":
                "{:.3f}",
        }
    )
)


print()
print(
    "JULY 18 CONTROLLED FOLD CONSTRUCTION: PASSED"
)

validation_origin,raw_training_rows,excluded_pre_activity_training_rows,eligible_training_rows,training_origin_min,training_origin_max,validation_rows,validation_promotion_rate,validation_mean_promo_count,validation_max_promo_count
2017-07-18 00:00:00,2574990,50226,2524764,2015-07-23 00:00:00,2017-07-11 00:00:00,24948,40.80%,6.600,519



JULY 18 CONTROLLED FOLD CONSTRUCTION: PASSED


In [14]:
# ---------------------------------------------------------------------
# Controlled A/B promotion-feature experiment on the July 18 fold.
#
# All four models use:
# - identical training rows;
# - identical validation rows;
# - identical structural hyperparameters;
# - identical categorical vocabularies;
# - exactly 3,200 boosting trees.
#
# Only the promotion representation changes.
# ---------------------------------------------------------------------

import gc
import time

import numpy as np
import pandas as pd

from xgboost import XGBRegressor


# ---------------------------------------------------------------------
# Frozen categorical vocabularies.
# ---------------------------------------------------------------------

STORE_CATEGORIES = sorted(
    promotion_sales[
        "store_nbr"
    ]
    .dropna()
    .unique()
    .tolist()
)


FAMILY_CATEGORIES = sorted(
    promotion_sales[
        "family"
    ]
    .dropna()
    .astype(
        str
    )
    .unique()
    .tolist()
)


CATEGORY_DTYPES_V1 = {
    "store_nbr":
        pd.CategoricalDtype(
            categories=
                STORE_CATEGORIES,

            ordered=
                False,
        ),

    "family":
        pd.CategoricalDtype(
            categories=
                FAMILY_CATEGORIES,

            ordered=
                False,
        ),

    "target_day_of_week":
        pd.CategoricalDtype(
            categories=
                list(
                    range(
                        7
                    )
                ),

            ordered=
                False,
        ),

    "target_month":
        pd.CategoricalDtype(
            categories=
                list(
                    range(
                        1,
                        13,
                    )
                ),

            ordered=
                False,
        ),
}


def prepare_xgboost_xy(
    frame,
    feature_columns,
):
    """
    Prepare one feature contract using the same native categorical
    representation used by point_forecast_v1.
    """

    X = (
        frame[
            feature_columns
        ]
        .copy()
    )


    y = (
        frame[
            "target_sales"
        ]
        .astype(
            "float32"
        )
        .copy()
    )


    for (
        column,
        categorical_dtype,
    ) in CATEGORY_DTYPES_V1.items():

        X[
            column
        ] = (
            X[
                column
            ]
            .astype(
                categorical_dtype
            )
        )


    return (
        X,
        y,
    )


def compute_forecast_metrics(
    actual,
    prediction,
):
    """
    Compute the project forecasting metrics.

    Positive normalized bias means overforecasting.
    Negative normalized bias means underforecasting.
    """

    actual = np.asarray(
        actual,
        dtype=float,
    )

    prediction = np.asarray(
        prediction,
        dtype=float,
    )


    error = (
        prediction
        - actual
    )


    absolute_error = (
        np.abs(
            error
        )
    )


    denominator = (
        actual.sum()
    )


    return {
        "mae":
            absolute_error.mean(),

        "rmse":
            np.sqrt(
                np.mean(
                    error ** 2
                )
            ),

        "wape":
            (
                absolute_error.sum()
                / denominator
                if denominator > 0
                else np.nan
            ),

        "bias":
            (
                error.sum()
                / denominator
                if denominator > 0
                else np.nan
            ),
    }


# ---------------------------------------------------------------------
# Frozen candidate_27 configuration.
# ---------------------------------------------------------------------

PROMOTION_EXPERIMENT_MODEL_PARAMS = {
    "objective":
        "reg:absoluteerror",

    "max_depth":
        8,

    "learning_rate":
        0.08,

    "min_child_weight":
        1,

    "subsample":
        0.8,

    "colsample_bytree":
        1.0,

    "n_estimators":
        3200,

    "tree_method":
        "hist",

    "enable_categorical":
        True,

    "random_state":
        42,

    "n_jobs":
        -1,

    "verbosity":
        0,
}


PROMOTION_MODEL_SPECS = {
    "A_no_promotion":
        MODEL_FEATURES_A,

    "B_flag":
        MODEL_FEATURES_B_FLAG,

    "B_count":
        MODEL_FEATURES_B_COUNT,

    "B_log_count":
        MODEL_FEATURES_B_LOG,
}


smoke_model_results = []

smoke_model_predictions = {}


for (
    model_name,
    feature_columns,
) in PROMOTION_MODEL_SPECS.items():

    print(
        "\n"
        + "=" * 72
    )

    print(
        "Model:",
        model_name,
    )


    (
        X_train,
        y_train,
    ) = prepare_xgboost_xy(
        smoke_train,
        feature_columns,
    )


    (
        X_validation,
        y_validation,
    ) = prepare_xgboost_xy(
        smoke_validation,
        feature_columns,
    )


    model = XGBRegressor(
        **PROMOTION_EXPERIMENT_MODEL_PARAMS
    )


    fit_start = (
        time.perf_counter()
    )


    model.fit(
        X_train,
        y_train,
    )


    fit_minutes = (
        (
            time.perf_counter()
            - fit_start
        )
        / 60.0
    )


    raw_predictions = (
        model.predict(
            X_validation
        )
        .astype(
            "float64"
        )
    )


    predictions = np.clip(
        raw_predictions,
        a_min=0.0,
        a_max=None,
    )


    actual = (
        y_validation
        .to_numpy(
            dtype="float64"
        )
    )


    # -------------------------------------------------------------
    # Portfolio metrics.
    # -------------------------------------------------------------

    global_metrics = (
        compute_forecast_metrics(
            actual,
            predictions,
        )
    )


    smoke_model_results.append(
        {
            "model":
                model_name,

            "segment":
                "all",

            "rows":
                len(
                    smoke_validation
                ),

            "actual_sales":
                actual.sum(),

            **global_metrics,

            "raw_negative_predictions":
                int(
                    (
                        raw_predictions
                        < 0
                    )
                    .sum()
                ),

            "fit_minutes":
                fit_minutes,
        }
    )


    # -------------------------------------------------------------
    # Promoted vs non-promoted target observations.
    # -------------------------------------------------------------

    for (
        segment_name,
        segment_mask,
    ) in {
        "promoted":
            (
                smoke_validation[
                    "promo_flag"
                ]
                == 1
            ),

        "non_promoted":
            (
                smoke_validation[
                    "promo_flag"
                ]
                == 0
            ),
    }.items():

        mask = (
            segment_mask
            .to_numpy()
        )


        segment_metrics = (
            compute_forecast_metrics(
                actual[
                    mask
                ],
                predictions[
                    mask
                ],
            )
        )


        smoke_model_results.append(
            {
                "model":
                    model_name,

                "segment":
                    segment_name,

                "rows":
                    int(
                        mask.sum()
                    ),

                "actual_sales":
                    actual[
                        mask
                    ]
                    .sum(),

                **segment_metrics,

                "raw_negative_predictions":
                    int(
                        (
                            raw_predictions[
                                mask
                            ]
                            < 0
                        )
                        .sum()
                    ),

                "fit_minutes":
                    fit_minutes,
            }
        )


    smoke_model_predictions[
        model_name
    ] = predictions


    print(
        "WAPE:",
        f"{global_metrics['wape']:.6f}",
    )

    print(
        "Bias:",
        f"{global_metrics['bias']:.6f}",
    )

    print(
        "Fit minutes:",
        f"{fit_minutes:.2f}",
    )


    del (
        X_train,
        y_train,
        X_validation,
        y_validation,
        model,
    )

    gc.collect()


smoke_model_results = (
    pd.DataFrame(
        smoke_model_results
    )
)


# ---------------------------------------------------------------------
# Model A parity check.
#
# Notebook 05 obtained approximately 0.122800 WAPE on this exact
# deployment with the final two-year policy at 3,200 trees.
# Small numerical differences are tolerated, but a material difference
# would indicate that the experimental pipeline is not reproducing A.
# ---------------------------------------------------------------------

model_a_wape = (
    smoke_model_results.loc[
        (
            smoke_model_results[
                "model"
            ]
            == "A_no_promotion"
        )
        &
        (
            smoke_model_results[
                "segment"
            ]
            == "all"
        ),
        "wape",
    ]
    .iloc[
        0
    ]
)


REFERENCE_MODEL_A_WAPE = (
    0.122800
)


assert (
    abs(
        model_a_wape
        - REFERENCE_MODEL_A_WAPE
    )
    < 0.002
), (
    "Model A does not reproduce the frozen July 18 benchmark closely "
    "enough. Do not interpret promotion-model results yet."
)


display(
    smoke_model_results
    .sort_values(
        [
            "segment",
            "wape",
        ]
    )
    .style.hide(
        axis="index"
    )
    .format(
        {
            "actual_sales":
                "{:,.1f}",

            "mae":
                "{:.3f}",

            "rmse":
                "{:.3f}",

            "wape":
                "{:.4f}",

            "bias":
                "{:+.4f}",

            "fit_minutes":
                "{:.2f}",
        }
    )
)


print()
print(
    "MODEL A PARITY CHECK: PASSED"
)

print(
    "JULY 18 PROMOTION FEATURE SMOKE TEST: PASSED"
)


Model: A_no_promotion
WAPE: 0.122800
Bias: -0.024697
Fit minutes: 4.73

Model: B_flag
WAPE: 0.122873
Bias: -0.025374
Fit minutes: 4.66

Model: B_count
WAPE: 0.120036
Bias: -0.029291
Fit minutes: 4.60

Model: B_log_count
WAPE: 0.120036
Bias: -0.029291
Fit minutes: 4.48


model,segment,rows,actual_sales,mae,rmse,wape,bias,raw_negative_predictions,fit_minutes
B_count,all,24948,"11,992,072.4",57.699,218.194,0.1200,-0.0293,1111,4.60
B_log_count,all,24948,"11,992,072.4",57.699,218.194,0.1200,-0.0293,1111,4.48
A_no_promotion,all,24948,"11,992,072.4",59.028,218.228,0.1228,-0.0247,865,4.73
B_flag,all,24948,"11,992,072.4",59.063,218.065,0.1229,-0.0254,1063,4.66
B_flag,non_promoted,14769,"1,028,370.4",9.329,32.945,0.1340,-0.0140,1063,4.66
B_count,non_promoted,14769,"1,028,370.4",9.465,36.328,0.1359,-0.0240,1111,4.60
B_log_count,non_promoted,14769,"1,028,370.4",9.465,36.328,0.1359,-0.0240,1111,4.48
A_no_promotion,non_promoted,14769,"1,028,370.4",9.469,32.402,0.1360,+0.0013,864,4.73
B_count,promoted,10179,"10,963,702.0",127.684,338.778,0.1185,-0.0298,0,4.60
B_log_count,promoted,10179,"10,963,702.0",127.684,338.778,0.1185,-0.0298,0,4.48



MODEL A PARITY CHECK: PASSED
JULY 18 PROMOTION FEATURE SMOKE TEST: PASSED


In [15]:
# ---------------------------------------------------------------------
# Check whether raw-count and log-count promotion representations
# produced numerically identical operational predictions on the smoke fold.
# ---------------------------------------------------------------------

count_log_absolute_difference = np.abs(
    smoke_model_predictions[
        "B_count"
    ]
    - smoke_model_predictions[
        "B_log_count"
    ]
)


count_log_comparison = pd.DataFrame(
    [
        {
            "max_absolute_difference":
                count_log_absolute_difference.max(),

            "mean_absolute_difference":
                count_log_absolute_difference.mean(),

            "different_predictions":
                int(
                    (
                        count_log_absolute_difference
                        > 1e-10
                    )
                    .sum()
                ),
        }
    ]
)


display(
    count_log_comparison
    .style.hide(
        axis="index"
    )
)


print(
    "COUNT VS LOG PREDICTION COMPARISON: PASSED"
)

max_absolute_difference,mean_absolute_difference,different_predictions
0.000000,0.000000,0


COUNT VS LOG PREDICTION COMPARISON: PASSED


#### July 18 controlled experiment — initial evidence

The frozen baseline is reproduced exactly on the July 18 deployment:

$$
WAPE_A = 0.122800.
$$

This validates the experimental pipeline before interpreting the
promotion-aware variants.

The binary promotion flag does not improve the baseline:

$$
WAPE_{B_{\mathrm{flag}}}=0.122873.
$$

In contrast, the promotion-count representation achieves:

$$
WAPE_{B_{\mathrm{count}}}=0.120036,
$$

corresponding to an approximately 2.25% relative WAPE improvement over Model A.

The gain is concentrated on promoted target observations, where WAPE decreases
from approximately 12.16% to 11.85%. Performance on non-promoted observations
is essentially unchanged.

This first fold therefore suggests that the intensity of the planned promotion
contains useful predictive information beyond simply knowing whether any
promotion exists.

The count and log-count specifications produce effectively identical results
on this fold, consistent with the ability of histogram-based tree models to
learn nonlinear thresholds from either monotonic representation.

One important descriptive result is that promoted observations account for
roughly 41% of rows but more than 90% of realized sales in this deployment.
This strong association must not be interpreted causally: promotion assignment
is clearly concentrated in high-volume commercial contexts.

Finally, the count-based model exhibits slightly stronger underforecasting
bias. Bias is therefore retained as a guardrail during the complete rolling
evaluation.

No promotion representation is selected from this single deployment.
Selection is deferred to the six-fold paired temporal comparison.

### 3.4 Rolling Temporal Evaluation

The July 18 smoke test establishes pipeline parity but is insufficient for
model selection.

The four feature contracts are therefore evaluated across the complete six-fold
recent temporal framework:

$$
\text{May 09},
\text{May 23},
\text{Jun 06},
\text{Jun 20},
\text{Jul 04},
\text{Jul 18}.
$$

For every deployment, the experiment preserves the frozen final forecasting
recipe:

- rolling two-year training history;
- pre-activity store exclusion;
- 14-day direct multi-horizon forecasting;
- `candidate_27` structural hyperparameters;
- fixed 3,200-tree boosting length;
- identical training and validation populations across A and all B variants.

This is a paired feature-ablation experiment.

The purpose is not to retune Model A but to determine whether adding planned
promotion information provides a temporally consistent improvement over the
frozen forecasting architecture.

The July 18 results already computed above are reused rather than retrained.

In [16]:
# ---------------------------------------------------------------------
# Precompute the reusable validation-layer inputs for all six recent
# deployments.
#
# The six 14-day windows are contiguous from May 10 through August 1,
# giving exactly:
#
#   6 × 1,782 × 14 = 149,688
#
# validation rows.
# ---------------------------------------------------------------------

ROLLING_VALIDATION_ORIGINS = pd.DatetimeIndex(
    VALIDATION_ORIGINS
)


ROLLING_VALIDATION_START = (
    ROLLING_VALIDATION_ORIGINS.min()
    + pd.Timedelta(
        days=1
    )
)


ROLLING_VALIDATION_END = (
    ROLLING_VALIDATION_ORIGINS.max()
    + pd.Timedelta(
        days=FORECAST_HORIZON
    )
)


# ---------------------------------------------------------------------
# Load origin states for the six historical deployments.
# ---------------------------------------------------------------------

origin_date_type = (
    origin_features_dataset
    .schema
    .field(
        "forecast_origin"
    )
    .type
)


def arrow_date_scalar(
    timestamp,
    arrow_type,
):
    """
    Convert a pandas Timestamp to a scalar compatible with the Arrow
    physical date type.
    """

    timestamp = pd.Timestamp(
        timestamp
    )


    if (
        pa.types.is_date32(
            arrow_type
        )
        or
        pa.types.is_date64(
            arrow_type
        )
    ):

        value = (
            timestamp.date()
        )

    elif pa.types.is_timestamp(
        arrow_type
    ):

        value = (
            timestamp.to_pydatetime()
        )

    else:

        raise TypeError(
            f"Unsupported Arrow date type: {arrow_type}"
        )


    return pa.scalar(
        value,
        type=arrow_type,
    )


origin_range_filter = (
    (
        pads.field(
            "forecast_origin"
        )
        >= arrow_date_scalar(
            ROLLING_VALIDATION_ORIGINS.min(),
            origin_date_type,
        )
    )
    &
    (
        pads.field(
            "forecast_origin"
        )
        <= arrow_date_scalar(
            ROLLING_VALIDATION_ORIGINS.max(),
            origin_date_type,
        )
    )
)


rolling_origin_states = (
    origin_features_dataset
    .to_table(
        filter=
            origin_range_filter
    )
    .to_pandas()
)


rolling_origin_states[
    "forecast_origin"
] = pd.to_datetime(
    rolling_origin_states[
        "forecast_origin"
    ]
)


rolling_origin_states = (
    rolling_origin_states.loc[
        rolling_origin_states[
            "forecast_origin"
        ]
        .isin(
            ROLLING_VALIDATION_ORIGINS
        )
    ]
    .reset_index(
        drop=True
    )
)


assert (
    len(
        rolling_origin_states
    )
    ==
    (
        len(
            ROLLING_VALIDATION_ORIGINS
        )
        * 1_782
    )
)


# ---------------------------------------------------------------------
# Recover target-aligned predictors for the complete contiguous
# May 10-August 1 validation period.
# ---------------------------------------------------------------------

target_date_type = (
    supervised_arrow_dataset
    .schema
    .field(
        "target_date"
    )
    .type
)


rolling_target_filter = (
    (
        pads.field(
            "target_date"
        )
        >= arrow_date_scalar(
            ROLLING_VALIDATION_START,
            target_date_type,
        )
    )
    &
    (
        pads.field(
            "target_date"
        )
        <= arrow_date_scalar(
            ROLLING_VALIDATION_END,
            target_date_type,
        )
    )
)


rolling_target_feature_rows = (
    supervised_arrow_dataset
    .to_table(
        columns=[
            "store_nbr",
            "family",
            "target_date",
        ]
        + TARGET_ALIGNED_FEATURES_V1,

        filter=
            rolling_target_filter,
    )
    .to_pandas()
)


rolling_target_feature_rows[
    "target_date"
] = pd.to_datetime(
    rolling_target_feature_rows[
        "target_date"
    ]
)


rolling_target_feature_lookup = (
    rolling_target_feature_rows
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


assert (
    rolling_target_feature_lookup
    .duplicated(
        [
            "store_nbr",
            "family",
            "target_date",
        ]
    )
    .sum()
    == 0
)


assert (
    len(
        rolling_target_feature_lookup
    )
    == 149_688
)


# ---------------------------------------------------------------------
# Labels and planned promotion information for the same evaluation
# period.
# ---------------------------------------------------------------------

rolling_target_lookup = (
    promotion_sales.loc[
        promotion_sales[
            "date"
        ]
        .between(
            ROLLING_VALIDATION_START,
            ROLLING_VALIDATION_END,
        ),
        [
            "store_nbr",
            "family",
            "date",
            "sales",
            "onpromotion",
        ],
    ]
    .rename(
        columns={
            "date":
                "target_date",

            "sales":
                "target_sales",
        }
    )
    .copy()
)


rolling_target_lookup[
    "target_date"
] = pd.to_datetime(
    rolling_target_lookup[
        "target_date"
    ]
)


assert (
    len(
        rolling_target_lookup
    )
    == 149_688
)


def build_promotion_validation_fold(
    validation_origin,
):
    """
    Reconstruct one complete 14-day historical deployment using the
    frozen Version 1 predictor contract plus target-date promotion data.
    """

    validation_origin = pd.Timestamp(
        validation_origin
    )


    validation_end = (
        validation_origin
        + pd.Timedelta(
            days=FORECAST_HORIZON
        )
    )


    # One point-in-time state per store-family.
    origin_state = (
        rolling_origin_states.loc[
            rolling_origin_states[
                "forecast_origin"
            ]
            .eq(
                validation_origin
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    assert (
        len(
            origin_state
        )
        == 1_782
    )


    # Expand each state into horizons 1-14.
    validation_frame = (
        origin_state.loc[
            origin_state
            .index
            .repeat(
                FORECAST_HORIZON
            )
        ]
        .reset_index(
            drop=True
        )
    )


    validation_frame[
        "horizon"
    ] = np.tile(
        np.arange(
            1,
            FORECAST_HORIZON + 1,
            dtype=np.int8,
        ),
        len(
            origin_state
        ),
    )


    validation_frame[
        "target_date"
    ] = (
        validation_frame[
            "forecast_origin"
        ]
        + pd.to_timedelta(
            validation_frame[
                "horizon"
            ],
            unit="D",
        )
    )


    # Future-known calendar predictors.
    validation_frame[
        "target_day_of_week"
    ] = (
        validation_frame[
            "target_date"
        ]
        .dt.dayofweek
        .astype(
            "int8"
        )
    )


    validation_frame[
        "target_month"
    ] = (
        validation_frame[
            "target_date"
        ]
        .dt.month
        .astype(
            "int8"
        )
    )


    validation_frame[
        "target_is_weekend"
    ] = (
        validation_frame[
            "target_day_of_week"
        ]
        .isin(
            [
                5,
                6,
            ]
        )
        .astype(
            "int8"
        )
    )


    # Target-aligned historical predictors.
    fold_target_features = (
        rolling_target_feature_lookup.loc[
            rolling_target_feature_lookup[
                "target_date"
            ]
            .between(
                validation_origin
                + pd.Timedelta(
                    days=1
                ),
                validation_end,
            )
        ]
    )


    validation_frame = (
        validation_frame
        .merge(
            fold_target_features,
            on=[
                "store_nbr",
                "family",
                "target_date",
            ],
            how="left",
            validate="many_to_one",
            sort=False,
        )
    )


    # Offline labels and future-known planned promotion.
    fold_targets = (
        rolling_target_lookup.loc[
            rolling_target_lookup[
                "target_date"
            ]
            .between(
                validation_origin
                + pd.Timedelta(
                    days=1
                ),
                validation_end,
            )
        ]
    )


    validation_frame = (
        validation_frame
        .merge(
            fold_targets,
            on=[
                "store_nbr",
                "family",
                "target_date",
            ],
            how="left",
            validate="many_to_one",
            sort=False,
        )
    )


    assert (
        validation_frame[
            "target_sales"
        ]
        .notna()
        .all()
    )


    assert (
        validation_frame[
            "onpromotion"
        ]
        .notna()
        .all()
    )


    # Promotion representations.
    validation_frame[
        "promo_flag"
    ] = (
        validation_frame[
            "onpromotion"
        ]
        .gt(
            0
        )
        .astype(
            "int8"
        )
    )


    validation_frame[
        "promo_count"
    ] = (
        validation_frame[
            "onpromotion"
        ]
        .astype(
            "int16"
        )
    )


    validation_frame[
        "promo_log_count"
    ] = (
        np.log1p(
            validation_frame[
                "onpromotion"
            ]
        )
        .astype(
            "float32"
        )
    )


    # Cross-layer Version 1 feature.
    seasonal_level = (
        validation_frame[
            "historical_seasonal_level"
        ]
    )


    validation_frame[
        "current_vs_seasonal_ratio"
    ] = np.where(
        seasonal_level.notna()
        &
        (
            seasonal_level.abs()
            > 1e-8
        ),

        (
            validation_frame[
                "rolling_mean_28d"
            ]
            / seasonal_level
        ),

        np.nan,
    ).astype(
        "float32"
    )


    (
        validation_frame,
        excluded_rows,
    ) = filter_to_started_store_scope(
        validation_frame
    )


    assert (
        excluded_rows
        == 0
    )


    assert (
        len(
            validation_frame
        )
        == 24_948
    )


    return (
        validation_frame
        .reset_index(
            drop=True
        )
    )


# Smoke the generalized builder itself.
generalized_smoke_validation = (
    build_promotion_validation_fold(
        SMOKE_VALIDATION_ORIGIN
    )
)


assert (
    len(
        generalized_smoke_validation
    )
    == len(
        smoke_validation
    )
)


print(
    "Rolling origin-state rows:",
    f"{len(rolling_origin_states):,}",
)

print(
    "Rolling target-feature rows:",
    f"{len(rolling_target_feature_lookup):,}",
)

print(
    "Rolling evaluation target rows:",
    f"{len(rolling_target_lookup):,}",
)

print()
print(
    "ROLLING VALIDATION BUILDER: PASSED"
)

Rolling origin-state rows: 10,692
Rolling target-feature rows: 149,688
Rolling evaluation target rows: 149,688

ROLLING VALIDATION BUILDER: PASSED


In [17]:
# ---------------------------------------------------------------------
# Six-fold paired rolling evaluation.
#
# July 18 is reused from the already completed smoke experiment.
# Every other fold fits all four frozen feature contracts.
#
# A checkpoint is written after each completed fold so that an
# interrupted notebook can resume without repeating finished fits.
# ---------------------------------------------------------------------

import gc
from pathlib import Path

import numpy as np
import pandas as pd

from xgboost import XGBRegressor


ROLLING_EXPECTED_TRAINING_ROWS = {
    pd.Timestamp("2017-05-09"):
        2_516_910,

    pd.Timestamp("2017-05-23"):
        2_502_852,

    pd.Timestamp("2017-06-06"):
        2_511_894,

    pd.Timestamp("2017-06-20"):
        2_531_562,

    pd.Timestamp("2017-07-04"):
        2_506_812,

    pd.Timestamp("2017-07-18"):
        2_524_764,
}


PREDICTION_COLUMNS = {
    "A_no_promotion":
        "prediction_a",

    "B_flag":
        "prediction_b_flag",

    "B_count":
        "prediction_b_count",

    "B_log_count":
        "prediction_b_log_count",
}


rolling_checkpoint_dir = (
    project_root
    / "artifacts"
    / "promotion_forecasting"
    / "rolling_checkpoints"
)


rolling_checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True,
)


rolling_prediction_frames = []


for (
    fold_number,
    validation_origin,
) in enumerate(
    ROLLING_VALIDATION_ORIGINS,
    start=1,
):

    validation_origin = pd.Timestamp(
        validation_origin
    )


    checkpoint_path = (
        rolling_checkpoint_dir
        / (
            "validation_origin="
            f"{validation_origin.date()}.parquet"
        )
    )


    print(
        "\n"
        + "=" * 80
    )

    print(
        f"Fold {fold_number}/"
        f"{len(ROLLING_VALIDATION_ORIGINS)}"
    )

    print(
        "Validation origin:",
        validation_origin.date(),
    )


    # -------------------------------------------------------------
    # Resume an already completed fold.
    # -------------------------------------------------------------

    if checkpoint_path.exists():

        fold_predictions = (
            pd.read_parquet(
                checkpoint_path
            )
        )


        required_checkpoint_columns = [
            "store_nbr",
            "family",
            "validation_origin",
            "horizon",
            "target_date",
            "target_sales",
            "promo_flag",
            "promo_count",
            "promo_log_count",
        ] + list(
            PREDICTION_COLUMNS.values()
        )


        assert all(
            column
            in fold_predictions.columns
            for column
            in required_checkpoint_columns
        )


        assert (
            len(
                fold_predictions
            )
            == 24_948
        )


        print(
            "Checkpoint loaded."
        )


        rolling_prediction_frames.append(
            fold_predictions
        )

        continue


    # -------------------------------------------------------------
    # Build the exact frozen two-year training population.
    # -------------------------------------------------------------

    minimum_training_origin = (
        validation_origin
        - pd.DateOffset(
            years=2
        )
    )


    fold_train = (
        supervised_promo.loc[
            (
                supervised_promo[
                    "target_date"
                ]
                <= validation_origin
            )
            &
            (
                supervised_promo[
                    "forecast_origin"
                ]
                >= minimum_training_origin
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    (
        fold_train,
        excluded_training_rows,
    ) = filter_to_started_store_scope(
        fold_train
    )


    expected_training_rows = (
        ROLLING_EXPECTED_TRAINING_ROWS[
            validation_origin
        ]
    )


    assert (
        len(
            fold_train
        )
        == expected_training_rows
    ), (
        "Training-population parity failure at "
        f"{validation_origin.date()}."
    )


    # -------------------------------------------------------------
    # Build the paired 14-day validation population.
    # -------------------------------------------------------------

    fold_validation = (
        build_promotion_validation_fold(
            validation_origin
        )
    )


    print(
        "Training rows:",
        f"{len(fold_train):,}",
    )

    print(
        "Excluded pre-activity rows:",
        f"{excluded_training_rows:,}",
    )

    print(
        "Validation rows:",
        f"{len(fold_validation):,}",
    )


    # -------------------------------------------------------------
    # Initialize compact row-level output used later for segment
    # diagnostics.
    # -------------------------------------------------------------

    fold_predictions = (
        fold_validation[
            [
                "store_nbr",
                "family",
                "horizon",
                "target_date",
                "target_sales",
                "promo_flag",
                "promo_count",
                "promo_log_count",
            ]
        ]
        .copy()
    )


    fold_predictions[
        "validation_origin"
    ] = validation_origin


    # -------------------------------------------------------------
    # Reuse the already completed July 18 smoke predictions.
    # -------------------------------------------------------------

    if (
        validation_origin
        == SMOKE_VALIDATION_ORIGIN
    ):

        for (
            model_name,
            prediction_column,
        ) in PREDICTION_COLUMNS.items():

            fold_predictions[
                prediction_column
            ] = (
                smoke_model_predictions[
                    model_name
                ]
            )


        print(
            "July 18 smoke predictions reused."
        )


    # -------------------------------------------------------------
    # All other folds: fit four paired models using the frozen recipe.
    # -------------------------------------------------------------

    else:

        for (
            model_name,
            feature_columns,
        ) in PROMOTION_MODEL_SPECS.items():

            print()
            print(
                "Model:",
                model_name,
            )


            (
                X_train,
                y_train,
            ) = prepare_xgboost_xy(
                fold_train,
                feature_columns,
            )


            (
                X_validation,
                y_validation,
            ) = prepare_xgboost_xy(
                fold_validation,
                feature_columns,
            )


            model = XGBRegressor(
                **PROMOTION_EXPERIMENT_MODEL_PARAMS
            )


            model.fit(
                X_train,
                y_train,
            )


            raw_predictions = (
                model.predict(
                    X_validation
                )
                .astype(
                    "float64"
                )
            )


            predictions = np.clip(
                raw_predictions,
                a_min=0.0,
                a_max=None,
            )


            prediction_column = (
                PREDICTION_COLUMNS[
                    model_name
                ]
            )


            fold_predictions[
                prediction_column
            ] = predictions


            fold_metrics = (
                compute_forecast_metrics(
                    y_validation.to_numpy(
                        dtype="float64"
                    ),
                    predictions,
                )
            )


            print(
                "WAPE:",
                f"{fold_metrics['wape']:.6f}",
            )

            print(
                "Bias:",
                f"{fold_metrics['bias']:+.6f}",
            )


            del (
                X_train,
                y_train,
                X_validation,
                y_validation,
                model,
                raw_predictions,
                predictions,
            )

            gc.collect()


    # -------------------------------------------------------------
    # Persist one atomic completed-fold checkpoint.
    # -------------------------------------------------------------

    assert all(
        column
        in fold_predictions.columns
        for column
        in PREDICTION_COLUMNS.values()
    )


    assert (
        fold_predictions[
            list(
                PREDICTION_COLUMNS.values()
            )
        ]
        .notna()
        .all()
        .all()
    )


    fold_predictions.to_parquet(
        checkpoint_path,
        index=False,
    )


    print(
        "Checkpoint written:",
        checkpoint_path.name,
    )


    rolling_prediction_frames.append(
        fold_predictions
    )


# ---------------------------------------------------------------------
# Combine the complete six-deployment prediction panel.
# ---------------------------------------------------------------------

promotion_rolling_predictions = (
    pd.concat(
        rolling_prediction_frames,
        ignore_index=True,
    )
)


promotion_rolling_predictions = (
    promotion_rolling_predictions
    .sort_values(
        [
            "validation_origin",
            "store_nbr",
            "family",
            "horizon",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    len(
        promotion_rolling_predictions
    )
    == 149_688
)


# ---------------------------------------------------------------------
# Fold-level paired metrics.
# ---------------------------------------------------------------------

promotion_rolling_metrics = []


for validation_origin in (
    ROLLING_VALIDATION_ORIGINS
):

    fold_frame = (
        promotion_rolling_predictions.loc[
            promotion_rolling_predictions[
                "validation_origin"
            ]
            .eq(
                validation_origin
            )
        ]
    )


    actual = (
        fold_frame[
            "target_sales"
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    for (
        model_name,
        prediction_column,
    ) in PREDICTION_COLUMNS.items():

        predictions = (
            fold_frame[
                prediction_column
            ]
            .to_numpy(
                dtype="float64"
            )
        )


        metrics = (
            compute_forecast_metrics(
                actual,
                predictions,
            )
        )


        promotion_rolling_metrics.append(
            {
                "validation_origin":
                    validation_origin,

                "model":
                    model_name,

                "rows":
                    len(
                        fold_frame
                    ),

                "actual_sales":
                    actual.sum(),

                **metrics,
            }
        )


promotion_rolling_metrics = pd.DataFrame(
    promotion_rolling_metrics
)


display(
    promotion_rolling_metrics
    .sort_values(
        [
            "validation_origin",
            "wape",
        ]
    )
    .style.hide(
        axis="index"
    )
    .format(
        {
            "actual_sales":
                "{:,.1f}",

            "mae":
                "{:.3f}",

            "rmse":
                "{:.3f}",

            "wape":
                "{:.4f}",

            "bias":
                "{:+.4f}",
        }
    )
)


# ---------------------------------------------------------------------
# Six-fold aggregate comparison.
# ---------------------------------------------------------------------

model_a_pooled_metrics = (
    compute_forecast_metrics(
        promotion_rolling_predictions[
            "target_sales"
        ],
        promotion_rolling_predictions[
            "prediction_a"
        ],
    )
)


model_a_pooled_wape = (
    model_a_pooled_metrics[
        "wape"
    ]
)


promotion_rolling_summary = []


for (
    model_name,
    prediction_column,
) in PREDICTION_COLUMNS.items():

    pooled_metrics = (
        compute_forecast_metrics(
            promotion_rolling_predictions[
                "target_sales"
            ],
            promotion_rolling_predictions[
                prediction_column
            ],
        )
    )


    model_fold_metrics = (
        promotion_rolling_metrics.loc[
            promotion_rolling_metrics[
                "model"
            ]
            .eq(
                model_name
            )
        ]
    )


    if (
        model_name
        == "A_no_promotion"
    ):

        wins_vs_a = np.nan

    else:

        model_wape_by_fold = (
            model_fold_metrics
            .set_index(
                "validation_origin"
            )[
                "wape"
            ]
        )


        a_wape_by_fold = (
            promotion_rolling_metrics.loc[
                promotion_rolling_metrics[
                    "model"
                ]
                .eq(
                    "A_no_promotion"
                )
            ]
            .set_index(
                "validation_origin"
            )[
                "wape"
            ]
        )


        wins_vs_a = int(
            (
                model_wape_by_fold
                < a_wape_by_fold
            )
            .sum()
        )


    promotion_rolling_summary.append(
        {
            "model":
                model_name,

            "pooled_wape":
                pooled_metrics[
                    "wape"
                ],

            "relative_wape_gain_vs_a":
                (
                    model_a_pooled_wape
                    - pooled_metrics[
                        "wape"
                    ]
                )
                / model_a_pooled_wape,

            "mean_fold_wape":
                model_fold_metrics[
                    "wape"
                ]
                .mean(),

            "median_fold_wape":
                model_fold_metrics[
                    "wape"
                ]
                .median(),

            "std_fold_wape":
                model_fold_metrics[
                    "wape"
                ]
                .std(),

            "worst_fold_wape":
                model_fold_metrics[
                    "wape"
                ]
                .max(),

            "pooled_bias":
                pooled_metrics[
                    "bias"
                ],

            "pooled_rmse":
                pooled_metrics[
                    "rmse"
                ],

            "wins_vs_a":
                wins_vs_a,
        }
    )


promotion_rolling_summary = (
    pd.DataFrame(
        promotion_rolling_summary
    )
    .sort_values(
        "pooled_wape"
    )
    .reset_index(
        drop=True
    )
)


display(
    promotion_rolling_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "pooled_wape":
                "{:.4f}",

            "relative_wape_gain_vs_a":
                "{:+.2%}",

            "mean_fold_wape":
                "{:.4f}",

            "median_fold_wape":
                "{:.4f}",

            "std_fold_wape":
                "{:.4f}",

            "worst_fold_wape":
                "{:.4f}",

            "pooled_bias":
                "{:+.4f}",

            "pooled_rmse":
                "{:.3f}",

            "wins_vs_a":
                lambda value:
                    "-"
                    if pd.isna(
                        value
                    )
                    else f"{int(value)}/6",
        }
    )
)


# ---------------------------------------------------------------------
# Check whether count and log-count remain equivalent across all six
# historical deployments.
# ---------------------------------------------------------------------

count_log_rolling_difference = np.abs(
    promotion_rolling_predictions[
        "prediction_b_count"
    ]
    - promotion_rolling_predictions[
        "prediction_b_log_count"
    ]
)


print()
print(
    "Max |count - log| prediction difference:",
    count_log_rolling_difference.max(),
)

print(
    "Rows with difference > 1e-10:",
    int(
        (
            count_log_rolling_difference
            > 1e-10
        )
        .sum()
    ),
)

print()
print(
    "SIX-FOLD PROMOTION ROLLING EVALUATION: PASSED"
)


Fold 1/6
Validation origin: 2017-05-09
Training rows: 2,516,910
Excluded pre-activity rows: 70,554
Validation rows: 24,948

Model: A_no_promotion
WAPE: 0.147075
Bias: +0.035183

Model: B_flag
WAPE: 0.147258
Bias: +0.035679

Model: B_count
WAPE: 0.149590
Bias: +0.045376

Model: B_log_count
WAPE: 0.149590
Bias: +0.045376
Checkpoint written: validation_origin=2017-05-09.parquet

Fold 2/6
Validation origin: 2017-05-23
Training rows: 2,502,852
Excluded pre-activity rows: 65,010
Validation rows: 24,948

Model: A_no_promotion
WAPE: 0.163806
Bias: -0.041659

Model: B_flag
WAPE: 0.164591
Bias: -0.043805

Model: B_count
WAPE: 0.164729
Bias: -0.051071

Model: B_log_count
WAPE: 0.164729
Bias: -0.051071
Checkpoint written: validation_origin=2017-05-23.parquet

Fold 3/6
Validation origin: 2017-06-06
Training rows: 2,511,894
Excluded pre-activity rows: 61,314
Validation rows: 24,948

Model: A_no_promotion
WAPE: 0.127854
Bias: +0.048620

Model: B_flag
WAPE: 0.128106
Bias: +0.046873

Model: B_count
WA

validation_origin,model,rows,actual_sales,mae,rmse,wape,bias
2017-05-09 00:00:00,A_no_promotion,24948,"11,794,945.2",69.534,275.165,0.1471,+0.0352
2017-05-09 00:00:00,B_flag,24948,"11,794,945.2",69.621,274.693,0.1473,+0.0357
2017-05-09 00:00:00,B_count,24948,"11,794,945.2",70.723,280.004,0.1496,+0.0454
2017-05-09 00:00:00,B_log_count,24948,"11,794,945.2",70.723,280.004,0.1496,+0.0454
2017-05-23 00:00:00,A_no_promotion,24948,"12,678,431.7",83.245,335.952,0.1638,-0.0417
2017-05-23 00:00:00,B_flag,24948,"12,678,431.7",83.644,334.733,0.1646,-0.0438
2017-05-23 00:00:00,B_count,24948,"12,678,431.7",83.714,341.093,0.1647,-0.0511
2017-05-23 00:00:00,B_log_count,24948,"12,678,431.7",83.714,341.093,0.1647,-0.0511
2017-06-06 00:00:00,A_no_promotion,24948,"11,931,126.6",61.145,244.879,0.1279,+0.0486
2017-06-06 00:00:00,B_flag,24948,"11,931,126.6",61.265,246.655,0.1281,+0.0469


model,pooled_wape,relative_wape_gain_vs_a,mean_fold_wape,median_fold_wape,std_fold_wape,worst_fold_wape,pooled_bias,pooled_rmse,wins_vs_a
A_no_promotion,0.1340,+0.00%,0.1337,0.1277,0.0184,0.1638,+0.0096,258.488,-
B_flag,0.1346,-0.43%,0.1343,0.1281,0.0183,0.1646,+0.0092,259.138,0/6
B_count,0.1347,-0.46%,0.1343,0.1293,0.0193,0.1647,+0.0133,259.441,2/6
B_log_count,0.1347,-0.46%,0.1343,0.1293,0.0193,0.1647,+0.0133,259.441,2/6



Max |count - log| prediction difference: 0.0
Rows with difference > 1e-10: 0

SIX-FOLD PROMOTION ROLLING EVALUATION: PASSED


### 3.5 Segment Diagnostics

The rolling experiment does not support promoting the promotion-aware model:

$$
WAPE_A = 0.1340
$$

versus:

$$
WAPE_{B_{\mathrm{count}}} = 0.1347.
$$

However, aggregate accuracy does not reveal where the additional promotion
signal helps or hurts.

The following diagnostics therefore examine whether the effect is concentrated
in specific parts of the forecasting problem.

Because the raw-count and log-count representations produced exactly identical
predictions across all 149,688 rolling-validation observations, the remaining
diagnostics retain only the raw-count representation.

Model B is not reconsidered or retuned in this section.

The objective is diagnostic:

- determine whether promotion information helps specifically on promoted
  targets;
- test whether predictive value varies with promotion intensity;
- examine whether any gain depends on forecast horizon;
- assess whether the promoted/non-promoted behavior is temporally stable.

These diagnostics do not override the primary pooled rolling-validation result.

In [18]:
# ---------------------------------------------------------------------
# Segment diagnostics for the completed six-fold promotion experiment.
#
# No model is retrained here.
#
# B_count is retained as the promotion-aware diagnostic model because:
#
# 1. count preserves more promotion information than the binary flag;
# 2. count and log-count produced exactly identical predictions across
#    the complete rolling evaluation;
# 3. the primary production decision remains based on the pooled
#    rolling-validation result, where Model A remains the champion.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


diagnostic_predictions = (
    promotion_rolling_predictions
    .copy()
)


# ---------------------------------------------------------------------
# Utility: compute standard project metrics for any subset.
# ---------------------------------------------------------------------

def segment_metrics(
    frame,
    actual_column,
    prediction_column,
):
    """
    Compute forecasting metrics for one diagnostic segment.
    """

    actual = (
        frame[
            actual_column
        ]
        .to_numpy(
            dtype="float64"
        )
    )

    predictions = (
        frame[
            prediction_column
        ]
        .to_numpy(
            dtype="float64"
        )
    )

    metrics = (
        compute_forecast_metrics(
            actual,
            predictions,
        )
    )

    return {
        "rows":
            len(
                frame
            ),

        "actual_sales":
            actual.sum(),

        **metrics,
    }


# =====================================================================
# 1. PROMOTED VS NON-PROMOTED
# =====================================================================

diagnostic_predictions[
    "promotion_status"
] = np.where(
    diagnostic_predictions[
        "promo_flag"
    ]
    .eq(
        1
    ),
    "promoted",
    "non_promoted",
)


status_rows = []


for promotion_status in [
    "non_promoted",
    "promoted",
]:

    status_frame = (
        diagnostic_predictions.loc[
            diagnostic_predictions[
                "promotion_status"
            ]
            .eq(
                promotion_status
            )
        ]
    )


    model_metrics = {}


    for (
        model_name,
        prediction_column,
    ) in {
        "A_no_promotion":
            "prediction_a",

        "B_flag":
            "prediction_b_flag",

        "B_count":
            "prediction_b_count",
    }.items():

        metrics = segment_metrics(
            status_frame,
            "target_sales",
            prediction_column,
        )


        model_metrics[
            model_name
        ] = metrics


        status_rows.append(
            {
                "promotion_status":
                    promotion_status,

                "model":
                    model_name,

                **metrics,
            }
        )


promotion_status_metrics = (
    pd.DataFrame(
        status_rows
    )
)


# Relative WAPE change against Model A within the same segment.
a_status_wape = (
    promotion_status_metrics.loc[
        promotion_status_metrics[
            "model"
        ]
        .eq(
            "A_no_promotion"
        )
    ]
    .set_index(
        "promotion_status"
    )[
        "wape"
    ]
)


promotion_status_metrics[
    "relative_wape_gain_vs_a"
] = (
    promotion_status_metrics
    .apply(
        lambda row:
            (
                a_status_wape.loc[
                    row[
                        "promotion_status"
                    ]
                ]
                - row[
                    "wape"
                ]
            )
            / a_status_wape.loc[
                row[
                    "promotion_status"
                ]
            ],
        axis=1,
    )
)


print(
    "=" * 80
)

print(
    "PROMOTED VS NON-PROMOTED"
)

print(
    "=" * 80
)


display(
    promotion_status_metrics
    .sort_values(
        [
            "promotion_status",
            "wape",
        ]
    )
    .style.hide(
        axis="index"
    )
    .format(
        {
            "actual_sales":
                "{:,.1f}",

            "mae":
                "{:.3f}",

            "rmse":
                "{:.3f}",

            "wape":
                "{:.4f}",

            "bias":
                "{:+.4f}",

            "relative_wape_gain_vs_a":
                "{:+.2%}",
        }
    )
)


# =====================================================================
# 2. PROMOTION-INTENSITY DIAGNOSTICS
# =====================================================================
#
# Business-oriented bins are used rather than quantile bins so that the
# ranges retain the same interpretation across folds.
#
# Zero is kept explicitly as the untreated state.
# ---------------------------------------------------------------------

promo_intensity_bins = [
    -0.5,
    0.5,
    1.5,
    4.5,
    9.5,
    24.5,
    np.inf,
]


promo_intensity_labels = [
    "0",
    "1",
    "2-4",
    "5-9",
    "10-24",
    "25+",
]


diagnostic_predictions[
    "promotion_intensity"
] = pd.cut(
    diagnostic_predictions[
        "promo_count"
    ],
    bins=
        promo_intensity_bins,
    labels=
        promo_intensity_labels,
    ordered=True,
)


intensity_rows = []


for intensity_level in (
    promo_intensity_labels
):

    intensity_frame = (
        diagnostic_predictions.loc[
            diagnostic_predictions[
                "promotion_intensity"
            ]
            .eq(
                intensity_level
            )
        ]
    )


    if (
        len(
            intensity_frame
        )
        == 0
    ):

        continue


    metrics_a = segment_metrics(
        intensity_frame,
        "target_sales",
        "prediction_a",
    )


    metrics_b = segment_metrics(
        intensity_frame,
        "target_sales",
        "prediction_b_count",
    )


    intensity_rows.append(
        {
            "promotion_intensity":
                intensity_level,

            "rows":
                len(
                    intensity_frame
                ),

            "actual_sales":
                intensity_frame[
                    "target_sales"
                ]
                .sum(),

            "A_wape":
                metrics_a[
                    "wape"
                ],

            "B_count_wape":
                metrics_b[
                    "wape"
                ],

            "relative_wape_gain_vs_a":
                (
                    metrics_a[
                        "wape"
                    ]
                    - metrics_b[
                        "wape"
                    ]
                )
                / metrics_a[
                    "wape"
                ],

            "A_bias":
                metrics_a[
                    "bias"
                ],

            "B_count_bias":
                metrics_b[
                    "bias"
                ],
        }
    )


promotion_intensity_metrics = (
    pd.DataFrame(
        intensity_rows
    )
)


promotion_intensity_metrics[
    "sales_share"
] = (
    promotion_intensity_metrics[
        "actual_sales"
    ]
    / promotion_intensity_metrics[
        "actual_sales"
    ]
    .sum()
)


print()
print(
    "=" * 80
)

print(
    "PROMOTION INTENSITY"
)

print(
    "=" * 80
)


display(
    promotion_intensity_metrics
    .style.hide(
        axis="index"
    )
    .format(
        {
            "actual_sales":
                "{:,.1f}",

            "sales_share":
                "{:.2%}",

            "A_wape":
                "{:.4f}",

            "B_count_wape":
                "{:.4f}",

            "relative_wape_gain_vs_a":
                "{:+.2%}",

            "A_bias":
                "{:+.4f}",

            "B_count_bias":
                "{:+.4f}",
        }
    )
)


# =====================================================================
# 3. FORECAST-HORIZON DIAGNOSTICS
# =====================================================================

horizon_rows = []


for horizon in range(
    1,
    FORECAST_HORIZON + 1,
):

    horizon_frame = (
        diagnostic_predictions.loc[
            diagnostic_predictions[
                "horizon"
            ]
            .eq(
                horizon
            )
        ]
    )


    metrics_a = segment_metrics(
        horizon_frame,
        "target_sales",
        "prediction_a",
    )


    metrics_b = segment_metrics(
        horizon_frame,
        "target_sales",
        "prediction_b_count",
    )


    horizon_rows.append(
        {
            "horizon":
                horizon,

            "rows":
                len(
                    horizon_frame
                ),

            "actual_sales":
                horizon_frame[
                    "target_sales"
                ]
                .sum(),

            "A_wape":
                metrics_a[
                    "wape"
                ],

            "B_count_wape":
                metrics_b[
                    "wape"
                ],

            "relative_wape_gain_vs_a":
                (
                    metrics_a[
                        "wape"
                    ]
                    - metrics_b[
                        "wape"
                    ]
                )
                / metrics_a[
                    "wape"
                ],

            "A_bias":
                metrics_a[
                    "bias"
                ],

            "B_count_bias":
                metrics_b[
                    "bias"
                ],
        }
    )


promotion_horizon_metrics = (
    pd.DataFrame(
        horizon_rows
    )
)


print()
print(
    "=" * 80
)

print(
    "FORECAST HORIZON"
)

print(
    "=" * 80
)


display(
    promotion_horizon_metrics
    .style.hide(
        axis="index"
    )
    .format(
        {
            "actual_sales":
                "{:,.1f}",

            "A_wape":
                "{:.4f}",

            "B_count_wape":
                "{:.4f}",

            "relative_wape_gain_vs_a":
                "{:+.2%}",

            "A_bias":
                "{:+.4f}",

            "B_count_bias":
                "{:+.4f}",
        }
    )
)


# =====================================================================
# 4. TEMPORAL STABILITY BY PROMOTION STATUS
# =====================================================================

fold_status_rows = []


for validation_origin in (
    ROLLING_VALIDATION_ORIGINS
):

    fold_frame = (
        diagnostic_predictions.loc[
            diagnostic_predictions[
                "validation_origin"
            ]
            .eq(
                validation_origin
            )
        ]
    )


    for promotion_status in [
        "non_promoted",
        "promoted",
    ]:

        segment_frame = (
            fold_frame.loc[
                fold_frame[
                    "promotion_status"
                ]
                .eq(
                    promotion_status
                )
            ]
        )


        metrics_a = segment_metrics(
            segment_frame,
            "target_sales",
            "prediction_a",
        )


        metrics_b = segment_metrics(
            segment_frame,
            "target_sales",
            "prediction_b_count",
        )


        fold_status_rows.append(
            {
                "validation_origin":
                    validation_origin,

                "promotion_status":
                    promotion_status,

                "rows":
                    len(
                        segment_frame
                    ),

                "actual_sales":
                    segment_frame[
                        "target_sales"
                    ]
                    .sum(),

                "A_wape":
                    metrics_a[
                        "wape"
                    ],

                "B_count_wape":
                    metrics_b[
                        "wape"
                    ],

                "relative_wape_gain_vs_a":
                    (
                        metrics_a[
                            "wape"
                        ]
                        - metrics_b[
                            "wape"
                        ]
                    )
                    / metrics_a[
                        "wape"
                    ],

                "A_bias":
                    metrics_a[
                        "bias"
                    ],

                "B_count_bias":
                    metrics_b[
                        "bias"
                    ],
            }
        )


promotion_fold_status_metrics = (
    pd.DataFrame(
        fold_status_rows
    )
)


print()
print(
    "=" * 80
)

print(
    "TEMPORAL STABILITY BY PROMOTION STATUS"
)

print(
    "=" * 80
)


display(
    promotion_fold_status_metrics
    .sort_values(
        [
            "validation_origin",
            "promotion_status",
        ]
    )
    .style.hide(
        axis="index"
    )
    .format(
        {
            "actual_sales":
                "{:,.1f}",

            "A_wape":
                "{:.4f}",

            "B_count_wape":
                "{:.4f}",

            "relative_wape_gain_vs_a":
                "{:+.2%}",

            "A_bias":
                "{:+.4f}",

            "B_count_bias":
                "{:+.4f}",
        }
    )
)


# ---------------------------------------------------------------------
# Compact diagnostic summary.
# ---------------------------------------------------------------------

promoted_rows = (
    diagnostic_predictions[
        "promo_flag"
    ]
    .eq(
        1
    )
)


promoted_sales_share = (
    diagnostic_predictions.loc[
        promoted_rows,
        "target_sales",
    ]
    .sum()
    / diagnostic_predictions[
        "target_sales"
    ]
    .sum()
)


count_horizon_wins = int(
    (
        promotion_horizon_metrics[
            "B_count_wape"
        ]
        <
        promotion_horizon_metrics[
            "A_wape"
        ]
    )
    .sum()
)


count_promoted_fold_wins = int(
    (
        promotion_fold_status_metrics.loc[
            promotion_fold_status_metrics[
                "promotion_status"
            ]
            .eq(
                "promoted"
            ),
            "relative_wape_gain_vs_a",
        ]
        > 0
    )
    .sum()
)


print()
print(
    "Promoted row share:",
    f"{promoted_rows.mean():.2%}",
)

print(
    "Promoted realized-sales share:",
    f"{promoted_sales_share:.2%}",
)

print(
    "B_count horizon wins vs A:",
    f"{count_horizon_wins}/"
    f"{FORECAST_HORIZON}",
)

print(
    "B_count promoted-segment fold wins vs A:",
    f"{count_promoted_fold_wins}/"
    f"{len(ROLLING_VALIDATION_ORIGINS)}",
)

print()
print(
    "PROMOTION SEGMENT DIAGNOSTICS: PASSED"
)

PROMOTED VS NON-PROMOTED


promotion_status,model,rows,actual_sales,mae,rmse,wape,bias,relative_wape_gain_vs_a
non_promoted,B_count,82556,"5,360,673.5",9.814,42.766,0.1511,-0.0084,+2.01%
non_promoted,B_flag,82556,"5,360,673.5",9.882,44.127,0.1522,-0.0036,+1.33%
non_promoted,A_no_promotion,82556,"5,360,673.5",10.016,43.503,0.1542,+0.0123,+0.00%
promoted,A_no_promotion,67132,"66,839,360.3",131.844,382.958,0.1324,+0.0094,+0.00%
promoted,B_flag,67132,"66,839,360.3",132.632,383.848,0.1332,+0.0102,-0.60%
promoted,B_count,67132,"66,839,360.3",132.758,384.493,0.1333,+0.0150,-0.69%



PROMOTION INTENSITY


promotion_intensity,rows,actual_sales,A_wape,B_count_wape,relative_wape_gain_vs_a,A_bias,B_count_bias,sales_share
0,82556,"5,360,673.5",0.1542,0.1511,+2.01%,+0.0123,-0.0084,7.42%
1,12977,"2,311,176.3",0.1343,0.1305,+2.81%,+0.0316,+0.0182,3.20%
2-4,14362,"3,335,706.1",0.1460,0.1438,+1.50%,+0.0242,+0.0172,4.62%
5-9,10586,"5,364,534.8",0.1322,0.1306,+1.16%,+0.0224,+0.0170,7.43%
10-24,13886,"12,199,195.1",0.1470,0.1452,+1.27%,+0.0037,+0.0045,16.90%
25+,15321,"43,628,747.9",0.1272,0.1297,-1.95%,+0.0071,+0.0174,60.43%



FORECAST HORIZON


horizon,rows,actual_sales,A_wape,B_count_wape,relative_wape_gain_vs_a,A_bias,B_count_bias
1,10692,"4,733,824.8",0.1058,0.1065,-0.62%,+0.0215,+0.0357
2,10692,"3,934,287.4",0.1276,0.1302,-2.03%,+0.0151,+0.0277
3,10692,"4,893,700.1",0.1369,0.1381,-0.85%,-0.0227,-0.0189
4,10692,"5,937,446.9",0.1272,0.1280,-0.65%,+0.0436,+0.0492
5,10692,"6,377,378.4",0.1571,0.1574,-0.19%,+0.0929,+0.0941
6,10692,"4,744,289.5",0.1307,0.1333,-1.98%,+0.0486,+0.0547
7,10692,"4,406,023.0",0.1311,0.1285,+1.98%,+0.0314,+0.0366
8,10692,"4,706,705.7",0.1284,0.1269,+1.20%,+0.0165,+0.0242
9,10692,"3,975,998.5",0.1381,0.1469,-6.40%,+0.0009,+0.0066
10,10692,"4,929,845.6",0.1410,0.1421,-0.76%,-0.0169,-0.0155



TEMPORAL STABILITY BY PROMOTION STATUS


validation_origin,promotion_status,rows,actual_sales,A_wape,B_count_wape,relative_wape_gain_vs_a,A_bias,B_count_bias
2017-05-09 00:00:00,non_promoted,12439,"531,943.9",0.2207,0.2084,+5.59%,+0.0317,+0.0029
2017-05-09 00:00:00,promoted,12509,"11,263,001.3",0.1436,0.1468,-2.24%,+0.0353,+0.0474
2017-05-23 00:00:00,non_promoted,13642,"1,389,058.6",0.1450,0.1478,-1.91%,-0.0519,-0.0671
2017-05-23 00:00:00,promoted,11306,"11,289,373.1",0.1661,0.1668,-0.42%,-0.0404,-0.0491
2017-06-06 00:00:00,non_promoted,13585,"899,090.0",0.1441,0.1381,+4.19%,+0.0625,+0.0485
2017-06-06 00:00:00,promoted,11363,"11,032,036.6",0.1265,0.1289,-1.90%,+0.0475,+0.0573
2017-06-20 00:00:00,non_promoted,13801,"699,634.3",0.1680,0.1648,+1.85%,+0.0254,+0.0024
2017-06-20 00:00:00,promoted,11147,"11,357,302.4",0.1250,0.1267,-1.43%,+0.0177,+0.0307
2017-07-04 00:00:00,non_promoted,14320,"812,576.4",0.1490,0.1414,+5.16%,+0.0564,+0.0319
2017-07-04 00:00:00,promoted,10628,"10,933,944.7",0.1107,0.1110,-0.30%,+0.0237,+0.0339



Promoted row share: 44.85%
Promoted realized-sales share: 92.58%
B_count horizon wins vs A: 4/14
B_count promoted-segment fold wins vs A: 1/6

PROMOTION SEGMENT DIAGNOSTICS: PASSED


### 3.6 Forecasting Conclusion

The six-fold rolling experiment does not support promoting any of the
promotion-aware specifications over the frozen Version 1 forecasting model.

Across the complete 149,688-row evaluation panel:

$$
WAPE_A = 0.1340,
$$

while:

$$
WAPE_{B_{\mathrm{flag}}}=0.1346,
$$

and:

$$
WAPE_{B_{\mathrm{count}}}
=
WAPE_{B_{\log}}
=
0.1347.
$$

The binary promotion model does not outperform Model A on any of the six
deployments.

The count-based model wins only two of six folds and produces an approximately
0.46% relative pooled-WAPE degradation.

The raw-count and log-count representations produce numerically identical
predictions across the complete rolling evaluation, so the logarithmic
transformation provides no additional predictive value in this tree-based
experiment.

---

#### Segment behavior

The aggregate result hides substantial differences across promotion regimes.

On non-promoted targets, the separately refitted count-based specification
achieves:

$$
WAPE_A = 0.1542
$$

versus:

$$
WAPE_{B_{\mathrm{count}}}=0.1511,
$$

an approximately 2.01% relative improvement.

This result should **not** be interpreted as promotion information creating
predictive value when no promotion is present.

Model B is a separately refitted tree ensemble:

$$
F_B(X,P),
$$

rather than an additive correction to the frozen baseline:

$$
F_A(X)+g(P).
$$

Introducing the promotion feature therefore changes the complete learned tree
partition, feature interactions and predictions even when:

$$
P=0.
$$

The non-promoted result describes the out-of-sample behavior of the refitted
specification rather than an isolated effect of promotion information.

This distinction is particularly important because non-promoted observations
represent approximately 55.15% of evaluation rows but only:

$$
7.42\%
$$

of realized sales.

Promoted observations represent approximately 44.85% of rows but:

$$
92.58\%
$$

of realized sales.

On this economically dominant promoted segment, Model A remains better:

$$
0.1324
\quad\text{versus}\quad
0.1333
$$

for the count-based specification.

---

#### Promotion intensity

The promotion-intensity diagnostic reveals additional heterogeneity.

The count-based specification improves WAPE relative to Model A for the
following target-date promotion-count ranges:

$$
1,\quad 2\text{-}4,\quad 5\text{-}9,\quad 10\text{-}24.
$$

The corresponding relative WAPE gains are approximately:

$$
2.81\%,\quad
1.50\%,\quad
1.16\%,\quad
1.27\%.
$$

However, performance reverses for:

$$
\text{onpromotion}\geq25,
$$

where Model B produces an approximately 1.95% relative WAPE degradation.

This high-intensity segment accounts for approximately:

$$
60.43\%
$$

of total realized sales across the six deployments.

The available target-date promotion count therefore contains localized
predictive information, but the chosen monolithic feature extension does not
translate that information into robust accuracy gains in the highest-volume
promotion regime.

A likely limitation is the semantics of `onpromotion`.

It records only the number of promoted items within a store-family-date state.
It does not identify:

- the promoted SKUs;
- discount depth;
- SKU-level baseline demand;
- relative SKU importance;
- promotion mechanics;
- prices;
- campaign visibility.

Two observations with the same promotion count may therefore represent
substantially different commercial interventions.

---

#### Temporal stability

The promoted-target result is not temporally stable.

The count-based specification improves promoted-target WAPE on only one of the
six rolling deployments:

$$
\text{July 18}.
$$

On that deployment, promoted-target WAPE improves by approximately:

$$
2.48\%.
$$

However, the other five deployments show relative degradations of approximately:

$$
2.24\%,\quad
0.42\%,\quad
1.90\%,\quad
1.43\%,\quad
0.30\%.
$$

The initial July 18 result is therefore an atypical validation outcome rather
than evidence of a robust promotion-aware forecasting improvement.

This illustrates why repeated temporal validation is necessary before a new
feature is promoted into a production forecasting contract.

---

#### Forecast-horizon stability

The count-based specification outperforms Model A on only four of the fourteen
forecast horizons.

No stable relationship is observed between forecast distance and the value of
the promotion feature.

In particular, the results do not support a simple strategy in which promotion
information is used only for short or long forecast horizons.

Post-hoc horizon-specific model selection would therefore not be justified by
the available validation evidence.

---

#### Final forecasting decision

The production forecasting champion remains:

$$
\boxed{
\texttt{point\_forecast\_v1}
}
$$

with no promotion feature added to its production predictor contract.

This conclusion does **not** imply that promotions have no effect on sales.

It establishes a narrower predictive result:

> The available future-known promotion signal does not provide a robust
> incremental out-of-time forecasting improvement when added as a feature to
> the frozen Version 1 global XGBoost architecture.

The failure of the monolithic promotion-aware forecasting extension also does
not imply that promotion effects cannot be modeled more effectively under a
different architecture.

A natural future extension would explicitly separate baseline demand from
incremental promotion response:

$$
\hat Y
=
\hat m_0(X)
+
D\hat\tau(X),
$$

where $\hat m_0(X)$ estimates sales under no promotion and
$\hat\tau(X)$ estimates promotion uplift.

Such a decomposition requires a causal treatment-effect framework and is
therefore deferred until after the causal analysis.

The count-based promotion model is retained only as a diagnostic predictive
model for the model-response analysis in Section 4.

It is not promoted to production and its learned promotion response is not
interpreted as causal evidence.

## 4. Model Counterfactual Promotion Response

The rolling experiment did not select the promotion-aware specification as the
production forecaster.

The production champion therefore remains:

$$
\texttt{point\_forecast\_v1}.
$$

Nevertheless, the count-based promotion model remains useful as a diagnostic
object.

Rather than asking whether it forecasts better, this section asks a different
question:

> How does the prediction of the promotion-aware model change when its
> promotion input is modified while the remaining predictor state is held
> fixed?

This is a **model-response diagnostic**, not a production forecast comparison
and not a causal estimate.

The distinction is essential:

$$
\text{forecast accuracy}
\neq
\text{model-implied response}
\neq
\text{causal treatment effect}.
$$

### 4.1 Prediction Under Promotion vs No Promotion

The promotion-aware count specification was not selected as the production
forecasting model.

Nevertheless, its learned response to promotion can be inspected directly.

For each of the six rolling deployments, the exact previously evaluated
`B_count` specification is reconstructed:

$$
\hat Y
=
F_B(X,P),
$$

where $X$ contains the Version 1 forecasting state and $P$ is the target-date
promotion count.

For every validation observation, the same fitted model is evaluated twice.

First, under the observed promotion input:

$$
\hat Y_i^{\mathrm{observed}}
=
F_B(X_i,P_i).
$$

Second, under an artificial no-promotion input:

$$
\hat Y_i^{P=0}
=
F_B(X_i,0).
$$

All other predictors are held fixed.

The resulting model-implied response is:

$$
\Delta_{\mathrm{model},i}
=
F_B(X_i,P_i)
-
F_B(X_i,0).
$$

Unlike the earlier Model A versus Model B forecasting comparison, this
experiment changes only the promotion input within the **same fitted model**.

For observations with:

$$
P_i=0,
$$

the two predictor states are identical and therefore:

$$
\Delta_{\mathrm{model},i}=0.
$$

This makes the experiment useful for understanding what response to promotion
the predictive model has learned.

However, the intervention is performed only inside the fitted prediction
function.

The resulting quantity is therefore a **model-implied predictive response**,
not yet a causal treatment effect.

In [19]:
# Inspect the current notebook state for fitted XGBoost model objects.
fitted_xgboost_objects = {
    name: type(value).__name__
    for name, value in globals().items()
    if (
        "XGB" in type(value).__name__
        or "model" in name.lower()
    )
}

fitted_xgboost_objects

{'MODEL_FEATURES_A': 'list',
 'missing_model_features': 'list',
 'MODEL_FEATURES_B_FLAG': 'list',
 'MODEL_FEATURES_B_COUNT': 'list',
 'MODEL_FEATURES_B_LOG': 'list',
 'PROMOTION_EXPERIMENT_MODEL_PARAMS': 'dict',
 'PROMOTION_MODEL_SPECS': 'dict',
 'smoke_model_results': 'DataFrame',
 'smoke_model_predictions': 'dict',
 'model_name': 'str',
 'model_a_wape': 'float64',
 'REFERENCE_MODEL_A_WAPE': 'float',
 'model_a_pooled_metrics': 'dict',
 'model_a_pooled_wape': 'float64',
 'model_fold_metrics': 'DataFrame',
 'model_wape_by_fold': 'Series',
 'model_metrics': 'dict'}

In [20]:
# ---------------------------------------------------------------------
# Refit the July 18 B_count diagnostic model.
#
# The original smoke-test model object was intentionally not retained.
# We therefore reproduce the exact deterministic fit once and verify
# prediction parity against the already stored smoke-test predictions.
#
# This model is retained only for predictive response diagnostics.
# It is NOT the production forecasting champion and is NOT a causal
# treatment-effect estimator.
# ---------------------------------------------------------------------

import numpy as np

from xgboost import XGBRegressor


# ---------------------------------------------------------------------
# Prepare the exact July 18 training and validation matrices used in
# the controlled promotion experiment.
# ---------------------------------------------------------------------

(
    X_b_count_train,
    y_b_count_train,
) = prepare_xgboost_xy(
    smoke_train,
    MODEL_FEATURES_B_COUNT,
)


(
    X_b_count_validation_observed,
    y_b_count_validation,
) = prepare_xgboost_xy(
    smoke_validation,
    MODEL_FEATURES_B_COUNT,
)


# ---------------------------------------------------------------------
# Refit the previously evaluated deterministic B_count specification.
# ---------------------------------------------------------------------

diagnostic_b_count_model = XGBRegressor(
    **PROMOTION_EXPERIMENT_MODEL_PARAMS
)


diagnostic_b_count_model.fit(
    X_b_count_train,
    y_b_count_train,
)


# ---------------------------------------------------------------------
# Reproduce the original observed-promotion predictions.
# ---------------------------------------------------------------------

diagnostic_observed_predictions = np.clip(
    diagnostic_b_count_model.predict(
        X_b_count_validation_observed
    ),
    a_min=0.0,
    a_max=None,
).astype(
    "float64"
)


stored_smoke_predictions = np.asarray(
    smoke_model_predictions[
        "B_count"
    ],
    dtype="float64",
)


refit_absolute_difference = np.abs(
    diagnostic_observed_predictions
    - stored_smoke_predictions
)


print(
    "Max refit prediction difference:",
    refit_absolute_difference.max(),
)

print(
    "Mean refit prediction difference:",
    refit_absolute_difference.mean(),
)


assert (
    refit_absolute_difference.max()
    < 1e-6
), (
    "The refitted B_count model does not reproduce the stored "
    "July 18 smoke-test predictions."
)


print()
print(
    "B_COUNT DIAGNOSTIC MODEL REFIT PARITY: PASSED"
)

Max refit prediction difference: 0.0
Mean refit prediction difference: 0.0

B_COUNT DIAGNOSTIC MODEL REFIT PARITY: PASSED


In [21]:
# ---------------------------------------------------------------------
# Six-fold model-implied promotion response.
#
# For each historical deployment:
#
#   1. reconstruct the exact previously evaluated B_count model;
#   2. verify parity with the stored rolling-validation predictions;
#   3. predict under the observed promotion count;
#   4. predict again after setting ONLY promo_count = 0;
#   5. retain the resulting within-model prediction difference.
#
# July 18 reuses the already parity-validated model currently in memory.
#
# These are predictive model-response diagnostics.
# They are NOT causal counterfactual estimates.
# ---------------------------------------------------------------------

import gc
from pathlib import Path

import numpy as np
import pandas as pd

from xgboost import XGBRegressor


MODEL_RESPONSE_CHECKPOINT_DIR = (
    project_root
    / "artifacts"
    / "promotion_forecasting"
    / "model_response_checkpoints"
)


MODEL_RESPONSE_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# Defensive contract check:
# B_count must contain exactly one promotion representation.
# ---------------------------------------------------------------------

assert (
    "promo_count"
    in MODEL_FEATURES_B_COUNT
)

assert (
    "promo_flag"
    not in MODEL_FEATURES_B_COUNT
)

assert (
    "promo_log_count"
    not in MODEL_FEATURES_B_COUNT
)


model_response_frames = []


for (
    fold_number,
    validation_origin,
) in enumerate(
    ROLLING_VALIDATION_ORIGINS,
    start=1,
):

    validation_origin = pd.Timestamp(
        validation_origin
    )


    checkpoint_path = (
        MODEL_RESPONSE_CHECKPOINT_DIR
        / (
            "validation_origin="
            f"{validation_origin.date()}.parquet"
        )
    )


    print(
        "\n"
        + "=" * 80
    )

    print(
        f"Model-response fold "
        f"{fold_number}/"
        f"{len(ROLLING_VALIDATION_ORIGINS)}"
    )

    print(
        "Validation origin:",
        validation_origin.date(),
    )


    # -------------------------------------------------------------
    # Resume completed diagnostic folds without retraining.
    # -------------------------------------------------------------

    if checkpoint_path.exists():

        fold_response = (
            pd.read_parquet(
                checkpoint_path
            )
        )


        required_columns = [
            "store_nbr",
            "family",
            "validation_origin",
            "horizon",
            "target_date",
            "target_sales",
            "promo_count",
            "prediction_observed_promo",
            "prediction_no_promo",
            "model_implied_uplift",
        ]


        assert all(
            column
            in fold_response.columns
            for column
            in required_columns
        )


        assert (
            len(
                fold_response
            )
            == 24_948
        )


        model_response_frames.append(
            fold_response
        )


        print(
            "Checkpoint loaded."
        )

        continue


    # -------------------------------------------------------------
    # Exact rolling two-year training population.
    # -------------------------------------------------------------

    minimum_training_origin = (
        validation_origin
        - pd.DateOffset(
            years=2
        )
    )


    fold_train = (
        supervised_promo.loc[
            (
                supervised_promo[
                    "target_date"
                ]
                <= validation_origin
            )
            &
            (
                supervised_promo[
                    "forecast_origin"
                ]
                >= minimum_training_origin
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    (
        fold_train,
        excluded_training_rows,
    ) = filter_to_started_store_scope(
        fold_train
    )


    assert (
        len(
            fold_train
        )
        ==
        ROLLING_EXPECTED_TRAINING_ROWS[
            validation_origin
        ]
    )


    # -------------------------------------------------------------
    # Exact 14-day validation deployment.
    # -------------------------------------------------------------

    fold_validation = (
        build_promotion_validation_fold(
            validation_origin
        )
    )


    # -------------------------------------------------------------
    # Prepare observed-promotion feature matrix.
    # -------------------------------------------------------------

    (
        X_train,
        y_train,
    ) = prepare_xgboost_xy(
        fold_train,
        MODEL_FEATURES_B_COUNT,
    )


    (
        X_validation_observed,
        y_validation,
    ) = prepare_xgboost_xy(
        fold_validation,
        MODEL_FEATURES_B_COUNT,
    )


    # -------------------------------------------------------------
    # Reuse July 18 fitted model.
    #
    # All other folds require one deterministic B_count refit.
    # -------------------------------------------------------------

    if (
        validation_origin
        == SMOKE_VALIDATION_ORIGIN
    ):

        fold_model = (
            diagnostic_b_count_model
        )


        print(
            "Reusing parity-validated July 18 model."
        )


    else:

        fold_model = XGBRegressor(
            **PROMOTION_EXPERIMENT_MODEL_PARAMS
        )


        fold_model.fit(
            X_train,
            y_train,
        )


    # -------------------------------------------------------------
    # Observed-promotion predictions.
    # -------------------------------------------------------------

    observed_predictions = np.clip(
        fold_model.predict(
            X_validation_observed
        ),
        a_min=0.0,
        a_max=None,
    ).astype(
        "float64"
    )


    # -------------------------------------------------------------
    # Verify exact reproduction of the predictions already used in
    # the six-fold forecasting evaluation.
    # -------------------------------------------------------------

    stored_fold_predictions = (
        promotion_rolling_predictions.loc[
            promotion_rolling_predictions[
                "validation_origin"
            ]
            .eq(
                validation_origin
            ),
            "prediction_b_count",
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    assert (
        len(
            stored_fold_predictions
        )
        == len(
            observed_predictions
        )
    )


    parity_difference = np.abs(
        observed_predictions
        - stored_fold_predictions
    )


    print(
        "Observed-prediction parity max diff:",
        parity_difference.max(),
    )


    assert (
        parity_difference.max()
        < 1e-6
    ), (
        "B_count refit failed to reproduce the stored "
        "rolling-validation predictions."
    )


    # -------------------------------------------------------------
    # Artificial no-promotion predictor state.
    #
    # IMPORTANT:
    # Only the model input `promo_count` is changed.
    # Every demand-history, seasonal, calendar and entity predictor
    # remains identical to the observed validation state.
    # -------------------------------------------------------------

    validation_no_promo = (
        fold_validation
        .copy()
    )


    validation_no_promo[
        "promo_count"
    ] = np.int16(
        0
    )


    (
        X_validation_no_promo,
        _,
    ) = prepare_xgboost_xy(
        validation_no_promo,
        MODEL_FEATURES_B_COUNT,
    )


    no_promo_predictions = np.clip(
        fold_model.predict(
            X_validation_no_promo
        ),
        a_min=0.0,
        a_max=None,
    ).astype(
        "float64"
    )


    # -------------------------------------------------------------
    # Model-implied promotion response.
    # -------------------------------------------------------------

    model_implied_uplift = (
        observed_predictions
        - no_promo_predictions
    )


    # For an actually non-promoted row, observed P is already zero.
    # Therefore the two feature vectors must be identical and the
    # model-implied response must be exactly zero.
    non_promoted_mask = (
        fold_validation[
            "promo_count"
        ]
        .eq(
            0
        )
        .to_numpy()
    )


    assert np.allclose(
        model_implied_uplift[
            non_promoted_mask
        ],
        0.0,
        atol=1e-10,
    )


    # -------------------------------------------------------------
    # Compact row-level diagnostic artifact.
    # -------------------------------------------------------------

    fold_response = (
        fold_validation[
            [
                "store_nbr",
                "family",
                "horizon",
                "target_date",
                "target_sales",
                "promo_flag",
                "promo_count",
            ]
        ]
        .copy()
    )


    fold_response[
        "validation_origin"
    ] = validation_origin


    fold_response[
        "prediction_observed_promo"
    ] = observed_predictions


    fold_response[
        "prediction_no_promo"
    ] = no_promo_predictions


    fold_response[
        "model_implied_uplift"
    ] = model_implied_uplift


    fold_response.to_parquet(
        checkpoint_path,
        index=False,
    )


    model_response_frames.append(
        fold_response
    )


    print(
        "Promoted rows:",
        f"{(~non_promoted_mask).sum():,}",
    )

    print(
        "Mean model-implied uplift on promoted rows:",
        f"{model_implied_uplift[~non_promoted_mask].mean():.3f}",
    )

    print(
        "Median model-implied uplift on promoted rows:",
        f"{np.median(model_implied_uplift[~non_promoted_mask]):.3f}",
    )

    print(
        "Checkpoint written:",
        checkpoint_path.name,
    )


    # Do not delete the persistent July 18 model.
    if (
        validation_origin
        != SMOKE_VALIDATION_ORIGIN
    ):

        del fold_model


    del (
        X_train,
        y_train,
        X_validation_observed,
        X_validation_no_promo,
        observed_predictions,
        no_promo_predictions,
        model_implied_uplift,
        fold_train,
        fold_validation,
        validation_no_promo,
    )


    gc.collect()


# ---------------------------------------------------------------------
# Complete six-fold model-response panel.
# ---------------------------------------------------------------------

promotion_model_response = (
    pd.concat(
        model_response_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "validation_origin",
            "store_nbr",
            "family",
            "horizon",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    len(
        promotion_model_response
    )
    == 149_688
)


assert (
    promotion_model_response.loc[
        promotion_model_response[
            "promo_count"
        ]
        .eq(
            0
        ),
        "model_implied_uplift",
    ]
    .abs()
    .max()
    < 1e-10
)


print()
print(
    "Complete model-response rows:",
    f"{len(promotion_model_response):,}",
)

print(
    "Promoted rows:",
    f"{promotion_model_response['promo_flag'].sum():,}",
)

print()
print(
    "SIX-FOLD MODEL-IMPLIED PROMOTION RESPONSE: PASSED"
)


Model-response fold 1/6
Validation origin: 2017-05-09
Observed-prediction parity max diff: 0.0
Promoted rows: 12,509
Mean model-implied uplift on promoted rows: 80.432
Median model-implied uplift on promoted rows: 11.935
Checkpoint written: validation_origin=2017-05-09.parquet

Model-response fold 2/6
Validation origin: 2017-05-23
Observed-prediction parity max diff: 0.0
Promoted rows: 11,306
Mean model-implied uplift on promoted rows: 45.472
Median model-implied uplift on promoted rows: 6.319
Checkpoint written: validation_origin=2017-05-23.parquet

Model-response fold 3/6
Validation origin: 2017-06-06
Observed-prediction parity max diff: 0.0
Promoted rows: 11,363
Mean model-implied uplift on promoted rows: 72.966
Median model-implied uplift on promoted rows: 13.884
Checkpoint written: validation_origin=2017-06-06.parquet

Model-response fold 4/6
Validation origin: 2017-06-20
Observed-prediction parity max diff: 0.0
Promoted rows: 11,147
Mean model-implied uplift on promoted rows: 69

### 4.2 Naive Model-Implied Uplift

Across all six rolling deployments, the promotion-aware predictive model
responds positively on average when the observed promotion count is retained
rather than artificially set to zero.

For every promoted validation observation:

$$
\Delta_{\mathrm{model},i}
=
F_B(X_i,P_i)
-
F_B(X_i,0).
$$

The mean model-implied response is positive on every deployment, ranging from
approximately 45 to 80 additional predicted units.

The median response is also positive on every deployment, but substantially
smaller than the mean.

This difference suggests a strongly right-skewed response distribution:
many promoted store-family-date observations receive modest prediction
adjustments, while a smaller number of high-volume observations receive much
larger adjustments.

This is already an important distinction from the forecasting-accuracy result.

The promotion-aware model has clearly learned an association between promotion
and higher sales predictions.

However, learning a positive promotion response is not sufficient to improve
forecast accuracy.

A model can learn the correct qualitative direction while estimating the
magnitude or heterogeneity of that response poorly enough to degrade
out-of-time WAPE.

The following diagnostics therefore examine:

- the distribution and sign of the learned response;
- temporal stability;
- variation with promotion intensity;
- variation across forecast horizons;
- variation across product families.

All quantities in this section remain predictive model responses rather than
causal treatment effects.

In [22]:
# ---------------------------------------------------------------------
# Model-implied promotion-response diagnostics.
#
# No model is trained in this section.
#
# The analysis is restricted to actually promoted validation rows
# because non-promoted rows have P = 0 in both predictor states and
# therefore exactly zero model-implied response by construction.
#
# IMPORTANT:
# `model_implied_uplift` is a response of the fitted predictive model.
# It must not be interpreted as a causal treatment effect.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


promoted_model_response = (
    promotion_model_response.loc[
        promotion_model_response[
            "promo_count"
        ]
        .gt(
            0
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert (
    len(
        promoted_model_response
    )
    == 67_132
)


# =====================================================================
# 1. OVERALL RESPONSE DISTRIBUTION
# =====================================================================

uplift = (
    promoted_model_response[
        "model_implied_uplift"
    ]
)


overall_response_summary = pd.DataFrame(
    [
        {
            "rows":
                len(
                    promoted_model_response
                ),

            "mean_uplift":
                uplift.mean(),

            "median_uplift":
                uplift.median(),

            "p10_uplift":
                uplift.quantile(
                    0.10
                ),

            "p25_uplift":
                uplift.quantile(
                    0.25
                ),

            "p75_uplift":
                uplift.quantile(
                    0.75
                ),

            "p90_uplift":
                uplift.quantile(
                    0.90
                ),

            "p95_uplift":
                uplift.quantile(
                    0.95
                ),

            "p99_uplift":
                uplift.quantile(
                    0.99
                ),

            "positive_share":
                uplift.gt(
                    0
                )
                .mean(),

            "negative_share":
                uplift.lt(
                    0
                )
                .mean(),

            "zero_share":
                uplift.eq(
                    0
                )
                .mean(),

            # Aggregate relative response is more stable than dividing
            # row-level uplift by potentially tiny baseline predictions.
            "aggregate_relative_response":
                (
                    uplift.sum()
                    /
                    promoted_model_response[
                        "prediction_no_promo"
                    ]
                    .sum()
                ),
        }
    ]
)


print(
    "=" * 80
)

print(
    "OVERALL MODEL-IMPLIED PROMOTION RESPONSE"
)

print(
    "=" * 80
)


display(
    overall_response_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "mean_uplift":
                "{:.3f}",

            "median_uplift":
                "{:.3f}",

            "p10_uplift":
                "{:.3f}",

            "p25_uplift":
                "{:.3f}",

            "p75_uplift":
                "{:.3f}",

            "p90_uplift":
                "{:.3f}",

            "p95_uplift":
                "{:.3f}",

            "p99_uplift":
                "{:.3f}",

            "positive_share":
                "{:.2%}",

            "negative_share":
                "{:.2%}",

            "zero_share":
                "{:.2%}",

            "aggregate_relative_response":
                "{:+.2%}",
        }
    )
)


# =====================================================================
# 2. TEMPORAL STABILITY
# =====================================================================

fold_response_rows = []


for validation_origin in (
    ROLLING_VALIDATION_ORIGINS
):

    fold_frame = (
        promoted_model_response.loc[
            promoted_model_response[
                "validation_origin"
            ]
            .eq(
                validation_origin
            )
        ]
    )


    fold_uplift = (
        fold_frame[
            "model_implied_uplift"
        ]
    )


    fold_response_rows.append(
        {
            "validation_origin":
                validation_origin,

            "rows":
                len(
                    fold_frame
                ),

            "mean_uplift":
                fold_uplift.mean(),

            "median_uplift":
                fold_uplift.median(),

            "positive_share":
                fold_uplift.gt(
                    0
                )
                .mean(),

            "negative_share":
                fold_uplift.lt(
                    0
                )
                .mean(),

            "aggregate_relative_response":
                (
                    fold_uplift.sum()
                    /
                    fold_frame[
                        "prediction_no_promo"
                    ]
                    .sum()
                ),
        }
    )


model_response_by_fold = pd.DataFrame(
    fold_response_rows
)


print()
print(
    "=" * 80
)

print(
    "TEMPORAL STABILITY"
)

print(
    "=" * 80
)


display(
    model_response_by_fold
    .style.hide(
        axis="index"
    )
    .format(
        {
            "mean_uplift":
                "{:.3f}",

            "median_uplift":
                "{:.3f}",

            "positive_share":
                "{:.2%}",

            "negative_share":
                "{:.2%}",

            "aggregate_relative_response":
                "{:+.2%}",
        }
    )
)


# =====================================================================
# 3. RESPONSE BY PROMOTION INTENSITY
# =====================================================================

promo_intensity_bins = [
    0.5,
    1.5,
    4.5,
    9.5,
    24.5,
    np.inf,
]


promo_intensity_labels = [
    "1",
    "2-4",
    "5-9",
    "10-24",
    "25+",
]


promoted_model_response[
    "promotion_intensity"
] = pd.cut(
    promoted_model_response[
        "promo_count"
    ],
    bins=
        promo_intensity_bins,
    labels=
        promo_intensity_labels,
    ordered=True,
)


intensity_response_rows = []


for intensity_level in (
    promo_intensity_labels
):

    intensity_frame = (
        promoted_model_response.loc[
            promoted_model_response[
                "promotion_intensity"
            ]
            .eq(
                intensity_level
            )
        ]
    )


    intensity_uplift = (
        intensity_frame[
            "model_implied_uplift"
        ]
    )


    intensity_response_rows.append(
        {
            "promotion_intensity":
                intensity_level,

            "rows":
                len(
                    intensity_frame
                ),

            "mean_promo_count":
                intensity_frame[
                    "promo_count"
                ]
                .mean(),

            "mean_uplift":
                intensity_uplift.mean(),

            "median_uplift":
                intensity_uplift.median(),

            "positive_share":
                intensity_uplift.gt(
                    0
                )
                .mean(),

            "negative_share":
                intensity_uplift.lt(
                    0
                )
                .mean(),

            "aggregate_relative_response":
                (
                    intensity_uplift.sum()
                    /
                    intensity_frame[
                        "prediction_no_promo"
                    ]
                    .sum()
                ),
        }
    )


model_response_by_intensity = pd.DataFrame(
    intensity_response_rows
)


print()
print(
    "=" * 80
)

print(
    "RESPONSE BY PROMOTION INTENSITY"
)

print(
    "=" * 80
)


display(
    model_response_by_intensity
    .style.hide(
        axis="index"
    )
    .format(
        {
            "mean_promo_count":
                "{:.2f}",

            "mean_uplift":
                "{:.3f}",

            "median_uplift":
                "{:.3f}",

            "positive_share":
                "{:.2%}",

            "negative_share":
                "{:.2%}",

            "aggregate_relative_response":
                "{:+.2%}",
        }
    )
)


# =====================================================================
# 4. RESPONSE BY FORECAST HORIZON
# =====================================================================

horizon_response_rows = []


for horizon in range(
    1,
    FORECAST_HORIZON + 1,
):

    horizon_frame = (
        promoted_model_response.loc[
            promoted_model_response[
                "horizon"
            ]
            .eq(
                horizon
            )
        ]
    )


    horizon_uplift = (
        horizon_frame[
            "model_implied_uplift"
        ]
    )


    horizon_response_rows.append(
        {
            "horizon":
                horizon,

            "rows":
                len(
                    horizon_frame
                ),

            "mean_uplift":
                horizon_uplift.mean(),

            "median_uplift":
                horizon_uplift.median(),

            "positive_share":
                horizon_uplift.gt(
                    0
                )
                .mean(),

            "aggregate_relative_response":
                (
                    horizon_uplift.sum()
                    /
                    horizon_frame[
                        "prediction_no_promo"
                    ]
                    .sum()
                ),
        }
    )


model_response_by_horizon = pd.DataFrame(
    horizon_response_rows
)


print()
print(
    "=" * 80
)

print(
    "RESPONSE BY FORECAST HORIZON"
)

print(
    "=" * 80
)


display(
    model_response_by_horizon
    .style.hide(
        axis="index"
    )
    .format(
        {
            "mean_uplift":
                "{:.3f}",

            "median_uplift":
                "{:.3f}",

            "positive_share":
                "{:.2%}",

            "aggregate_relative_response":
                "{:+.2%}",
        }
    )
)


# =====================================================================
# 5. RESPONSE BY PRODUCT FAMILY
# =====================================================================
#
# This is descriptive only. We do not select or retune models by family.
# ---------------------------------------------------------------------

family_response_rows = []


for (
    family,
    family_frame,
) in promoted_model_response.groupby(
    "family",
    observed=True,
    sort=False,
):

    family_uplift = (
        family_frame[
            "model_implied_uplift"
        ]
    )


    family_response_rows.append(
        {
            "family":
                family,

            "rows":
                len(
                    family_frame
                ),

            "actual_sales":
                family_frame[
                    "target_sales"
                ]
                .sum(),

            "mean_uplift":
                family_uplift.mean(),

            "median_uplift":
                family_uplift.median(),

            "positive_share":
                family_uplift.gt(
                    0
                )
                .mean(),

            "aggregate_relative_response":
                (
                    family_uplift.sum()
                    /
                    family_frame[
                        "prediction_no_promo"
                    ]
                    .sum()
                ),
        }
    )


model_response_by_family = (
    pd.DataFrame(
        family_response_rows
    )
    .sort_values(
        "actual_sales",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print()
print(
    "=" * 80
)

print(
    "RESPONSE BY PRODUCT FAMILY"
)

print(
    "=" * 80
)


display(
    model_response_by_family
    .style.hide(
        axis="index"
    )
    .format(
        {
            "actual_sales":
                "{:,.1f}",

            "mean_uplift":
                "{:.3f}",

            "median_uplift":
                "{:.3f}",

            "positive_share":
                "{:.2%}",

            "aggregate_relative_response":
                "{:+.2%}",
        }
    )
)


# =====================================================================
# 6. MONOTONIC ASSOCIATION BETWEEN PROMOTION COUNT AND MODEL RESPONSE
# =====================================================================
#
# Spearman correlation examines rank association rather than assuming a
# linear relationship between promotion count and prediction response.
# ---------------------------------------------------------------------

promo_uplift_spearman = (
    promoted_model_response[
        [
            "promo_count",
            "model_implied_uplift",
        ]
    ]
    .corr(
        method="spearman"
    )
    .loc[
        "promo_count",
        "model_implied_uplift",
    ]
)


print()
print(
    "Spearman correlation between promo_count "
    "and model-implied uplift:",
    f"{promo_uplift_spearman:.4f}",
)

print()
print(
    "MODEL-IMPLIED UPLIFT DIAGNOSTICS: PASSED"
)

OVERALL MODEL-IMPLIED PROMOTION RESPONSE


rows,mean_uplift,median_uplift,p10_uplift,p25_uplift,p75_uplift,p90_uplift,p95_uplift,p99_uplift,positive_share,negative_share,zero_share,aggregate_relative_response
67132,69.116,12.095,0.684,2.362,52.940,192.908,334.416,777.762,93.24%,6.74%,0.02%,+7.34%



TEMPORAL STABILITY


validation_origin,rows,mean_uplift,median_uplift,positive_share,negative_share,aggregate_relative_response
2017-05-09 00:00:00,12509,80.432,11.935,94.97%,4.93%,+9.32%
2017-05-23 00:00:00,11306,45.472,6.319,86.30%,13.70%,+5.03%
2017-06-06 00:00:00,11363,72.966,13.884,93.82%,6.17%,+7.65%
2017-06-20 00:00:00,11147,69.121,10.591,92.37%,7.63%,+7.05%
2017-07-04 00:00:00,10628,77.498,20.508,96.97%,3.03%,+7.86%
2017-07-18 00:00:00,10179,68.417,13.316,95.25%,4.75%,+7.01%



RESPONSE BY PROMOTION INTENSITY


promotion_intensity,rows,mean_promo_count,mean_uplift,median_uplift,positive_share,negative_share,aggregate_relative_response
1,12977,1.00,2.163,1.542,93.00%,6.92%,+1.21%
2-4,14362,2.78,7.342,4.595,91.07%,8.92%,+3.21%
5-9,10586,6.78,17.412,11.081,88.53%,11.46%,+3.50%
10-24,13886,15.34,48.830,32.155,94.34%,5.66%,+5.86%
25+,15321,50.48,237.843,146.927,97.75%,2.25%,+8.94%


... (2 outputs left out)


RESPONSE BY PRODUCT FAMILY


/var/folders/9b/jy4h3b490nx9py6bch99p2tm0000gr/T/ipykernel_69197/711653668.py:630: RuntimeWarning: divide by zero encountered in scalar divide
  family_uplift.sum()


family,rows,actual_sales,mean_uplift,median_uplift,positive_share,aggregate_relative_response
GROCERY I,4530,"21,264,259.9",326.547,254.929,98.19%,+7.38%
BEVERAGES,4530,"16,198,913.0",244.492,193.691,97.95%,+7.18%
PRODUCE,3984,"9,847,664.8",199.539,45.412,90.84%,+8.54%
CLEANING,4530,"5,943,976.0",67.517,48.083,93.16%,+5.63%
DAIRY,4520,"4,160,541.0",39.099,29.186,88.23%,+4.20%
BREAD/BAKERY,3452,"1,815,526.5",17.072,9.345,87.66%,+3.24%
PERSONAL CARE,4519,"1,420,367.0",13.601,9.290,87.52%,+4.45%
HOME CARE,4530,"1,386,658.0",32.846,27.969,97.11%,+11.74%
DELI,4253,"1,371,893.8",17.625,6.952,71.62%,+5.63%
MEATS,1830,"863,879.6",84.990,41.314,97.38%,+21.41%



Spearman correlation between promo_count and model-implied uplift: 0.7993

MODEL-IMPLIED UPLIFT DIAGNOSTICS: PASSED


In [23]:
# ---------------------------------------------------------------------
# Clean the family-level relative-response diagnostic.
#
# Very small families can have a zero aggregate no-promotion prediction,
# making the relative response undefined rather than infinite.
# ---------------------------------------------------------------------

family_no_promo_totals = (
    promoted_model_response
    .groupby(
        "family",
        observed=True,
    )[
        "prediction_no_promo"
    ]
    .sum()
)


model_response_by_family[
    "aggregate_relative_response"
] = (
    model_response_by_family
    .apply(
        lambda row:
            (
                row[
                    "mean_uplift"
                ]
                * row[
                    "rows"
                ]
                / family_no_promo_totals.loc[
                    row[
                        "family"
                    ]
                ]
            )
            if (
                family_no_promo_totals.loc[
                    row[
                        "family"
                    ]
                ]
                > 1e-8
            )
            else np.nan,
        axis=1,
    )
)


display(
    model_response_by_family
    .style.hide(
        axis="index"
    )
    .format(
        {
            "actual_sales":
                "{:,.1f}",

            "mean_uplift":
                "{:.3f}",

            "median_uplift":
                "{:.3f}",

            "positive_share":
                "{:.2%}",

            "aggregate_relative_response":
                "{:+.2%}",
        },
        na_rep="N/A",
    )
)

family,rows,actual_sales,mean_uplift,median_uplift,positive_share,aggregate_relative_response
GROCERY I,4530,"21,264,259.9",326.547,254.929,98.19%,+7.38%
BEVERAGES,4530,"16,198,913.0",244.492,193.691,97.95%,+7.18%
PRODUCE,3984,"9,847,664.8",199.539,45.412,90.84%,+8.54%
CLEANING,4530,"5,943,976.0",67.517,48.083,93.16%,+5.63%
DAIRY,4520,"4,160,541.0",39.099,29.186,88.23%,+4.20%
BREAD/BAKERY,3452,"1,815,526.5",17.072,9.345,87.66%,+3.24%
PERSONAL CARE,4519,"1,420,367.0",13.601,9.290,87.52%,+4.45%
HOME CARE,4530,"1,386,658.0",32.846,27.969,97.11%,+11.74%
DELI,4253,"1,371,893.8",17.625,6.952,71.62%,+5.63%
MEATS,1830,"863,879.6",84.990,41.314,97.38%,+21.41%


#### Model-response interpretation

The promotion-aware predictive model has learned a clear and stable positive
response to the target-date promotion count.

Across the 67,132 promoted validation observations:

- mean model-implied response is approximately 69.1 sales units;
- median response is approximately 12.1 units;
- approximately 93.24% of responses are positive;
- aggregate predictions are approximately 7.34% higher under the observed
  promotion counts than under the artificial zero-promotion input.

The substantial difference between the mean and median indicates a strongly
right-skewed response distribution, with a relatively small number of
high-volume observations receiving very large prediction adjustments.

The response is also temporally stable. All six rolling deployments exhibit a
positive mean and median response, and the positive-response share ranges from
approximately 86% to 97%.

Most importantly, the learned response increases strongly with promotion
intensity.

Mean model-implied response rises from approximately:

$$
2.2
$$

units when one item is promoted to approximately:

$$
237.8
$$

units for observations with at least 25 promoted items.

The rank correlation between promotion count and model-implied response is:

$$
\rho_{\mathrm{Spearman}}
=
0.7993.
$$

The promotion-aware model has therefore learned a strong monotonic association
between promotion intensity and higher sales predictions.

---

This finding clarifies the earlier forecasting result.

The failure of Model B to improve pooled rolling WAPE is **not** evidence that
the model ignored the promotion feature.

Instead, Model B uses the feature extensively and in a directionally coherent
way.

The largest learned prediction adjustments occur precisely in the
high-intensity promotion regime, where the forecasting experiment previously
showed the largest deterioration relative to Model A.

For observations with:

$$
\text{onpromotion}\geq25,
$$

the model applies an average prediction increase of approximately 238 units,
yet this same segment experiences worse out-of-time WAPE than the frozen
baseline.

This suggests that the main forecasting limitation is more likely related to
the calibration and heterogeneity of the learned promotion response than to
the absence of a promotion signal.

The available `onpromotion` count may be too coarse to distinguish commercial
interventions with very different SKU composition, discount depth and economic
importance.

The result motivates more structured forecasting approaches, including a
frozen-baseline residual correction and, after causal estimation, an explicit
baseline-plus-uplift decomposition.

These remain future modeling extensions rather than changes to the frozen
forecasting champion.

### 4.3 Why Predictive Counterfactuals Are Not Yet Causal

The model-response experiment produces a quantity that resembles an uplift:

$$
\Delta_{\mathrm{model}}
=
F_B(X,P)
-
F_B(X,0).
$$

However, this quantity must not be interpreted as:

$$
Y(P)-Y(0).
$$

The distinction follows from the objective used to train the predictive model.

The promotion-aware forecaster learns the observational conditional
relationship:

$$
F_B(X,P)
\approx
E[Y\mid X,P].
$$

A causal treatment-effect analysis instead requires potential outcomes such as:

$$
E[Y(p)\mid X].
$$

These quantities are not generally equal:

$$
E[Y\mid X,P=p]
\neq
E[Y(p)\mid X].
$$

---

#### Observational promotion assignment

Promotions are not randomly assigned.

Commercial context can simultaneously influence both promotion decisions and
sales.

For example:

$$
\text{expected demand}
\rightarrow
P
$$

and:

$$
\text{expected demand}
\rightarrow
Y.
$$

A predictive model can therefore learn a strong positive relationship between
promotion count and sales even if part of that relationship is generated by
the contexts in which promotions are selected.

The strong model-implied response observed in Section 4.2 establishes that the
predictive model uses the promotion feature.

It does not establish how much of that response is caused by promotion itself.

---

#### Changing a feature is not the same as intervening in the world

The artificial no-promotion experiment changes only one input to the fitted
prediction function:

$$
P
\leftarrow
0.
$$

All other feature values remain fixed.

This creates a valid **model counterfactual**:

> What would this fitted function predict if its promotion input were changed?

It does not automatically create a valid causal counterfactual:

> What would actually have happened to sales if the retailer had not run the
> promotion?

The latter question requires assumptions about treatment assignment,
confounding, overlap and treatment consistency that the forecasting model was
not designed to enforce.

---

#### Support and extrapolation

Some observed commercial contexts may almost always receive promotion.

Setting:

$$
P=0
$$

inside the prediction function can therefore create combinations of $X$ and
$P$ that are rare or unsupported in the historical data.

A flexible predictive model can still return a numerical prediction in these
regions.

The existence of a prediction does not imply the existence of credible
counterfactual support.

This is why positivity and empirical overlap must be examined explicitly in
the causal analysis.

---

#### Treatment definition

The predictive experiment uses the full promotion count:

$$
P=\text{onpromotion},
$$

whereas the primary causal analysis begins with the binary treatment:

$$
D
=
\mathbb{1}
[
\text{onpromotion}>0
].
$$

The binary treatment deliberately avoids interpreting the raw count as a
well-defined continuous causal dose.

`onpromotion` measures the number of promoted items but does not contain
discount depth, exact SKU identity, promotion mechanics or campaign exposure.

A continuous causal dose-response analysis would therefore require additional
assumptions and methodology beyond the primary analysis.

---

#### Conclusion

The predictive experiment establishes three separate facts:

1. the promotion-aware model has learned a strong positive association between
   promotion intensity and predicted sales;

2. this learned response is stable across time and increases strongly with
   promotion count;

3. the learned response does not translate into a robust improvement in
   out-of-time forecasting accuracy.

None of these facts identifies the causal promotion effect.

The causal question is instead:

> For comparable store-family-date contexts, how much would realized sales
> change if the promotion state itself were changed?

Answering that question requires explicit treatment definition, confounder
adjustment, overlap diagnostics and causal effect estimation.

The following sections therefore leave the predictive forecasting framework
and begin a separate causal-inference pipeline.

## 5. Causal Problem Formulation

### 5.1 Treatment, Outcome and Potential Outcomes

The forecasting analysis asks whether planned promotion information improves
prediction accuracy.

The causal question is fundamentally different:

> How much does promotion change realized sales relative to what would have
> happened in the same commercial context without promotion?

The causal observation unit is a store-family-date state:

$$
i=(s,f,t).
$$

For the primary causal analysis, treatment is binary:

$$
D_i
=
\mathbb{1}
[
\text{onpromotion}_i>0
].
$$

Therefore:

- $D_i=1$ means that at least one item in the store-family is under promotion;
- $D_i=0$ means that no item in the store-family is recorded as being under
  promotion.

The observed outcome is:

$$
Y_i=\text{sales}_i.
$$

This is realized sales rather than latent unconstrained demand. Inventory
availability and stock-outs are not observed, so the analysis cannot identify
the causal effect on latent consumer demand separately from supply constraints.

---

#### Potential outcomes

For every observation $i$, define two theoretical outcomes:

$$
Y_i(1)
$$

as the sales that would be realized under the promoted state, and:

$$
Y_i(0)
$$

as the sales that would be realized for the same store-family-date state under
no promotion.

The individual causal effect is therefore:

$$
\tau_i
=
Y_i(1)-Y_i(0).
$$

Only one potential outcome can be observed for a given observation.

The realized outcome satisfies:

$$
Y_i
=
D_iY_i(1)
+
(1-D_i)Y_i(0).
$$

If $D_i=1$, $Y_i(1)$ is observed and $Y_i(0)$ is counterfactual.

If $D_i=0$, $Y_i(0)$ is observed and $Y_i(1)$ is counterfactual.

This missing counterfactual is the fundamental causal-inference problem.

A causal estimator therefore cannot simply compare average sales between
promoted and non-promoted rows. It must attempt to construct valid comparisons
between observations that differ in promotion exposure but are otherwise
comparable in factors that jointly influence promotion assignment and sales.

---

#### Treatment-version limitation

The binary treatment deliberately collapses different promotion intensities:

$$
\text{onpromotion}=1,\;5,\;50,\;200
\quad\Rightarrow\quad
D=1.
$$

The dataset also does not contain discount depth, exact promoted SKUs, campaign
mechanics or promotion prices.

Consequently, the binary causal estimand should not be interpreted as the
effect of one precisely defined promotion policy such as a 20% discount.

Instead, it represents the average effect of entering the observed
"any-promotion" state relative to no promotion, averaging over the promotion
versions and intensities represented in the estimation population.

A continuous or multi-valued promotion dose-response analysis is a natural
future extension.

### 5.2 Confounders, Mediators and Colliders

Estimating promotion uplift requires more than predicting sales accurately.

The main causal-design question is:

> Which variables should be conditioned on so that promoted and non-promoted
> observations become comparable?

Not every predictive variable should be included.

---

#### Confounders

A confounder is a pre-treatment variable $X$ that influences both treatment
assignment and the outcome:

$$
X \rightarrow D
$$

and:

$$
X \rightarrow Y.
$$

For example, seasonality may affect both promotion decisions and baseline
demand:

$$
\text{season}
\rightarrow
\text{promotion},
$$

$$
\text{season}
\rightarrow
\text{sales}.
$$

A naive comparison:

$$
E[Y\mid D=1]
-
E[Y\mid D=0]
$$

would then mix the causal promotion effect with pre-existing differences
between promoted and non-promoted commercial contexts.

Relevant pre-treatment adjustment variables in this problem include:

- store identity;
- product family;
- calendar and seasonal context;
- holidays known before the treatment date;
- lagged demand;
- rolling demand-state features;
- potentially lagged transaction activity.

These variables help characterize the commercial context in which promotion
decisions are made.

A variable does not need to materially improve forecasting accuracy to be
important for causal adjustment. Prediction asks whether $X$ helps predict
$Y$, whereas causal adjustment asks whether omitting $X$ creates an invalid
comparison between treated and untreated observations.

---

#### Mediators

A mediator lies on the causal pathway from treatment to outcome:

$$
D
\rightarrow
M
\rightarrow
Y.
$$

Same-day transaction activity is a potential example:

$$
\text{promotion}
\rightarrow
\text{customer transactions}
\rightarrow
\text{sales}.
$$

If promotion attracts additional customers, controlling for same-day
transactions would remove part of the promotion effect that the analysis is
trying to estimate.

The primary estimand in this notebook is the **total promotion effect**.
Therefore, variables that may be caused by the promotion itself should not be
included blindly in the adjustment set.

Lagged transactions may still be useful because they are observed before the
current treatment and can proxy historical store demand.

---

#### Colliders

A collider is a variable caused by two other variables:

$$
D
\rightarrow
C
\leftarrow
U.
$$

Conditioning on a collider can create a statistical association between its
causes even when no such association existed beforehand.

This means that adding more variables to a causal model is not automatically
safer.

Post-treatment operational variables such as realized inventory,
replenishment decisions or other same-day business responses may therefore
require particular care: depending on the underlying causal structure, they
could behave as mediators, colliders, or both.

---

#### Adjustment principle

The primary causal adjustment set will therefore prioritize variables that:

1. are determined before treatment;
2. plausibly influence promotion assignment;
3. plausibly influence realized sales;
4. are not consequences of the current promotion.

The objective is not to maximize predictive accuracy.

It is to block non-causal backdoor paths between promotion and sales without
blocking the causal effect that the analysis aims to estimate.

### 5.3 Identification Assumptions

Potential outcomes define the causal estimand, but observational data identify
that estimand only under additional assumptions.

---

#### Conditional exchangeability

The central observational assumption is:

$$
\{Y_i(1),Y_i(0)\}
\perp
D_i
\mid X_i.
$$

Conditional on the pre-treatment context $X_i$, treatment assignment must not
contain additional information about the potential outcomes.

In this application, $X_i$ should capture relevant pre-treatment commercial
context such as:

- store and family identity;
- calendar and seasonal conditions;
- holidays and events known before treatment;
- lagged sales;
- recent demand state;
- other defensible pre-treatment demand proxies.

This assumption is often described as **no unmeasured confounding**.

It cannot be proven from the observational data.

If an unobserved variable affects both promotion assignment and sales, the
estimated promotion effect may remain biased even after adjustment.

The results of this notebook must therefore be interpreted as causal only under
the stated observational assumptions rather than as equivalent to evidence
from a randomized experiment.

---

#### Consistency and treatment versions

For the observed treatment:

$$
D_i=d,
$$

consistency requires:

$$
Y_i=Y_i(d).
$$

The treatment definition used here is:

$$
D_i
=
\mathbb{1}
[
\text{onpromotion}_i>0
].
$$

However, the promoted state contains multiple treatment versions because
different observations may have very different numbers of promoted items,
discount depths, campaign designs and promoted SKUs.

The binary estimand therefore represents the effect of entering the observed
mix of "any-promotion" states relative to no promotion.

It should not be interpreted as the effect of one uniquely defined discount or
promotion policy.

---

#### Positivity

For every relevant covariate state $X=x$, both treatment states must have
positive probability:

$$
0
<
P(D=1\mid X=x)
<
1.
$$

Without comparable promoted and non-promoted observations in a region of the
covariate space, the corresponding counterfactual cannot be estimated reliably
from the observed data.

Positivity and empirical overlap are examined explicitly in the next section.

---

#### Interference

Standard potential-outcome identification also assumes that one observation's
outcome is not materially changed by the treatment assigned to another
observation.

This assumption may be imperfect in retail.

Promotions can generate:

- substitution across product families;
- cannibalization;
- basket-level spillovers;
- intertemporal purchase displacement.

The primary analysis therefore targets the direct store-family-date promotion
effect and does not explicitly model cross-family or intertemporal spillovers.

This limitation must be retained when interpreting the results.

---

#### Temporal ordering

Adjustment variables should be determined before treatment assignment and must
not themselves be consequences of the promotion decision.

The dataset records realized daily promotion status but does not contain the
exact historical decision time at which each promotion was planned.

Therefore, variables close to the treatment date require particular caution:
being observed before realized sales does not necessarily imply that they were
determined before the promotion decision.

The primary adjustment set will prioritize variables with a defensible
pre-treatment interpretation.

### 5.4 Positivity and Overlap

Conditional exchangeability is not sufficient for causal identification.

Even if all relevant confounders were observed, the data must contain
meaningful treatment variation within comparable pre-treatment contexts.

Define the propensity score:

$$
e(X)
=
P(D=1\mid X).
$$

It represents the probability that a store-family-date observation receives
the promotion treatment given its pre-treatment commercial context.

It does **not** represent the probability that a promotion will be successful.

---

#### Theoretical positivity

For every relevant covariate state $X=x$ in the target population, both
treatment states must be possible:

$$
0
<
P(D=1\mid X=x)
<
1.
$$

Equivalently:

$$
0<e(X)<1.
$$

If:

$$
e(X)=1,
$$

all observations with that commercial context are promoted.

The data then contain no untreated observation from which the corresponding
no-promotion counterfactual can be learned.

Similarly, if:

$$
e(X)=0,
$$

the promoted counterfactual is unsupported by observed data.

Causal identification therefore requires more than variation in promotion
status at the aggregate dataset level.

It requires variation **within sufficiently comparable commercial contexts**.

---

#### Practical overlap

Exact positivity is a theoretical condition.

In finite observational data, propensity scores very close to zero or one are
also problematic.

For example:

$$
e(X)=0.50
$$

means that both promoted and non-promoted observations are common in that
context.

This provides strong empirical support for comparing the two treatment states.

In contrast:

$$
e(X)=0.99
$$

means that almost every comparable observation is promoted.

Although the theoretical probability of observing an untreated state is not
zero, the no-promotion counterfactual is supported by very few observations.

The resulting causal estimate may therefore depend heavily on extrapolation
and become statistically unstable.

This distinction motivates the empirical concept of **overlap** or
**common support**.

---

#### What the promotion audit already tells us

The descriptive promotion audit in Section 2 provides encouraging but
insufficient evidence about overlap.

Across the complete historical dataset:

- approximately 20.37% of store-family-date observations have
  `onpromotion > 0`;
- approximately 79.63% have no promoted item;
- 1,600 of the 1,782 store-family series, or approximately 89.79%, experience
  both promoted and non-promoted states at some point in their history;
- 182 series are never observed under promotion;
- no series is permanently promoted.

Therefore, the dataset contains substantial unconditional treatment variation.

However, the promotion process changes materially over time.

The first positive `onpromotion` observations appear on 2014-04-01 and
promotion coverage subsequently expands progressively across stores and
families.

The annual share of promoted observations rises from approximately:

$$
10.02\%\quad\text{in 2014}
$$

to:

$$
20.86\%\quad\text{in 2015},
$$

$$
35.05\%\quad\text{in 2016},
$$

and:

$$
45.26\%\quad\text{in 2017}.
$$

The early years therefore represent a different and potentially ambiguous
promotion-tracking regime.

For the primary causal analysis, the estimation population is consequently
restricted to the recent two-year operating regime:

$$
2015\text{-}08\text{-}16
\quad\text{through}\quad
2017\text{-}08\text{-}15.
$$

This restriction is motivated by causal comparability and data-regime
stability, independently of the two-year training window selected earlier for
forecasting.

---

#### Why unconditional treatment variation is not enough

The fact that a store-family series experiences both treatment states does not
establish conditional positivity.

For example, consider a hypothetical family that is:

- never promoted on ordinary days;
- always promoted during major commercial events.

The series contains both:

$$
D=0
$$

and:

$$
D=1,
$$

but conditional on the event context:

$$
P(D=1\mid X=\text{major event})=1.
$$

There is therefore no untreated counterfactual support for major-event
observations.

Similarly, treatment variation across different stores or families cannot
replace treatment variation within sufficiently comparable contexts.

---

#### Empirical implication

Section 2 establishes that the dataset contains broad promotion variation.

It does **not** establish causal overlap.

The empirical overlap question is:

> After accounting for store, family, calendar context, seasonality and
> pre-treatment demand state, do both promoted and non-promoted observations
> still occur with meaningful frequency?

This question requires an estimated propensity model:

$$
\hat e(X)
=
\widehat{P}(D=1\mid X),
$$

which is introduced in Section 6.

The resulting propensity-score distributions will be used to identify regions
of strong common support and observations for which causal inference would
require excessive extrapolation.

## 6. Propensity Modeling and Diagnostics

The causal analysis now moves from identification assumptions to empirical
treatment-assignment modeling.

The primary binary treatment is:

$$
D_i
=
\mathbb{1}
[
\text{onpromotion}_i>0
].
$$

For each store-family-date observation, define the propensity score:

$$
e(X_i)
=
P(D_i=1\mid X_i).
$$

The propensity score describes how likely Favorita was to assign promotion
given the observable pre-treatment commercial context.

It does **not** estimate whether promotion works and does not predict sales.

A high propensity score means that observations with similar pre-treatment
characteristics are usually promoted.

A low propensity score means that comparable observations are usually not
promoted.

The propensity model therefore serves two causal purposes:

1. diagnose whether promoted and non-promoted observations coexist in
   comparable parts of the covariate space;
2. provide treatment-assignment probabilities required later by the
   doubly robust estimator.

The objective is not to maximize classification accuracy.

A treatment process that is extremely easy to predict may instead indicate
weak overlap and limited counterfactual support.

### 6.1 Propensity Score Model

#### Causal estimation population

The causal unit differs from the forecasting training unit.

Forecasting used long-format:

$$
(store,\ family,\ forecast\ origin,\ horizon)
$$

observations.

The causal analysis instead uses one observation per:

$$
(store,\ family,\ date).
$$

The primary estimation population is restricted to:

$$
2015\text{-}08\text{-}16
\quad\text{through}\quad
2017\text{-}08\text{-}15.
$$

This recent two-year period was selected earlier because it represents the
more mature promotion-tracking regime while remaining large enough to support
treatment/control comparisons.

Pre-activity store states are excluded using the same point-in-time store
eligibility rule used by the forecasting pipeline.

---

#### Pre-treatment covariate design

The propensity covariates $X$ must describe commercial context without using
information caused by the current treatment.

The exact historical promotion decision time is unavailable.

To reduce the risk of conditioning on variables observed after promotion
planning has already occurred, the primary causal specification uses a
conservative 14-day gap for dynamic historical features.

Historical demand and promotion summaries therefore use information dated no
later than:

$$
t-14.
$$

This 14-day gap is an operational design assumption rather than a known
historical promotion-decision timestamp and remains an explicit limitation.

The primary covariate set contains:

- store and product-family identity;
- target-date calendar context;
- known holiday/event context;
- historical sales levels and seasonality;
- historical demand-state summaries ending at least 14 days before treatment;
- prior promotion history ending at least 14 days before treatment.

The following variables are explicitly excluded from $X$:

- current `onpromotion`, because it defines treatment;
- current sales, because it is the outcome;
- same-day transactions;
- future sales or future operational information.

The propensity model is therefore designed to approximate:

$$
e(X)
=
P(D=1\mid X)
$$

using only defensible pre-treatment information.

In [11]:
# ---------------------------------------------------------------------
# Build the daily causal estimation population.
#
# Causal unit:
#
#   store × family × date
#
# Primary treatment:
#
#   D = 1[onpromotion > 0]
#
# Outcome:
#
#   realized sales
#
# Dynamic adjustment features use a conservative 14-day gap so that
# they are less likely to be consequences of the current promotion
# planning decision.
#
# IMPORTANT:
# The dataset does not expose the actual historical promotion-decision
# timestamp. The 14-day gap is therefore an explicit design assumption,
# not a proven property of every historical campaign.
# ---------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


CAUSAL_START_DATE = pd.Timestamp(
    "2015-08-16"
)

CAUSAL_END_DATE = pd.Timestamp(
    "2017-08-15"
)

CAUSAL_HISTORY_GAP_DAYS = 14


# =====================================================================
# 1. BASE DAILY SALES / PROMOTION TABLE
# =====================================================================

causal_source = (
    promotion_sales[
        [
            "date",
            "store_nbr",
            "family",
            "sales",
            "onpromotion",
        ]
    ]
    .copy()
)


causal_source[
    "date"
] = pd.to_datetime(
    causal_source[
        "date"
    ]
)


causal_source = (
    causal_source
    .sort_values(
        [
            "store_nbr",
            "family",
            "date",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    causal_source
    .duplicated(
        [
            "store_nbr",
            "family",
            "date",
        ]
    )
    .sum()
    == 0
)


# =====================================================================
# 2. REGULARIZE THE CALENDAR
# =====================================================================
#
# The raw competition table omits a small number of global calendar
# dates, including Christmas dates.
#
# We create a complete calendar before computing lags so that a 14-day
# lag always means fourteen calendar days rather than fourteen observed
# rows.
# =====================================================================

all_stores = np.sort(
    causal_source[
        "store_nbr"
    ]
    .unique()
)


all_families = np.sort(
    causal_source[
        "family"
    ]
    .unique()
)


full_calendar = pd.date_range(
    causal_source[
        "date"
    ]
    .min(),
    causal_source[
        "date"
    ]
    .max(),
    freq="D",
)


complete_index = pd.MultiIndex.from_product(
    [
        all_stores,
        all_families,
        full_calendar,
    ],
    names=[
        "store_nbr",
        "family",
        "date",
    ],
)


causal_regularized = (
    causal_source
    .set_index(
        [
            "store_nbr",
            "family",
            "date",
        ]
    )
    .reindex(
        complete_index
    )
    .reset_index()
)


causal_regularized[
    "sales_observed"
] = (
    causal_regularized[
        "sales"
    ]
    .notna()
)


causal_regularized[
    "promotion_observed"
] = (
    causal_regularized[
        "onpromotion"
    ]
    .notna()
)


# =====================================================================
# 3. CURRENT TREATMENT AND OUTCOME
# =====================================================================

causal_regularized[
    "treatment"
] = np.where(
    causal_regularized[
        "onpromotion"
    ]
    .notna(),
    causal_regularized[
        "onpromotion"
    ]
    .gt(
        0
    )
    .astype(
        "int8"
    ),
    np.nan,
)


causal_regularized[
    "target_sales"
] = (
    causal_regularized[
        "sales"
    ]
    .astype(
        "float32"
    )
)


# =====================================================================
# 4. TARGET-DATE CALENDAR FEATURES
# =====================================================================
#
# Calendar state is known independently of the promotion decision.
# =====================================================================

causal_regularized[
    "day_of_week"
] = (
    causal_regularized[
        "date"
    ]
    .dt.dayofweek
    .astype(
        "int8"
    )
)


causal_regularized[
    "month"
] = (
    causal_regularized[
        "date"
    ]
    .dt.month
    .astype(
        "int8"
    )
)


causal_regularized[
    "is_weekend"
] = (
    causal_regularized[
        "day_of_week"
    ]
    .isin(
        [
            5,
            6,
        ]
    )
    .astype(
        "int8"
    )
)


causal_regularized[
    "calendar_year"
] = (
    causal_regularized[
        "date"
    ]
    .dt.year
    .astype(
        "int16"
    )
)


causal_regularized[
    "time_index_days"
] = (
    causal_regularized[
        "date"
    ]
    - CAUSAL_START_DATE
).dt.days.astype(
    "int16"
)


# =====================================================================
# 5. STORE-SPECIFIC HOLIDAY / EVENT CONTEXT
# =====================================================================

stores_path = (
    project_root
    / "data"
    / "raw"
    / "favorita"
    / "stores"
    / "stores.csv"
)


holidays_path = (
    project_root
    / "data"
    / "raw"
    / "favorita"
    / "holidays_events"
    / "holidays_events.csv"
)


stores_metadata = pd.read_csv(
    stores_path
)


holiday_events = pd.read_csv(
    holidays_path,
    parse_dates=[
        "date"
    ],
)


# The original date of a transferred holiday is not treated as the
# realized holiday. Rows explicitly representing the transferred date
# remain active.
active_holiday_events = (
    holiday_events.loc[
        ~holiday_events[
            "transferred"
        ]
        .fillna(
            False
        )
    ]
    .copy()
)


store_calendar = pd.MultiIndex.from_product(
    [
        all_stores,
        full_calendar,
    ],
    names=[
        "store_nbr",
        "date",
    ],
).to_frame(
    index=False
)


store_calendar = (
    store_calendar
    .merge(
        stores_metadata[
            [
                "store_nbr",
                "city",
                "state",
            ]
        ],
        on="store_nbr",
        how="left",
        validate="many_to_one",
    )
)


store_calendar[
    "is_special_event"
] = np.int8(
    0
)


store_calendar[
    "is_work_day_override"
] = np.int8(
    0
)


def apply_calendar_event_flag(
    calendar_frame,
    events_frame,
    locale,
    location_column,
    output_column,
    event_types,
):
    """
    Flag store-date rows affected by selected calendar-event types.
    """

    subset = (
        events_frame.loc[
            (
                events_frame[
                    "locale"
                ]
                .eq(
                    locale
                )
            )
            &
            (
                events_frame[
                    "type"
                ]
                .isin(
                    event_types
                )
            )
        ]
        .copy()
    )


    if (
        locale
        == "National"
    ):

        affected_dates = (
            subset[
                [
                    "date"
                ]
            ]
            .drop_duplicates()
            .assign(
                _event_flag=
                    np.int8(
                        1
                    )
            )
        )


        calendar_frame = (
            calendar_frame
            .merge(
                affected_dates,
                on="date",
                how="left",
                validate="many_to_one",
            )
        )


    else:

        affected_dates = (
            subset[
                [
                    "date",
                    "locale_name",
                ]
            ]
            .drop_duplicates()
            .rename(
                columns={
                    "locale_name":
                        location_column,
                }
            )
            .assign(
                _event_flag=
                    np.int8(
                        1
                    )
            )
        )


        calendar_frame = (
            calendar_frame
            .merge(
                affected_dates,
                on=[
                    "date",
                    location_column,
                ],
                how="left",
                validate="many_to_one",
            )
        )


    calendar_frame[
        output_column
    ] = np.maximum(
        calendar_frame[
            output_column
        ],
        calendar_frame[
            "_event_flag"
        ]
        .fillna(
            0
        )
        .astype(
            "int8"
        ),
    ).astype(
        "int8"
    )


    calendar_frame = (
        calendar_frame
        .drop(
            columns=[
                "_event_flag"
            ]
        )
    )


    return calendar_frame


SPECIAL_EVENT_TYPES = [
    "Holiday",
    "Event",
    "Additional",
    "Transfer",
    "Bridge",
]


for (
    locale,
    location_column,
) in [
    (
        "National",
        None,
    ),
    (
        "Regional",
        "state",
    ),
    (
        "Local",
        "city",
    ),
]:

    store_calendar = (
        apply_calendar_event_flag(
            store_calendar,
            active_holiday_events,
            locale=
                locale,
            location_column=
                location_column,
            output_column=
                "is_special_event",
            event_types=
                SPECIAL_EVENT_TYPES,
        )
    )


for (
    locale,
    location_column,
) in [
    (
        "National",
        None,
    ),
    (
        "Regional",
        "state",
    ),
    (
        "Local",
        "city",
    ),
]:

    store_calendar = (
        apply_calendar_event_flag(
            store_calendar,
            active_holiday_events,
            locale=
                locale,
            location_column=
                location_column,
            output_column=
                "is_work_day_override",
            event_types=[
                "Work Day"
            ],
        )
    )


causal_regularized = (
    causal_regularized
    .merge(
        store_calendar[
            [
                "store_nbr",
                "date",
                "is_special_event",
                "is_work_day_override",
            ]
        ],
        on=[
            "store_nbr",
            "date",
        ],
        how="left",
        validate="many_to_one",
    )
)


# =====================================================================
# 6. HISTORICAL DEMAND FEATURES
# =====================================================================
#
# All dynamic features below are derived from observations at least
# fourteen calendar days before the current treatment date.
# =====================================================================

series_keys = [
    "store_nbr",
    "family",
]


series_group = (
    causal_regularized
    .groupby(
        series_keys,
        observed=True,
        sort=False,
    )
)


for lag_days in [
    14,
    21,
    28,
    35,
    364,
    728,
]:

    causal_regularized[
        f"sales_lag_{lag_days}d"
    ] = (
        series_group[
            "sales"
        ]
        .shift(
            lag_days
        )
        .astype(
            "float32"
        )
    )


# Create the history state as of t-14 and summarize the preceding
# 28 calendar days.
causal_regularized[
    "_sales_pre14"
] = (
    series_group[
        "sales"
    ]
    .shift(
        CAUSAL_HISTORY_GAP_DAYS
    )
)


causal_regularized[
    "_positive_sales_pre14"
] = (
    causal_regularized[
        "_sales_pre14"
    ]
    .gt(
        0
    )
    .where(
        causal_regularized[
            "_sales_pre14"
        ]
        .notna()
    )
    .astype(
        "float32"
    )
)


pre14_group = (
    causal_regularized
    .groupby(
        series_keys,
        observed=True,
        sort=False,
    )
)


causal_regularized[
    "sales_mean_28d_pre14"
] = (
    pre14_group[
        "_sales_pre14"
    ]
    .transform(
        lambda values:
            values
            .rolling(
                window=28,
                min_periods=14,
            )
            .mean()
    )
    .astype(
        "float32"
    )
)


causal_regularized[
    "sales_std_28d_pre14"
] = (
    pre14_group[
        "_sales_pre14"
    ]
    .transform(
        lambda values:
            values
            .rolling(
                window=28,
                min_periods=14,
            )
            .std()
    )
    .astype(
        "float32"
    )
)


causal_regularized[
    "positive_rate_28d_pre14"
] = (
    pre14_group[
        "_positive_sales_pre14"
    ]
    .transform(
        lambda values:
            values
            .rolling(
                window=28,
                min_periods=14,
            )
            .mean()
    )
    .astype(
        "float32"
    )
)


# Historical annual level helps distinguish structurally high- and
# low-demand seasonal contexts without using current sales.
causal_regularized[
    "historical_seasonal_level"
] = (
    causal_regularized[
        [
            "sales_lag_364d",
            "sales_lag_728d",
        ]
    ]
    .mean(
        axis=1,
        skipna=True,
    )
    .astype(
        "float32"
    )
)


# =====================================================================
# 7. HISTORICAL PROMOTION FEATURES
# =====================================================================
#
# Current onpromotion is the treatment and is NEVER included in X.
#
# Historical promotion state can help describe campaign persistence and
# the retailer's previous treatment-assignment behavior.
# =====================================================================

causal_regularized[
    "_promo_flag_history"
] = (
    causal_regularized[
        "onpromotion"
    ]
    .gt(
        0
    )
    .where(
        causal_regularized[
            "onpromotion"
        ]
        .notna()
    )
    .astype(
        "float32"
    )
)


promotion_history_group = (
    causal_regularized
    .groupby(
        series_keys,
        observed=True,
        sort=False,
    )
)


causal_regularized[
    "promo_flag_lag_14d"
] = (
    promotion_history_group[
        "_promo_flag_history"
    ]
    .shift(
        14
    )
    .astype(
        "float32"
    )
)


causal_regularized[
    "promo_flag_lag_28d"
] = (
    promotion_history_group[
        "_promo_flag_history"
    ]
    .shift(
        28
    )
    .astype(
        "float32"
    )
)


causal_regularized[
    "_promo_flag_pre14"
] = (
    promotion_history_group[
        "_promo_flag_history"
    ]
    .shift(
        CAUSAL_HISTORY_GAP_DAYS
    )
)


causal_regularized[
    "_promo_count_pre14"
] = (
    promotion_history_group[
        "onpromotion"
    ]
    .shift(
        CAUSAL_HISTORY_GAP_DAYS
    )
)


promotion_pre14_group = (
    causal_regularized
    .groupby(
        series_keys,
        observed=True,
        sort=False,
    )
)


causal_regularized[
    "promo_rate_28d_pre14"
] = (
    promotion_pre14_group[
        "_promo_flag_pre14"
    ]
    .transform(
        lambda values:
            values
            .rolling(
                window=28,
                min_periods=14,
            )
            .mean()
    )
    .astype(
        "float32"
    )
)


causal_regularized[
    "promo_count_mean_28d_pre14"
] = (
    promotion_pre14_group[
        "_promo_count_pre14"
    ]
    .transform(
        lambda values:
            values
            .rolling(
                window=28,
                min_periods=14,
            )
            .mean()
    )
    .astype(
        "float32"
    )
)


# =====================================================================
# 8. POINT-IN-TIME STORE ACTIVITY
# =====================================================================

store_daily_sales = (
    causal_regularized
    .groupby(
        [
            "store_nbr",
            "date",
        ],
        observed=True,
        sort=False,
    )[
        "sales"
    ]
    .sum(
        min_count=1
    )
    .rename(
        "store_total_sales"
    )
    .reset_index()
)


store_daily_sales[
    "store_positive_day"
] = (
    store_daily_sales[
        "store_total_sales"
    ]
    .fillna(
        0.0
    )
    .gt(
        0.0
    )
)


store_daily_sales[
    "store_started"
] = (
    store_daily_sales
    .groupby(
        "store_nbr",
        observed=True,
        sort=False,
    )[
        "store_positive_day"
    ]
    .cummax()
)


causal_regularized = (
    causal_regularized
    .merge(
        store_daily_sales[
            [
                "store_nbr",
                "date",
                "store_started",
            ]
        ],
        on=[
            "store_nbr",
            "date",
        ],
        how="left",
        validate="many_to_one",
    )
)


# =====================================================================
# 9. PRIMARY CAUSAL ESTIMATION POPULATION
# =====================================================================

causal_date_mask = (
    causal_regularized[
        "date"
    ]
    .between(
        CAUSAL_START_DATE,
        CAUSAL_END_DATE,
    )
)


observed_causal_mask = (
    causal_regularized[
        "sales_observed"
    ]
    &
    causal_regularized[
        "promotion_observed"
    ]
)


pre_activity_mask = (
    causal_date_mask
    &
    observed_causal_mask
    &
    ~causal_regularized[
        "store_started"
    ]
)


excluded_pre_activity_rows = int(
    pre_activity_mask.sum()
)


causal_population = (
    causal_regularized.loc[
        causal_date_mask
        &
        observed_causal_mask
        &
        causal_regularized[
            "store_started"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


causal_population[
    "treatment"
] = (
    causal_population[
        "treatment"
    ]
    .astype(
        "int8"
    )
)


# =====================================================================
# 10. FROZEN PRIMARY PROPENSITY FEATURE CONTRACT
# =====================================================================

CAUSAL_CATEGORICAL_FEATURES = [
    "store_nbr",
    "family",
    "day_of_week",
    "month",
]


CAUSAL_NUMERIC_FEATURES = [
    "is_weekend",
    "calendar_year",
    "time_index_days",
    "is_special_event",
    "is_work_day_override",

    "sales_lag_14d",
    "sales_lag_21d",
    "sales_lag_28d",
    "sales_lag_35d",
    "sales_lag_364d",
    "sales_lag_728d",

    "sales_mean_28d_pre14",
    "sales_std_28d_pre14",
    "positive_rate_28d_pre14",
    "historical_seasonal_level",

    "promo_flag_lag_14d",
    "promo_flag_lag_28d",
    "promo_rate_28d_pre14",
    "promo_count_mean_28d_pre14",
]


CAUSAL_PROPENSITY_FEATURES = (
    CAUSAL_CATEGORICAL_FEATURES
    + CAUSAL_NUMERIC_FEATURES
)


# Current treatment and outcome must never leak into X.
assert (
    "onpromotion"
    not in CAUSAL_PROPENSITY_FEATURES
)

assert (
    "treatment"
    not in CAUSAL_PROPENSITY_FEATURES
)

assert (
    "sales"
    not in CAUSAL_PROPENSITY_FEATURES
)

assert (
    "target_sales"
    not in CAUSAL_PROPENSITY_FEATURES
)


# =====================================================================
# 11. AUDIT
# =====================================================================

causal_population_audit = pd.DataFrame(
    [
        {
            "rows":
                len(
                    causal_population
                ),

            "date_min":
                causal_population[
                    "date"
                ]
                .min(),

            "date_max":
                causal_population[
                    "date"
                ]
                .max(),

            "unique_dates":
                causal_population[
                    "date"
                ]
                .nunique(),

            "stores":
                causal_population[
                    "store_nbr"
                ]
                .nunique(),

            "families":
                causal_population[
                    "family"
                ]
                .nunique(),

            "treatment_rate":
                causal_population[
                    "treatment"
                ]
                .mean(),

            "treated_rows":
                causal_population[
                    "treatment"
                ]
                .sum(),

            "control_rows":
                (
                    causal_population[
                        "treatment"
                    ]
                    .eq(
                        0
                    )
                    .sum()
                ),

            "excluded_pre_activity_rows":
                excluded_pre_activity_rows,
        }
    ]
)


display(
    causal_population_audit
    .style.hide(
        axis="index"
    )
    .format(
        {
            "treatment_rate":
                "{:.2%}",
        }
    )
)


causal_feature_missingness = (
    causal_population[
        CAUSAL_PROPENSITY_FEATURES
    ]
    .isna()
    .mean()
    .mul(
        100
    )
    .rename(
        "missing_pct"
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "feature",
        }
    )
    .sort_values(
        "missing_pct",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


display(
    causal_feature_missingness
    .style.hide(
        axis="index"
    )
    .format(
        {
            "missing_pct":
                "{:.2f}%",
        }
    )
)


assert (
    causal_population
    .duplicated(
        [
            "store_nbr",
            "family",
            "date",
        ]
    )
    .sum()
    == 0
)


assert (
    causal_population[
        "treatment"
    ]
    .isin(
        [
            0,
            1,
        ]
    )
    .all()
)


print(
    "Causal propensity feature count:",
    len(
        CAUSAL_PROPENSITY_FEATURES
    ),
)

print()
print(
    "CAUSAL ESTIMATION POPULATION: PASSED"
)

rows,date_min,date_max,unique_dates,stores,families,treatment_rate,treated_rows,control_rows,excluded_pre_activity_rows
1276968,2015-08-16 00:00:00,2017-08-15 00:00:00,729,54,33,36.56%,466815,810153,22110


feature,missing_pct
sales_lag_28d,0.27%
sales_lag_364d,0.27%
promo_flag_lag_28d,0.27%
promo_flag_lag_14d,0.27%
sales_lag_14d,0.27%
sales_lag_21d,0.27%
sales_lag_728d,0.27%
sales_lag_35d,0.27%
promo_rate_28d_pre14,0.00%
historical_seasonal_level,0.00%


Causal propensity feature count: 23

CAUSAL ESTIMATION POPULATION: PASSED


In [12]:
# ---------------------------------------------------------------------
# Audit the small amount of missingness in lagged causal features.
#
# We do not impute these values automatically. XGBoost can handle NaN
# natively, and the missingness may reflect genuinely unobserved global
# calendar dates rather than missing values within an otherwise observed
# commercial state.
# ---------------------------------------------------------------------

lag_features_with_missingness = [
    feature
    for feature in CAUSAL_PROPENSITY_FEATURES
    if (
        causal_population[
            feature
        ]
        .isna()
        .any()
    )
]


missing_lag_dates = (
    causal_population.loc[
        causal_population[
            lag_features_with_missingness
        ]
        .isna()
        .any(
            axis=1
        ),
        [
            "date",
        ]
        + lag_features_with_missingness,
    ]
)


missingness_by_target_date = (
    missing_lag_dates
    .groupby(
        "date",
        observed=True,
    )
    .size()
    .rename(
        "rows_with_any_missing_lag"
    )
    .reset_index()
    .sort_values(
        "date"
    )
    .reset_index(
        drop=True
    )
)


display(
    missingness_by_target_date
    .style.hide(
        axis="index"
    )
)


print(
    "Dates affected:",
    missingness_by_target_date[
        "date"
    ]
    .nunique(),
)

print(
    "Rows with at least one missing lag:",
    f"{len(missing_lag_dates):,}",
)

date,rows_with_any_missing_lag
2015-12-23 00:00:00,1749
2015-12-24 00:00:00,1749
2016-01-08 00:00:00,1749
2016-01-15 00:00:00,1749
2016-01-22 00:00:00,1749
2016-01-29 00:00:00,1749
2016-12-22 00:00:00,1749
2016-12-23 00:00:00,1749
2017-01-08 00:00:00,1749
2017-01-15 00:00:00,1749


Dates affected: 12
Rows with at least one missing lag: 20,988


#### Cross-fitted propensity estimation

Propensity scores used for causal estimation should not be obtained from a
flexible model evaluated on the same observations on which it was trained.

A highly adaptive classifier could otherwise overfit treatment assignment and
produce excessively extreme propensity scores.

The primary propensity specification therefore uses **cross-fitting**.

The two-year causal population is divided into five contiguous temporal
blocks.

For each block:

1. the propensity model is trained on the other four blocks;
2. treatment probability is predicted only for the held-out block;
3. the held-out predictions are stored as that observation's propensity score.

Every causal observation therefore receives:

$$
\hat e_i
=
\widehat{P}(D_i=1\mid X_i)
$$

from a model that did not use observation $i$ during fitting.

Unlike forecasting validation, causal cross-fitting is retrospective.
Training a nuisance model on observations occurring after a held-out date does
not constitute target leakage because the model is not being evaluated as a
historical forecasting deployment.

The temporal blocking is retained to preserve the time structure of the
treatment process and avoid randomly mixing near-identical dates across folds.

No class reweighting is used.

The treatment prevalence is approximately 36.6%, and the objective is to
estimate the actual conditional treatment probability rather than optimize
minority-class classification performance.

The retrospective cross-fitting design is used here to estimate and diagnose
the causal nuisance functions over the historical causal population.

The resulting fitted model objects are not reused directly in later
forecasting experiments because some retrospective folds may have been trained
using observations occurring after another held-out date.

However, the **causal architecture itself is reused**.

The later causally structured forecasting experiment reconstructs the same
pipeline:

$$
\hat e(X),
\qquad
\hat m_0(X),
\qquad
\hat m_1(X),
\qquad
\hat\tau(X),
$$

under a strict point-in-time training protocol.

For a forecast origin $T$, every model component used to predict
$T+1,\ldots,T+14$ is fitted exclusively from information available at or
before $T$.

The retrospective causal analysis therefore serves both as the causal-effect
study and as the methodological foundation for the later leakage-safe
causally structured forecasting experiment.

**Causally structured forecasting protocol lock**

The later V4 forecasting experiment reuses the causal architecture developed
in Sections 6–8, but not the retrospectively fitted nuisance models.
To prevent future causal diagnostics from influencing the historical
forecasting backtest, the V4 modeling contract is frozen before inspecting
propensity, outcome-model or treatment-effect results.

V4 uses:
- binary treatment `1[onpromotion > 0]`;
- the frozen 23-feature pre-treatment covariate contract;
- the 14-day historical-information gap;
- XGBoost nuisance learners;
- doubly robust treatment-effect estimation;
- the decomposition

$$
\hat Y
=
\hat m_0(X)
+
D\hat\tau(X).
$$

For every forecast origin $T$, all propensity, outcome and uplift models are refitted exclusively from information available at or before $T$.

Results from the retrospective causal analysis may be used for causal interpretation and diagnostics, but not to change this frozen V4 forecasting specification.

In [13]:
# ---------------------------------------------------------------------
# Cross-fitted XGBoost propensity model.
#
# Goal:
#
#   e(X) = P(D = 1 | X)
#
# Each observation receives a probability from a model that was trained
# without that observation.
#
# IMPORTANT:
# This is a nuisance model for causal adjustment, not a forecasting
# model and not a treatment-effect model.
# ---------------------------------------------------------------------

import gc
import time

import numpy as np
import pandas as pd

from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

from xgboost import XGBClassifier


N_PROPENSITY_FOLDS = 5


# =====================================================================
# 1. FIXED PROPENSITY MODEL SPECIFICATION
# =====================================================================
#
# We intentionally use one reasonable, regularized specification rather
# than tuning extensively against the causal population.
#
# Probability quality and overlap diagnostics matter more here than
# squeezing out maximum classification accuracy.
# =====================================================================

PROPENSITY_MODEL_PARAMS = {
    "objective":
        "binary:logistic",

    "n_estimators":
        800,

    "max_depth":
        6,

    "learning_rate":
        0.05,

    "min_child_weight":
        10,

    "subsample":
        0.80,

    "colsample_bytree":
        0.80,

    "reg_lambda":
        5.0,

    "tree_method":
        "hist",

    "enable_categorical":
        True,

    "eval_metric":
        "logloss",

    "random_state":
        42,

    "n_jobs":
        -1,

    "verbosity":
        0,
}


# =====================================================================
# 2. CONSISTENT CATEGORICAL VOCABULARIES
# =====================================================================
#
# Every cross-fitting fold must use exactly the same category vocabulary.
# This prevents categorical codes from changing between train and holdout
# blocks.
# =====================================================================

CAUSAL_CATEGORY_DTYPES = {
    "store_nbr":
        pd.CategoricalDtype(
            categories=
                sorted(
                    causal_population[
                        "store_nbr"
                    ]
                    .unique()
                    .tolist()
                )
        ),

    "family":
        pd.CategoricalDtype(
            categories=
                sorted(
                    causal_population[
                        "family"
                    ]
                    .unique()
                    .tolist()
                )
        ),

    "day_of_week":
        pd.CategoricalDtype(
            categories=
                list(
                    range(
                        7
                    )
                )
        ),

    "month":
        pd.CategoricalDtype(
            categories=
                list(
                    range(
                        1,
                        13,
                    )
                )
        ),
}


def prepare_propensity_X(
    frame,
):
    """
    Prepare the frozen causal propensity feature matrix.

    Categorical vocabularies are held fixed across cross-fitting folds.
    Numeric NaN values are preserved because XGBoost handles missing
    values natively.
    """

    X = (
        frame[
            CAUSAL_PROPENSITY_FEATURES
        ]
        .copy()
    )


    for (
        feature,
        categorical_dtype,
    ) in CAUSAL_CATEGORY_DTYPES.items():

        X[
            feature
        ] = (
            X[
                feature
            ]
            .astype(
                categorical_dtype
            )
        )


    return X


# =====================================================================
# 3. BUILD FIVE CONTIGUOUS TEMPORAL BLOCKS
# =====================================================================

causal_dates = (
    pd.DatetimeIndex(
        sorted(
            causal_population[
                "date"
            ]
            .unique()
        )
    )
)


date_blocks = np.array_split(
    causal_dates,
    N_PROPENSITY_FOLDS,
)


propensity_fold_lookup = {}


for (
    fold_id,
    block_dates,
) in enumerate(
    date_blocks,
    start=1,
):

    for date in block_dates:

        propensity_fold_lookup[
            pd.Timestamp(
                date
            )
        ] = fold_id


causal_population[
    "propensity_fold"
] = (
    causal_population[
        "date"
    ]
    .map(
        propensity_fold_lookup
    )
    .astype(
        "int8"
    )
)


propensity_fold_audit = (
    causal_population
    .groupby(
        "propensity_fold",
        observed=True,
    )
    .agg(
        date_min=(
            "date",
            "min",
        ),
        date_max=(
            "date",
            "max",
        ),
        unique_dates=(
            "date",
            "nunique",
        ),
        rows=(
            "treatment",
            "size",
        ),
        treated_rows=(
            "treatment",
            "sum",
        ),
        treatment_rate=(
            "treatment",
            "mean",
        ),
    )
    .reset_index()
)


display(
    propensity_fold_audit
    .style.hide(
        axis="index"
    )
    .format(
        {
            "treatment_rate":
                "{:.2%}",
        }
    )
)


assert (
    causal_population[
        "propensity_fold"
    ]
    .notna()
    .all()
)


# =====================================================================
# 4. CROSS-FITTED PROPENSITY PREDICTIONS
# =====================================================================

cross_fitted_propensity = np.full(
    len(
        causal_population
    ),
    np.nan,
    dtype="float64",
)


propensity_fold_metrics = []


for fold_id in range(
    1,
    N_PROPENSITY_FOLDS + 1,
):

    fold_start_time = time.time()


    print(
        "\n"
        + "=" * 80
    )

    print(
        f"Propensity fold "
        f"{fold_id}/"
        f"{N_PROPENSITY_FOLDS}"
    )


    validation_mask = (
        causal_population[
            "propensity_fold"
        ]
        .eq(
            fold_id
        )
        .to_numpy()
    )


    training_mask = (
        ~validation_mask
    )


    training_frame = (
        causal_population.loc[
            training_mask
        ]
    )


    validation_frame = (
        causal_population.loc[
            validation_mask
        ]
    )


    X_train = (
        prepare_propensity_X(
            training_frame
        )
    )


    y_train = (
        training_frame[
            "treatment"
        ]
        .to_numpy(
            dtype="int8"
        )
    )


    X_validation = (
        prepare_propensity_X(
            validation_frame
        )
    )


    y_validation = (
        validation_frame[
            "treatment"
        ]
        .to_numpy(
            dtype="int8"
        )
    )


    propensity_model = XGBClassifier(
        **PROPENSITY_MODEL_PARAMS
    )


    propensity_model.fit(
        X_train,
        y_train,
    )


    validation_propensity = (
        propensity_model
        .predict_proba(
            X_validation
        )[
            :,
            1
        ]
        .astype(
            "float64"
        )
    )


    assert np.isfinite(
        validation_propensity
    ).all()


    assert (
        (
            validation_propensity
            >= 0.0
        )
        &
        (
            validation_propensity
            <= 1.0
        )
    ).all()


    cross_fitted_propensity[
        validation_mask
    ] = validation_propensity


    fold_auc = roc_auc_score(
        y_validation,
        validation_propensity,
    )


    fold_log_loss = log_loss(
        y_validation,
        validation_propensity,
    )


    fold_brier = brier_score_loss(
        y_validation,
        validation_propensity,
    )


    fit_minutes = (
        time.time()
        - fold_start_time
    ) / 60.0


    propensity_fold_metrics.append(
        {
            "propensity_fold":
                fold_id,

            "date_min":
                validation_frame[
                    "date"
                ]
                .min(),

            "date_max":
                validation_frame[
                    "date"
                ]
                .max(),

            "rows":
                len(
                    validation_frame
                ),

            "treatment_rate":
                y_validation.mean(),

            "mean_predicted_propensity":
                validation_propensity.mean(),

            "auc":
                fold_auc,

            "log_loss":
                fold_log_loss,

            "brier_score":
                fold_brier,

            "fit_minutes":
                fit_minutes,
        }
    )


    print(
        "Validation rows:",
        f"{len(validation_frame):,}",
    )

    print(
        "Observed treatment rate:",
        f"{y_validation.mean():.4f}",
    )

    print(
        "Mean predicted propensity:",
        f"{validation_propensity.mean():.4f}",
    )

    print(
        "AUC:",
        f"{fold_auc:.4f}",
    )

    print(
        "Log loss:",
        f"{fold_log_loss:.4f}",
    )

    print(
        "Brier score:",
        f"{fold_brier:.4f}",
    )

    print(
        "Fit minutes:",
        f"{fit_minutes:.2f}",
    )


    del (
        X_train,
        y_train,
        X_validation,
        y_validation,
        propensity_model,
        validation_propensity,
        training_frame,
        validation_frame,
    )

    gc.collect()


assert np.isfinite(
    cross_fitted_propensity
).all()


causal_population[
    "propensity_score"
] = (
    cross_fitted_propensity
    .astype(
        "float32"
    )
)


propensity_fold_metrics = pd.DataFrame(
    propensity_fold_metrics
)


# =====================================================================
# 5. GLOBAL OUT-OF-FOLD PROPENSITY METRICS
# =====================================================================

observed_treatment = (
    causal_population[
        "treatment"
    ]
    .to_numpy(
        dtype="int8"
    )
)


global_propensity = (
    causal_population[
        "propensity_score"
    ]
    .to_numpy(
        dtype="float64"
    )
)


GLOBAL_PROPENSITY_AUC = roc_auc_score(
    observed_treatment,
    global_propensity,
)


GLOBAL_PROPENSITY_LOG_LOSS = log_loss(
    observed_treatment,
    global_propensity,
)


GLOBAL_PROPENSITY_BRIER = brier_score_loss(
    observed_treatment,
    global_propensity,
)


print()
print(
    "=" * 80
)

print(
    "CROSS-FITTED PROPENSITY SUMMARY"
)

print(
    "=" * 80
)


display(
    propensity_fold_metrics
    .style.hide(
        axis="index"
    )
    .format(
        {
            "treatment_rate":
                "{:.2%}",

            "mean_predicted_propensity":
                "{:.2%}",

            "auc":
                "{:.4f}",

            "log_loss":
                "{:.4f}",

            "brier_score":
                "{:.4f}",

            "fit_minutes":
                "{:.2f}",
        }
    )
)


print(
    "Global treatment rate:",
    f"{observed_treatment.mean():.4f}",
)

print(
    "Global mean predicted propensity:",
    f"{global_propensity.mean():.4f}",
)

print(
    "Global out-of-fold AUC:",
    f"{GLOBAL_PROPENSITY_AUC:.4f}",
)

print(
    "Global out-of-fold log loss:",
    f"{GLOBAL_PROPENSITY_LOG_LOSS:.4f}",
)

print(
    "Global out-of-fold Brier score:",
    f"{GLOBAL_PROPENSITY_BRIER:.4f}",
)


print()
print(
    "Propensity min:",
    f"{global_propensity.min():.6f}",
)

print(
    "Propensity p01:",
    f"{np.quantile(global_propensity, 0.01):.6f}",
)

print(
    "Propensity p05:",
    f"{np.quantile(global_propensity, 0.05):.6f}",
)

print(
    "Propensity median:",
    f"{np.median(global_propensity):.6f}",
)

print(
    "Propensity p95:",
    f"{np.quantile(global_propensity, 0.95):.6f}",
)

print(
    "Propensity p99:",
    f"{np.quantile(global_propensity, 0.99):.6f}",
)

print(
    "Propensity max:",
    f"{global_propensity.max():.6f}",
)


print()
print(
    "CROSS-FITTED PROPENSITY MODEL: PASSED"
)

propensity_fold,date_min,date_max,unique_dates,rows,treated_rows,treatment_rate
1,2015-08-16 00:00:00,2016-01-09 00:00:00,146,253407,59036,23.30%
2,2016-01-10 00:00:00,2016-06-03 00:00:00,146,255354,71623,28.05%
3,2016-06-04 00:00:00,2016-10-27 00:00:00,146,255354,100121,39.21%
4,2016-10-28 00:00:00,2017-03-23 00:00:00,146,255354,119631,46.85%
5,2017-03-24 00:00:00,2017-08-15 00:00:00,145,257499,116404,45.21%



Propensity fold 1/5
Validation rows: 253,407
Observed treatment rate: 0.2330
Mean predicted propensity: 0.2605
AUC: 0.9357
Log loss: 0.2803
Brier score: 0.0858
Fit minutes: 0.37

Propensity fold 2/5
Validation rows: 255,354
Observed treatment rate: 0.2805
Mean predicted propensity: 0.3365
AUC: 0.9412
Log loss: 0.2977
Brier score: 0.0918
Fit minutes: 0.37

Propensity fold 3/5
Validation rows: 255,354
Observed treatment rate: 0.3921
Mean predicted propensity: 0.4571
AUC: 0.9474
Log loss: 0.3225
Brier score: 0.1002
Fit minutes: 0.37

Propensity fold 4/5
Validation rows: 255,354
Observed treatment rate: 0.4685
Mean predicted propensity: 0.4236
AUC: 0.9558
Log loss: 0.2856
Brier score: 0.0875
Fit minutes: 0.37

Propensity fold 5/5
Validation rows: 257,499
Observed treatment rate: 0.4521
Mean predicted propensity: 0.4707
AUC: 0.9436
Log loss: 0.3236
Brier score: 0.0959
Fit minutes: 0.38

CROSS-FITTED PROPENSITY SUMMARY


propensity_fold,date_min,date_max,rows,treatment_rate,mean_predicted_propensity,auc,log_loss,brier_score,fit_minutes
1,2015-08-16 00:00:00,2016-01-09 00:00:00,253407,23.30%,26.05%,0.9357,0.2803,0.0858,0.37
2,2016-01-10 00:00:00,2016-06-03 00:00:00,255354,28.05%,33.65%,0.9412,0.2977,0.0918,0.37
3,2016-06-04 00:00:00,2016-10-27 00:00:00,255354,39.21%,45.71%,0.9474,0.3225,0.1002,0.37
4,2016-10-28 00:00:00,2017-03-23 00:00:00,255354,46.85%,42.36%,0.9558,0.2856,0.0875,0.37
5,2017-03-24 00:00:00,2017-08-15 00:00:00,257499,45.21%,47.07%,0.9436,0.3236,0.0959,0.38


Global treatment rate: 0.3656
Global mean predicted propensity: 0.3900
Global out-of-fold AUC: 0.9443
Global out-of-fold log loss: 0.3020
Global out-of-fold Brier score: 0.0923

Propensity min: 0.000001
Propensity p01: 0.000018
Propensity p05: 0.000139
Propensity median: 0.181456
Propensity p95: 0.996856
Propensity p99: 0.999261
Propensity max: 0.999941

CROSS-FITTED PROPENSITY MODEL: PASSED


### 6.2 Overlap

Estimating a propensity score is not sufficient for causal identification.

The causal comparison also requires treated and control observations to coexist
within sufficiently similar regions of the observed covariate space.

The relevant condition is:

$$
0 < e(X) < 1.
$$

In practice, the concern is not merely whether some propensity scores are
numerically close to zero or one.

The dangerous cases are specifically:

$$
D=1
\quad\text{with}\quad
e(X)\approx0,
$$

and:

$$
D=0
\quad\text{with}\quad
e(X)\approx1.
$$

These observations represent treatment assignments that are extremely rare
given their observed covariates.

They also generate extreme inverse-probability weights:

$$
\frac{D}{e(X)}
+
\frac{1-D}{1-e(X)}.
$$

Large weights imply that a small number of unusual observations can dominate
the causal estimate.

The overlap analysis therefore examines:

- propensity distributions separately for treated and control observations;
- the amount of population mass in conventional overlap regions;
- the frequency of treated observations with very low propensity;
- the frequency of controls with very high propensity;
- the resulting inverse-probability weight distribution.

No trimming rule is imposed before inspecting these diagnostics.

#### From Propensity Scores to Inverse-Probability Weighting

The propensity score describes the treatment-assignment probability for each
causal observation:

$$
\hat e_i
=
\widehat P(D_i=1\mid X_i),
$$

where one observation $i$ corresponds to one:

$$
(store,\ family,\ date)
$$

unit.

The propensity score can be used to reweight observations so that treated and
control groups better represent the same target covariate population.

For estimation of the Average Treatment Effect, inverse-probability weighting
(IPW) assigns:

$$
w_i^{ATE}
=
\frac{D_i}{\hat e_i}
+
\frac{1-D_i}{1-\hat e_i}.
$$

Therefore, a treated observation receives weight:

$$
\frac{1}{\hat e_i},
$$

while a control observation receives:

$$
\frac{1}{1-\hat e_i}.
$$

The intuition is that observations receiving a treatment state that is rare
for their covariates must represent a larger number of comparable observations.

For example, if:

$$
\hat e_i=0.20
$$

for a treated observation, its IPW weight is:

$$
\frac{1}{0.20}=5.
$$

Roughly speaking, this observation represents a context in which only one in
five comparable units receives treatment.

This reweighting can reduce confounding by making the weighted covariate
distributions of treated and control observations more similar.

However, propensity scores close to zero for treated observations, or close to
one for control observations, generate very large weights.

These extreme weights can make pure IPW estimates highly unstable.

IPW is therefore introduced here primarily as a diagnostic and as a building
block for the doubly robust AIPW estimator used later in Section 7.

In [14]:
# ---------------------------------------------------------------------
# Propensity-score overlap diagnostics.
#
# These diagnostics use the retrospective cross-fitted propensity scores
# from Section 6.1.
#
# No trimming or clipping decision is made yet.
# The purpose is first to understand the empirical support available for
# treated-versus-control comparisons.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


# =====================================================================
# 1. BASIC PROPENSITY DISTRIBUTIONS BY TREATMENT STATE
# =====================================================================

propensity_quantiles = [
    0.00,
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    1.00,
]


overlap_distribution_rows = []


for treatment_value in [
    0,
    1,
]:

    treatment_frame = (
        causal_population.loc[
            causal_population[
                "treatment"
            ]
            .eq(
                treatment_value
            )
        ]
    )


    scores = (
        treatment_frame[
            "propensity_score"
        ]
        .astype(
            "float64"
        )
    )


    quantile_values = (
        scores.quantile(
            propensity_quantiles
        )
    )


    overlap_distribution_rows.append(
        {
            "treatment":
                treatment_value,

            "rows":
                len(
                    treatment_frame
                ),

            "mean":
                scores.mean(),

            "p01":
                quantile_values.loc[
                    0.01
                ],

            "p05":
                quantile_values.loc[
                    0.05
                ],

            "p10":
                quantile_values.loc[
                    0.10
                ],

            "p25":
                quantile_values.loc[
                    0.25
                ],

            "median":
                quantile_values.loc[
                    0.50
                ],

            "p75":
                quantile_values.loc[
                    0.75
                ],

            "p90":
                quantile_values.loc[
                    0.90
                ],

            "p95":
                quantile_values.loc[
                    0.95
                ],

            "p99":
                quantile_values.loc[
                    0.99
                ],

            "min":
                scores.min(),

            "max":
                scores.max(),
        }
    )


propensity_distribution_by_treatment = (
    pd.DataFrame(
        overlap_distribution_rows
    )
)


print(
    "=" * 80
)

print(
    "PROPENSITY DISTRIBUTION BY TREATMENT STATE"
)

print(
    "=" * 80
)


display(
    propensity_distribution_by_treatment
    .style.hide(
        axis="index"
    )
    .format(
        {
            "mean":
                "{:.6f}",

            "p01":
                "{:.6f}",

            "p05":
                "{:.6f}",

            "p10":
                "{:.6f}",

            "p25":
                "{:.6f}",

            "median":
                "{:.6f}",

            "p75":
                "{:.6f}",

            "p90":
                "{:.6f}",

            "p95":
                "{:.6f}",

            "p99":
                "{:.6f}",

            "min":
                "{:.6f}",

            "max":
                "{:.6f}",
        }
    )
)


# =====================================================================
# 2. PRACTICAL OVERLAP REGIONS
# =====================================================================

propensity_score = (
    causal_population[
        "propensity_score"
    ]
    .astype(
        "float64"
    )
)


treatment = (
    causal_population[
        "treatment"
    ]
    .astype(
        "int8"
    )
)


overlap_region_rows = []


for lower_threshold in [
    0.01,
    0.025,
    0.05,
    0.10,
]:

    upper_threshold = (
        1.0
        - lower_threshold
    )


    inside_overlap = (
        propensity_score.between(
            lower_threshold,
            upper_threshold,
            inclusive="both",
        )
    )


    treated_inside = (
        inside_overlap
        &
        treatment.eq(
            1
        )
    )


    control_inside = (
        inside_overlap
        &
        treatment.eq(
            0
        )
    )


    overlap_region_rows.append(
        {
            "overlap_region":
                (
                    f"[{lower_threshold:.3f}, "
                    f"{upper_threshold:.3f}]"
                ),

            "overall_share":
                inside_overlap.mean(),

            "treated_share_retained":
                (
                    treated_inside.sum()
                    /
                    treatment.eq(
                        1
                    )
                    .sum()
                ),

            "control_share_retained":
                (
                    control_inside.sum()
                    /
                    treatment.eq(
                        0
                    )
                    .sum()
                ),

            "rows_retained":
                inside_overlap.sum(),
        }
    )


overlap_region_summary = (
    pd.DataFrame(
        overlap_region_rows
    )
)


print()
print(
    "=" * 80
)

print(
    "PRACTICAL OVERLAP REGIONS"
)

print(
    "=" * 80
)


display(
    overlap_region_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "overall_share":
                "{:.2%}",

            "treated_share_retained":
                "{:.2%}",

            "control_share_retained":
                "{:.2%}",
        }
    )
)


# =====================================================================
# 3. DIRECTLY DIAGNOSE THE DANGEROUS TAILS
# =====================================================================
#
# Low propensity is particularly dangerous for treated observations.
# High propensity is particularly dangerous for control observations.
# =====================================================================

tail_diagnostic_rows = []


for threshold in [
    0.01,
    0.025,
    0.05,
    0.10,
]:

    treated_low = (
        treatment.eq(
            1
        )
        &
        propensity_score.lt(
            threshold
        )
    )


    control_high = (
        treatment.eq(
            0
        )
        &
        propensity_score.gt(
            1.0
            - threshold
        )
    )


    tail_diagnostic_rows.append(
        {
            "threshold":
                threshold,

            "treated_below_threshold_rows":
                treated_low.sum(),

            "treated_below_threshold_share":
                (
                    treated_low.sum()
                    /
                    treatment.eq(
                        1
                    )
                    .sum()
                ),

            "control_above_complement_rows":
                control_high.sum(),

            "control_above_complement_share":
                (
                    control_high.sum()
                    /
                    treatment.eq(
                        0
                    )
                    .sum()
                ),
        }
    )


extreme_assignment_summary = (
    pd.DataFrame(
        tail_diagnostic_rows
    )
)


print()
print(
    "=" * 80
)

print(
    "EXTREME ASSIGNMENT DIAGNOSTICS"
)

print(
    "=" * 80
)


display(
    extreme_assignment_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "threshold":
                "{:.3f}",

            "treated_below_threshold_share":
                "{:.3%}",

            "control_above_complement_share":
                "{:.3%}",
        }
    )
)


# =====================================================================
# 4. RAW INVERSE-PROBABILITY WEIGHTS
# =====================================================================
#
# For a treated row:
#
#     weight = 1 / e(X)
#
# For a control row:
#
#     weight = 1 / (1 - e(X))
#
# No clipping is applied here because the purpose is to diagnose the
# raw instability implied by the estimated propensity model.
# =====================================================================

causal_population[
    "raw_ipw"
] = np.where(
    treatment.eq(
        1
    ),
    1.0
    / propensity_score,
    1.0
    / (
        1.0
        - propensity_score
    ),
)


raw_ipw = (
    causal_population[
        "raw_ipw"
    ]
    .astype(
        "float64"
    )
)


weight_quantiles = (
    raw_ipw.quantile(
        [
            0.50,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999,
            1.00,
        ]
    )
)


ipw_summary = pd.DataFrame(
    [
        {
            "mean":
                raw_ipw.mean(),

            "median":
                weight_quantiles.loc[
                    0.50
                ],

            "p90":
                weight_quantiles.loc[
                    0.90
                ],

            "p95":
                weight_quantiles.loc[
                    0.95
                ],

            "p99":
                weight_quantiles.loc[
                    0.99
                ],

            "p995":
                weight_quantiles.loc[
                    0.995
                ],

            "p999":
                weight_quantiles.loc[
                    0.999
                ],

            "max":
                weight_quantiles.loc[
                    1.00
                ],
        }
    ]
)


print()
print(
    "=" * 80
)

print(
    "RAW INVERSE-PROBABILITY WEIGHT DISTRIBUTION"
)

print(
    "=" * 80
)


display(
    ipw_summary
    .style.hide(
        axis="index"
    )
    .format(
        "{:.3f}"
    )
)


# =====================================================================
# 5. EFFECTIVE SAMPLE SIZE
# =====================================================================
#
# ESS measures how much information remains after unequal weighting.
#
# If all weights were equal:
#
#     ESS ≈ N
#
# If a small number of rows dominate the weighted estimator:
#
#     ESS << N
# =====================================================================

def effective_sample_size(
    weights,
):
    """
    Kish effective sample size.
    """

    weights = np.asarray(
        weights,
        dtype="float64",
    )


    return (
        weights.sum() ** 2
        /
        np.square(
            weights
        )
        .sum()
    )


treated_weights = (
    causal_population.loc[
        treatment.eq(
            1
        ),
        "raw_ipw",
    ]
)


control_weights = (
    causal_population.loc[
        treatment.eq(
            0
        ),
        "raw_ipw",
    ]
)


ipw_ess_summary = pd.DataFrame(
    [
        {
            "group":
                "treated",

            "raw_rows":
                len(
                    treated_weights
                ),

            "effective_sample_size":
                effective_sample_size(
                    treated_weights
                ),
        },

        {
            "group":
                "control",

            "raw_rows":
                len(
                    control_weights
                ),

            "effective_sample_size":
                effective_sample_size(
                    control_weights
                ),
        },

        {
            "group":
                "overall",

            "raw_rows":
                len(
                    raw_ipw
                ),

            "effective_sample_size":
                effective_sample_size(
                    raw_ipw
                ),
        },
    ]
)


ipw_ess_summary[
    "ess_fraction"
] = (
    ipw_ess_summary[
        "effective_sample_size"
    ]
    /
    ipw_ess_summary[
        "raw_rows"
    ]
)


print()
print(
    "=" * 80
)

print(
    "INVERSE-PROBABILITY EFFECTIVE SAMPLE SIZE"
)

print(
    "=" * 80
)


display(
    ipw_ess_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "effective_sample_size":
                "{:,.0f}",

            "ess_fraction":
                "{:.2%}",
        }
    )
)


print()
print(
    "OVERLAP DIAGNOSTICS: PASSED"
)

PROPENSITY DISTRIBUTION BY TREATMENT STATE


treatment,rows,mean,p01,p05,p10,p25,median,p75,p90,p95,p99,min,max
0,810153,0.148466,0.000012,0.000069,0.000210,0.001698,0.022140,0.175051,0.554221,0.766298,0.967735,0.000001,0.999558
1,466815,0.809156,0.021244,0.155829,0.325915,0.723432,0.955176,0.992488,0.997740,0.998799,0.999582,0.000068,0.999941



PRACTICAL OVERLAP REGIONS


overlap_region,overall_share,treated_share_retained,control_share_retained,rows_retained
"[0.010, 0.990]",62.93%,70.34%,58.65%,803549
"[0.025, 0.975]",51.17%,56.93%,47.84%,653366
"[0.050, 0.950]",42.12%,46.68%,39.49%,537835
"[0.100, 0.900]",32.24%,35.75%,30.22%,411712



EXTREME ASSIGNMENT DIAGNOSTICS


threshold,treated_below_threshold_rows,treated_below_threshold_share,control_above_complement_rows,control_above_complement_share
0.010,2650,0.568%,2943,0.363%
0.025,5259,1.127%,6651,0.821%
0.050,8574,1.837%,11154,1.377%
0.100,15362,3.291%,18695,2.308%



RAW INVERSE-PROBABILITY WEIGHT DISTRIBUTION


mean,median,p90,p95,p99,p995,p999,max
3.505,1.031,2.522,5.031,36.475,86.306,321.888,14722.454



INVERSE-PROBABILITY EFFECTIVE SAMPLE SIZE


group,raw_rows,effective_sample_size,ess_fraction
treated,466815,"1,569",0.34%
control,810153,"15,089",1.86%
overall,1276968,"5,882",0.46%



OVERLAP DIAGNOSTICS: PASSED


#### Overlap conclusion

The retrospective propensity model reveals substantial practical overlap
limitations.

Treatment assignment is strongly separated across the observed covariate
space.

The median estimated propensity is approximately:

$$
0.022
$$

among control observations, compared with approximately:

$$
0.955
$$

among treated observations.

Only approximately 51% of the causal population lies within:

$$
0.025 \leq \hat e(X) \leq 0.975,
$$

and approximately 42% lies within:

$$
0.05 \leq \hat e(X) \leq 0.95.
$$

This indicates that a substantial fraction of the target population belongs
to covariate regions in which one treatment state is rarely observed.

Extreme propensity scores are problematic even when the observed treatment
state is the expected one, because the opposite potential outcome has little
empirical support in that region.

The most severe numerical instability occurs when an observation receives the
rare treatment state for its covariates.

At a 5% threshold:

- 8,574 treated observations have estimated propensity below 0.05;
- 11,154 control observations have estimated propensity above 0.95.

These observations generate very large inverse-probability weights.

The raw ATE weight distribution has a 99.9th percentile of approximately 322
and a maximum weight above 14,700.

As a consequence, the Kish effective sample size collapses from approximately
1.28 million raw observations to fewer than 6,000 effective observations
under raw inverse-probability weighting.

Raw IPW is therefore too unstable to use as the primary causal estimator.

The primary analysis will use doubly robust estimation, but extreme
propensities can still destabilize its residual correction terms.

Before choosing a stabilization or common-support rule, Section 6.3 therefore
examines covariate balance under several pre-specified propensity clipping
levels.

The stabilization choice will be frozen before estimating treatment effects.

### 6.3 Covariate Balance

A propensity model is useful for causal adjustment only if it helps create
treated and control groups that are comparable on observed pre-treatment
covariates.

Balance is evaluated using the standardized mean difference:

$$
SMD
=
\frac{
\bar X_1-\bar X_0
}{
\sqrt{
\frac{s_1^2+s_0^2}{2}
}
}.
$$

Unlike a hypothesis-test p-value, the SMD measures the magnitude of imbalance
and does not mechanically become significant because of the very large sample
size.

As a practical diagnostic:

$$
|SMD|<0.10
$$

is treated as a useful balance benchmark rather than a strict identification
rule.

Because the raw propensity diagnostics reveal extreme weights, balance is
compared under several propensity specifications before any treatment effect
is estimated:

- no weighting;
- raw inverse-probability weighting;
- propensity clipping at 1%;
- propensity clipping at 2.5%;
- propensity clipping at 5%.

The stabilization rule is selected using balance and weight stability rather
than the resulting treatment-effect estimate.

In [15]:
# ---------------------------------------------------------------------
# Covariate-balance diagnostics under alternative ATE weighting rules.
#
# IMPORTANT:
# No treatment-effect estimate is computed in this section.
#
# The purpose is to select a defensible propensity stabilization rule
# using only:
#
#   - covariate balance;
#   - weight concentration;
#   - effective sample size.
#
# This prevents the clipping threshold from being chosen because it
# produces a preferred causal-effect estimate.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


# =====================================================================
# 1. BUILD A BALANCE-DIAGNOSTIC FEATURE MATRIX
# =====================================================================
#
# Categorical variables are expanded into indicators.
#
# For numeric variables with rare missing values, the missingness itself
# is represented by an indicator and the numerical diagnostic copy is
# median-filled.
#
# This does NOT alter the XGBoost propensity model. It is only a
# convenient representation for balance diagnostics.
# =====================================================================

balance_frame = (
    causal_population[
        CAUSAL_PROPENSITY_FEATURES
    ]
    .copy()
)


for feature in CAUSAL_NUMERIC_FEATURES:

    if (
        balance_frame[
            feature
        ]
        .isna()
        .any()
    ):

        balance_frame[
            f"{feature}__missing"
        ] = (
            balance_frame[
                feature
            ]
            .isna()
            .astype(
                "int8"
            )
        )


        balance_frame[
            feature
        ] = (
            balance_frame[
                feature
            ]
            .fillna(
                balance_frame[
                    feature
                ]
                .median()
            )
        )


balance_frame = pd.get_dummies(
    balance_frame,
    columns=
        CAUSAL_CATEGORICAL_FEATURES,
    prefix=
        CAUSAL_CATEGORICAL_FEATURES,
    dtype="float32",
)


balance_frame = (
    balance_frame
    .astype(
        "float64"
    )
)


balance_feature_names = (
    balance_frame
    .columns
    .tolist()
)


print(
    "Balance-diagnostic feature count:",
    len(
        balance_feature_names
    ),
)


# =====================================================================
# 2. WEIGHTING SPECIFICATIONS
# =====================================================================

treatment_array = (
    causal_population[
        "treatment"
    ]
    .to_numpy(
        dtype="int8"
    )
)


propensity_array = (
    causal_population[
        "propensity_score"
    ]
    .to_numpy(
        dtype="float64"
    )
)


def ate_ipw_from_propensity(
    propensity,
    treatment,
):
    """
    Construct unstabilized ATE inverse-probability weights.
    """

    return np.where(
        treatment
        == 1,
        1.0
        / propensity,
        1.0
        / (
            1.0
            - propensity
        ),
    )


weighting_specs = {
    "unweighted":
        np.ones_like(
            propensity_array,
            dtype="float64",
        ),

    "raw_ipw":
        ate_ipw_from_propensity(
            propensity_array,
            treatment_array,
        ),
}


for clip_threshold in [
    0.01,
    0.025,
    0.05,
]:

    clipped_propensity = np.clip(
        propensity_array,
        clip_threshold,
        1.0
        - clip_threshold,
    )


    weighting_specs[
        f"clip_{clip_threshold:g}"
    ] = ate_ipw_from_propensity(
        clipped_propensity,
        treatment_array,
    )


# =====================================================================
# 3. WEIGHTED STANDARDIZED MEAN DIFFERENCE
# =====================================================================

def weighted_mean_and_variance(
    values,
    weights,
):
    """
    Weighted population mean and variance.
    """

    values = np.asarray(
        values,
        dtype="float64",
    )

    weights = np.asarray(
        weights,
        dtype="float64",
    )


    weight_sum = (
        weights.sum()
    )


    mean = (
        np.sum(
            weights
            * values
        )
        / weight_sum
    )


    variance = (
        np.sum(
            weights
            * np.square(
                values
                - mean
            )
        )
        / weight_sum
    )


    return (
        mean,
        variance,
    )


def weighted_smd(
    values,
    treatment,
    weights,
):
    """
    Absolute standardized mean difference between treated and control.
    """

    treated_mask = (
        treatment
        == 1
    )

    control_mask = (
        treatment
        == 0
    )


    (
        treated_mean,
        treated_variance,
    ) = weighted_mean_and_variance(
        values[
            treated_mask
        ],
        weights[
            treated_mask
        ],
    )


    (
        control_mean,
        control_variance,
    ) = weighted_mean_and_variance(
        values[
            control_mask
        ],
        weights[
            control_mask
        ],
    )


    pooled_sd = np.sqrt(
        (
            treated_variance
            + control_variance
        )
        / 2.0
    )


    if (
        pooled_sd
        < 1e-12
    ):

        return 0.0


    return (
        (
            treated_mean
            - control_mean
        )
        / pooled_sd
    )


# =====================================================================
# 4. COMPUTE BALANCE FOR EVERY SPECIFICATION
# =====================================================================

balance_rows = []


for (
    specification,
    weights,
) in weighting_specs.items():

    print(
        "Computing balance:",
        specification,
    )


    for feature in balance_feature_names:

        feature_values = (
            balance_frame[
                feature
            ]
            .to_numpy(
                dtype="float64"
            )
        )


        feature_smd = weighted_smd(
            feature_values,
            treatment_array,
            weights,
        )


        balance_rows.append(
            {
                "specification":
                    specification,

                "feature":
                    feature,

                "smd":
                    feature_smd,

                "abs_smd":
                    abs(
                        feature_smd
                    ),
            }
        )


covariate_balance = pd.DataFrame(
    balance_rows
)


# =====================================================================
# 5. SUMMARIZE BALANCE QUALITY
# =====================================================================

balance_summary = (
    covariate_balance
    .groupby(
        "specification",
        observed=True,
    )
    .agg(
        max_abs_smd=(
            "abs_smd",
            "max",
        ),

        median_abs_smd=(
            "abs_smd",
            "median",
        ),

        mean_abs_smd=(
            "abs_smd",
            "mean",
        ),

        features_above_010=(
            "abs_smd",
            lambda values:
                int(
                    (
                        values
                        > 0.10
                    )
                    .sum()
                ),
        ),

        features_above_020=(
            "abs_smd",
            lambda values:
                int(
                    (
                        values
                        > 0.20
                    )
                    .sum()
                ),
        ),
    )
    .reset_index()
)


balance_summary[
    "share_above_010"
] = (
    balance_summary[
        "features_above_010"
    ]
    /
    len(
        balance_feature_names
    )
)


# =====================================================================
# 6. WEIGHT-STABILITY SUMMARY
# =====================================================================

def effective_sample_size(
    weights,
):
    """
    Kish effective sample size.
    """

    weights = np.asarray(
        weights,
        dtype="float64",
    )


    return (
        weights.sum() ** 2
        /
        np.square(
            weights
        )
        .sum()
    )


weight_stability_rows = []


for (
    specification,
    weights,
) in weighting_specs.items():

    treated_weights = (
        weights[
            treatment_array
            == 1
        ]
    )


    control_weights = (
        weights[
            treatment_array
            == 0
        ]
    )


    weight_stability_rows.append(
        {
            "specification":
                specification,

            "max_weight":
                weights.max(),

            "p99_weight":
                np.quantile(
                    weights,
                    0.99,
                ),

            "p999_weight":
                np.quantile(
                    weights,
                    0.999,
                ),

            "treated_ess":
                effective_sample_size(
                    treated_weights
                ),

            "control_ess":
                effective_sample_size(
                    control_weights
                ),

            "overall_ess":
                effective_sample_size(
                    weights
                ),
        }
    )


weight_stability = pd.DataFrame(
    weight_stability_rows
)


weight_stability[
    "treated_ess_fraction"
] = (
    weight_stability[
        "treated_ess"
    ]
    /
    treatment_array.sum()
)


weight_stability[
    "control_ess_fraction"
] = (
    weight_stability[
        "control_ess"
    ]
    /
    (
        treatment_array
        == 0
    )
    .sum()
)


weight_stability[
    "overall_ess_fraction"
] = (
    weight_stability[
        "overall_ess"
    ]
    /
    len(
        treatment_array
    )
)


# =====================================================================
# 7. DISPLAY
# =====================================================================

print()
print(
    "=" * 80
)

print(
    "COVARIATE BALANCE SUMMARY"
)

print(
    "=" * 80
)


display(
    balance_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "max_abs_smd":
                "{:.3f}",

            "median_abs_smd":
                "{:.3f}",

            "mean_abs_smd":
                "{:.3f}",

            "share_above_010":
                "{:.2%}",
        }
    )
)


print()
print(
    "=" * 80
)

print(
    "WEIGHT STABILITY BY SPECIFICATION"
)

print(
    "=" * 80
)


display(
    weight_stability
    .style.hide(
        axis="index"
    )
    .format(
        {
            "max_weight":
                "{:.2f}",

            "p99_weight":
                "{:.2f}",

            "p999_weight":
                "{:.2f}",

            "treated_ess":
                "{:,.0f}",

            "control_ess":
                "{:,.0f}",

            "overall_ess":
                "{:,.0f}",

            "treated_ess_fraction":
                "{:.2%}",

            "control_ess_fraction":
                "{:.2%}",

            "overall_ess_fraction":
                "{:.2%}",
        }
    )
)


# Show the largest residual imbalances for every specification.
largest_balance_issues = (
    covariate_balance
    .sort_values(
        [
            "specification",
            "abs_smd",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "specification",
        observed=True,
    )
    .head(
        10
    )
    .reset_index(
        drop=True
    )
)


print()
print(
    "=" * 80
)

print(
    "TEN LARGEST ABSOLUTE SMDs PER SPECIFICATION"
)

print(
    "=" * 80
)


display(
    largest_balance_issues
    .style.hide(
        axis="index"
    )
    .format(
        {
            "smd":
                "{:+.3f}",

            "abs_smd":
                "{:.3f}",
        }
    )
)


print()
print(
    "COVARIATE BALANCE DIAGNOSTICS: PASSED"
)

Balance-diagnostic feature count: 133
Computing balance: unweighted
Computing balance: raw_ipw
Computing balance: clip_0.01
Computing balance: clip_0.025
Computing balance: clip_0.05

COVARIATE BALANCE SUMMARY


specification,max_abs_smd,median_abs_smd,mean_abs_smd,features_above_010,features_above_020,share_above_010
clip_0.01,0.316,0.021,0.045,18,5,13.53%
clip_0.025,0.300,0.019,0.052,30,7,22.56%
clip_0.05,0.528,0.018,0.070,32,22,24.06%
raw_ipw,0.848,0.045,0.114,41,27,30.83%
unweighted,2.183,0.033,0.181,41,40,30.83%



WEIGHT STABILITY BY SPECIFICATION


specification,max_weight,p99_weight,p999_weight,treated_ess,control_ess,overall_ess,treated_ess_fraction,control_ess_fraction,overall_ess_fraction
unweighted,1.00,1.00,1.00,"466,815","810,153","1,276,968",100.00%,100.00%,100.00%
raw_ipw,14722.45,36.47,321.89,"1,569","15,089","5,882",0.34%,1.86%,0.46%
clip_0.01,100.00,36.47,100.00,"38,289","64,035","101,683",8.20%,7.90%,7.96%
clip_0.025,40.00,36.47,40.00,"81,280","137,496","217,772",17.41%,16.97%,17.05%
clip_0.05,20.00,20.00,20.00,"138,989","244,677","381,372",29.77%,30.20%,29.87%



TEN LARGEST ABSOLUTE SMDs PER SPECIFICATION


specification,feature,smd,abs_smd
clip_0.01,family_FROZEN FOODS,-0.316,0.316
clip_0.01,family_BREAD/BAKERY,-0.285,0.285
clip_0.01,family_LINGERIE,+0.247,0.247
clip_0.01,family_GROCERY II,+0.212,0.212
clip_0.01,family_BOOKS,-0.209,0.209
clip_0.01,promo_flag_lag_28d,-0.194,0.194
clip_0.01,promo_rate_28d_pre14,-0.191,0.191
clip_0.01,month_5,+0.175,0.175
clip_0.01,family_HOME APPLIANCES,-0.160,0.160
clip_0.01,month_7,-0.156,0.156



COVARIATE BALANCE DIAGNOSTICS: PASSED


#### Balance and stabilization decision

The observational treatment groups are strongly imbalanced before propensity
adjustment.

Several historical promotion and demand-state covariates have absolute
standardized mean differences well above one before weighting, confirming that
promotion assignment is highly structured rather than approximately random.

Raw inverse-probability weighting reduces much of this imbalance but is
numerically unstable.

Its maximum weight exceeds 14,000 and the overall effective sample size falls
below 0.5% of the raw causal population.

Propensity clipping substantially improves this bias-variance trade-off.

Among the pre-specified stabilization rules, clipping estimated propensity
scores to:

$$
[0.01,\ 0.99]
$$

provides the strongest overall covariate balance.

Under this specification:

- median absolute SMD falls to approximately 0.021;
- mean absolute SMD falls to approximately 0.045;
- the number of diagnostic features with absolute SMD above 0.10 falls from
  41 under raw IPW to 18;
- the overall effective sample size increases from approximately 5,900 under
  raw IPW to approximately 102,000;
- the maximum inverse-probability weight is bounded at 100.

More aggressive clipping increases effective sample size further but weakens
covariate balance, particularly for historical promotion-state variables.

The primary retrospective causal analysis therefore freezes:

$$
\boxed{
\hat e^{\,stab}(X)
=
\operatorname{clip}
\left(
\hat e(X),
0.01,
0.99
\right)
}
$$

before observing any treatment-effect estimate.

Residual imbalance remains on several covariates, so propensity weighting alone
is not considered sufficient.

The treatment-effect analysis therefore proceeds with doubly robust outcome
adjustment through AIPW, while alternative clipping levels are retained for
sensitivity analysis.

## 7. Doubly Robust Promotion Effect Estimation

Propensity-score diagnostics describe how treatment was assigned, but they do
not estimate sales outcomes.

The causal analysis now introduces outcome regression.

For each pre-treatment context $X$, define:

$$
m_0(X)
=
E[Y\mid D=0,X],
$$

and:

$$
m_1(X)
=
E[Y\mid D=1,X].
$$

Under conditional exchangeability:

$$
\{Y(1),Y(0)\}
\perp
D
\mid X,
$$

these observational conditional means identify the corresponding conditional
potential-outcome means:

$$
E[Y(0)\mid X]
=
m_0(X),
$$

and:

$$
E[Y(1)\mid X]
=
m_1(X).
$$

The outcome models therefore provide a second route to counterfactual
estimation in addition to propensity weighting.

They will subsequently be combined through the Augmented Inverse Probability
Weighting estimator.

### 7.1 Outcome Models

A separate nonlinear regression model is estimated for each treatment state.

The control outcome model:

$$
\hat m_0(X)
$$

is trained only on observations with:

$$
D=0.
$$

The treated outcome model:

$$
\hat m_1(X)
$$

is trained only on observations with:

$$
D=1.
$$

Both models use the same frozen pre-treatment covariate set as the propensity
model.

For every causal observation $i$, both predictions are generated:

$$
\hat m_{0,i}
=
\hat m_0(X_i),
$$

and:

$$
\hat m_{1,i}
=
\hat m_1(X_i).
$$

Only one potential outcome is factually observed.

If $D_i=1$, the observed outcome can be used to evaluate
$\hat m_1(X_i)$, while $\hat m_0(X_i)$ remains a counterfactual conditional-mean
prediction.

If $D_i=0$, the reverse holds.

These models are cross-fitted using the same five retrospective folds as the
propensity model, ensuring that every nuisance prediction is produced by a
model that did not train on observation $i$.

Squared-error regression is used because the causal nuisance functions target
conditional means rather than conditional medians.

In [16]:
# ---------------------------------------------------------------------
# Cross-fitted outcome regression.
#
# Estimate:
#
#   m0(X) = E[Y | D = 0, X]
#   m1(X) = E[Y | D = 1, X]
#
# Two separate XGBoost regressors are fitted in every cross-fitting fold:
#
#   - one using only control observations;
#   - one using only treated observations.
#
# Both models predict every row of the held-out fold.
#
# Therefore every observation receives two out-of-sample nuisance
# predictions:
#
#   m0_hat_i
#   m1_hat_i
#
# Only the prediction corresponding to the observed treatment state can
# be directly evaluated against the factual outcome.
# ---------------------------------------------------------------------

import gc
import time

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)

from xgboost import XGBRegressor


# =====================================================================
# 1. FIXED OUTCOME-MODEL SPECIFICATION
# =====================================================================
#
# Squared-error regression targets the conditional mean required by
# m_d(X) = E[Y | D=d, X].
#
# This differs deliberately from the absolute-error forecasting
# objective used by the production forecasting model.
# =====================================================================

CAUSAL_OUTCOME_MODEL_PARAMS = {
    "objective":
        "reg:squarederror",

    "n_estimators":
        1000,

    "max_depth":
        8,

    "learning_rate":
        0.05,

    "min_child_weight":
        10,

    "subsample":
        0.80,

    "colsample_bytree":
        0.80,

    "reg_lambda":
        5.0,

    "tree_method":
        "hist",

    "enable_categorical":
        True,

    "random_state":
        42,

    "n_jobs":
        -1,

    "verbosity":
        0,
}


# =====================================================================
# 2. PREPARE OUTCOME FEATURE MATRIX
# =====================================================================

def prepare_outcome_X(
    frame,
):
    """
    Prepare the frozen causal covariate matrix for outcome regression.

    The feature contract is intentionally identical to the propensity
    model contract.

    Current treatment, current promotion count and current sales are
    excluded from X.
    """

    X = (
        frame[
            CAUSAL_PROPENSITY_FEATURES
        ]
        .copy()
    )


    for (
        feature,
        categorical_dtype,
    ) in CAUSAL_CATEGORY_DTYPES.items():

        X[
            feature
        ] = (
            X[
                feature
            ]
            .astype(
                categorical_dtype
            )
        )


    return X


CAUSAL_OUTCOME_FEATURES = (
    CAUSAL_PROPENSITY_FEATURES.copy()
)


assert (
    "treatment"
    not in CAUSAL_OUTCOME_FEATURES
)

assert (
    "onpromotion"
    not in CAUSAL_OUTCOME_FEATURES
)

assert (
    "target_sales"
    not in CAUSAL_OUTCOME_FEATURES
)


# =====================================================================
# 3. STORAGE FOR CROSS-FITTED NUISANCE PREDICTIONS
# =====================================================================

n_causal_rows = len(
    causal_population
)


cross_fitted_m0 = np.full(
    n_causal_rows,
    np.nan,
    dtype="float64",
)


cross_fitted_m1 = np.full(
    n_causal_rows,
    np.nan,
    dtype="float64",
)


outcome_fold_metrics = []


# =====================================================================
# 4. CROSS-FITTING
# =====================================================================

for fold_id in range(
    1,
    N_PROPENSITY_FOLDS + 1,
):

    fold_start_time = time.time()


    print(
        "\n"
        + "=" * 80
    )

    print(
        f"Outcome-model fold "
        f"{fold_id}/"
        f"{N_PROPENSITY_FOLDS}"
    )


    holdout_mask = (
        causal_population[
            "propensity_fold"
        ]
        .eq(
            fold_id
        )
        .to_numpy()
    )


    training_mask = (
        ~holdout_mask
    )


    training_frame = (
        causal_population.loc[
            training_mask
        ]
    )


    holdout_frame = (
        causal_population.loc[
            holdout_mask
        ]
    )


    # -------------------------------------------------------------
    # Separate treatment-state training populations.
    # -------------------------------------------------------------

    control_training_frame = (
        training_frame.loc[
            training_frame[
                "treatment"
            ]
            .eq(
                0
            )
        ]
    )


    treated_training_frame = (
        training_frame.loc[
            training_frame[
                "treatment"
            ]
            .eq(
                1
            )
        ]
    )


    print(
        "Control training rows:",
        f"{len(control_training_frame):,}",
    )

    print(
        "Treated training rows:",
        f"{len(treated_training_frame):,}",
    )

    print(
        "Holdout rows:",
        f"{len(holdout_frame):,}",
    )


    # -------------------------------------------------------------
    # Prepare common held-out covariates.
    # -------------------------------------------------------------

    X_holdout = (
        prepare_outcome_X(
            holdout_frame
        )
    )


    # -------------------------------------------------------------
    # CONTROL OUTCOME MODEL
    #
    # m0(X) = E[Y | D=0, X]
    # -------------------------------------------------------------

    X_control_train = (
        prepare_outcome_X(
            control_training_frame
        )
    )


    y_control_train = (
        control_training_frame[
            "target_sales"
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    model_m0 = XGBRegressor(
        **CAUSAL_OUTCOME_MODEL_PARAMS
    )


    model_m0.fit(
        X_control_train,
        y_control_train,
    )


    m0_holdout = (
        model_m0
        .predict(
            X_holdout
        )
        .astype(
            "float64"
        )
    )


    # Sales are non-negative by construction.
    m0_holdout = np.clip(
        m0_holdout,
        a_min=0.0,
        a_max=None,
    )


    # -------------------------------------------------------------
    # TREATED OUTCOME MODEL
    #
    # m1(X) = E[Y | D=1, X]
    # -------------------------------------------------------------

    X_treated_train = (
        prepare_outcome_X(
            treated_training_frame
        )
    )


    y_treated_train = (
        treated_training_frame[
            "target_sales"
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    model_m1 = XGBRegressor(
        **CAUSAL_OUTCOME_MODEL_PARAMS
    )


    model_m1.fit(
        X_treated_train,
        y_treated_train,
    )


    m1_holdout = (
        model_m1
        .predict(
            X_holdout
        )
        .astype(
            "float64"
        )
    )


    m1_holdout = np.clip(
        m1_holdout,
        a_min=0.0,
        a_max=None,
    )


    # -------------------------------------------------------------
    # Store both nuisance predictions for every held-out observation.
    # -------------------------------------------------------------

    cross_fitted_m0[
        holdout_mask
    ] = m0_holdout


    cross_fitted_m1[
        holdout_mask
    ] = m1_holdout


    # -------------------------------------------------------------
    # Factual model evaluation.
    #
    # m0 can only be directly evaluated where D=0.
    # m1 can only be directly evaluated where D=1.
    # -------------------------------------------------------------

    holdout_treatment = (
        holdout_frame[
            "treatment"
        ]
        .to_numpy(
            dtype="int8"
        )
    )


    holdout_sales = (
        holdout_frame[
            "target_sales"
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    control_holdout_mask = (
        holdout_treatment
        == 0
    )


    treated_holdout_mask = (
        holdout_treatment
        == 1
    )


    factual_prediction = np.where(
        treated_holdout_mask,
        m1_holdout,
        m0_holdout,
    )


    # -------------------------------------------------------------
    # Group-specific nuisance diagnostics.
    # -------------------------------------------------------------

    control_actual = (
        holdout_sales[
            control_holdout_mask
        ]
    )


    control_prediction = (
        m0_holdout[
            control_holdout_mask
        ]
    )


    treated_actual = (
        holdout_sales[
            treated_holdout_mask
        ]
    )


    treated_prediction = (
        m1_holdout[
            treated_holdout_mask
        ]
    )


    control_mae = mean_absolute_error(
        control_actual,
        control_prediction,
    )


    control_rmse = np.sqrt(
        mean_squared_error(
            control_actual,
            control_prediction,
        )
    )


    control_bias = (
        control_prediction
        - control_actual
    ).mean()


    treated_mae = mean_absolute_error(
        treated_actual,
        treated_prediction,
    )


    treated_rmse = np.sqrt(
        mean_squared_error(
            treated_actual,
            treated_prediction,
        )
    )


    treated_bias = (
        treated_prediction
        - treated_actual
    ).mean()


    factual_mae = mean_absolute_error(
        holdout_sales,
        factual_prediction,
    )


    factual_rmse = np.sqrt(
        mean_squared_error(
            holdout_sales,
            factual_prediction,
        )
    )


    factual_bias = (
        factual_prediction
        - holdout_sales
    ).mean()


    fit_minutes = (
        time.time()
        - fold_start_time
    ) / 60.0


    outcome_fold_metrics.append(
        {
            "fold":
                fold_id,

            "date_min":
                holdout_frame[
                    "date"
                ]
                .min(),

            "date_max":
                holdout_frame[
                    "date"
                ]
                .max(),

            "holdout_rows":
                len(
                    holdout_frame
                ),

            "control_rows":
                control_holdout_mask.sum(),

            "treated_rows":
                treated_holdout_mask.sum(),

            "m0_control_mae":
                control_mae,

            "m0_control_rmse":
                control_rmse,

            "m0_control_bias":
                control_bias,

            "m1_treated_mae":
                treated_mae,

            "m1_treated_rmse":
                treated_rmse,

            "m1_treated_bias":
                treated_bias,

            "factual_mae":
                factual_mae,

            "factual_rmse":
                factual_rmse,

            "factual_bias":
                factual_bias,

            "fit_minutes":
                fit_minutes,
        }
    )


    print(
        "m0 factual RMSE:",
        f"{control_rmse:.3f}",
    )

    print(
        "m1 factual RMSE:",
        f"{treated_rmse:.3f}",
    )

    print(
        "Combined factual RMSE:",
        f"{factual_rmse:.3f}",
    )

    print(
        "Combined factual bias:",
        f"{factual_bias:+.3f}",
    )

    print(
        "Fit minutes:",
        f"{fit_minutes:.2f}",
    )


    del (
        training_frame,
        holdout_frame,
        control_training_frame,
        treated_training_frame,
        X_holdout,
        X_control_train,
        X_treated_train,
        y_control_train,
        y_treated_train,
        model_m0,
        model_m1,
        m0_holdout,
        m1_holdout,
        holdout_treatment,
        holdout_sales,
        factual_prediction,
        control_actual,
        control_prediction,
        treated_actual,
        treated_prediction,
    )


    gc.collect()


# =====================================================================
# 5. STORE COMPLETE CROSS-FITTED OUTCOME PREDICTIONS
# =====================================================================

assert np.isfinite(
    cross_fitted_m0
).all()


assert np.isfinite(
    cross_fitted_m1
).all()


causal_population[
    "m0_hat"
] = (
    cross_fitted_m0
    .astype(
        "float32"
    )
)


causal_population[
    "m1_hat"
] = (
    cross_fitted_m1
    .astype(
        "float32"
    )
)


causal_population[
    "factual_outcome_hat"
] = np.where(
    causal_population[
        "treatment"
    ]
    .eq(
        1
    ),
    causal_population[
        "m1_hat"
    ],
    causal_population[
        "m0_hat"
    ],
).astype(
    "float32"
)


causal_population[
    "factual_outcome_residual"
] = (
    causal_population[
        "target_sales"
    ]
    - causal_population[
        "factual_outcome_hat"
    ]
).astype(
    "float32"
)


outcome_fold_metrics = pd.DataFrame(
    outcome_fold_metrics
)


# =====================================================================
# 6. GLOBAL FACTUAL DIAGNOSTICS
# =====================================================================

global_control_mask = (
    causal_population[
        "treatment"
    ]
    .eq(
        0
    )
)


global_treated_mask = (
    causal_population[
        "treatment"
    ]
    .eq(
        1
    )
)


GLOBAL_M0_MAE = mean_absolute_error(
    causal_population.loc[
        global_control_mask,
        "target_sales",
    ],
    causal_population.loc[
        global_control_mask,
        "m0_hat",
    ],
)


GLOBAL_M0_RMSE = np.sqrt(
    mean_squared_error(
        causal_population.loc[
            global_control_mask,
            "target_sales",
        ],
        causal_population.loc[
            global_control_mask,
            "m0_hat",
        ],
    )
)


GLOBAL_M1_MAE = mean_absolute_error(
    causal_population.loc[
        global_treated_mask,
        "target_sales",
    ],
    causal_population.loc[
        global_treated_mask,
        "m1_hat",
    ],
)


GLOBAL_M1_RMSE = np.sqrt(
    mean_squared_error(
        causal_population.loc[
            global_treated_mask,
            "target_sales",
        ],
        causal_population.loc[
            global_treated_mask,
            "m1_hat",
        ],
    )
)


GLOBAL_FACTUAL_MAE = mean_absolute_error(
    causal_population[
        "target_sales"
    ],
    causal_population[
        "factual_outcome_hat"
    ],
)


GLOBAL_FACTUAL_RMSE = np.sqrt(
    mean_squared_error(
        causal_population[
            "target_sales"
        ],
        causal_population[
            "factual_outcome_hat"
        ],
    )
)


GLOBAL_FACTUAL_BIAS = (
    causal_population[
        "factual_outcome_hat"
    ]
    - causal_population[
        "target_sales"
    ]
).mean()


print()
print(
    "=" * 80
)

print(
    "CROSS-FITTED OUTCOME MODEL SUMMARY"
)

print(
    "=" * 80
)


display(
    outcome_fold_metrics
    .style.hide(
        axis="index"
    )
    .format(
        {
            "m0_control_mae":
                "{:.3f}",

            "m0_control_rmse":
                "{:.3f}",

            "m0_control_bias":
                "{:+.3f}",

            "m1_treated_mae":
                "{:.3f}",

            "m1_treated_rmse":
                "{:.3f}",

            "m1_treated_bias":
                "{:+.3f}",

            "factual_mae":
                "{:.3f}",

            "factual_rmse":
                "{:.3f}",

            "factual_bias":
                "{:+.3f}",

            "fit_minutes":
                "{:.2f}",
        }
    )
)


print()
print(
    "Global m0 factual MAE:",
    f"{GLOBAL_M0_MAE:.3f}",
)

print(
    "Global m0 factual RMSE:",
    f"{GLOBAL_M0_RMSE:.3f}",
)

print(
    "Global m1 factual MAE:",
    f"{GLOBAL_M1_MAE:.3f}",
)

print(
    "Global m1 factual RMSE:",
    f"{GLOBAL_M1_RMSE:.3f}",
)

print(
    "Global factual MAE:",
    f"{GLOBAL_FACTUAL_MAE:.3f}",
)

print(
    "Global factual RMSE:",
    f"{GLOBAL_FACTUAL_RMSE:.3f}",
)

print(
    "Global factual bias:",
    f"{GLOBAL_FACTUAL_BIAS:+.3f}",
)


print()
print(
    "CROSS-FITTED OUTCOME MODELS: PASSED"
)


Outcome-model fold 1/5
Control training rows: 615,782
Treated training rows: 407,779
Holdout rows: 253,407
m0 factual RMSE: 210.303
m1 factual RMSE: 655.020
Combined factual RMSE: 365.895
Combined factual bias: +40.715
Fit minutes: 0.38

Outcome-model fold 2/5
Control training rows: 626,422
Treated training rows: 395,192
Holdout rows: 255,354
m0 factual RMSE: 156.635
m1 factual RMSE: 898.296
Combined factual RMSE: 493.950
Combined factual bias: +22.936
Fit minutes: 0.40

Outcome-model fold 3/5
Control training rows: 654,920
Treated training rows: 366,694
Holdout rows: 255,354
m0 factual RMSE: 243.730
m1 factual RMSE: 375.846
Combined factual RMSE: 302.488
Combined factual bias: +16.783
Fit minutes: 0.41

Outcome-model fold 4/5
Control training rows: 674,430
Treated training rows: 347,184
Holdout rows: 255,354
m0 factual RMSE: 218.906
m1 factual RMSE: 471.448
Combined factual RMSE: 359.998
Combined factual bias: +2.397
Fit minutes: 0.39

Outcome-model fold 5/5
Control training rows: 66

fold,date_min,date_max,holdout_rows,control_rows,treated_rows,m0_control_mae,m0_control_rmse,m0_control_bias,m1_treated_mae,m1_treated_rmse,m1_treated_bias,factual_mae,factual_rmse,factual_bias,fit_minutes
1,2015-08-16 00:00:00,2016-01-09 00:00:00,253407,194371,59036,43.341,210.303,+22.834,252.517,655.020,+99.589,92.073,365.895,+40.715,0.38
2,2016-01-10 00:00:00,2016-06-03 00:00:00,255354,183731,71623,30.357,156.635,+3.131,240.201,898.296,+73.741,89.215,493.950,+22.936,0.40
3,2016-06-04 00:00:00,2016-10-27 00:00:00,255354,155233,100121,15.956,243.730,+4.143,139.839,375.846,+36.380,64.529,302.488,+16.783,0.41
4,2016-10-28 00:00:00,2017-03-23 00:00:00,255354,135723,119631,23.116,218.906,+11.535,163.483,471.448,-7.969,88.877,359.998,+2.397,0.39
5,2017-03-24 00:00:00,2017-08-15 00:00:00,257499,141095,116404,11.937,78.231,-2.353,173.525,499.748,-54.893,84.984,340.960,-26.104,0.39



Global m0 factual MAE: 26.292
Global m0 factual RMSE: 191.445
Global m1 factual MAE: 183.947
Global m1 factual RMSE: 572.364
Global factual MAE: 83.925
Global factual RMSE: 378.170
Global factual bias: +11.238

CROSS-FITTED OUTCOME MODELS: PASSED


In [17]:
# ---------------------------------------------------------------------
# Normalized factual diagnostics for the cross-fitted outcome models.
#
# Absolute MAE/RMSE values are not directly comparable between treated
# and control observations because the sales scales can differ strongly.
#
# WAPE and normalized bias provide scale-aware diagnostics.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


factual_diagnostic_rows = []


for treatment_value, model_column in [
    (0, "m0_hat"),
    (1, "m1_hat"),
]:

    group_frame = (
        causal_population.loc[
            causal_population[
                "treatment"
            ]
            .eq(
                treatment_value
            )
        ]
        .copy()
    )


    actual = (
        group_frame[
            "target_sales"
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    prediction = (
        group_frame[
            model_column
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    residual = (
        prediction
        - actual
    )


    factual_diagnostic_rows.append(
        {
            "treatment":
                treatment_value,

            "rows":
                len(
                    group_frame
                ),

            "mean_actual_sales":
                actual.mean(),

            "total_actual_sales":
                actual.sum(),

            "mae":
                np.abs(
                    residual
                )
                .mean(),

            "rmse":
                np.sqrt(
                    np.mean(
                        residual ** 2
                    )
                ),

            "wape":
                (
                    np.abs(
                        residual
                    )
                    .sum()
                    /
                    actual.sum()
                ),

            "normalized_bias":
                (
                    residual.sum()
                    /
                    actual.sum()
                ),
        }
    )


factual_outcome_diagnostics = (
    pd.DataFrame(
        factual_diagnostic_rows
    )
)


display(
    factual_outcome_diagnostics
    .style.hide(
        axis="index"
    )
    .format(
        {
            "mean_actual_sales":
                "{:,.3f}",

            "total_actual_sales":
                "{:,.1f}",

            "mae":
                "{:.3f}",

            "rmse":
                "{:.3f}",

            "wape":
                "{:.2%}",

            "normalized_bias":
                "{:+.2%}",
        }
    )
)

treatment,rows,mean_actual_sales,total_actual_sales,mae,rmse,wape,normalized_bias
0,810153,100.633,"81,528,287.5",26.292,191.445,26.13%,+8.45%
1,466815,"1,092.625","510,053,876.4",183.947,572.364,16.84%,+1.46%


### 7.2 AIPW Estimator

Outcome regression alone provides the plug-in conditional contrast:

$$
\hat m_1(X_i)
-
\hat m_0(X_i).
$$

However, this contrast depends directly on the quality of the two outcome
models.

Augmented Inverse Probability Weighting (AIPW) combines outcome regression
with propensity-score adjustment and the factual outcome actually observed for
each unit.

For the treated potential-outcome mean, define:

$$
\phi_{1,i}
=
\hat m_1(X_i)
+
\frac{D_i}{\hat e_i}
\left[
Y_i-\hat m_1(X_i)
\right].
$$

For the untreated potential-outcome mean:

$$
\phi_{0,i}
=
\hat m_0(X_i)
+
\frac{1-D_i}{1-\hat e_i}
\left[
Y_i-\hat m_0(X_i)
\right].
$$

Their difference defines the observation-level AIPW score:

$$
\psi_i
=
\phi_{1,i}
-
\phi_{0,i}.
$$

Equivalently:

$$
\psi_i
=
\hat m_1(X_i)
-
\hat m_0(X_i)
+
\frac{D_i}{\hat e_i}
\left[
Y_i-\hat m_1(X_i)
\right]
-
\frac{1-D_i}{1-\hat e_i}
\left[
Y_i-\hat m_0(X_i)
\right].
$$

The first term is the outcome-regression contrast.

The remaining terms use factual residuals to augment or correct that
model-based contrast.

The AIPW estimator is called **doubly robust** because, under the causal
identification assumptions and standard regularity conditions, it remains
consistent if either:

- the treatment-assignment model is correctly specified; or
- the relevant outcome-regression models are correctly specified.

Cross-fitting ensures that all nuisance quantities entering an observation's
AIPW score were generated by models that did not train on that observation.

Because the empirical propensity distribution contains extreme values, the
primary finite-sample specification uses the propensity stabilization rule
frozen in Section 6:

$$
\hat e_i^{stab}
=
\operatorname{clip}
(
\hat e_i,
0.01,
0.99
).
$$

This clipping improves numerical stability at the cost of modifying the exact
raw-IPW correction in extreme-support regions.

Alternative clipping thresholds are therefore retained for sensitivity
analysis.

An individual AIPW score $\psi_i$ should not be interpreted as an identified
individual treatment effect.

The causal estimand is obtained by aggregating these scores across the target
population.

In [18]:
# ---------------------------------------------------------------------
# Construct cross-fitted AIPW components.
#
# No headline treatment-effect estimate is reported yet.
#
# For each causal observation i:
#
#   phi1_i =
#       m1_hat_i
#       + D_i / e_i * (Y_i - m1_hat_i)
#
#   phi0_i =
#       m0_hat_i
#       + (1-D_i) / (1-e_i) * (Y_i - m0_hat_i)
#
#   psi_i =
#       phi1_i - phi0_i
#
# The primary finite-sample specification uses the propensity clipping
# rule frozen in Section 6:
#
#   e_stab = clip(e_hat, 0.01, 0.99)
#
# Individual psi_i values are influence-function-style pseudo-outcomes;
# they are NOT identified individual causal effects.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


PRIMARY_PROPENSITY_CLIP = 0.01


# =====================================================================
# 1. EXTRACT CROSS-FITTED NUISANCE QUANTITIES
# =====================================================================

y = (
    causal_population[
        "target_sales"
    ]
    .to_numpy(
        dtype="float64"
    )
)


d = (
    causal_population[
        "treatment"
    ]
    .to_numpy(
        dtype="float64"
    )
)


e_hat = (
    causal_population[
        "propensity_score"
    ]
    .to_numpy(
        dtype="float64"
    )
)


m0_hat = (
    causal_population[
        "m0_hat"
    ]
    .to_numpy(
        dtype="float64"
    )
)


m1_hat = (
    causal_population[
        "m1_hat"
    ]
    .to_numpy(
        dtype="float64"
    )
)


assert np.isfinite(
    y
).all()


assert np.isfinite(
    e_hat
).all()


assert np.isfinite(
    m0_hat
).all()


assert np.isfinite(
    m1_hat
).all()


# =====================================================================
# 2. PRIMARY STABILIZED PROPENSITY
# =====================================================================

e_stab = np.clip(
    e_hat,
    PRIMARY_PROPENSITY_CLIP,
    1.0
    - PRIMARY_PROPENSITY_CLIP,
)


causal_population[
    "propensity_score_stabilized"
] = (
    e_stab.astype(
        "float32"
    )
)


clipped_mask = (
    e_hat
    != e_stab
)


print(
    "Rows affected by primary propensity clipping:",
    f"{clipped_mask.sum():,}",
)


print(
    "Share affected by primary propensity clipping:",
    f"{clipped_mask.mean():.2%}",
)


# =====================================================================
# 3. OUTCOME-REGRESSION CONTRAST
# =====================================================================

outcome_regression_contrast = (
    m1_hat
    - m0_hat
)


# =====================================================================
# 4. FACTUAL RESIDUAL CORRECTIONS
# =====================================================================
#
# A treated row contributes only to the treated correction.
#
# A control row contributes only to the control correction.
# =====================================================================

treated_residual_correction = (
    d
    / e_stab
    * (
        y
        - m1_hat
    )
)


control_residual_correction = (
    (
        1.0
        - d
    )
    / (
        1.0
        - e_stab
    )
    * (
        y
        - m0_hat
    )
)


# =====================================================================
# 5. AUGMENTED POTENTIAL-OUTCOME PSEUDO-OUTCOMES
# =====================================================================

aipw_phi1 = (
    m1_hat
    + treated_residual_correction
)


aipw_phi0 = (
    m0_hat
    + control_residual_correction
)


aipw_score = (
    aipw_phi1
    - aipw_phi0
)


assert np.isfinite(
    aipw_score
).all()


# =====================================================================
# 6. STORE ROW-LEVEL COMPONENTS
# =====================================================================

causal_population[
    "or_contrast"
] = (
    outcome_regression_contrast
    .astype(
        "float32"
    )
)


causal_population[
    "aipw_treated_correction"
] = (
    treated_residual_correction
    .astype(
        "float32"
    )
)


causal_population[
    "aipw_control_correction"
] = (
    control_residual_correction
    .astype(
        "float32"
    )
)


causal_population[
    "aipw_phi1"
] = (
    aipw_phi1
    .astype(
        "float32"
    )
)


causal_population[
    "aipw_phi0"
] = (
    aipw_phi0
    .astype(
        "float32"
    )
)


causal_population[
    "aipw_score"
] = (
    aipw_score
    .astype(
        "float32"
    )
)


# =====================================================================
# 7. DIAGNOSTIC DISTRIBUTIONS
#
# These diagnostics assess whether the augmentation terms remain
# numerically dominated by a small number of observations.
#
# The mean AIPW score is deliberately not printed here; that is the ATE
# estimand reported in Section 7.3.
# =====================================================================

def distribution_summary(
    values,
    name,
):
    """
    Compact distribution summary for a numerical AIPW component.
    """

    values = np.asarray(
        values,
        dtype="float64",
    )


    return {
        "component":
            name,

        "mean":
            values.mean(),

        "median":
            np.median(
                values
            ),

        "p01":
            np.quantile(
                values,
                0.01,
            ),

        "p05":
            np.quantile(
                values,
                0.05,
            ),

        "p95":
            np.quantile(
                values,
                0.95,
            ),

        "p99":
            np.quantile(
                values,
                0.99,
            ),

        "min":
            values.min(),

        "max":
            values.max(),
    }


aipw_component_summary = pd.DataFrame(
    [
        distribution_summary(
            outcome_regression_contrast,
            "outcome_regression_contrast",
        ),

        distribution_summary(
            treated_residual_correction,
            "treated_residual_correction",
        ),

        distribution_summary(
            control_residual_correction,
            "control_residual_correction",
        ),

        distribution_summary(
            aipw_phi1,
            "phi1",
        ),

        distribution_summary(
            aipw_phi0,
            "phi0",
        ),
    ]
)


display(
    aipw_component_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "mean":
                "{:,.3f}",

            "median":
                "{:,.3f}",

            "p01":
                "{:,.3f}",

            "p05":
                "{:,.3f}",

            "p95":
                "{:,.3f}",

            "p99":
                "{:,.3f}",

            "min":
                "{:,.3f}",

            "max":
                "{:,.3f}",
        }
    )
)


# =====================================================================
# 8. CHECK THE ALGEBRAIC IDENTITY
# =====================================================================

direct_aipw_score = (
    m1_hat
    - m0_hat
    + d
    / e_stab
    * (
        y
        - m1_hat
    )
    - (
        1.0
        - d
    )
    / (
        1.0
        - e_stab
    )
    * (
        y
        - m0_hat
    )
)


max_identity_difference = np.max(
    np.abs(
        direct_aipw_score
        - aipw_score
    )
)


print()
print(
    "Max AIPW identity difference:",
    max_identity_difference,
)


assert (
    max_identity_difference
    < 1e-10
)


print()
print(
    "AIPW COMPONENT CONSTRUCTION: PASSED"
)

Rows affected by primary propensity clipping: 473,419
Share affected by primary propensity clipping: 37.07%


component,mean,median,p01,p05,p95,p99,min,max
outcome_regression_contrast,115.883,9.518,-516.681,-63.520,835.585,"2,778.043","-50,970.068","20,843.630"
treated_residual_correction,-18.412,0.000,"-1,001.264",-229.724,134.024,840.450,"-360,318.604","244,472.761"
control_residual_correction,-98.001,0.000,-794.959,-57.778,20.361,291.297,"-1,377,742.480","791,353.516"
phi1,524.292,59.346,-76.138,0.000,"2,832.723","6,828.359","-354,874.417","250,376.882"
phi0,328.820,21.869,-175.543,-0.027,"2,529.828","5,988.049","-1,363,965.056","800,400.980"



Max AIPW identity difference: 1.4551915228366852e-11

AIPW COMPONENT CONSTRUCTION: PASSED


### 7.3 ATE and ATT

The causal analysis now aggregates the cross-fitted nuisance predictions into
population-level treatment-effect estimands.

The **Average Treatment Effect** is:

$$
ATE
=
E[Y(1)-Y(0)].
$$

It targets the complete causal population.

In this project, the estimand asks:

> Across eligible store-family-date observations in the 2015–2017 causal
> population, how much would realized sales differ on average under the binary
> promotion state versus no promotion?

The primary AIPW estimator is:

$$
\widehat{ATE}_{AIPW}
=
\frac{1}{N}
\sum_{i=1}^N
\psi_i,
$$

where $\psi_i$ is the cross-fitted AIPW score constructed in Section 7.2.

---

The **Average Treatment Effect on the Treated** is:

$$
ATT
=
E[Y(1)-Y(0)\mid D=1].
$$

Its target population contains only observations that actually received
promotion.

For these units, $Y(1)$ is factually observed. The main counterfactual problem
is therefore to estimate what those same treated commercial contexts would
have produced under no promotion.

A doubly robust ATT estimator is:

$$
\widehat{ATT}
=
\frac{1}{N_1}
\sum_{i=1}^N
\left[
D_i
\left(
Y_i-\hat m_0(X_i)
\right)
-
(1-D_i)
\frac{\hat e_i}{1-\hat e_i}
\left(
Y_i-\hat m_0(X_i)
\right)
\right],
$$

where:

$$
N_1=\sum_i D_i.
$$

The first term compares factual treated outcomes with their estimated
no-promotion outcome.

The second term uses weighted control residuals to augment that counterfactual
estimate.

The primary finite-sample specification uses the previously frozen propensity
stabilization:

$$
\hat e_i^{stab}
=
\operatorname{clip}
(
\hat e_i,
0.01,
0.99
).
$$

---

Neither estimand represents the effect of a specific discount or campaign.

The binary treatment:

$$
D
=
\mathbb{1}
[
\text{onpromotion}>0
]
$$

combines heterogeneous promotion intensities and promotion versions.

The resulting estimands should therefore be interpreted as average effects of
entering the observed binary promotion state under the historical treatment
regime.

Because observations form a repeated store-family panel and share common
calendar shocks, uncertainty is not computed under an independent-row
assumption.

Approximate confidence intervals use two-way clustered influence-function
standard errors across:

- store-family series;
- calendar date.

In [19]:
# ---------------------------------------------------------------------
# Primary ATE and ATT estimation.
#
# Primary propensity specification:
#
#   e_stab = clip(e_hat, 0.01, 0.99)
#
# Uncertainty is estimated from cross-fitted influence-function-style
# scores with two-way clustering by:
#
#   1. store-family series;
#   2. calendar date.
#
# This avoids treating the ~1.28 million panel rows as independent.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


# =====================================================================
# 1. PRIMARY ATE
# =====================================================================

primary_aipw_score = (
    causal_population[
        "aipw_score"
    ]
    .to_numpy(
        dtype="float64"
    )
)


PRIMARY_ATE = (
    primary_aipw_score.mean()
)


# Outcome-regression plug-in estimate retained only as a diagnostic
# comparator. It is not the primary causal estimator.
OR_PLUGIN_ATE = (
    causal_population[
        "or_contrast"
    ]
    .mean()
)


# =====================================================================
# 2. PRIMARY DOUBLY ROBUST ATT
# =====================================================================

y = (
    causal_population[
        "target_sales"
    ]
    .to_numpy(
        dtype="float64"
    )
)


d = (
    causal_population[
        "treatment"
    ]
    .to_numpy(
        dtype="float64"
    )
)


e_stab = (
    causal_population[
        "propensity_score_stabilized"
    ]
    .to_numpy(
        dtype="float64"
    )
)


m0_hat = (
    causal_population[
        "m0_hat"
    ]
    .to_numpy(
        dtype="float64"
    )
)


n_treated = (
    d.sum()
)


treated_factual_contribution = (
    d
    * (
        y
        - m0_hat
    )
)


control_att_augmentation = (
    (
        1.0
        - d
    )
    * (
        e_stab
        / (
            1.0
            - e_stab
        )
    )
    * (
        y
        - m0_hat
    )
)


att_numerator_contribution = (
    treated_factual_contribution
    - control_att_augmentation
)


PRIMARY_ATT = (
    att_numerator_contribution.sum()
    / n_treated
)


causal_population[
    "att_numerator_contribution"
] = (
    att_numerator_contribution
    .astype(
        "float32"
    )
)


# =====================================================================
# 3. TWO-WAY CLUSTERED STANDARD ERROR
# =====================================================================
#
# For the sample mean of an influence score IF_i:
#
#   Var(mean IF)
#
# is estimated using cluster sums.
#
# We use:
#
#   cluster A = store-family series
#   cluster B = calendar date
#
# and subtract the intersection contribution.
#
# In this panel, a series-date intersection corresponds to one causal
# observation.
# =====================================================================

series_id = (
    causal_population[
        "store_nbr"
    ]
    .astype(
        "string"
    )
    + "::"
    + causal_population[
        "family"
    ]
    .astype(
        "string"
    )
)


date_id = (
    causal_population[
        "date"
    ]
)


def cluster_sum_of_squares(
    influence_values,
    cluster_labels,
):
    """
    Sum squared cluster totals of an influence-function vector.
    """

    frame = pd.DataFrame(
        {
            "cluster":
                np.asarray(
                    cluster_labels
                ),

            "influence":
                np.asarray(
                    influence_values,
                    dtype="float64",
                ),
        }
    )


    cluster_totals = (
        frame
        .groupby(
            "cluster",
            sort=False,
            observed=True,
        )[
            "influence"
        ]
        .sum()
        .to_numpy(
            dtype="float64"
        )
    )


    return np.square(
        cluster_totals
    ).sum()


def two_way_clustered_se(
    influence_values,
    cluster_a,
    cluster_b,
):
    """
    Two-way clustered standard error for an estimator represented as
    the sample mean of influence-function values.

    Uses the inclusion-exclusion cluster variance:

        V_A + V_B - V_intersection

    Here each store-family-date intersection contains one observation,
    so the intersection term is the row-level sum of squares.
    """

    influence_values = np.asarray(
        influence_values,
        dtype="float64",
    )


    n = len(
        influence_values
    )


    cluster_a_ss = cluster_sum_of_squares(
        influence_values,
        cluster_a,
    )


    cluster_b_ss = cluster_sum_of_squares(
        influence_values,
        cluster_b,
    )


    intersection_ss = np.square(
        influence_values
    ).sum()


    variance = (
        cluster_a_ss
        + cluster_b_ss
        - intersection_ss
    ) / (
        n ** 2
    )


    # Numerical guard only.
    variance = max(
        variance,
        0.0,
    )


    return np.sqrt(
        variance
    )


# =====================================================================
# 4. ATE INFLUENCE VALUES
# =====================================================================

ate_influence = (
    primary_aipw_score
    - PRIMARY_ATE
)


ATE_CLUSTERED_SE = (
    two_way_clustered_se(
        ate_influence,
        series_id,
        date_id,
    )
)


ATE_CI_LOW = (
    PRIMARY_ATE
    - 1.96
    * ATE_CLUSTERED_SE
)


ATE_CI_HIGH = (
    PRIMARY_ATE
    + 1.96
    * ATE_CLUSTERED_SE
)


# =====================================================================
# 5. ATT INFLUENCE VALUES
# =====================================================================
#
# ATT is a ratio:
#
#   mean(q_i) / mean(D_i)
#
# where q_i is the ATT numerator contribution.
#
# Its first-order influence representation is:
#
#   IF_i =
#       (q_i - ATT * D_i) / P(D=1)
# =====================================================================

treatment_prevalence = (
    d.mean()
)


att_influence = (
    (
        att_numerator_contribution
        - PRIMARY_ATT
        * d
    )
    / treatment_prevalence
)


ATT_CLUSTERED_SE = (
    two_way_clustered_se(
        att_influence,
        series_id,
        date_id,
    )
)


ATT_CI_LOW = (
    PRIMARY_ATT
    - 1.96
    * ATT_CLUSTERED_SE
)


ATT_CI_HIGH = (
    PRIMARY_ATT
    + 1.96
    * ATT_CLUSTERED_SE
)


# =====================================================================
# 6. OBSERVATIONAL DIFFERENCE FOR CONTEXT ONLY
# =====================================================================
#
# This is intentionally NOT a causal estimator.
# It illustrates how strongly treatment selection changes the raw
# treated-versus-control comparison.
# =====================================================================

observed_treated_mean = (
    causal_population.loc[
        causal_population[
            "treatment"
        ]
        .eq(
            1
        ),
        "target_sales",
    ]
    .mean()
)


observed_control_mean = (
    causal_population.loc[
        causal_population[
            "treatment"
        ]
        .eq(
            0
        ),
        "target_sales",
    ]
    .mean()
)


OBSERVED_MEAN_DIFFERENCE = (
    observed_treated_mean
    - observed_control_mean
)


# =====================================================================
# 7. SUMMARY
# =====================================================================

causal_effect_summary = pd.DataFrame(
    [
        {
            "estimand":
                "Observed treated - control difference",

            "estimate":
                OBSERVED_MEAN_DIFFERENCE,

            "clustered_se":
                np.nan,

            "ci_low":
                np.nan,

            "ci_high":
                np.nan,

            "causal_estimator":
                False,
        },

        {
            "estimand":
                "Outcome-regression plug-in ATE",

            "estimate":
                OR_PLUGIN_ATE,

            "clustered_se":
                np.nan,

            "ci_low":
                np.nan,

            "ci_high":
                np.nan,

            "causal_estimator":
                True,
        },

        {
            "estimand":
                "Primary clipped AIPW ATE",

            "estimate":
                PRIMARY_ATE,

            "clustered_se":
                ATE_CLUSTERED_SE,

            "ci_low":
                ATE_CI_LOW,

            "ci_high":
                ATE_CI_HIGH,

            "causal_estimator":
                True,
        },

        {
            "estimand":
                "Primary clipped DR ATT",

            "estimate":
                PRIMARY_ATT,

            "clustered_se":
                ATT_CLUSTERED_SE,

            "ci_low":
                ATT_CI_LOW,

            "ci_high":
                ATT_CI_HIGH,

            "causal_estimator":
                True,
        },
    ]
)


display(
    causal_effect_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "estimate":
                "{:+,.3f}",

            "clustered_se":
                "{:.3f}",

            "ci_low":
                "{:+,.3f}",

            "ci_high":
                "{:+,.3f}",
        },
        na_rep="—",
    )
)


print()
print(
    "ATE target population rows:",
    f"{len(causal_population):,}",
)

print(
    "ATT target population treated rows:",
    f"{int(n_treated):,}",
)

print(
    "Two-way clustering series:",
    f"{series_id.nunique():,}",
)

print(
    "Two-way clustering dates:",
    f"{date_id.nunique():,}",
)

print()
print(
    "PRIMARY ATE AND ATT ESTIMATION: PASSED"
)

estimand,estimate,clustered_se,ci_low,ci_high,causal_estimator
Observed treated - control difference,+991.992,—,—,—,False
Outcome-regression plug-in ATE,+115.883,—,—,—,True
Primary clipped AIPW ATE,+195.472,62.804,+72.376,+318.567,True
Primary clipped DR ATT,+367.789,167.395,+39.695,+695.882,True



ATE target population rows: 1,276,968
ATT target population treated rows: 466,815
Two-way clustering series: 1,782
Two-way clustering dates: 729

PRIMARY ATE AND ATT ESTIMATION: PASSED


### 7.4 Diagnostics and Sensitivity

The primary causal estimates depend on an observational treatment-assignment
process with substantial practical overlap limitations.

The primary finite-sample analysis stabilized propensity scores through:

$$
\hat e^{stab}(X)
=
\operatorname{clip}
(
\hat e(X),
0.01,
0.99
).
$$

Because approximately 37% of causal observations are affected by this rule,
the resulting treatment-effect estimate should not be interpreted without
sensitivity analysis.

Two distinct robustness operations are considered.

#### Propensity clipping

Clipping retains the complete causal population but limits the magnitude of
inverse-probability corrections.

Alternative clipping thresholds therefore assess the sensitivity of the
full-population estimator to weight stabilization.

#### Common-support trimming

Trimming instead removes observations whose estimated treatment probabilities
fall outside:

$$
[\alpha,1-\alpha].
$$

Unlike clipping, trimming changes the target population.

The corresponding estimand is therefore an ATE for the retained overlap
population rather than the original full causal population.

Sensitivity analysis compares:

- raw, unclipped AIPW;
- propensity clipping at 1%, 2.5% and 5%;
- common-support trimming at 1%, 2.5% and 5%;
- temporal stability of the primary 1%-clipped AIPW score.

The objective is not to select the specification producing the preferred
treatment effect.

The objective is to determine whether the qualitative causal conclusion is
stable across reasonable overlap and stabilization choices.

In [20]:
# ---------------------------------------------------------------------
# AIPW sensitivity analysis.
#
# We compare:
#
#   1. raw, unclipped propensity;
#   2. clipping at 1%, 2.5%, and 5%;
#   3. common-support trimming at 1%, 2.5%, and 5%;
#   4. temporal stability of the primary 1%-clipped AIPW estimate.
#
# IMPORTANT:
# Clipping and trimming are not equivalent.
#
# Clipping:
#   - retains all observations;
#   - regularizes extreme weights;
#   - nominally targets the full population.
#
# Trimming:
#   - removes weak-support observations;
#   - changes the target population;
#   - estimates an overlap-population ATE.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


# =====================================================================
# 1. REUSABLE AIPW SCORE FUNCTION
# =====================================================================

def compute_aipw_score(
    y,
    d,
    e,
    m0,
    m1,
):
    """
    Observation-level AIPW score for the ATE.
    """

    return (
        m1
        - m0
        + d
        / e
        * (
            y
            - m1
        )
        - (
            1.0
            - d
        )
        / (
            1.0
            - e
        )
        * (
            y
            - m0
        )
    )


# =====================================================================
# 2. REUSABLE ATT FUNCTION
# =====================================================================

def compute_dr_att(
    y,
    d,
    e,
    m0,
):
    """
    Doubly robust ATT estimator.

    The target population is the actually treated population.
    """

    numerator = (
        d
        * (
            y
            - m0
        )
        - (
            1.0
            - d
        )
        * (
            e
            / (
                1.0
                - e
            )
        )
        * (
            y
            - m0
        )
    )


    return (
        numerator.sum()
        / d.sum()
    )


# =====================================================================
# 3. PROPENSITY-CLIPPING SENSITIVITY
# =====================================================================

clipping_specs = [
    (
        "raw",
        None,
    ),
    (
        "clip_0.01",
        0.01,
    ),
    (
        "clip_0.025",
        0.025,
    ),
    (
        "clip_0.05",
        0.05,
    ),
]


clipping_sensitivity_rows = []


for (
    specification,
    threshold,
) in clipping_specs:

    if threshold is None:

        e_spec = (
            e_hat.copy()
        )


    else:

        e_spec = np.clip(
            e_hat,
            threshold,
            1.0
            - threshold,
        )


    score = compute_aipw_score(
        y,
        d,
        e_spec,
        m0_hat,
        m1_hat,
    )


    ate = (
        score.mean()
    )


    att = compute_dr_att(
        y,
        d,
        e_spec,
        m0_hat,
    )


    influence = (
        score
        - ate
    )


    clustered_se = (
        two_way_clustered_se(
            influence,
            series_id,
            date_id,
        )
    )


    ci_low = (
        ate
        - 1.96
        * clustered_se
    )


    ci_high = (
        ate
        + 1.96
        * clustered_se
    )


    if threshold is None:

        affected_share = 0.0

    else:

        affected_share = (
            (
                e_hat
                != e_spec
            )
            .mean()
        )


    clipping_sensitivity_rows.append(
        {
            "specification":
                specification,

            "rows":
                len(
                    y
                ),

            "share_propensity_modified":
                affected_share,

            "ate":
                ate,

            "clustered_se":
                clustered_se,

            "ci_low":
                ci_low,

            "ci_high":
                ci_high,

            "att":
                att,
        }
    )


clipping_sensitivity = pd.DataFrame(
    clipping_sensitivity_rows
)


print(
    "=" * 80
)

print(
    "PROPENSITY-CLIPPING SENSITIVITY"
)

print(
    "=" * 80
)


display(
    clipping_sensitivity
    .style.hide(
        axis="index"
    )
    .format(
        {
            "share_propensity_modified":
                "{:.2%}",

            "ate":
                "{:+,.3f}",

            "clustered_se":
                "{:.3f}",

            "ci_low":
                "{:+,.3f}",

            "ci_high":
                "{:+,.3f}",

            "att":
                "{:+,.3f}",
        }
    )
)


# =====================================================================
# 4. COMMON-SUPPORT TRIMMING SENSITIVITY
# =====================================================================
#
# Here we retain only rows satisfying:
#
#   threshold <= e_hat <= 1 - threshold
#
# No additional propensity clipping is applied inside the retained
# population.
#
# The resulting estimand is an ATE for the retained overlap population.
# =====================================================================

trimming_sensitivity_rows = []


for threshold in [
    0.01,
    0.025,
    0.05,
]:

    retained_mask = (
        (
            e_hat
            >= threshold
        )
        &
        (
            e_hat
            <= (
                1.0
                - threshold
            )
        )
    )


    y_trim = (
        y[
            retained_mask
        ]
    )


    d_trim = (
        d[
            retained_mask
        ]
    )


    e_trim = (
        e_hat[
            retained_mask
        ]
    )


    m0_trim = (
        m0_hat[
            retained_mask
        ]
    )


    m1_trim = (
        m1_hat[
            retained_mask
        ]
    )


    series_trim = (
        series_id.loc[
            retained_mask
        ]
    )


    date_trim = (
        date_id.loc[
            retained_mask
        ]
    )


    trimmed_score = compute_aipw_score(
        y_trim,
        d_trim,
        e_trim,
        m0_trim,
        m1_trim,
    )


    trimmed_ate = (
        trimmed_score.mean()
    )


    trimmed_influence = (
        trimmed_score
        - trimmed_ate
    )


    trimmed_se = (
        two_way_clustered_se(
            trimmed_influence,
            series_trim,
            date_trim,
        )
    )


    trimming_sensitivity_rows.append(
        {
            "support_region":
                (
                    f"[{threshold:.3f}, "
                    f"{1.0 - threshold:.3f}]"
                ),

            "rows_retained":
                retained_mask.sum(),

            "population_share_retained":
                retained_mask.mean(),

            "treatment_rate":
                d_trim.mean(),

            "ate":
                trimmed_ate,

            "clustered_se":
                trimmed_se,

            "ci_low":
                (
                    trimmed_ate
                    - 1.96
                    * trimmed_se
                ),

            "ci_high":
                (
                    trimmed_ate
                    + 1.96
                    * trimmed_se
                ),
        }
    )


trimming_sensitivity = pd.DataFrame(
    trimming_sensitivity_rows
)


print()
print(
    "=" * 80
)

print(
    "COMMON-SUPPORT TRIMMING SENSITIVITY"
)

print(
    "=" * 80
)


display(
    trimming_sensitivity
    .style.hide(
        axis="index"
    )
    .format(
        {
            "population_share_retained":
                "{:.2%}",

            "treatment_rate":
                "{:.2%}",

            "ate":
                "{:+,.3f}",

            "clustered_se":
                "{:.3f}",

            "ci_low":
                "{:+,.3f}",

            "ci_high":
                "{:+,.3f}",
        }
    )
)


# =====================================================================
# 5. TEMPORAL STABILITY OF THE PRIMARY AIPW SCORE
# =====================================================================
#
# These are descriptive period-specific averages of the same primary
# cross-fitted AIPW score.
#
# They reveal whether the estimated promotion effect is concentrated in
# a particular historical regime.
# =====================================================================

temporal_effect_rows = []


for fold_id in range(
    1,
    N_PROPENSITY_FOLDS + 1,
):

    fold_mask = (
        causal_population[
            "propensity_fold"
        ]
        .eq(
            fold_id
        )
        .to_numpy()
    )


    fold_score = (
        primary_aipw_score[
            fold_mask
        ]
    )


    fold_ate = (
        fold_score.mean()
    )


    fold_influence = (
        fold_score
        - fold_ate
    )


    fold_series = (
        series_id.loc[
            fold_mask
        ]
    )


    fold_dates = (
        date_id.loc[
            fold_mask
        ]
    )


    fold_se = (
        two_way_clustered_se(
            fold_influence,
            fold_series,
            fold_dates,
        )
    )


    fold_frame = (
        causal_population.loc[
            fold_mask
        ]
    )


    temporal_effect_rows.append(
        {
            "fold":
                fold_id,

            "date_min":
                fold_frame[
                    "date"
                ]
                .min(),

            "date_max":
                fold_frame[
                    "date"
                ]
                .max(),

            "rows":
                len(
                    fold_frame
                ),

            "treatment_rate":
                fold_frame[
                    "treatment"
                ]
                .mean(),

            "ate":
                fold_ate,

            "clustered_se":
                fold_se,

            "ci_low":
                (
                    fold_ate
                    - 1.96
                    * fold_se
                ),

            "ci_high":
                (
                    fold_ate
                    + 1.96
                    * fold_se
                ),
        }
    )


temporal_effect_stability = pd.DataFrame(
    temporal_effect_rows
)


print()
print(
    "=" * 80
)

print(
    "TEMPORAL STABILITY OF PRIMARY AIPW EFFECT"
)

print(
    "=" * 80
)


display(
    temporal_effect_stability
    .style.hide(
        axis="index"
    )
    .format(
        {
            "treatment_rate":
                "{:.2%}",

            "ate":
                "{:+,.3f}",

            "clustered_se":
                "{:.3f}",

            "ci_low":
                "{:+,.3f}",

            "ci_high":
                "{:+,.3f}",
        }
    )
)


print()
print(
    "CAUSAL SENSITIVITY ANALYSIS: PASSED"
)

PROPENSITY-CLIPPING SENSITIVITY


specification,rows,share_propensity_modified,ate,clustered_se,ci_low,ci_high,att
raw,1276968,0.00%,+191.592,135.226,-73.452,+456.636,+426.522
clip_0.01,1276968,37.07%,+195.472,62.804,+72.376,+318.567,+367.788
clip_0.025,1276968,48.83%,+158.913,30.928,+98.295,+219.531,+254.454
clip_0.05,1276968,57.88%,+141.554,18.969,+104.374,+178.734,+200.817



COMMON-SUPPORT TRIMMING SENSITIVITY


support_region,rows_retained,population_share_retained,treatment_rate,ate,clustered_se,ci_low,ci_high
"[0.010, 0.990]",803549,62.93%,40.86%,+79.392,28.721,+23.100,+135.684
"[0.025, 0.975]",653366,51.17%,40.68%,+44.854,11.053,+23.191,+66.517
"[0.050, 0.950]",537835,42.12%,40.51%,+35.675,6.860,+22.230,+49.120



TEMPORAL STABILITY OF PRIMARY AIPW EFFECT


fold,date_min,date_max,rows,treatment_rate,ate,clustered_se,ci_low,ci_high
1,2015-08-16 00:00:00,2016-01-09 00:00:00,253407,23.30%,+187.411,87.833,+15.258,+359.564
2,2016-01-10 00:00:00,2016-06-03 00:00:00,255354,28.05%,+131.940,22.923,+87.010,+176.870
3,2016-06-04 00:00:00,2016-10-27 00:00:00,255354,39.21%,+202.577,48.713,+107.100,+298.054
4,2016-10-28 00:00:00,2017-03-23 00:00:00,255354,46.85%,+358.963,291.890,-213.141,+931.067
5,2017-03-24 00:00:00,2017-08-15 00:00:00,257499,45.21%,+97.231,15.107,+67.621,+126.841



CAUSAL SENSITIVITY ANALYSIS: PASSED


#### Sensitivity conclusion

The causal sensitivity analysis supports a positive qualitative promotion
effect, but not a precisely identified single effect magnitude.

The raw and 1%-clipped full-population AIPW specifications produce very similar
point estimates:

$$
191.6
\quad\text{and}\quad
195.5
$$

sales units per store-family-date observation.

However, raw propensity weighting produces substantially greater uncertainty.
Its approximate 95% confidence interval includes zero:

$$
[-73.5,\ 456.6],
$$

whereas 1% propensity clipping reduces the clustered standard error from
approximately 135 to 63 while leaving the point estimate nearly unchanged.

More aggressive clipping produces smaller estimates:

$$
195.5
\rightarrow
158.9
\rightarrow
141.6,
$$

indicating that estimated effect magnitude remains sensitive to treatment
assignment in low-overlap regions.

Common-support trimming changes the target population rather than merely
stabilizing the estimator.

Across increasingly restrictive overlap populations, the estimated ATE remains
positive:

$$
79.4,\qquad
44.9,\qquad
35.7.
$$

The substantially smaller overlap-population effects suggest that low-support
regions contribute materially to the larger full-population estimate.

These trimmed effects are not alternative estimates of exactly the same
estimand: each corresponds to a different retained causal population.

Temporal diagnostics provide a similar conclusion.

All five period-specific point estimates are positive, but their magnitude and
precision vary considerably. One historical period exhibits particularly large
uncertainty and an interval spanning zero.

The causal evidence therefore supports the qualitative conclusion that
promotion increases realized sales on average under the observed historical
promotion regime.

The exact magnitude of that effect is less robust and remains sensitive to
overlap, propensity stabilization and historical regime.

The primary 1%-clipped AIPW estimate is retained as the headline retrospective
ATE, while overlap-population and alternative-clipping estimates are reported
as sensitivity analyses rather than competing headline estimates.

## 8. Heterogeneous Promotion Effects

The ATE and ATT summarize promotion effects across broad target populations.

They do not imply that promotion has the same effect in every commercial
context.

Define the Conditional Average Treatment Effect:

$$
\tau(x)
=
E[Y(1)-Y(0)\mid X=x].
$$

Treatment-effect heterogeneity is particularly relevant in retail because the
commercial response to promotion may depend on product family, store
characteristics, baseline demand and historical promotion behavior.

The cross-fitted AIPW score constructed in Section 7 provides a useful
pseudo-outcome for this analysis.

Under the causal identification assumptions and correctly estimated nuisance
functions:

$$
E[\psi_i\mid X_i=x]
=
\tau(x).
$$

An individual $\psi_i$ remains a noisy pseudo-outcome rather than an identified
individual treatment effect.

Heterogeneity is therefore studied through averages across sufficiently large
groups and, subsequently, through a regularized conditional-effect model.

### 8.1 Effects by Store / Family / Demand Regime

The first heterogeneity analysis remains descriptive and highly interpretable.

Rather than immediately fitting another machine-learning model, the primary
AIPW score is averaged across pre-defined commercial segments.

The analysis considers:

- product family;
- store;
- historical demand regime;
- historical promotion regime.

These subgroup estimates answer questions such as:

> Is the estimated promotion effect systematically larger for high-demand
> store-family-date contexts?

and:

> Do some product families exhibit substantially stronger promotion response
> than others?

Only groups with sufficient observations are interpreted.

Differences across subgroup estimates are treated as evidence of effect
heterogeneity rather than exact individual-level treatment effects.

In [21]:
# ---------------------------------------------------------------------
# Interpretable heterogeneous-treatment-effect diagnostics.
#
# The cross-fitted primary AIPW score is averaged across predefined
# commercial subgroups.
#
# IMPORTANT:
# - Individual AIPW scores are noisy pseudo-outcomes.
# - Group averages are interpreted, not individual scores.
# - Segmentation variables are strictly pre-treatment covariates.
# - Current sales and current promotion intensity are not used to define
#   causal subgroups.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


PRIMARY_CAUSAL_SCORE_COLUMN = (
    "aipw_score"
)


# =====================================================================
# 1. PRE-TREATMENT DEMAND REGIME
# =====================================================================
#
# Historical demand is continuous enough for standard quintile-based
# segmentation.
#
# Quantile categories are created first without fixed labels so that the
# code remains robust if duplicated empirical quantile boundaries ever
# reduce the number of available bins.
# =====================================================================

demand_regime_raw = pd.qcut(
    causal_population[
        "sales_mean_28d_pre14"
    ],
    q=5,
    duplicates="drop",
)


n_demand_bins = (
    demand_regime_raw
    .cat
    .categories
    .size
)


if n_demand_bins == 1:

    demand_labels = [
        "R1 only regime",
    ]

else:

    demand_labels = [
        (
            "R1 lowest"
            if regime_index == 0
            else
            (
                f"R{regime_index + 1} highest"
                if regime_index
                == n_demand_bins - 1
                else
                f"R{regime_index + 1}"
            )
        )
        for regime_index
        in range(
            n_demand_bins
        )
    ]


causal_population[
    "demand_regime"
] = (
    demand_regime_raw
    .cat
    .rename_categories(
        demand_labels
    )
)


# =====================================================================
# 2. PRE-TREATMENT HISTORICAL PROMOTION REGIME
# =====================================================================
#
# promo_rate_28d_pre14 has substantial mass at exactly zero and is also
# discrete because it is constructed from a finite historical window.
#
# A direct five-way qcut would therefore either:
#
#   - fail because several quantile boundaries coincide; or
#   - require arbitrary splitting of identical values.
#
# We instead preserve zero as a meaningful business state:
#
#   "No recent promo"
#
# and divide strictly positive historical promotion rates into four
# empirical quantile regimes.
# =====================================================================

historical_promo_rate = (
    causal_population[
        "promo_rate_28d_pre14"
    ]
)


positive_promo_mask = (
    historical_promo_rate
    > 0
)


positive_promo_regime_raw = pd.qcut(
    historical_promo_rate.loc[
        positive_promo_mask
    ],
    q=4,
    duplicates="drop",
)


n_positive_promo_bins = (
    positive_promo_regime_raw
    .cat
    .categories
    .size
)


if n_positive_promo_bins == 1:

    positive_promo_labels = [
        "Positive promo",
    ]

else:

    positive_promo_labels = [
        (
            "Positive Q1 lowest"
            if regime_index == 0
            else
            (
                f"Positive Q{regime_index + 1} highest"
                if regime_index
                == n_positive_promo_bins - 1
                else
                f"Positive Q{regime_index + 1}"
            )
        )
        for regime_index
        in range(
            n_positive_promo_bins
        )
    ]


positive_promo_regime = (
    positive_promo_regime_raw
    .cat
    .rename_categories(
        positive_promo_labels
    )
)


historical_promo_regime = pd.Series(
    "No recent promo",
    index=
        causal_population.index,
    dtype="string",
)


historical_promo_regime.loc[
    positive_promo_mask
] = (
    positive_promo_regime
    .astype(
        "string"
    )
)


historical_promo_regime.loc[
    historical_promo_rate.isna()
] = pd.NA


promo_category_order = [
    "No recent promo",
    *positive_promo_labels,
]


causal_population[
    "historical_promo_regime"
] = pd.Categorical(
    historical_promo_regime,
    categories=
        promo_category_order,
    ordered=True,
)


# =====================================================================
# 3. SEGMENT-CONSTRUCTION DIAGNOSTICS
# =====================================================================

print(
    "=" * 80
)

print(
    "PRE-TREATMENT HETEROGENEITY REGIMES"
)

print(
    "=" * 80
)


print(
    "Demand regimes created:",
    n_demand_bins,
)


print(
    "Positive-promotion regimes created:",
    n_positive_promo_bins,
)


print()
print(
    "Demand-regime row counts:"
)


display(
    causal_population[
        "demand_regime"
    ]
    .value_counts(
        sort=False,
        dropna=False,
    )
    .rename(
        "rows"
    )
    .to_frame()
    .reset_index()
    .style.hide(
        axis="index"
    )
)


print()
print(
    "Historical-promotion-regime row counts:"
)


display(
    causal_population[
        "historical_promo_regime"
    ]
    .value_counts(
        sort=False,
        dropna=False,
    )
    .rename(
        "rows"
    )
    .to_frame()
    .reset_index()
    .style.hide(
        axis="index"
    )
)


# =====================================================================
# 4. REUSABLE GROUP-EFFECT SUMMARY
# =====================================================================

def summarize_group_effects(
    frame,
    group_column,
    minimum_rows=1000,
):
    """
    Summarize group-average primary AIPW scores.

    The estimated_effect column is the average cross-fitted AIPW
    pseudo-outcome within the specified subgroup.

    Standard errors below are naive row-level descriptive quantities.
    They do NOT replace the clustered causal inference used for the
    headline ATE in Section 7.
    """

    grouped = (
        frame
        .groupby(
            group_column,
            observed=True,
        )
        .agg(
            rows=(
                PRIMARY_CAUSAL_SCORE_COLUMN,
                "size",
            ),

            treated_rows=(
                "treatment",
                "sum",
            ),

            treatment_rate=(
                "treatment",
                "mean",
            ),

            mean_actual_sales=(
                "target_sales",
                "mean",
            ),

            mean_propensity=(
                "propensity_score",
                "mean",
            ),

            estimated_effect=(
                PRIMARY_CAUSAL_SCORE_COLUMN,
                "mean",
            ),

            effect_std=(
                PRIMARY_CAUSAL_SCORE_COLUMN,
                "std",
            ),
        )
        .reset_index()
    )


    grouped[
        "naive_effect_se"
    ] = (
        grouped[
            "effect_std"
        ]
        /
        np.sqrt(
            grouped[
                "rows"
            ]
        )
    )


    grouped[
        "large_enough"
    ] = (
        grouped[
            "rows"
        ]
        >= minimum_rows
    )


    return grouped


# =====================================================================
# 5. FAMILY-LEVEL EFFECTS
# =====================================================================

family_effects = summarize_group_effects(
    causal_population,
    "family",
    minimum_rows=5000,
)


family_effects = (
    family_effects
    .sort_values(
        "estimated_effect",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print()
print(
    "=" * 80
)

print(
    "ESTIMATED EFFECT BY PRODUCT FAMILY"
)

print(
    "=" * 80
)


display(
    family_effects
    .style.hide(
        axis="index"
    )
    .format(
        {
            "treatment_rate":
                "{:.2%}",

            "mean_actual_sales":
                "{:,.2f}",

            "mean_propensity":
                "{:.3f}",

            "estimated_effect":
                "{:+,.2f}",

            "effect_std":
                "{:,.2f}",

            "naive_effect_se":
                "{:,.2f}",
        }
    )
)


# =====================================================================
# 6. STORE-LEVEL EFFECTS
# =====================================================================

store_effects = summarize_group_effects(
    causal_population,
    "store_nbr",
    minimum_rows=5000,
)


store_effects = (
    store_effects
    .sort_values(
        "estimated_effect",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print()
print(
    "=" * 80
)

print(
    "ESTIMATED EFFECT BY STORE"
)

print(
    "=" * 80
)


display(
    store_effects
    .style.hide(
        axis="index"
    )
    .format(
        {
            "treatment_rate":
                "{:.2%}",

            "mean_actual_sales":
                "{:,.2f}",

            "mean_propensity":
                "{:.3f}",

            "estimated_effect":
                "{:+,.2f}",

            "effect_std":
                "{:,.2f}",

            "naive_effect_se":
                "{:,.2f}",
        }
    )
)


# =====================================================================
# 7. DEMAND-REGIME EFFECTS
# =====================================================================

demand_regime_effects = summarize_group_effects(
    causal_population,
    "demand_regime",
    minimum_rows=1000,
)


print()
print(
    "=" * 80
)

print(
    "ESTIMATED EFFECT BY PRE-TREATMENT DEMAND REGIME"
)

print(
    "=" * 80
)


display(
    demand_regime_effects
    .style.hide(
        axis="index"
    )
    .format(
        {
            "treatment_rate":
                "{:.2%}",

            "mean_actual_sales":
                "{:,.2f}",

            "mean_propensity":
                "{:.3f}",

            "estimated_effect":
                "{:+,.2f}",

            "effect_std":
                "{:,.2f}",

            "naive_effect_se":
                "{:,.2f}",
        }
    )
)


# =====================================================================
# 8. HISTORICAL-PROMOTION-REGIME EFFECTS
# =====================================================================

historical_promo_effects = summarize_group_effects(
    causal_population,
    "historical_promo_regime",
    minimum_rows=1000,
)


print()
print(
    "=" * 80
)

print(
    "ESTIMATED EFFECT BY HISTORICAL PROMOTION REGIME"
)

print(
    "=" * 80
)


display(
    historical_promo_effects
    .style.hide(
        axis="index"
    )
    .format(
        {
            "treatment_rate":
                "{:.2%}",

            "mean_actual_sales":
                "{:,.2f}",

            "mean_propensity":
                "{:.3f}",

            "estimated_effect":
                "{:+,.2f}",

            "effect_std":
                "{:,.2f}",

            "naive_effect_se":
                "{:,.2f}",
        }
    )
)


print()
print(
    "INTERPRETABLE HETEROGENEITY DIAGNOSTICS: PASSED"
)

PRE-TREATMENT HETEROGENEITY REGIMES
Demand regimes created: 5
Positive-promotion regimes created: 4

Demand-regime row counts:


demand_regime,rows
R1 lowest,255489
R2,255485
R3,255227
R4,255373
R5 highest,255394



Historical-promotion-regime row counts:


historical_promo_regime,rows
No recent promo,463949
Positive Q1 lowest,217253
Positive Q2,201220
Positive Q3,211997
Positive Q4 highest,182549



ESTIMATED EFFECT BY PRODUCT FAMILY


family,rows,treated_rows,treatment_rate,mean_actual_sales,mean_propensity,estimated_effect,effect_std,naive_effect_se,large_enough
BOOKS,38696,0,0.00%,0.17,0.000,"+1,903.14",666.95,3.39,True
GROCERY I,38696,37670,97.35%,"4,521.26",0.982,"+1,763.26","30,184.87",153.45,True
BEVERAGES,38696,36932,95.44%,"3,345.94",0.963,"+1,074.40","24,141.40",122.72,True
CLEANING,38696,35394,91.47%,"1,208.40",0.944,+426.37,"8,531.28",43.37,True
DAIRY,38696,32775,84.70%,885.99,0.915,+376.62,"6,956.33",35.36,True
PRODUCE,38696,26101,67.45%,"2,331.84",0.705,+199.15,"6,105.32",31.04,True
PERSONAL CARE,38696,32035,82.79%,321.93,0.874,+128.74,"2,972.96",15.11,True
HOME CARE,38696,28683,74.12%,296.34,0.761,+91.29,"1,919.98",9.76,True
POULTRY,38696,11906,30.77%,396.15,0.373,+82.96,"2,978.79",15.14,True
MEATS,38696,13468,34.80%,364.11,0.352,+74.31,"2,684.53",13.65,True


... (3 outputs left out)

demand_regime,rows,treated_rows,treatment_rate,mean_actual_sales,mean_propensity,estimated_effect,effect_std,naive_effect_se,large_enough
R1 lowest,255489,4841,1.89%,4.81,0.023,+290.92,"3,145.77",6.22,True
R2,255485,24107,9.44%,6.57,0.113,+6.98,191.35,0.38,True
R3,255227,76396,29.93%,37.20,0.313,+8.25,267.96,0.53,True
R4,255373,151898,59.48%,216.73,0.642,+40.79,"1,147.65",2.27,True
R5 highest,255394,209573,82.06%,"2,051.07",0.860,+630.32,"15,699.72",31.07,True



ESTIMATED EFFECT BY HISTORICAL PROMOTION REGIME


historical_promo_regime,rows,treated_rows,treatment_rate,mean_actual_sales,mean_propensity,estimated_effect,effect_std,naive_effect_se,large_enough
No recent promo,463949,10502,2.26%,10.93,0.040,+165.94,"2,344.74",3.44,True
Positive Q1 lowest,217253,39803,18.32%,131.98,0.217,+19.94,971.41,2.08,True
Positive Q2,201220,81563,40.53%,337.19,0.460,+56.49,"1,685.93",3.76,True
Positive Q3,211997,160185,75.56%,810.40,0.775,+132.27,"7,201.69",15.64,True
Positive Q4 highest,182549,174762,95.73%,"1,743.02",0.962,+706.03,"16,801.99",39.33,True



INTERPRETABLE HETEROGENEITY DIAGNOSTICS: PASSED


#### Support-aware interpretation of subgroup effects

A subgroup AIPW mean is mathematically computable even when treatment
variation within that subgroup is extremely limited.

However, computability does not imply credible subgroup identification.

For example, a product family with no treated observations cannot provide
direct empirical information about its treated potential outcome within that
family. Any estimated treated outcome must rely heavily on extrapolation from
other covariate regions.

Subgroup heterogeneity is therefore interpreted only after examining:

- the number of treated observations;
- the number of control observations;
- the treatment rate;
- the share of observations lying inside the primary practical-overlap region.

The subgroup tables are exploratory heterogeneity diagnostics rather than
proof that every reported subgroup effect is independently identified.

In [22]:
# ---------------------------------------------------------------------
# Support diagnostics for heterogeneous-effect summaries.
#
# A large subgroup is not necessarily causally informative.
#
# For subgroup interpretation we additionally require:
#
#   - sufficient treated observations;
#   - sufficient control observations;
#   - substantial representation inside the empirical overlap region.
#
# The thresholds below are transparent descriptive guardrails, not
# identification theorems.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


SUBGROUP_MIN_TREATED = 500
SUBGROUP_MIN_CONTROL = 500

PRIMARY_OVERLAP_LOW = 0.01
PRIMARY_OVERLAP_HIGH = 0.99

MIN_SUBGROUP_OVERLAP_SHARE = 0.50


# =====================================================================
# 1. REUSABLE SUPPORT DIAGNOSTIC
# =====================================================================

def add_subgroup_support(
    effect_summary,
    frame,
    group_column,
):
    """
    Add treatment-variation and practical-overlap diagnostics to an
    existing subgroup-effect table.
    """

    support_frame = (
        frame[
            [
                group_column,
                "treatment",
                "propensity_score",
            ]
        ]
        .copy()
    )


    support_frame[
        "inside_primary_overlap"
    ] = (
        support_frame[
            "propensity_score"
        ]
        .between(
            PRIMARY_OVERLAP_LOW,
            PRIMARY_OVERLAP_HIGH,
            inclusive="both",
        )
    )


    support_summary = (
        support_frame
        .groupby(
            group_column,
            observed=True,
        )
        .agg(
            treated_rows_support=(
                "treatment",
                "sum",
            ),

            total_rows_support=(
                "treatment",
                "size",
            ),

            overlap_share=(
                "inside_primary_overlap",
                "mean",
            ),
        )
        .reset_index()
    )


    support_summary[
        "control_rows_support"
    ] = (
        support_summary[
            "total_rows_support"
        ]
        - support_summary[
            "treated_rows_support"
        ]
    )


    support_summary[
        "support_ok"
    ] = (
        (
            support_summary[
                "treated_rows_support"
            ]
            >= SUBGROUP_MIN_TREATED
        )
        &
        (
            support_summary[
                "control_rows_support"
            ]
            >= SUBGROUP_MIN_CONTROL
        )
        &
        (
            support_summary[
                "overlap_share"
            ]
            >= MIN_SUBGROUP_OVERLAP_SHARE
        )
    )


    result = (
        effect_summary
        .merge(
            support_summary[
                [
                    group_column,
                    "control_rows_support",
                    "overlap_share",
                    "support_ok",
                ]
            ],
            on=group_column,
            how="left",
            validate="one_to_one",
        )
    )


    return result


# =====================================================================
# 2. APPLY TO ALL HETEROGENEITY TABLES
# =====================================================================

family_effects = add_subgroup_support(
    family_effects,
    causal_population,
    "family",
)


store_effects = add_subgroup_support(
    store_effects,
    causal_population,
    "store_nbr",
)


demand_regime_effects = add_subgroup_support(
    demand_regime_effects,
    causal_population,
    "demand_regime",
)


historical_promo_effects = add_subgroup_support(
    historical_promo_effects,
    causal_population,
    "historical_promo_regime",
)


# =====================================================================
# 3. DISPLAY SUPPORT-AWARE SUMMARIES
# =====================================================================

def display_support_summary(
    frame,
    group_column,
    title,
):
    """
    Display the causal-effect estimate together with subgroup support.
    """

    print()
    print(
        "=" * 80
    )

    print(
        title
    )

    print(
        "=" * 80
    )


    display_columns = [
        group_column,
        "rows",
        "treated_rows",
        "control_rows_support",
        "treatment_rate",
        "mean_propensity",
        "overlap_share",
        "estimated_effect",
        "support_ok",
    ]


    display(
        frame[
            display_columns
        ]
        .style.hide(
            axis="index"
        )
        .format(
            {
                "treatment_rate":
                    "{:.2%}",

                "mean_propensity":
                    "{:.3f}",

                "overlap_share":
                    "{:.2%}",

                "estimated_effect":
                    "{:+,.2f}",
            }
        )
    )


display_support_summary(
    family_effects,
    "family",
    "SUPPORT-AWARE FAMILY EFFECTS",
)


display_support_summary(
    demand_regime_effects,
    "demand_regime",
    "SUPPORT-AWARE DEMAND-REGIME EFFECTS",
)


display_support_summary(
    historical_promo_effects,
    "historical_promo_regime",
    "SUPPORT-AWARE HISTORICAL-PROMOTION EFFECTS",
)


print()
print(
    "Supported family groups:",
    int(
        family_effects[
            "support_ok"
        ]
        .sum()
    ),
    "/",
    len(
        family_effects
    ),
)


print(
    "Supported stores:",
    int(
        store_effects[
            "support_ok"
        ]
        .sum()
    ),
    "/",
    len(
        store_effects
    ),
)


print(
    "Supported demand regimes:",
    int(
        demand_regime_effects[
            "support_ok"
        ]
        .sum()
    ),
    "/",
    len(
        demand_regime_effects
    ),
)


print(
    "Supported historical-promotion regimes:",
    int(
        historical_promo_effects[
            "support_ok"
        ]
        .sum()
    ),
    "/",
    len(
        historical_promo_effects
    ),
)


print()
print(
    "SUBGROUP SUPPORT DIAGNOSTICS: PASSED"
)


SUPPORT-AWARE FAMILY EFFECTS


family,rows,treated_rows,control_rows_support,treatment_rate,mean_propensity,overlap_share,estimated_effect,support_ok
BOOKS,38696,0,38696,0.00%,0.000,0.02%,"+1,903.14",False
GROCERY I,38696,37670,1026,97.35%,0.982,25.90%,"+1,763.26",False
BEVERAGES,38696,36932,1764,95.44%,0.963,37.17%,"+1,074.40",False
CLEANING,38696,35394,3302,91.47%,0.944,44.69%,+426.37,False
DAIRY,38696,32775,5921,84.70%,0.915,61.54%,+376.62,True
PRODUCE,38696,26101,12595,67.45%,0.705,96.38%,+199.15,True
PERSONAL CARE,38696,32035,6661,82.79%,0.874,64.48%,+128.74,True
HOME CARE,38696,28683,10013,74.12%,0.761,71.89%,+91.29,True
POULTRY,38696,11906,26790,30.77%,0.373,93.29%,+82.96,True
MEATS,38696,13468,25228,34.80%,0.352,97.86%,+74.31,True



SUPPORT-AWARE DEMAND-REGIME EFFECTS


demand_regime,rows,treated_rows,control_rows_support,treatment_rate,mean_propensity,overlap_share,estimated_effect,support_ok
R1 lowest,255489,4841,250648,1.89%,0.023,20.61%,+290.92,False
R2,255485,24107,231378,9.44%,0.113,63.79%,+6.98,True
R3,255227,76396,178831,29.93%,0.313,83.93%,+8.25,True
R4,255373,151898,103475,59.48%,0.642,86.44%,+40.79,True
R5 highest,255394,209573,45821,82.06%,0.860,59.90%,+630.32,True



SUPPORT-AWARE HISTORICAL-PROMOTION EFFECTS


historical_promo_regime,rows,treated_rows,control_rows_support,treatment_rate,mean_propensity,overlap_share,estimated_effect,support_ok
No recent promo,463949,10502,453447,2.26%,0.040,35.29%,+165.94,False
Positive Q1 lowest,217253,39803,177450,18.32%,0.217,82.92%,+19.94,True
Positive Q2,201220,81563,119657,40.53%,0.460,98.06%,+56.49,True
Positive Q3,211997,160185,51812,75.56%,0.775,89.70%,+132.27,True
Positive Q4 highest,182549,174762,7787,95.73%,0.962,39.53%,+706.03,False



Supported family groups: 22 / 33
Supported stores: 54 / 54
Supported demand regimes: 4 / 5
Supported historical-promotion regimes: 3 / 5

SUBGROUP SUPPORT DIAGNOSTICS: PASSED


#### Heterogeneity conclusion

The subgroup analysis reveals substantial apparent promotion-effect
heterogeneity, but the credibility of that heterogeneity varies strongly with
empirical treatment support.

Large subgroup size alone is not sufficient for causal interpretation.

Several of the most extreme family-level estimates occur in product families
with almost no treatment variation. For example, some families contain almost
exclusively treated or almost exclusively control observations.

Those effects rely heavily on extrapolation and are therefore not interpreted
as credible standalone subgroup treatment effects.

After applying transparent support diagnostics:

- 22 of 33 product families satisfy the subgroup support guardrails;
- all 54 stores satisfy them;
- 4 of 5 pre-treatment demand regimes satisfy them;
- 3 of 5 historical-promotion regimes satisfy them.

The least-supported regimes are generally those in which treatment assignment
is already nearly deterministic.

Among better-supported segments, the results still suggest economically
meaningful effect heterogeneity, particularly across stores and demand levels.

The analysis therefore supports modeling heterogeneous uplift, but only inside
regions with defensible empirical treatment overlap.

The next section estimates a regularized conditional treatment-effect model on
the practical-overlap population rather than extrapolating promotion effects
into unsupported regions.

### 8.2 Policy-Oriented Uplift

Subgroup averages provide interpretable evidence of treatment-effect
heterogeneity, but they impose coarse commercial partitions.

A conditional treatment-effect model instead targets:

$$
\tau(X)
=
E[Y(1)-Y(0)\mid X].
$$

The cross-fitted AIPW pseudo-outcome from Section 7 satisfies, under the causal
identification and nuisance-model assumptions:

$$
E[\psi_i\mid X_i=x]
=
\tau(x).
$$

This motivates a second-stage **DR-learner**:

$$
X_i
\longrightarrow
\psi_i.
$$

The resulting prediction:

$$
\hat\tau(X_i)
$$

is an estimate of the conditional average treatment effect associated with the
commercial context $X_i$.

It is not an identified individual treatment effect.

---

The conditional-effect model is deliberately restricted to the primary
practical-overlap population:

$$
0.01
\leq
\hat e(X)
\leq
0.99.
$$

This prevents the model from being interpreted in regions where one treatment
state is almost absent and the causal contrast would rely mainly on
extrapolation.

No conditional uplift estimate is reported outside this support region.

---

The DR learner is itself cross-fitted across the five retrospective folds.

For every held-out fold, the second-stage model is trained on the remaining
overlap observations and predicts conditional uplift for the held-out overlap
observations.

These out-of-fold predictions are used to evaluate whether the model can rank
commercial contexts by estimated promotion response.

The objective at this stage is not to define an optimal promotion policy.

Promotion cost, discount depth, margin and operational constraints are not
available.

The model therefore provides an **uplift-ranking signal in sales units**, not
a profit-maximizing decision rule.

In [23]:
# ---------------------------------------------------------------------
# Cross-fitted DR learner for conditional promotion uplift.
#
# Target:
#
#   psi_i = cross-fitted AIPW pseudo-outcome
#
# Under the causal assumptions:
#
#   E[psi_i | X_i=x] = tau(x)
#
# The model is trained and interpreted ONLY inside the primary
# practical-overlap region:
#
#   0.01 <= propensity_score <= 0.99
#
# No causal uplift prediction is reported outside this region.
#
# IMPORTANT:
# This remains a retrospective heterogeneity analysis.
#
# It is NOT the later operational V4 experiment, where an outer
# forecast-origin boundary T will ensure that every causal model is
# estimated only from information available by T.
# ---------------------------------------------------------------------

import gc
import time

import numpy as np
import pandas as pd

from xgboost import XGBRegressor


# =====================================================================
# 1. FREEZE CONDITIONAL-EFFECT MODEL CONTRACT
# =====================================================================

TAU_FEATURES = (
    CAUSAL_PROPENSITY_FEATURES.copy()
)


TAU_MODEL_PARAMS = {
    "objective":
        "reg:squarederror",

    "n_estimators":
        800,

    "max_depth":
        5,

    "learning_rate":
        0.03,

    "min_child_weight":
        50,

    "subsample":
        0.80,

    "colsample_bytree":
        0.80,

    "reg_lambda":
        20.0,

    "reg_alpha":
        1.0,

    "tree_method":
        "hist",

    "enable_categorical":
        True,

    "random_state":
        42,

    "n_jobs":
        -1,

    "verbosity":
        0,
}


def prepare_tau_X(
    frame,
):
    """
    Prepare the frozen pre-treatment covariates for the DR learner.
    """

    X = (
        frame[
            TAU_FEATURES
        ]
        .copy()
    )


    for (
        feature,
        categorical_dtype,
    ) in CAUSAL_CATEGORY_DTYPES.items():

        X[
            feature
        ] = (
            X[
                feature
            ]
            .astype(
                categorical_dtype
            )
        )


    return X


# =====================================================================
# 2. PRIMARY PRACTICAL-OVERLAP POPULATION
# =====================================================================

tau_overlap_mask = (
    causal_population[
        "propensity_score"
    ]
    .between(
        0.01,
        0.99,
        inclusive="both",
    )
    .to_numpy()
)


n_tau_overlap_rows = (
    tau_overlap_mask.sum()
)


print(
    "DR-learner overlap rows:",
    f"{n_tau_overlap_rows:,}",
)


print(
    "DR-learner overlap population share:",
    f"{tau_overlap_mask.mean():.2%}",
)


# No conditional treatment-effect estimate will be stored outside
# empirical support.

tau_hat_oof = np.full(
    len(
        causal_population
    ),
    np.nan,
    dtype="float64",
)


tau_fold_metrics = []


# =====================================================================
# 3. CROSS-FITTED SECOND-STAGE DR LEARNER
# =====================================================================

for fold_id in range(
    1,
    N_PROPENSITY_FOLDS + 1,
):

    fold_start_time = time.time()


    print(
        "\n"
        + "=" * 80
    )

    print(
        f"DR-learner fold "
        f"{fold_id}/"
        f"{N_PROPENSITY_FOLDS}"
    )


    fold_array = (
        causal_population[
            "propensity_fold"
        ]
        .to_numpy()
    )


    training_mask = (
        tau_overlap_mask
        &
        (
            fold_array
            != fold_id
        )
    )


    holdout_mask = (
        tau_overlap_mask
        &
        (
            fold_array
            == fold_id
        )
    )


    training_frame = (
        causal_population.loc[
            training_mask
        ]
    )


    holdout_frame = (
        causal_population.loc[
            holdout_mask
        ]
    )


    print(
        "Training overlap rows:",
        f"{len(training_frame):,}",
    )

    print(
        "Holdout overlap rows:",
        f"{len(holdout_frame):,}",
    )


    # -------------------------------------------------------------
    # Prepare X and the cross-fitted AIPW pseudo-outcome target.
    # -------------------------------------------------------------

    X_train = (
        prepare_tau_X(
            training_frame
        )
    )


    X_holdout = (
        prepare_tau_X(
            holdout_frame
        )
    )


    y_tau_train = (
        training_frame[
            "aipw_score"
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    y_tau_holdout = (
        holdout_frame[
            "aipw_score"
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    # -------------------------------------------------------------
    # Fit deliberately regularized second-stage regression.
    # -------------------------------------------------------------

    tau_model = XGBRegressor(
        **TAU_MODEL_PARAMS
    )


    tau_model.fit(
        X_train,
        y_tau_train,
    )


    tau_holdout = (
        tau_model
        .predict(
            X_holdout
        )
        .astype(
            "float64"
        )
    )


    tau_hat_oof[
        holdout_mask
    ] = (
        tau_holdout
    )


    # -------------------------------------------------------------
    # Descriptive second-stage diagnostics.
    #
    # The AIPW pseudo-outcome is intrinsically very noisy, so RMSE is
    # not interpreted like a standard forecasting metric.
    #
    # The more important downstream diagnostic is whether predicted
    # uplift ranks groups with progressively larger average DR scores.
    # -------------------------------------------------------------

    residual = (
        tau_holdout
        - y_tau_holdout
    )


    pseudo_mae = (
        np.abs(
            residual
        )
        .mean()
    )


    pseudo_rmse = np.sqrt(
        np.mean(
            residual ** 2
        )
    )


    prediction_mean = (
        tau_holdout.mean()
    )


    target_mean = (
        y_tau_holdout.mean()
    )


    fit_minutes = (
        time.time()
        - fold_start_time
    ) / 60.0


    tau_fold_metrics.append(
        {
            "fold":
                fold_id,

            "date_min":
                holdout_frame[
                    "date"
                ]
                .min(),

            "date_max":
                holdout_frame[
                    "date"
                ]
                .max(),

            "training_rows":
                len(
                    training_frame
                ),

            "holdout_rows":
                len(
                    holdout_frame
                ),

            "target_mean":
                target_mean,

            "prediction_mean":
                prediction_mean,

            "pseudo_mae":
                pseudo_mae,

            "pseudo_rmse":
                pseudo_rmse,

            "fit_minutes":
                fit_minutes,
        }
    )


    print(
        "Held-out mean DR score:",
        f"{target_mean:+.3f}",
    )

    print(
        "Held-out mean tau prediction:",
        f"{prediction_mean:+.3f}",
    )

    print(
        "Pseudo-outcome MAE:",
        f"{pseudo_mae:.3f}",
    )

    print(
        "Fit minutes:",
        f"{fit_minutes:.2f}",
    )


    del (
        training_frame,
        holdout_frame,
        X_train,
        X_holdout,
        y_tau_train,
        y_tau_holdout,
        tau_model,
        tau_holdout,
        residual,
    )


    gc.collect()


# =====================================================================
# 4. VALIDATE AND STORE OOF CONDITIONAL UPLIFT
# =====================================================================

assert np.isfinite(
    tau_hat_oof[
        tau_overlap_mask
    ]
).all()


assert np.isnan(
    tau_hat_oof[
        ~tau_overlap_mask
    ]
).all()


causal_population[
    "tau_hat_oof"
] = (
    tau_hat_oof
    .astype(
        "float32"
    )
)


tau_fold_metrics = pd.DataFrame(
    tau_fold_metrics
)


print()
print(
    "=" * 80
)

print(
    "CROSS-FITTED DR-LEARNER SUMMARY"
)

print(
    "=" * 80
)


display(
    tau_fold_metrics
    .style.hide(
        axis="index"
    )
    .format(
        {
            "target_mean":
                "{:+,.3f}",

            "prediction_mean":
                "{:+,.3f}",

            "pseudo_mae":
                "{:,.3f}",

            "pseudo_rmse":
                "{:,.3f}",

            "fit_minutes":
                "{:.2f}",
        }
    )
)


# =====================================================================
# 5. OUT-OF-FOLD UPLIFT RANKING
# =====================================================================
#
# If tau_hat contains useful heterogeneity signal, observations assigned
# to higher predicted-uplift groups should also exhibit progressively
# larger mean DR pseudo-outcomes.
#
# Quintiles are defined only over OOF predictions inside empirical
# support.
# =====================================================================

uplift_evaluation = (
    causal_population.loc[
        tau_overlap_mask,
        [
            "treatment",
            "target_sales",
            "aipw_score",
            "tau_hat_oof",
            "propensity_score",
        ],
    ]
    .copy()
)


uplift_evaluation[
    "predicted_uplift_quintile"
] = pd.qcut(
    uplift_evaluation[
        "tau_hat_oof"
    ],
    q=5,
    labels=[
        "Q1 lowest",
        "Q2",
        "Q3",
        "Q4",
        "Q5 highest",
    ],
    duplicates="drop",
)


uplift_quintile_summary = (
    uplift_evaluation
    .groupby(
        "predicted_uplift_quintile",
        observed=True,
    )
    .agg(
        rows=(
            "tau_hat_oof",
            "size",
        ),

        treatment_rate=(
            "treatment",
            "mean",
        ),

        mean_propensity=(
            "propensity_score",
            "mean",
        ),

        mean_predicted_uplift=(
            "tau_hat_oof",
            "mean",
        ),

        mean_dr_effect=(
            "aipw_score",
            "mean",
        ),

        median_predicted_uplift=(
            "tau_hat_oof",
            "median",
        ),

        mean_actual_sales=(
            "target_sales",
            "mean",
        ),
    )
    .reset_index()
)


print()
print(
    "=" * 80
)

print(
    "OUT-OF-FOLD UPLIFT RANKING BY PREDICTED QUINTILE"
)

print(
    "=" * 80
)


display(
    uplift_quintile_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "treatment_rate":
                "{:.2%}",

            "mean_propensity":
                "{:.3f}",

            "mean_predicted_uplift":
                "{:+,.2f}",

            "mean_dr_effect":
                "{:+,.2f}",

            "median_predicted_uplift":
                "{:+,.2f}",

            "mean_actual_sales":
                "{:,.2f}",
        }
    )
)


# =====================================================================
# 6. POLICY-ORIENTED TOP-SHARE DIAGNOSTIC
# =====================================================================
#
# This is NOT a profit policy evaluation.
#
# It asks only:
#
#   If commercial contexts were ranked by tau_hat, how large is the
#   average DR-estimated sales effect among the top-ranked contexts?
#
# No promotion cost or margin information is available.
# =====================================================================

uplift_evaluation = (
    uplift_evaluation
    .sort_values(
        "tau_hat_oof",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


policy_share_rows = []


for top_share in [
    0.10,
    0.20,
    0.40,
    0.60,
    0.80,
    1.00,
]:

    n_selected = int(
        np.ceil(
            len(
                uplift_evaluation
            )
            * top_share
        )
    )


    selected = (
        uplift_evaluation
        .iloc[
            :n_selected
        ]
    )


    policy_share_rows.append(
        {
            "top_share":
                top_share,

            "rows_selected":
                len(
                    selected
                ),

            "mean_predicted_uplift":
                selected[
                    "tau_hat_oof"
                ]
                .mean(),

            "mean_dr_effect":
                selected[
                    "aipw_score"
                ]
                .mean(),

            "treatment_rate":
                selected[
                    "treatment"
                ]
                .mean(),
        }
    )


policy_uplift_summary = pd.DataFrame(
    policy_share_rows
)


print()
print(
    "=" * 80
)

print(
    "TOP-RANKED UPLIFT DIAGNOSTIC"
)

print(
    "=" * 80
)


display(
    policy_uplift_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "top_share":
                "{:.0%}",

            "mean_predicted_uplift":
                "{:+,.2f}",

            "mean_dr_effect":
                "{:+,.2f}",

            "treatment_rate":
                "{:.2%}",
        }
    )
)


# =====================================================================
# 7. SIMPLE RANKING QUALITY SUMMARY
# =====================================================================

lowest_quintile_effect = (
    uplift_quintile_summary
    .iloc[
        0
    ][
        "mean_dr_effect"
    ]
)


highest_quintile_effect = (
    uplift_quintile_summary
    .iloc[
        -1
    ][
        "mean_dr_effect"
    ]
)


top_bottom_effect_spread = (
    highest_quintile_effect
    - lowest_quintile_effect
)


print()
print(
    "Top-minus-bottom DR-effect spread:",
    f"{top_bottom_effect_spread:+,.3f}",
)


print()
print(
    "CROSS-FITTED POLICY-ORIENTED UPLIFT MODEL: PASSED"
)

DR-learner overlap rows: 803,549
DR-learner overlap population share: 62.93%

DR-learner fold 1/5
Training overlap rows: 651,252
Holdout overlap rows: 152,297
Held-out mean DR score: +138.464
Held-out mean tau prediction: +22.772
Pseudo-outcome MAE: 721.485
Fit minutes: 0.11

DR-learner fold 2/5
Training overlap rows: 634,101
Holdout overlap rows: 169,448
Held-out mean DR score: +13.600
Held-out mean tau prediction: +177.795
Pseudo-outcome MAE: 452.631
Fit minutes: 0.11

DR-learner fold 3/5
Training overlap rows: 631,375
Holdout overlap rows: 172,174
Held-out mean DR score: +100.555
Held-out mean tau prediction: +45.201
Pseudo-outcome MAE: 264.374
Fit minutes: 0.11

DR-learner fold 4/5
Training overlap rows: 638,458
Holdout overlap rows: 165,091
Held-out mean DR score: +135.617
Held-out mean tau prediction: +142.207
Pseudo-outcome MAE: 417.772
Fit minutes: 0.11

DR-learner fold 5/5
Training overlap rows: 659,010
Holdout overlap rows: 144,539
Held-out mean DR score: +4.850
Held-out mean

fold,date_min,date_max,training_rows,holdout_rows,target_mean,prediction_mean,pseudo_mae,pseudo_rmse,fit_minutes
1,2015-08-16 00:00:00,2016-01-09 00:00:00,651252,152297,+138.464,+22.772,721.485,"5,245.852",0.11
2,2016-01-10 00:00:00,2016-06-03 00:00:00,634101,169448,+13.600,+177.795,452.631,"3,832.107",0.11
3,2016-06-04 00:00:00,2016-10-27 00:00:00,631375,172174,+100.555,+45.201,264.374,"3,203.958",0.11
4,2016-10-28 00:00:00,2017-03-23 00:00:00,638458,165091,+135.617,+142.207,417.772,"5,614.841",0.11
5,2017-03-24 00:00:00,2017-08-15 00:00:00,659010,144539,+4.850,+4.587,170.271,"1,413.462",0.11



OUT-OF-FOLD UPLIFT RANKING BY PREDICTED QUINTILE


predicted_uplift_quintile,rows,treatment_rate,mean_propensity,mean_predicted_uplift,mean_dr_effect,median_predicted_uplift,mean_actual_sales
Q1 lowest,160722,40.21%,0.434,-294.76,+162.48,-40.22,636.05
Q2,160718,31.95%,0.359,-6.45,+14.18,-6.19,78.90
Q3,160689,27.06%,0.314,+3.17,+10.87,+3.12,68.90
Q4,160710,41.75%,0.457,+12.83,+20.64,+11.75,133.98
Q5 highest,160710,63.34%,0.670,+692.91,+188.78,+86.15,903.87



TOP-RANKED UPLIFT DIAGNOSTIC


top_share,rows_selected,mean_predicted_uplift,mean_dr_effect,treatment_rate
10%,80355,"+1,337.72",+313.30,71.02%
20%,160710,+692.91,+188.78,63.34%
40%,321420,+352.87,+104.71,52.55%
60%,482130,+236.30,+73.42,44.05%
80%,642840,+175.61,+58.62,41.03%
100%,803549,+81.54,+79.39,40.86%



Top-minus-bottom DR-effect spread: +26.302

CROSS-FITTED POLICY-ORIENTED UPLIFT MODEL: PASSED


#### Fold-relative uplift ranking

The initial pooled ranking compares raw predictions produced by five different
second-stage models.

This comparison implicitly assumes that the conditional-effect predictions are
equally calibrated across cross-fitting folds.

The fold diagnostics show that this assumption does not hold: predicted and
observed mean DR effects differ substantially across several historical folds.

A raw prediction of the same numerical magnitude is therefore not necessarily
comparable across two separately fitted fold models.

For out-of-fold ranking evaluation, observations are consequently ranked
relative to other observations predicted by the same held-out-fold model.

Within each fold, conditional-effect predictions are converted into uplift
quintiles and percentile ranks.

The fold-relative ranks are then pooled across folds.

This evaluates the economically relevant question:

> Within a given historical regime and model fit, can the DR learner rank
> commercial contexts from relatively low to relatively high promotion
> response?

No model is refitted and no prediction is modified by this diagnostic.

In [24]:
# ---------------------------------------------------------------------
# Fold-relative evaluation of the cross-fitted DR learner.
#
# Raw OOF tau predictions are produced by five separately trained
# second-stage models and are not perfectly calibrated across folds.
#
# Therefore, ranking quality is evaluated WITHIN each held-out fold.
#
# This preserves the out-of-fold structure while preventing systematic
# differences in model calibration across folds from dominating the
# ranking diagnostic.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


# =====================================================================
# 1. BUILD OVERLAP EVALUATION FRAME
# =====================================================================

fold_relative_uplift = (
    causal_population.loc[
        tau_overlap_mask,
        [
            "propensity_fold",
            "treatment",
            "target_sales",
            "aipw_score",
            "tau_hat_oof",
            "propensity_score",
        ],
    ]
    .copy()
)


# =====================================================================
# 2. WITHIN-FOLD PERCENTILE RANK
# =====================================================================
#
# pct=True creates a rank between approximately 0 and 1 inside each
# held-out fold.
#
# A value near 1 means that the observation is among the largest
# predicted treatment effects produced by that fold's model.
# =====================================================================

fold_relative_uplift[
    "within_fold_uplift_rank"
] = (
    fold_relative_uplift
    .groupby(
        "propensity_fold",
        observed=True,
    )[
        "tau_hat_oof"
    ]
    .rank(
        method="average",
        pct=True,
    )
)


# =====================================================================
# 3. DEFINE COMMON WITHIN-FOLD QUINTILES
# =====================================================================

fold_relative_uplift[
    "within_fold_uplift_quintile"
] = pd.cut(
    fold_relative_uplift[
        "within_fold_uplift_rank"
    ],
    bins=[
        0.0,
        0.2,
        0.4,
        0.6,
        0.8,
        1.0,
    ],
    labels=[
        "Q1 lowest",
        "Q2",
        "Q3",
        "Q4",
        "Q5 highest",
    ],
    include_lowest=True,
)


# =====================================================================
# 4. POOLED FOLD-RELATIVE QUINTILE SUMMARY
# =====================================================================

fold_relative_quintile_summary = (
    fold_relative_uplift
    .groupby(
        "within_fold_uplift_quintile",
        observed=True,
    )
    .agg(
        rows=(
            "tau_hat_oof",
            "size",
        ),

        mean_within_fold_rank=(
            "within_fold_uplift_rank",
            "mean",
        ),

        treatment_rate=(
            "treatment",
            "mean",
        ),

        mean_propensity=(
            "propensity_score",
            "mean",
        ),

        mean_predicted_uplift=(
            "tau_hat_oof",
            "mean",
        ),

        mean_dr_effect=(
            "aipw_score",
            "mean",
        ),

        mean_actual_sales=(
            "target_sales",
            "mean",
        ),
    )
    .reset_index()
)


print(
    "=" * 80
)

print(
    "FOLD-RELATIVE OUT-OF-FOLD UPLIFT RANKING"
)

print(
    "=" * 80
)


display(
    fold_relative_quintile_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "mean_within_fold_rank":
                "{:.3f}",

            "treatment_rate":
                "{:.2%}",

            "mean_propensity":
                "{:.3f}",

            "mean_predicted_uplift":
                "{:+,.2f}",

            "mean_dr_effect":
                "{:+,.2f}",

            "mean_actual_sales":
                "{:,.2f}",
        }
    )
)


# =====================================================================
# 5. WITHIN-FOLD TOP-SHARE POLICY-ORIENTED DIAGNOSTIC
# =====================================================================
#
# Selection is performed separately inside each fold.
#
# Example:
#
#   top 10%
#
# means the top 10% predicted uplift observations from EACH held-out
# historical fold, not the globally largest raw predictions.
# =====================================================================

fold_relative_policy_rows = []


for top_share in [
    0.10,
    0.20,
    0.40,
    0.60,
    0.80,
    1.00,
]:

    rank_threshold = (
        1.0
        - top_share
    )


    selected_mask = (
        fold_relative_uplift[
            "within_fold_uplift_rank"
        ]
        > rank_threshold
    )


    selected = (
        fold_relative_uplift.loc[
            selected_mask
        ]
    )


    fold_relative_policy_rows.append(
        {
            "top_share_within_each_fold":
                top_share,

            "rows_selected":
                len(
                    selected
                ),

            "mean_predicted_uplift":
                selected[
                    "tau_hat_oof"
                ]
                .mean(),

            "mean_dr_effect":
                selected[
                    "aipw_score"
                ]
                .mean(),

            "treatment_rate":
                selected[
                    "treatment"
                ]
                .mean(),
        }
    )


fold_relative_policy_summary = pd.DataFrame(
    fold_relative_policy_rows
)


print()
print(
    "=" * 80
)

print(
    "FOLD-RELATIVE TOP-RANKED UPLIFT DIAGNOSTIC"
)

print(
    "=" * 80
)


display(
    fold_relative_policy_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "top_share_within_each_fold":
                "{:.0%}",

            "mean_predicted_uplift":
                "{:+,.2f}",

            "mean_dr_effect":
                "{:+,.2f}",

            "treatment_rate":
                "{:.2%}",
        }
    )
)


# =====================================================================
# 6. RANKING QUALITY INSIDE EACH FOLD
# =====================================================================
#
# Spearman correlation is computed between:
#
#   tau_hat_oof
#   aipw_score
#
# within each fold.
#
# Because individual pseudo-outcomes are extremely noisy, these
# correlations are expected to be modest. Their sign and consistency
# are more informative than their absolute magnitude.
# =====================================================================

fold_rank_rows = []


for fold_id in range(
    1,
    N_PROPENSITY_FOLDS + 1,
):

    fold_frame = (
        fold_relative_uplift.loc[
            fold_relative_uplift[
                "propensity_fold"
            ]
            .eq(
                fold_id
            )
        ]
    )


    spearman_correlation = (
        fold_frame[
            [
                "tau_hat_oof",
                "aipw_score",
            ]
        ]
        .corr(
            method="spearman"
        )
        .iloc[
            0,
            1
        ]
    )


    fold_rank_rows.append(
        {
            "fold":
                fold_id,

            "rows":
                len(
                    fold_frame
                ),

            "mean_dr_effect":
                fold_frame[
                    "aipw_score"
                ]
                .mean(),

            "mean_predicted_uplift":
                fold_frame[
                    "tau_hat_oof"
                ]
                .mean(),

            "spearman_tau_vs_dr":
                spearman_correlation,
        }
    )


fold_rank_diagnostics = pd.DataFrame(
    fold_rank_rows
)


print()
print(
    "=" * 80
)

print(
    "WITHIN-FOLD RANK CORRELATION"
)

print(
    "=" * 80
)


display(
    fold_rank_diagnostics
    .style.hide(
        axis="index"
    )
    .format(
        {
            "mean_dr_effect":
                "{:+,.2f}",

            "mean_predicted_uplift":
                "{:+,.2f}",

            "spearman_tau_vs_dr":
                "{:+.4f}",
        }
    )
)


# =====================================================================
# 7. TOP-VERSUS-BOTTOM EFFECT SPREAD
# =====================================================================

fold_relative_bottom_effect = (
    fold_relative_quintile_summary
    .iloc[
        0
    ][
        "mean_dr_effect"
    ]
)


fold_relative_top_effect = (
    fold_relative_quintile_summary
    .iloc[
        -1
    ][
        "mean_dr_effect"
    ]
)


fold_relative_spread = (
    fold_relative_top_effect
    - fold_relative_bottom_effect
)


print()
print(
    "Fold-relative top-minus-bottom DR-effect spread:",
    f"{fold_relative_spread:+,.3f}",
)


print()
print(
    "FOLD-RELATIVE UPLIFT EVALUATION: PASSED"
)

FOLD-RELATIVE OUT-OF-FOLD UPLIFT RANKING


within_fold_uplift_quintile,rows,mean_within_fold_rank,treatment_rate,mean_propensity,mean_predicted_uplift,mean_dr_effect,mean_actual_sales
Q1 lowest,160515,0.100,41.92%,0.450,-278.92,+155.56,599.93
Q2,160882,0.300,26.36%,0.304,-8.30,+8.66,62.30
Q3,160760,0.500,32.62%,0.371,-3.20,+12.00,96.02
Q4,160666,0.700,39.89%,0.447,+12.10,+27.37,173.12
Q5 highest,160726,0.900,63.55%,0.663,+685.60,+193.52,890.96



FOLD-RELATIVE TOP-RANKED UPLIFT DIAGNOSTIC


top_share_within_each_fold,rows_selected,mean_predicted_uplift,mean_dr_effect,treatment_rate
10%,80360,"+1,325.26",+310.64,73.49%
20%,160726,+685.60,+193.52,63.55%
40%,321392,+348.91,+110.46,51.72%
60%,482152,+231.51,+77.63,45.35%
80%,643034,+171.51,+60.38,40.60%
100%,803549,+81.54,+79.39,40.86%



WITHIN-FOLD RANK CORRELATION


fold,rows,mean_dr_effect,mean_predicted_uplift,spearman_tau_vs_dr
1,152297,+138.46,+22.77,+0.0156
2,169448,+13.60,+177.79,+0.0102
3,172174,+100.55,+45.20,+0.0424
4,165091,+135.62,+142.21,+0.0490
5,144539,+4.85,+4.59,+0.0633



Fold-relative top-minus-bottom DR-effect spread: +37.961

FOLD-RELATIVE UPLIFT EVALUATION: PASSED


#### Conditional-uplift model conclusion

The cross-fitted DR learner captures some heterogeneous promotion-response
signal, but it does not provide a sufficiently reliable global ranking of
conditional treatment effects.

The average DR pseudo-outcome in the practical-overlap population is
approximately:

$$
79.4
$$

sales units per store-family-date observation.

Commercial contexts ranked in the top 10% by predicted uplift have a much
larger average DR score:

$$
310.6,
$$

and the top 20% average approximately:

$$
193.5.
$$

This indicates that the model can isolate some high-response regions.

However, uplift ranking is not monotonic across predicted quintiles.

The pooled fold-relative DR effects are approximately:

$$
155.6,\quad
8.7,\quad
12.0,\quad
27.4,\quad
193.5
$$

from the lowest to the highest predicted-uplift quintile.

The lowest predicted-uplift quintile therefore also contains a large average
DR effect.

Within-fold Spearman correlations between predicted conditional uplift and the
individual DR pseudo-outcome are consistently positive but small, ranging from
approximately 0.01 to 0.06.

The extreme predicted-uplift groups also correspond to substantially larger
sales scales than the middle groups.

This suggests that the second-stage learner is partly interacting with the
strong heteroskedasticity and scale variation of the sales-unit pseudo-outcome,
rather than recovering a clean one-dimensional ordering of causal response.

The retrospective analysis therefore supports:

- a positive average promotion effect;
- meaningful treatment-effect heterogeneity across several supported
  commercial segments;
- some predictive signal in the upper uplift tail.

It does **not** support interpreting the fitted DR learner as a precise
individualized treatment-effect model or a production-ready promotion policy.

The conditional-effect model is retained as an exploratory causal modeling
component rather than a validated decision rule.

#### Relative causal uplift formulation

Absolute treatment effects in sales units are strongly influenced by the
economic scale of each store-family-date context.

For promotion analysis and MROI-oriented interpretation, a relative treatment
effect is often more meaningful.

Define:

$$
\mu_0(x)
=
E[Y(0)\mid X=x],
$$

$$
\mu_1(x)
=
E[Y(1)\mid X=x],
$$

and the conditional relative promotion uplift:

$$
\rho(x)
=
\frac{
\mu_1(x)-\mu_0(x)
}{
\mu_0(x)
}.
$$

Equivalently:

$$
\rho(x)
=
\frac{\mu_1(x)}{\mu_0(x)}-1.
$$

For example:

$$
\rho(x)=0.20
$$

means that expected sales under promotion are 20% higher than the
counterfactual expected sales without promotion for commercial contexts with
covariates $X=x$.

This multiplicative representation separates:

- baseline demand scale, $\mu_0(x)$;
- relative causal promotion response, $\rho(x)$.

Absolute incremental demand can then be recovered through:

$$
\tau(x)
=
\mu_0(x)\rho(x).
$$

This decomposition is particularly useful for downstream profitability
analysis because incremental units can subsequently be combined with price,
margin and promotion-cost information.

However, relative effects become unstable when the no-promotion baseline is
close to zero.

Before estimating a relative-uplift model, the empirical distribution of the
cross-fitted baseline prediction $\hat m_0(X)$ is therefore audited inside the
practical-overlap population.

In [25]:
# ---------------------------------------------------------------------
# Audit the cross-fitted no-promotion baseline before constructing a
# relative causal-uplift target.
#
# Relative uplift requires division by:
#
#   mu0(X)
#
# so contexts with near-zero baseline demand can generate extremely
# unstable or economically meaningless percentages.
#
# This cell does NOT choose a threshold yet.
# It only characterizes the empirical baseline distribution inside the
# already-defined practical-overlap population.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


# =====================================================================
# 1. RESTRICT TO THE PRIMARY PRACTICAL-OVERLAP POPULATION
# =====================================================================

relative_uplift_audit = (
    causal_population.loc[
        tau_overlap_mask,
        [
            "store_nbr",
            "family",
            "date",
            "treatment",
            "target_sales",
            "propensity_score",
            "m0_hat",
            "m1_hat",
            "aipw_phi0",
            "aipw_phi1",
            "aipw_score",
        ],
    ]
    .copy()
)


m0_overlap = (
    relative_uplift_audit[
        "m0_hat"
    ]
    .to_numpy(
        dtype="float64"
    )
)


# =====================================================================
# 2. BASELINE DISTRIBUTION
# =====================================================================

baseline_quantiles = [
    0.00,
    0.01,
    0.025,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.975,
    0.99,
    1.00,
]


baseline_distribution = pd.DataFrame(
    {
        "quantile":
            baseline_quantiles,

        "m0_hat":
            np.quantile(
                m0_overlap,
                baseline_quantiles,
            ),
    }
)


print(
    "=" * 80
)

print(
    "CROSS-FITTED NO-PROMOTION BASELINE DISTRIBUTION"
)

print(
    "=" * 80
)


display(
    baseline_distribution
    .style.hide(
        axis="index"
    )
    .format(
        {
            "quantile":
                "{:.1%}",

            "m0_hat":
                "{:,.4f}",
        }
    )
)


# =====================================================================
# 3. SHARE OF OBSERVATIONS BELOW CANDIDATE BASELINE LEVELS
# =====================================================================
#
# These are diagnostics only.
#
# No threshold is selected in this cell.
# =====================================================================

candidate_baseline_levels = [
    0.0,
    0.1,
    0.5,
    1.0,
    2.0,
    5.0,
    10.0,
    20.0,
    50.0,
    100.0,
]


baseline_threshold_rows = []


for baseline_level in candidate_baseline_levels:

    below_or_equal = (
        m0_overlap
        <= baseline_level
    )


    baseline_threshold_rows.append(
        {
            "baseline_level":
                baseline_level,

            "rows_at_or_below":
                int(
                    below_or_equal.sum()
                ),

            "share_at_or_below":
                below_or_equal.mean(),
        }
    )


baseline_threshold_summary = pd.DataFrame(
    baseline_threshold_rows
)


print()
print(
    "=" * 80
)

print(
    "SHARE OF OVERLAP POPULATION AT LOW BASELINE LEVELS"
)

print(
    "=" * 80
)


display(
    baseline_threshold_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "baseline_level":
                "{:,.1f}",

            "share_at_or_below":
                "{:.2%}",
        }
    )
)


# =====================================================================
# 4. RAW MODEL-BASED RELATIVE CONTRAST DIAGNOSTIC
# =====================================================================
#
# This is NOT yet the final relative causal estimator.
#
# It is useful only to visualize the instability introduced by dividing
# the model-based absolute contrast by small baseline predictions.
# =====================================================================

tiny_epsilon = 1e-12


raw_model_relative_uplift = (
    (
        relative_uplift_audit[
            "m1_hat"
        ]
        .to_numpy(
            dtype="float64"
        )
        - m0_overlap
    )
    /
    np.maximum(
        m0_overlap,
        tiny_epsilon,
    )
)


relative_uplift_audit[
    "raw_model_relative_uplift"
] = (
    raw_model_relative_uplift
)


relative_diagnostic_quantiles = [
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
]


relative_contrast_distribution = pd.DataFrame(
    {
        "quantile":
            relative_diagnostic_quantiles,

        "relative_uplift":
            np.quantile(
                raw_model_relative_uplift,
                relative_diagnostic_quantiles,
            ),
    }
)


print()
print(
    "=" * 80
)

print(
    "RAW MODEL-BASED RELATIVE CONTRAST DIAGNOSTIC"
)

print(
    "=" * 80
)


display(
    relative_contrast_distribution
    .style.hide(
        axis="index"
    )
    .format(
        {
            "quantile":
                "{:.1%}",

            "relative_uplift":
                "{:+,.2%}",
        }
    )
)


# =====================================================================
# 5. RELATIVE UPLIFT EXTREMES BY BASELINE DEMAND
# =====================================================================
#
# This table helps determine whether extreme percentage predictions are
# concentrated almost entirely in near-zero baseline contexts.
# =====================================================================

baseline_bands = pd.cut(
    relative_uplift_audit[
        "m0_hat"
    ],
    bins=[
        -np.inf,
        1.0,
        5.0,
        10.0,
        25.0,
        50.0,
        100.0,
        250.0,
        500.0,
        1000.0,
        np.inf,
    ],
    labels=[
        "<=1",
        "(1,5]",
        "(5,10]",
        "(10,25]",
        "(25,50]",
        "(50,100]",
        "(100,250]",
        "(250,500]",
        "(500,1000]",
        ">1000",
    ],
)


relative_uplift_audit[
    "baseline_band"
] = (
    baseline_bands
)


baseline_band_summary = (
    relative_uplift_audit
    .groupby(
        "baseline_band",
        observed=True,
    )
    .agg(
        rows=(
            "m0_hat",
            "size",
        ),

        mean_m0=(
            "m0_hat",
            "mean",
        ),

        median_m0=(
            "m0_hat",
            "median",
        ),

        mean_absolute_contrast=(
            "aipw_score",
            lambda values:
                np.mean(
                    np.abs(
                        values
                    )
                ),
        ),

        median_raw_relative_uplift=(
            "raw_model_relative_uplift",
            "median",
        ),

        p90_abs_raw_relative_uplift=(
            "raw_model_relative_uplift",
            lambda values:
                np.quantile(
                    np.abs(
                        values
                    ),
                    0.90,
                ),
        ),

        p99_abs_raw_relative_uplift=(
            "raw_model_relative_uplift",
            lambda values:
                np.quantile(
                    np.abs(
                        values
                    ),
                    0.99,
                ),
        ),
    )
    .reset_index()
)


baseline_band_summary[
    "population_share"
] = (
    baseline_band_summary[
        "rows"
    ]
    /
    len(
        relative_uplift_audit
    )
)


print()
print(
    "=" * 80
)

print(
    "RELATIVE-UPLIFT STABILITY BY BASELINE-DEMAND BAND"
)

print(
    "=" * 80
)


display(
    baseline_band_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "population_share":
                "{:.2%}",

            "mean_m0":
                "{:,.2f}",

            "median_m0":
                "{:,.2f}",

            "mean_absolute_contrast":
                "{:,.2f}",

            "median_raw_relative_uplift":
                "{:+,.2%}",

            "p90_abs_raw_relative_uplift":
                "{:,.2%}",

            "p99_abs_raw_relative_uplift":
                "{:,.2%}",
        }
    )
)


print()
print(
    "RELATIVE UPLIFT BASELINE AUDIT: PASSED"
)

CROSS-FITTED NO-PROMOTION BASELINE DISTRIBUTION


quantile,m0_hat
0.0%,0.0000
1.0%,0.0000
2.5%,0.4429
5.0%,1.2402
10.0%,3.1725
25.0%,9.9989
50.0%,60.4658
75.0%,284.8134
90.0%,924.7657
95.0%,"1,732.1462"



SHARE OF OVERLAP POPULATION AT LOW BASELINE LEVELS


baseline_level,rows_at_or_below,share_at_or_below
0.0,10436,1.30%
0.1,12114,1.51%
0.5,21587,2.69%
1.0,34279,4.27%
2.0,56891,7.08%
5.0,117826,14.66%
10.0,200906,25.00%
20.0,284956,35.46%
50.0,383319,47.70%
100.0,454103,56.51%



RAW MODEL-BASED RELATIVE CONTRAST DIAGNOSTIC


quantile,relative_uplift
1.0%,-100.00%
5.0%,-89.54%
10.0%,-31.75%
25.0%,-5.15%
50.0%,+9.89%
75.0%,+52.79%
90.0%,+259.41%
95.0%,+748.66%
99.0%,"+65,231.50%"



RELATIVE-UPLIFT STABILITY BY BASELINE-DEMAND BAND


baseline_band,rows,mean_m0,median_m0,mean_absolute_contrast,median_raw_relative_uplift,p90_abs_raw_relative_uplift,p99_abs_raw_relative_uplift,population_share
<=1,34279,0.36,0.33,56.20,"+2,524.28%","2,895,486,755,371,096.00%","25,369,727,905,273,456.00%",4.27%
"(1,5]",83547,2.96,2.96,23.35,+201.77%,"1,219.86%","4,426.40%",10.40%
"(5,10]",83080,7.30,7.19,23.38,+62.94%,354.97%,"1,102.41%",10.34%
"(10,25]",110372,16.32,15.77,30.29,+31.14%,172.15%,630.41%,13.74%
"(25,50]",72041,35.80,35.00,52.35,+19.71%,96.32%,342.97%,8.97%
"(50,100]",70784,72.81,71.74,79.63,+13.52%,60.42%,189.85%,8.81%
"(100,250]",129384,166.73,162.63,120.92,+7.24%,33.66%,100.00%,16.10%
"(250,500]",87191,353.23,342.54,223.90,+2.99%,23.26%,73.63%,10.85%
"(500,1000]",58183,702.84,678.11,458.65,+1.24%,23.40%,85.17%,7.24%
>1000,74688,"2,503.40","1,838.56","1,854.66",+1.65%,38.42%,104.45%,9.29%



RELATIVE UPLIFT BASELINE AUDIT: PASSED


The scale sensitivity observed in the absolute-uplift learner motivates the
relative treatment-effect formulation evaluated in Section 8.2 bis.

### 8.2 bis - Policy-Oriented Relative Uplift

For promotion targeting and downstream MROI analysis, relative causal response
is more interpretable across store-family contexts with very different demand
scales.

Define:

$$
\mu_0(x)
=
E[Y(0)\mid X=x],
$$

$$
\mu_1(x)
=
E[Y(1)\mid X=x],
$$

and:

$$
\rho(x)
=
\frac{\mu_1(x)}{\mu_0(x)}-1.
$$

A value:

$$
\rho(x)=0.20
$$

means that expected sales under promotion are 20% higher than expected
counterfactual sales without promotion for commercial contexts with
covariates $X=x$.

Absolute incremental demand remains:

$$
\tau(x)
=
\mu_0(x)\rho(x).
$$

---

An initial experiment attempted to learn the augmented potential-outcome
pseudo-outcomes directly.

That approach was unstable because the AIPW pseudo-outcomes are highly
heteroskedastic and can take extreme positive or negative values.

The final relative-uplift model therefore estimates the potential-outcome
means directly from observed non-negative sales.

A cross-fitted T-learner fits:

$$
\hat\mu_0(X)
$$

using control observations and:

$$
\hat\mu_1(X)
$$

using treated observations.

The conditional relative uplift is then:

$$
\hat\rho(X)
=
\frac{\hat\mu_1(X)}{\hat\mu_0(X)}-1.
$$

Tweedie regression is used for the outcome models because retail sales are
non-negative, strongly heteroskedastic and contain zero-demand observations.
Its positive mean specification is also naturally compatible with a
multiplicative relative-effect representation.

---

The model is trained only inside the practical propensity-overlap region:

$$
0.01
\leq
\hat e(X)
\leq
0.99.
$$

The denominator audit additionally showed severe relative-effect instability
for near-zero baseline demand.

Before inspecting relative-uplift model results, the primary interpretation
domain was therefore frozen at:

$$
\hat m_0(X)\geq10.
$$

This retains approximately 75% of the overlap population and is a numerical
interpretability guardrail rather than a promotion-policy rule.

---

The T-learner itself is an outcome-regression estimator rather than a doubly
robust estimator.

Its heterogeneous-uplift ranking is therefore evaluated against the
independent AIPW potential-outcome pseudo-outcomes constructed in Section 7.

For any predicted-uplift group $G$, the doubly robust evaluation target is:

$$
\widehat\rho_G^{DR}
=
\frac{
\frac{1}{|G|}
\sum_{i\in G}\phi_{1,i}
}{
\frac{1}{|G|}
\sum_{i\in G}\phi_{0,i}
}
-1.
$$

This is a ratio of group-level potential-outcome means, not an average of
individual ratios.

In [28]:
# ---------------------------------------------------------------------
# Cross-fitted relative-uplift T-learner.
#
# Two positive conditional-mean models are estimated directly from
# observed sales:
#
#   mu0(X) = E[Y | D=0, X]
#   mu1(X) = E[Y | D=1, X]
#
# Relative uplift is then:
#
#   rho(X) = mu1(X) / mu0(X) - 1
#
# This avoids learning the extremely noisy AIPW pseudo-outcomes at the
# individual-row level.
#
# The model is trained only inside practical propensity overlap.
#
# Its predictions are subsequently evaluated against independent
# group-level AIPW potential-outcome estimates.
# ---------------------------------------------------------------------

import gc
import time

import numpy as np
import pandas as pd

from xgboost import XGBRegressor


# =====================================================================
# 1. FREEZE MODEL CONTRACT
# =====================================================================

RELATIVE_OVERLAP_LOW = 0.01
RELATIVE_OVERLAP_HIGH = 0.99

RELATIVE_BASELINE_FLOOR = 10.0


RELATIVE_T_FEATURES = (
    CAUSAL_PROPENSITY_FEATURES.copy()
)


RELATIVE_T_MODEL_PARAMS = {
    "objective":
        "reg:tweedie",

    "tweedie_variance_power":
        1.5,

    "n_estimators":
        1000,

    "max_depth":
        6,

    "learning_rate":
        0.05,

    "min_child_weight":
        20,

    "subsample":
        0.80,

    "colsample_bytree":
        0.80,

    "reg_lambda":
        10.0,

    "reg_alpha":
        1.0,

    "tree_method":
        "hist",

    "enable_categorical":
        True,

    "random_state":
        42,

    "n_jobs":
        -1,

    "verbosity":
        0,
}


def prepare_relative_t_X(
    frame,
):
    """
    Prepare frozen pre-treatment covariates for relative-uplift
    outcome regression.
    """

    X = (
        frame[
            RELATIVE_T_FEATURES
        ]
        .copy()
    )


    for (
        feature,
        categorical_dtype,
    ) in CAUSAL_CATEGORY_DTYPES.items():

        X[
            feature
        ] = (
            X[
                feature
            ]
            .astype(
                categorical_dtype
            )
        )


    return X


# =====================================================================
# 2. TRAINING AND INTERPRETATION POPULATIONS
# =====================================================================

relative_overlap_mask = (
    causal_population[
        "propensity_score"
    ]
    .between(
        RELATIVE_OVERLAP_LOW,
        RELATIVE_OVERLAP_HIGH,
        inclusive="both",
    )
    .to_numpy()
)


relative_baseline_domain_mask = (
    causal_population[
        "m0_hat"
    ]
    .ge(
        RELATIVE_BASELINE_FLOOR
    )
    .to_numpy()
)


primary_relative_domain_mask = (
    relative_overlap_mask
    &
    relative_baseline_domain_mask
)


print(
    "Relative T-learner overlap rows:",
    f"{relative_overlap_mask.sum():,}",
)


print(
    "Relative T-learner overlap share:",
    f"{relative_overlap_mask.mean():.2%}",
)


print(
    "Primary relative interpretation rows:",
    f"{primary_relative_domain_mask.sum():,}",
)


print(
    "Share of overlap retained:",
    f"{primary_relative_domain_mask.sum() / relative_overlap_mask.sum():.2%}",
)


# =====================================================================
# 3. STORAGE
# =====================================================================

n_causal_rows = len(
    causal_population
)


relative_mu0_oof = np.full(
    n_causal_rows,
    np.nan,
    dtype="float64",
)


relative_mu1_oof = np.full(
    n_causal_rows,
    np.nan,
    dtype="float64",
)


relative_t_fold_metrics = []


fold_array = (
    causal_population[
        "propensity_fold"
    ]
    .to_numpy()
)


# =====================================================================
# 4. CROSS-FITTED T-LEARNER
# =====================================================================

for fold_id in range(
    1,
    N_PROPENSITY_FOLDS + 1,
):

    start_time = time.time()


    print(
        "\n"
        + "=" * 80
    )

    print(
        f"Relative T-learner fold "
        f"{fold_id}/"
        f"{N_PROPENSITY_FOLDS}"
    )


    training_mask = (
        relative_overlap_mask
        &
        (
            fold_array
            != fold_id
        )
    )


    holdout_mask = (
        relative_overlap_mask
        &
        (
            fold_array
            == fold_id
        )
    )


    training_frame = (
        causal_population.loc[
            training_mask
        ]
    )


    holdout_frame = (
        causal_population.loc[
            holdout_mask
        ]
    )


    control_training_frame = (
        training_frame.loc[
            training_frame[
                "treatment"
            ]
            .eq(
                0
            )
        ]
    )


    treated_training_frame = (
        training_frame.loc[
            training_frame[
                "treatment"
            ]
            .eq(
                1
            )
        ]
    )


    print(
        "Control training rows:",
        f"{len(control_training_frame):,}",
    )


    print(
        "Treated training rows:",
        f"{len(treated_training_frame):,}",
    )


    print(
        "Holdout rows:",
        f"{len(holdout_frame):,}",
    )


    X_holdout = (
        prepare_relative_t_X(
            holdout_frame
        )
    )


    # -------------------------------------------------------------
    # No-promotion potential-outcome model.
    # -------------------------------------------------------------

    X_control_train = (
        prepare_relative_t_X(
            control_training_frame
        )
    )


    y_control_train = (
        control_training_frame[
            "target_sales"
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    mu0_model = XGBRegressor(
        **RELATIVE_T_MODEL_PARAMS
    )


    mu0_model.fit(
        X_control_train,
        y_control_train,
    )


    mu0_holdout = (
        mu0_model
        .predict(
            X_holdout
        )
        .astype(
            "float64"
        )
    )


    # -------------------------------------------------------------
    # Promotion potential-outcome model.
    # -------------------------------------------------------------

    X_treated_train = (
        prepare_relative_t_X(
            treated_training_frame
        )
    )


    y_treated_train = (
        treated_training_frame[
            "target_sales"
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    mu1_model = XGBRegressor(
        **RELATIVE_T_MODEL_PARAMS
    )


    mu1_model.fit(
        X_treated_train,
        y_treated_train,
    )


    mu1_holdout = (
        mu1_model
        .predict(
            X_holdout
        )
        .astype(
            "float64"
        )
    )


    # Tweedie mean predictions should be strictly positive.
    assert (
        mu0_holdout
        > 0
    ).all()


    assert (
        mu1_holdout
        > 0
    ).all()


    relative_mu0_oof[
        holdout_mask
    ] = (
        mu0_holdout
    )


    relative_mu1_oof[
        holdout_mask
    ] = (
        mu1_holdout
    )


    # -------------------------------------------------------------
    # Factual diagnostics.
    # -------------------------------------------------------------

    holdout_treatment = (
        holdout_frame[
            "treatment"
        ]
        .to_numpy(
            dtype="int8"
        )
    )


    holdout_sales = (
        holdout_frame[
            "target_sales"
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    control_holdout_mask = (
        holdout_treatment
        == 0
    )


    treated_holdout_mask = (
        holdout_treatment
        == 1
    )


    factual_prediction = np.where(
        treated_holdout_mask,
        mu1_holdout,
        mu0_holdout,
    )


    factual_wape = (
        np.abs(
            factual_prediction
            - holdout_sales
        )
        .sum()
        /
        holdout_sales.sum()
    )


    factual_bias = (
        factual_prediction
        - holdout_sales
    ).sum() / holdout_sales.sum()


    # -------------------------------------------------------------
    # Fold-level causal calibration against AIPW group means.
    # -------------------------------------------------------------

    target_mu0 = (
        holdout_frame[
            "aipw_phi0"
        ]
        .mean()
    )


    target_mu1 = (
        holdout_frame[
            "aipw_phi1"
        ]
        .mean()
    )


    predicted_mu0 = (
        mu0_holdout.mean()
    )


    predicted_mu1 = (
        mu1_holdout.mean()
    )


    target_relative_uplift = (
        target_mu1
        / target_mu0
        - 1.0
    )


    predicted_relative_uplift = (
        predicted_mu1
        / predicted_mu0
        - 1.0
    )


    relative_t_fold_metrics.append(
        {
            "fold":
                fold_id,

            "date_min":
                holdout_frame[
                    "date"
                ]
                .min(),

            "date_max":
                holdout_frame[
                    "date"
                ]
                .max(),

            "training_rows":
                len(
                    training_frame
                ),

            "holdout_rows":
                len(
                    holdout_frame
                ),

            "target_mu0":
                target_mu0,

            "predicted_mu0":
                predicted_mu0,

            "target_mu1":
                target_mu1,

            "predicted_mu1":
                predicted_mu1,

            "target_relative_uplift":
                target_relative_uplift,

            "predicted_relative_uplift":
                predicted_relative_uplift,

            "factual_wape":
                factual_wape,

            "factual_bias":
                factual_bias,

            "fit_minutes":
                (
                    time.time()
                    - start_time
                )
                / 60.0,
        }
    )


    print(
        "Target DR relative uplift:",
        f"{target_relative_uplift:+.2%}",
    )


    print(
        "Predicted relative uplift:",
        f"{predicted_relative_uplift:+.2%}",
    )


    print(
        "Factual WAPE:",
        f"{factual_wape:.2%}",
    )


    del (
        training_frame,
        holdout_frame,
        control_training_frame,
        treated_training_frame,
        X_holdout,
        X_control_train,
        X_treated_train,
        y_control_train,
        y_treated_train,
        mu0_model,
        mu1_model,
        mu0_holdout,
        mu1_holdout,
        holdout_treatment,
        holdout_sales,
        factual_prediction,
    )


    gc.collect()


# =====================================================================
# 5. VALIDATE AND STORE
# =====================================================================

assert np.isfinite(
    relative_mu0_oof[
        relative_overlap_mask
    ]
).all()


assert np.isfinite(
    relative_mu1_oof[
        relative_overlap_mask
    ]
).all()


causal_population[
    "relative_mu0_oof"
] = (
    relative_mu0_oof
    .astype(
        "float32"
    )
)


causal_population[
    "relative_mu1_oof"
] = (
    relative_mu1_oof
    .astype(
        "float32"
    )
)


relative_uplift_hat_oof = np.full(
    n_causal_rows,
    np.nan,
    dtype="float64",
)


relative_uplift_hat_oof[
    primary_relative_domain_mask
] = (
    relative_mu1_oof[
        primary_relative_domain_mask
    ]
    /
    relative_mu0_oof[
        primary_relative_domain_mask
    ]
    - 1.0
)


causal_population[
    "relative_uplift_hat_oof"
] = (
    relative_uplift_hat_oof
    .astype(
        "float32"
    )
)


relative_t_fold_metrics = pd.DataFrame(
    relative_t_fold_metrics
)


print()
print(
    "=" * 80
)

print(
    "CROSS-FITTED RELATIVE T-LEARNER SUMMARY"
)

print(
    "=" * 80
)


display(
    relative_t_fold_metrics
    .style.hide(
        axis="index"
    )
    .format(
        {
            "target_mu0":
                "{:,.3f}",

            "predicted_mu0":
                "{:,.3f}",

            "target_mu1":
                "{:,.3f}",

            "predicted_mu1":
                "{:,.3f}",

            "target_relative_uplift":
                "{:+.2%}",

            "predicted_relative_uplift":
                "{:+.2%}",

            "factual_wape":
                "{:.2%}",

            "factual_bias":
                "{:+.2%}",

            "fit_minutes":
                "{:.2f}",
        }
    )
)


print()
print(
    "RELATIVE T-LEARNER: PASSED"
)

Relative T-learner overlap rows: 803,549
Relative T-learner overlap share: 62.93%
Primary relative interpretation rows: 602,643
Share of overlap retained: 75.00%

Relative T-learner fold 1/5
Control training rows: 375,004
Treated training rows: 276,248
Holdout rows: 152,297
Target DR relative uplift: +32.05%
Predicted relative uplift: -8.61%
Factual WAPE: 25.00%

Relative T-learner fold 2/5
Control training rows: 365,150
Treated training rows: 268,951
Holdout rows: 169,448
Target DR relative uplift: +3.17%
Predicted relative uplift: +19.82%
Factual WAPE: 19.22%

Relative T-learner fold 3/5
Control training rows: 371,848
Treated training rows: 259,527
Holdout rows: 172,174
Target DR relative uplift: +66.57%
Predicted relative uplift: +11.18%
Factual WAPE: 15.66%

Relative T-learner fold 4/5
Control training rows: 399,057
Treated training rows: 239,401
Holdout rows: 165,091
Target DR relative uplift: +61.62%
Predicted relative uplift: +32.46%
Factual WAPE: 17.70%

Relative T-learner fold

fold,date_min,date_max,training_rows,holdout_rows,target_mu0,predicted_mu0,target_mu1,predicted_mu1,target_relative_uplift,predicted_relative_uplift,factual_wape,factual_bias,fit_minutes
1,2015-08-16 00:00:00,2016-01-09 00:00:00,651252,152297,432.014,690.293,570.478,630.874,+32.05%,-8.61%,25.00%,+13.80%,0.33
2,2016-01-10 00:00:00,2016-06-03 00:00:00,634101,169448,429.145,381.824,442.745,457.518,+3.17%,+19.82%,19.22%,+1.66%,0.31
3,2016-06-04 00:00:00,2016-10-27 00:00:00,631375,172174,151.042,232.087,251.597,258.026,+66.57%,+11.18%,15.66%,+2.93%,0.32
4,2016-10-28 00:00:00,2017-03-23 00:00:00,638458,165091,220.099,256.939,355.716,340.335,+61.62%,+32.46%,17.70%,-3.59%,0.32
5,2017-03-24 00:00:00,2017-08-15 00:00:00,659010,144539,224.119,162.779,228.968,204.584,+2.16%,+25.68%,18.00%,-10.45%,0.32



RELATIVE T-LEARNER: PASSED


In [29]:
# ---------------------------------------------------------------------
# Group-level causal evaluation of the cross-fitted relative T-learner.
#
# Relative predictions are ranked WITHIN each held-out fold.
#
# The external causal evaluation target for every group is:
#
#   mean(phi1) / mean(phi0) - 1
#
# using the AIPW pseudo-outcomes constructed independently in Section 7.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


relative_eval = (
    causal_population.loc[
        primary_relative_domain_mask,
        [
            "propensity_fold",
            "treatment",
            "target_sales",
            "m0_hat",
            "aipw_phi0",
            "aipw_phi1",
            "relative_mu0_oof",
            "relative_mu1_oof",
            "relative_uplift_hat_oof",
        ],
    ]
    .copy()
)


# =====================================================================
# 1. WITHIN-FOLD RANK
# =====================================================================

relative_eval[
    "relative_rank"
] = (
    relative_eval
    .groupby(
        "propensity_fold",
        observed=True,
    )[
        "relative_uplift_hat_oof"
    ]
    .rank(
        method="average",
        pct=True,
    )
)


relative_eval[
    "relative_quintile"
] = pd.cut(
    relative_eval[
        "relative_rank"
    ],
    bins=[
        0.0,
        0.2,
        0.4,
        0.6,
        0.8,
        1.0,
    ],
    labels=[
        "Q1 lowest",
        "Q2",
        "Q3",
        "Q4",
        "Q5 highest",
    ],
    include_lowest=True,
)


# =====================================================================
# 2. QUINTILE-LEVEL EVALUATION
# =====================================================================

relative_quintile_summary = (
    relative_eval
    .groupby(
        "relative_quintile",
        observed=True,
    )
    .agg(
        rows=(
            "relative_uplift_hat_oof",
            "size",
        ),

        treatment_rate=(
            "treatment",
            "mean",
        ),

        mean_baseline=(
            "m0_hat",
            "mean",
        ),

        mean_predicted_mu0=(
            "relative_mu0_oof",
            "mean",
        ),

        mean_predicted_mu1=(
            "relative_mu1_oof",
            "mean",
        ),

        median_predicted_relative_uplift=(
            "relative_uplift_hat_oof",
            "median",
        ),

        mean_dr_phi0=(
            "aipw_phi0",
            "mean",
        ),

        mean_dr_phi1=(
            "aipw_phi1",
            "mean",
        ),

        mean_actual_sales=(
            "target_sales",
            "mean",
        ),
    )
    .reset_index()
)


relative_quintile_summary[
    "predicted_group_relative_uplift"
] = (
    relative_quintile_summary[
        "mean_predicted_mu1"
    ]
    /
    relative_quintile_summary[
        "mean_predicted_mu0"
    ]
    - 1.0
)


relative_quintile_summary[
    "dr_group_relative_uplift"
] = (
    relative_quintile_summary[
        "mean_dr_phi1"
    ]
    /
    relative_quintile_summary[
        "mean_dr_phi0"
    ]
    - 1.0
)


print(
    "=" * 80
)

print(
    "RELATIVE-UPLIFT QUINTILE EVALUATION"
)

print(
    "=" * 80
)


display(
    relative_quintile_summary[
        [
            "relative_quintile",
            "rows",
            "treatment_rate",
            "mean_baseline",
            "median_predicted_relative_uplift",
            "predicted_group_relative_uplift",
            "dr_group_relative_uplift",
            "mean_actual_sales",
        ]
    ]
    .style.hide(
        axis="index"
    )
    .format(
        {
            "treatment_rate":
                "{:.2%}",

            "mean_baseline":
                "{:,.2f}",

            "median_predicted_relative_uplift":
                "{:+.2%}",

            "predicted_group_relative_uplift":
                "{:+.2%}",

            "dr_group_relative_uplift":
                "{:+.2%}",

            "mean_actual_sales":
                "{:,.2f}",
        }
    )
)


# =====================================================================
# 3. TOP-SHARE EVALUATION
# =====================================================================

relative_policy_rows = []


for top_share in [
    0.10,
    0.20,
    0.40,
    0.60,
    0.80,
    1.00,
]:

    selected = (
        relative_eval.loc[
            relative_eval[
                "relative_rank"
            ]
            > (
                1.0
                - top_share
            )
        ]
    )


    predicted_mu0 = (
        selected[
            "relative_mu0_oof"
        ]
        .mean()
    )


    predicted_mu1 = (
        selected[
            "relative_mu1_oof"
        ]
        .mean()
    )


    dr_mu0 = (
        selected[
            "aipw_phi0"
        ]
        .mean()
    )


    dr_mu1 = (
        selected[
            "aipw_phi1"
        ]
        .mean()
    )


    relative_policy_rows.append(
        {
            "top_share":
                top_share,

            "rows":
                len(
                    selected
                ),

            "predicted_relative_uplift":
                (
                    predicted_mu1
                    / predicted_mu0
                    - 1.0
                ),

            "dr_relative_uplift":
                (
                    dr_mu1
                    / dr_mu0
                    - 1.0
                ),

            "mean_baseline":
                selected[
                    "m0_hat"
                ]
                .mean(),

            "treatment_rate":
                selected[
                    "treatment"
                ]
                .mean(),
        }
    )


relative_policy_summary = pd.DataFrame(
    relative_policy_rows
)


print()
print(
    "=" * 80
)

print(
    "TOP-RANKED RELATIVE-UPLIFT DIAGNOSTIC"
)

print(
    "=" * 80
)


display(
    relative_policy_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "top_share":
                "{:.0%}",

            "predicted_relative_uplift":
                "{:+.2%}",

            "dr_relative_uplift":
                "{:+.2%}",

            "mean_baseline":
                "{:,.2f}",

            "treatment_rate":
                "{:.2%}",
        }
    )
)


# =====================================================================
# 4. SUMMARY
# =====================================================================

dr_relative_quintiles = (
    relative_quintile_summary[
        "dr_group_relative_uplift"
    ]
    .to_numpy(
        dtype="float64"
    )
)


adjacent_increases = (
    np.diff(
        dr_relative_quintiles
    )
    > 0
).sum()


top_bottom_spread = (
    dr_relative_quintiles[
        -1
    ]
    - dr_relative_quintiles[
        0
    ]
)


full_dr_mu0 = (
    relative_eval[
        "aipw_phi0"
    ]
    .mean()
)


full_dr_mu1 = (
    relative_eval[
        "aipw_phi1"
    ]
    .mean()
)


full_dr_relative_uplift = (
    full_dr_mu1
    / full_dr_mu0
    - 1.0
)


print()
print(
    "Adjacent DR-relative quintile increases:",
    f"{adjacent_increases}/4",
)


print(
    "Top-minus-bottom DR relative-uplift spread:",
    f"{top_bottom_spread:+.2%}",
)


print(
    "Full primary-domain DR relative uplift:",
    f"{full_dr_relative_uplift:+.2%}",
)


print()
print(
    "RELATIVE T-LEARNER EVALUATION: PASSED"
)

RELATIVE-UPLIFT QUINTILE EVALUATION


relative_quintile,rows,treatment_rate,mean_baseline,median_predicted_relative_uplift,predicted_group_relative_uplift,dr_group_relative_uplift,mean_actual_sales
Q1 lowest,120526,44.62%,628.11,-7.05%,-15.87%,+21.51%,570.34
Q2,120528,48.08%,479.39,+2.58%,-3.37%,+18.28%,459.24
Q3,120530,50.55%,416.21,+9.72%,+4.19%,+25.75%,407.49
Q4,120528,52.92%,389.63,+19.87%,+16.36%,+27.89%,399.22
Q5 highest,120531,54.61%,490.84,+55.75%,+86.55%,+42.46%,582.58



TOP-RANKED RELATIVE-UPLIFT DIAGNOSTIC


top_share,rows,predicted_relative_uplift,dr_relative_uplift,mean_baseline,treatment_rate
10%,60267,+141.85%,+44.29%,567.83,55.31%
20%,120531,+86.55%,+42.46%,490.84,54.61%
40%,241059,+48.96%,+36.08%,440.24,53.76%
60%,361589,+32.02%,+32.86%,432.23,52.69%
80%,482117,+21.00%,+28.89%,444.02,51.54%
100%,602643,+9.86%,+27.07%,480.83,50.16%



Adjacent DR-relative quintile increases: 3/4
Top-minus-bottom DR relative-uplift spread: +20.95%
Full primary-domain DR relative uplift: +27.07%

RELATIVE T-LEARNER EVALUATION: PASSED


#### Relative-uplift conclusion

The relative treatment-effect formulation substantially improves the
policy-oriented heterogeneity signal compared with the absolute-uplift
learner.

Across the primary interpretation population, the doubly robust relative
promotion effect is approximately:

$$
27.1\%.
$$

When observations are ranked out-of-fold by the relative T-learner, the
highest-ranked contexts exhibit materially larger causal relative effects.

The estimated doubly robust uplift is approximately:

$$
44.3\%
$$

for the top 10% of predicted-response contexts and:

$$
42.5\%
$$

for the top 20%, compared with approximately:

$$
27.1\%
$$

for the complete eligible population.

Across predicted-uplift quintiles, the doubly robust relative effects are:

$$
21.5\%,\quad
18.3\%,\quad
25.8\%,\quad
27.9\%,\quad
42.5\%.
$$

Three of the four adjacent quintile transitions increase, and the highest
predicted quintile exceeds the lowest by approximately 21 percentage points.

The ranking is therefore not perfectly monotonic, but it contains a
substantially clearer heterogeneous-response signal than the earlier
absolute-effect learner.

---

The raw percentage predictions themselves are not well calibrated.

For example, the top 10% of contexts receive a much larger model-implied
relative uplift than the approximately 44% effect supported by the doubly
robust group-level evaluation.

Fold-level predicted relative effects also differ substantially from their
corresponding doubly robust estimates.

The T-learner is therefore retained primarily as a **relative-uplift ranking
model**, rather than as a calibrated row-level causal percentage estimator.

The practical interpretation is:

> the machine-learning model identifies commercial contexts that appear more
> promotion-responsive, while doubly robust estimation is used to quantify
> the causal effect of the resulting groups.

This separation is particularly appropriate for downstream promotion and MROI
analysis.

The model can identify higher-response populations, while profitability
analysis should use causally evaluated group-level uplift together with
baseline demand, margins, discount depth and promotion costs.

### 8.3 Business Interpretation and MROI Extension

The causal analysis provides two complementary representations of promotion
response.

The absolute treatment effect measures incremental sales units:

$$
\tau_{\text{units}}(X)
=
E[Y(1)-Y(0)\mid X].
$$

The relative treatment effect measures percentage sales uplift:

$$
\rho(X)
=
\frac{
E[Y(1)\mid X]
}{
E[Y(0)\mid X]
}
-1.
$$

These quantities are directly related.

If:

$$
\mu_0(X)
=
E[Y(0)\mid X]
$$

denotes expected baseline sales without promotion, then:

$$
\tau_{\text{units}}(X)
=
\mu_0(X)\rho(X).
$$

For example, if expected baseline sales are 500 units and the estimated
relative causal uplift is 20%, then expected incremental demand is:

$$
500\times0.20
=
100
$$

additional units.

---

The relative-uplift model developed in Section 8.2 bis is primarily used to
**rank commercial contexts by expected promotion responsiveness**.

Its raw row-level percentage predictions are not interpreted literally because
their calibration is imperfect.

Instead, doubly robust estimates are used to quantify the average causal uplift
of selected groups.

This creates a useful separation between:

1. identifying **where** promotion response appears strongest;
2. estimating **how large** the causal response is for the selected population.

---

A positive causal sales uplift does not necessarily imply that a promotion is
profitable.

The economically relevant quantity is incremental profit:

$$
\Delta Profit(X)
=
E[Profit(1)-Profit(0)\mid X].
$$

A general simplified representation is:

$$
\Delta Profit(X)
\approx
\mu_1(X)\times CM_1(X)
-
\mu_0(X)\times CM_0(X)
-
Incremental\ Promotion\ Cost(X),
$$

where:

- $\mu_0(X)$ is expected sales without promotion;
- $\mu_1(X)$ is expected sales under promotion;
- $CM_0(X)$ is contribution margin per unit without promotion;
- $CM_1(X)$ is contribution margin per unit under promotion.

Using:

$$
\mu_1(X)
=
\mu_0(X)\left(1+\rho(X)\right),
$$

the causal relative-uplift estimate can therefore be translated into an
economic profitability analysis once price and margin information are
available.

If contribution margin per unit were unchanged by the promotion, the expression
would simplify approximately to:

$$
\Delta Profit(X)
\approx
\tau_{\text{units}}(X)
\times
Contribution\ Margin
-
Incremental\ Promotion\ Cost(X).
$$

In practice, however, promotions usually affect selling price and therefore
unit margin, so the more general formulation is preferable.

A retailer could then evaluate a metric such as Marginal Return on Investment:

$$
MROI(X)
=
\frac{
\Delta Profit(X)
}{
Promotion\ Investment(X)
}.
$$

---

The Favorita dataset does not contain enough information to estimate promotion
profitability credibly.

In particular, it does not identify:

- promotion discount depth;
- regular and promotional selling prices;
- gross or contribution margins;
- campaign and display costs;
- supplier funding;
- cannibalization across related products;
- stock availability and lost sales.

The present analysis therefore estimates **incremental sales response**, not
incremental profit.

---

A production promotion-decision system would extend the current framework in
four stages:

1. estimate baseline demand without promotion;
2. estimate relative causal promotion response;
3. convert relative uplift into incremental demand:

$$
Incremental\ Units
=
Baseline\ Demand
\times
Relative\ Uplift;
$$

4. combine incremental demand with price, margins, inventory constraints and
promotion costs to optimize expected incremental profit or MROI.

The present binary treatment is also a coarse representation of a promotion.

Because:

$$
D
=
\mathbb{1}
[
\text{onpromotion}>0
]
$$

combines promotions with different numbers of promoted SKUs and unknown
discount depths, the estimated treatment effect should be interpreted as the
effect of entering the observed historical **any-promotion state**, rather than
the effect of a specific promotion design.

A richer production system would require multi-valued or continuous treatment
modeling once promotion intensity, discount depth and economic data become
available.

## 9. Causal Forecasting Integration - V4

### 9.1 Objective and Point-in-Time Protocol

The previous sections addressed a retrospective causal question:

> What promotion effects are supported by the historical data when the full
> analysis period is available?

V4 addresses a different operational question:

> At a historical forecast origin $T$, could a causal decomposition of
> baseline demand and promotion response improve the next 14 days of demand
> forecasts using only information available by $T$?

This distinction is fundamental.

The retrospective causal models from Sections 6–8 use the complete causal
analysis period and are therefore **not reused directly for historical
forecasting**.

At every V4 forecast origin $T$, all causal components are re-estimated using
only data available at or before $T$.

No observation with:

$$
date>T
$$

may affect propensity estimation, outcome models, treatment-effect estimation,
uplift calibration or any numerical support threshold used by the V4 forecast.

---

V4 is evaluated on exactly the same six 14-day rolling origins used for the
previous forecasting models:

- 2017-05-09;
- 2017-05-23;
- 2017-06-06;
- 2017-06-20;
- 2017-07-04;
- 2017-07-18.

For each origin $T$, the forecast horizon is:

$$
T+1,\ldots,T+14.
$$

The causal training population uses the same two-year rolling historical
window used by the final forecasting architecture:

$$
[T-2\text{ years},T].
$$

The purpose is to compare forecasting architectures under approximately the
same historical-information budget rather than giving V4 access to a larger
training history.

---

#### Point-in-time causal features

The historical causal model uses only pre-treatment covariates.

Dynamic sales and promotion features retain the conservative 14-day gap used
in the retrospective causal analysis.

Because the forecast horizon is at most 14 days, for every target date:

$$
t=T+h,
\qquad
1\leq h\leq14,
$$

a dynamic feature lagged by at least 14 days references a date:

$$
t-14\leq T.
$$

Therefore, the same causal feature contract can be constructed for future
forecast rows without accessing realized post-origin sales.

Calendar information for the future target date is allowed because the
calendar is known in advance.

The planned binary promotion treatment:

$$
D
=
\mathbb{1}
[
\text{onpromotion}>0
]
$$

is also treated as known for the 14-day planning horizon.

This is a production assumption: future promotion plans must be available when
the forecast is generated.

---

#### Specification revision after retrospective causal diagnostics

The original causal-forecasting protocol was frozen before the retrospective
treatment-effect analysis and proposed the additive decomposition:

$$
\hat Y
=
\hat m_0(X)
+
D\hat\tau(X).
$$

The later retrospective heterogeneity analysis revealed two important
properties.

First, absolute uplift in sales units was strongly affected by the sales scale
of each store-family context.

Second, relative uplift became numerically unstable when the predicted
no-promotion baseline was very small.

Section 8.2 bis therefore motivated a documented revision of the V4
architecture **before any V4 forecasting performance was inspected**.

The revised representation is:

$$
\hat Y_{V4}
=
\hat\mu_0(X)
\left[
1+D\hat\rho(X)
\right],
$$

where:

$$
\hat\mu_0(X)
$$

represents predicted no-promotion demand and:

$$
\hat\rho(X)
$$

represents relative promotion response.

The retrospective relative-uplift analysis showed that the raw percentage
predictions contain useful ranking information but are not sufficiently
well calibrated to be interpreted directly as causal percentages.

V4 therefore uses the relative treatment model primarily as a ranking signal
and estimates causal response magnitudes from historical doubly robust
group-level effects.

---

#### Baseline-floor provenance and point-in-time implementation

The retrospective denominator audit showed that relative effects became highly
unstable in very-low-baseline contexts.

In that retrospective sample, approximately the lowest quarter of the
practical-overlap population had predicted no-promotion baseline demand below:

$$
10
$$

sales units.

The numerical value 10 is **not reused as a fixed V4 threshold**.

Doing so would use information obtained from the later retrospective analysis
when evaluating earlier historical forecast origins.

Instead, V4 freezes only the rule motivated by that diagnostic:

> exclude the lowest-baseline quarter from relative-uplift interpretation.

At every forecast origin $T$, the numerical threshold is recomputed using only
historical out-of-fold no-promotion baseline predictions available by $T$:

$$
c_T
=
Q_{0.25}
\left(
\hat m_{0,T}^{OOF}(X_i)
\right).
$$

A future promoted context is eligible for a relative causal adjustment only
when its predicted baseline satisfies:

$$
\hat\mu_{0,T}(X)
\geq
c_T.
$$

Therefore, the numerical support threshold applied at each historical origin
does not use data after that origin.

The choice of the 25% rule itself was informed by the earlier retrospective
causal analysis.

V4 should consequently be interpreted as a **development backtest under a
specification informed by prior causal diagnostics**, rather than as a fully
untouched external validation.

Crucially, no V4 forecast errors or future outcomes are used to modify this
rule once V4 evaluation begins.

---

#### Point-in-time causal architecture

For every forecast origin $T$, V4 estimates:

$$
\mu_{0,T}(X)
=
E[Y(0)\mid X]
$$

from historical non-promoted observations and:

$$
\mu_{1,T}(X)
=
E[Y(1)\mid X]
$$

from historical promoted observations.

The raw relative-response score is:

$$
s_T(X)
=
\frac{
\hat\mu_{1,T}(X)
}{
\hat\mu_{0,T}(X)
}
-1.
$$

Because the retrospective analysis showed that raw relative percentages are
not sufficiently calibrated, V4 does not directly interpret $s_T(X)$ as the
causal percentage effect.

Historical observations available by $T$ are instead used to estimate a
point-in-time mapping between uplift-score ranks and doubly robust group-level
relative effects.

Conceptually:

$$
s_T(X)
\longrightarrow
\hat\rho_T^{calibrated}(X).
$$

Both the ranking model and its causal calibration are learned exclusively from
information available by $T$.

---

#### Support-aware forecasting

A causal promotion adjustment is applied only when the target context lies
inside the historical treatment-support region:

$$
0.01
\leq
\hat e_T(X)
\leq
0.99,
$$

and when predicted no-promotion baseline demand satisfies the point-in-time
baseline rule:

$$
\hat\mu_{0,T}(X)
\geq
c_T.
$$

For promoted target rows outside this causal-support domain, V4 does not
extrapolate a treatment effect.

The conservative fallback is:

$$
\hat\rho_T(X)=0.
$$

Therefore:

$$
\hat Y_{V4}(X,D)
=
\hat\mu_{0,T}(X)
\left[
1+D\hat\rho_T(X)
\right].
$$

For a non-promoted target row:

$$
D=0
\quad\Rightarrow\quad
\hat Y_{V4}
=
\hat\mu_{0,T}(X).
$$

For a promoted and causally supported target row:

$$
D=1
\quad\Rightarrow\quad
\hat Y_{V4}
=
\hat\mu_{0,T}(X)
\left[
1+\hat\rho_T(X)
\right].
$$

For a promoted row without sufficient causal support:

$$
\hat Y_{V4}
=
\hat\mu_{0,T}(X).
$$

This deliberately prefers no causal adjustment to unsupported treatment-effect
extrapolation.

---

#### Inner estimation versus outer forecasting

The six forecast origins define the **outer temporal evaluation**.

At each origin $T$, all information dated at or before $T$ is already
historically observable.

Inner cross-fitting may therefore partition and mix observations inside this
past-only window to generate out-of-sample nuisance predictions and causal
calibration estimates.

This does not violate the forecasting boundary because no observation after
$T$ enters the procedure.

The outer rule remains:

$$
\boxed{
\text{everything used to build the V4 forecast at }T
\text{ must be known by }T
}
$$

This creates a strict separation between retrospective causal explanation,
point-in-time causal learning and future forecasting evaluation.

In [33]:
# ---------------------------------------------------------------------
# V4 causal-forecasting protocol.
#
# This cell freezes the OUTER point-in-time evaluation contract before
# any V4 forecast model is fitted or evaluated.
#
# Every model, nuisance estimate, causal calibration and numerical
# support threshold used for a forecast origin T must use information
# available no later than T.
#
# IMPORTANT:
#
# The retrospective relative-uplift analysis motivated excluding the
# lowest-baseline quarter from relative-effect interpretation.
#
# The retrospective numerical value around 10 sales units is NOT reused
# directly in V4.
#
# Instead, each forecast origin T will estimate its own historical
# baseline threshold:
#
#   c_T = 25th percentile of historical OOF m0 predictions.
#
# This prevents future information from determining the numerical V4
# support threshold.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


# =====================================================================
# 1. OUTER FORECAST ORIGINS
# =====================================================================

V4_FORECAST_ORIGINS = (
    pd.Timestamp("2017-05-09"),
    pd.Timestamp("2017-05-23"),
    pd.Timestamp("2017-06-06"),
    pd.Timestamp("2017-06-20"),
    pd.Timestamp("2017-07-04"),
    pd.Timestamp("2017-07-18"),
)


V4_HORIZON_DAYS = 14

V4_ROLLING_YEARS = 2


# =====================================================================
# 2. POINT-IN-TIME FEATURE-SAFETY CONTRACT
# =====================================================================
#
# All dynamic causal features use at least a 14-day historical gap.
#
# For:
#
#   target_date = T + h
#
# with:
#
#   1 <= h <= 14
#
# every dynamic feature lagged by at least 14 days references:
#
#   target_date - 14 <= T
#
# and therefore does not require realized post-origin sales.
# =====================================================================

V4_MIN_DYNAMIC_LAG_DAYS = 14


assert (
    V4_HORIZON_DAYS
    <= V4_MIN_DYNAMIC_LAG_DAYS
), (
    "The V4 forecast horizon exceeds the minimum causal-feature lag. "
    "Point-in-time feature safety would no longer be guaranteed."
)


# =====================================================================
# 3. PROPENSITY SUPPORT CONTRACT
# =====================================================================

V4_PROPENSITY_SUPPORT_LOW = 0.01

V4_PROPENSITY_SUPPORT_HIGH = 0.99


# =====================================================================
# 4. BASELINE-SUPPORT CONTRACT
# =====================================================================
#
# Retrospective Section 8.2 bis showed severe relative-uplift instability
# in the lowest-baseline quarter.
#
# The RULE is frozen here:
#
#   exclude the lowest 25% of historical OOF baseline predictions.
#
# The numerical threshold itself is NOT frozen.
#
# At every forecast origin T it will be recomputed using only historical
# OOF no-promotion baseline predictions available by T:
#
#   c_T = Q_0.25(m0_OOF)
#
# This makes the numerical floor point-in-time safe.
# =====================================================================

V4_BASELINE_FLOOR_QUANTILE = 0.25


# =====================================================================
# 5. UNSUPPORTED CAUSAL-ADJUSTMENT CONTRACT
# =====================================================================
#
# If a promoted target row lies outside practical propensity support or
# below the origin-specific baseline floor, V4 does not extrapolate a
# causal uplift.
#
# It falls back to zero relative uplift.
# =====================================================================

V4_UNSUPPORTED_RELATIVE_UPLIFT = 0.0


# =====================================================================
# 6. RELATIVE-UPLIFT CALIBRATION CONTRACT
# =====================================================================
#
# Raw relative T-learner percentages are treated primarily as a ranking
# score rather than as directly trusted causal percentages.
#
# Historical OOF relative scores will be divided into five rank groups.
#
# Every group will receive a past-only doubly robust relative-effect
# estimate.
#
# Future rows will subsequently be mapped into the corresponding
# historical score-rank group.
# =====================================================================

V4_UPLIFT_CALIBRATION_BINS = 5

V4_INNER_FOLDS = 5


# =====================================================================
# 7. FORECAST-ORIGIN PROTOCOL TABLE
# =====================================================================

v4_protocol_rows = []


for forecast_origin in V4_FORECAST_ORIGINS:

    training_start = (
        forecast_origin
        - pd.DateOffset(
            years=V4_ROLLING_YEARS
        )
    )


    training_end = (
        forecast_origin
    )


    forecast_start = (
        forecast_origin
        + pd.Timedelta(
            days=1
        )
    )


    forecast_end = (
        forecast_origin
        + pd.Timedelta(
            days=V4_HORIZON_DAYS
        )
    )


    v4_protocol_rows.append(
        {
            "forecast_origin":
                forecast_origin,

            "causal_training_start":
                training_start,

            "causal_training_end":
                training_end,

            "forecast_start":
                forecast_start,

            "forecast_end":
                forecast_end,

            "forecast_days":
                V4_HORIZON_DAYS,

            "max_information_date":
                forecast_origin,

            "baseline_floor_quantile":
                V4_BASELINE_FLOOR_QUANTILE,
        }
    )


v4_protocol = pd.DataFrame(
    v4_protocol_rows
)


# =====================================================================
# 8. OUTER TEMPORAL-SAFETY ASSERTIONS
# =====================================================================

assert (
    v4_protocol[
        "causal_training_end"
    ]
    <=
    v4_protocol[
        "forecast_origin"
    ]
).all()


assert (
    v4_protocol[
        "forecast_start"
    ]
    >
    v4_protocol[
        "forecast_origin"
    ]
).all()


assert (
    v4_protocol[
        "forecast_end"
    ]
    >
    v4_protocol[
        "forecast_origin"
    ]
).all()


assert (
    (
        v4_protocol[
            "forecast_end"
        ]
        -
        v4_protocol[
            "forecast_start"
        ]
    )
    .dt.days
    .add(
        1
    )
    .eq(
        V4_HORIZON_DAYS
    )
    .all()
)


# =====================================================================
# 9. DISPLAY FROZEN V4 PROTOCOL
# =====================================================================

print(
    "=" * 80
)

print(
    "V4 POINT-IN-TIME CAUSAL FORECASTING PROTOCOL"
)

print(
    "=" * 80
)


display(
    v4_protocol
    .style.hide(
        axis="index"
    )
    .format(
        {
            "baseline_floor_quantile":
                "{:.0%}",
        }
    )
)


print()
print(
    "Forecast origins:",
    len(
        V4_FORECAST_ORIGINS
    ),
)


print(
    "Forecast horizon:",
    f"{V4_HORIZON_DAYS} days",
)


print(
    "Rolling causal history:",
    f"{V4_ROLLING_YEARS} years",
)


print(
    "Minimum dynamic-feature lag:",
    f"{V4_MIN_DYNAMIC_LAG_DAYS} days",
)


print(
    "Propensity support:",
    (
        f"[{V4_PROPENSITY_SUPPORT_LOW:.2f}, "
        f"{V4_PROPENSITY_SUPPORT_HIGH:.2f}]"
    ),
)


print(
    "Relative-uplift baseline-floor rule:",
    (
        f"historical OOF "
        f"{V4_BASELINE_FLOOR_QUANTILE:.0%} "
        f"quantile per forecast origin"
    ),
)


print(
    "Relative-uplift calibration bins:",
    V4_UPLIFT_CALIBRATION_BINS,
)


print(
    "Inner causal folds:",
    V4_INNER_FOLDS,
)


print()
print(
    "IMPORTANT: the retrospective ~10-unit threshold is not reused "
    "numerically in V4."
)


print(
    "Each origin will estimate its own baseline floor using past-only "
    "OOF baseline predictions."
)


print()
print(
    "V4 OUTER POINT-IN-TIME PROTOCOL: PASSED"
)

V4 POINT-IN-TIME CAUSAL FORECASTING PROTOCOL


forecast_origin,causal_training_start,causal_training_end,forecast_start,forecast_end,forecast_days,max_information_date,baseline_floor_quantile
2017-05-09 00:00:00,2015-05-09 00:00:00,2017-05-09 00:00:00,2017-05-10 00:00:00,2017-05-23 00:00:00,14,2017-05-09 00:00:00,25%
2017-05-23 00:00:00,2015-05-23 00:00:00,2017-05-23 00:00:00,2017-05-24 00:00:00,2017-06-06 00:00:00,14,2017-05-23 00:00:00,25%
2017-06-06 00:00:00,2015-06-06 00:00:00,2017-06-06 00:00:00,2017-06-07 00:00:00,2017-06-20 00:00:00,14,2017-06-06 00:00:00,25%
2017-06-20 00:00:00,2015-06-20 00:00:00,2017-06-20 00:00:00,2017-06-21 00:00:00,2017-07-04 00:00:00,14,2017-06-20 00:00:00,25%
2017-07-04 00:00:00,2015-07-04 00:00:00,2017-07-04 00:00:00,2017-07-05 00:00:00,2017-07-18 00:00:00,14,2017-07-04 00:00:00,25%
2017-07-18 00:00:00,2015-07-18 00:00:00,2017-07-18 00:00:00,2017-07-19 00:00:00,2017-08-01 00:00:00,14,2017-07-18 00:00:00,25%



Forecast origins: 6
Forecast horizon: 14 days
Rolling causal history: 2 years
Minimum dynamic-feature lag: 14 days
Propensity support: [0.01, 0.99]
Relative-uplift baseline-floor rule: historical OOF 25% quantile per forecast origin
Relative-uplift calibration bins: 5
Inner causal folds: 5

IMPORTANT: the retrospective ~10-unit threshold is not reused numerically in V4.
Each origin will estimate its own baseline floor using past-only OOF baseline predictions.

V4 OUTER POINT-IN-TIME PROTOCOL: PASSED


### 9.2 Historical Causal Model at Each Forecast Origin

The V4 experiment requires a new causal model at every historical forecast
origin.

The retrospective `causal_population` used in Sections 5–8 is not sufficient
for this purpose because its fixed analysis window begins on 2015-08-16,
whereas the earliest V4 origin requires a two-year training window beginning
on 2015-05-09.

V4 therefore returns to the complete causal-ready feature table and applies
the rolling training window separately at every forecast origin.

For each origin $T$, the historical sample is:

$$
T-2\text{ years}
\leq
date
\leq
T.
$$

The future evaluation sample is:

$$
T+1
\leq
date
\leq
T+14.
$$

Only the frozen pre-treatment feature contract from the retrospective causal
analysis is reused.

The retrospective fitted propensity scores, outcome predictions, AIPW scores
and treatment-effect estimates are **not** reused.

Each V4 origin will independently reconstruct:

1. historical treatment assignment;
2. inner out-of-fold propensity estimates;
3. inner out-of-fold no-promotion and promotion outcome models;
4. doubly robust historical potential-outcome estimates;
5. relative-uplift ranking and group-level causal calibration;
6. final past-only models used to forecast the next 14 days.

Before model fitting, this section verifies that the complete source table
covers every required historical and future date and reconstructs the
point-in-time store-activity scope.

In [34]:
# ---------------------------------------------------------------------
# Build the canonical V4 point-in-time modeling source.
#
# `causal_regularized` contains the complete daily causal feature grid.
#
# It also contains synthetic calendar rows inserted during temporal
# regularization for historically missing dates (notably Christmas).
#
# Those synthetic rows are necessary for correct lag / rolling-feature
# construction, but they do NOT represent observed store-family-days.
#
# Therefore:
#
#   - they are retained in the already-computed feature history;
#   - they are NOT converted into control observations;
#   - they are excluded from causal model fitting and forecast scoring.
#
# No missing treatment value is filled with zero.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


# =====================================================================
# 1. FREEZE V4 DATE RANGE
# =====================================================================

V4_EARLIEST_REQUIRED_DATE = (
    v4_protocol[
        "causal_training_start"
    ]
    .min()
)


V4_LATEST_REQUIRED_DATE = (
    v4_protocol[
        "forecast_end"
    ]
    .max()
)


# =====================================================================
# 2. BUILD CANONICAL SOURCE FROM THE KNOWN FULL FEATURE TABLE
# =====================================================================

V4_SOURCE_OBJECT_NAME = (
    "causal_regularized"
)


V4_SOURCE_SALES_COLUMN = (
    "target_sales"
)


V4_SOURCE_PROMOTION_COLUMN = (
    "treatment"
)


v4_source_columns = list(
    dict.fromkeys(
        [
            "date",
            "store_nbr",
            "family",
            V4_SOURCE_SALES_COLUMN,
            V4_SOURCE_PROMOTION_COLUMN,
            *CAUSAL_PROPENSITY_FEATURES,
        ]
    )
)


v4_source_frame = (
    causal_regularized[
        v4_source_columns
    ]
    .copy()
)


v4_source_frame[
    "date"
] = pd.to_datetime(
    v4_source_frame[
        "date"
    ]
)


v4_source_frame[
    "target_sales"
] = pd.to_numeric(
    v4_source_frame[
        V4_SOURCE_SALES_COLUMN
    ],
    errors="coerce",
)


v4_source_frame[
    "treatment"
] = pd.to_numeric(
    v4_source_frame[
        V4_SOURCE_PROMOTION_COLUMN
    ],
    errors="coerce",
)


# =====================================================================
# 3. AUDIT OBSERVED VS SYNTHETIC CALENDAR ROWS
# =====================================================================
#
# An observed causal row requires BOTH:
#
#   - realized sales;
#   - observed promotion treatment.
#
# Missing synthetic calendar rows must not be interpreted as D=0.
# =====================================================================

v4_source_frame[
    "has_observed_sales"
] = (
    v4_source_frame[
        "target_sales"
    ]
    .notna()
)


v4_source_frame[
    "has_observed_treatment"
] = (
    v4_source_frame[
        "treatment"
    ]
    .notna()
)


v4_source_frame[
    "is_observed_row"
] = (
    v4_source_frame[
        "has_observed_sales"
    ]
    &
    v4_source_frame[
        "has_observed_treatment"
    ]
)


missing_sales_rows = int(
    (
        ~v4_source_frame[
            "has_observed_sales"
        ]
    )
    .sum()
)


missing_treatment_rows = int(
    (
        ~v4_source_frame[
            "has_observed_treatment"
        ]
    )
    .sum()
)


fully_unobserved_rows = int(
    (
        ~v4_source_frame[
            "has_observed_sales"
        ]
        &
        ~v4_source_frame[
            "has_observed_treatment"
        ]
    )
    .sum()
)


partial_missing_mask = (
    v4_source_frame[
        "has_observed_sales"
    ]
    ^
    v4_source_frame[
        "has_observed_treatment"
    ]
)


partial_missing_rows = int(
    partial_missing_mask.sum()
)


synthetic_unobserved_dates = (
    v4_source_frame.loc[
        ~v4_source_frame[
            "is_observed_row"
        ],
        "date",
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)


print(
    "=" * 80
)

print(
    "V4 OBSERVED / SYNTHETIC ROW AUDIT"
)

print(
    "=" * 80
)


print(
    "Complete regularized rows:",
    f"{len(v4_source_frame):,}",
)


print(
    "Missing-sales rows:",
    f"{missing_sales_rows:,}",
)


print(
    "Missing-treatment rows:",
    f"{missing_treatment_rows:,}",
)


print(
    "Fully unobserved rows:",
    f"{fully_unobserved_rows:,}",
)


print(
    "Partially missing rows:",
    f"{partial_missing_rows:,}",
)


print(
    "Unobserved calendar dates:",
    [
        timestamp.date()
        for timestamp in synthetic_unobserved_dates
    ],
)


# Non-missing treatment values must represent the frozen binary
# treatment definition.

observed_treatment_values = set(
    v4_source_frame.loc[
        v4_source_frame[
            "has_observed_treatment"
        ],
        "treatment",
    ]
    .unique()
)


assert observed_treatment_values.issubset(
    {
        0.0,
        1.0,
    }
), (
    "Observed treatment contains values outside {0, 1}."
)


# Within the V4 period, an observed outcome without an observed treatment
# (or vice versa) would be a genuine data-contract problem rather than a
# synthetic calendar row.

v4_required_period_mask = (
    v4_source_frame[
        "date"
    ]
    .between(
        V4_EARLIEST_REQUIRED_DATE,
        V4_LATEST_REQUIRED_DATE,
        inclusive="both",
    )
)


required_period_partial_missing = int(
    (
        partial_missing_mask
        &
        v4_required_period_mask
    )
    .sum()
)


assert (
    required_period_partial_missing
    == 0
), (
    "The V4 period contains rows with only one of sales/treatment "
    "observed. These require investigation before modeling."
)


# =====================================================================
# 4. RECONSTRUCT POINT-IN-TIME STORE ACTIVITY
# =====================================================================
#
# Store activity is based on observed sales only.
#
# Synthetic calendar rows never create or terminate store activity.
# =====================================================================

observed_sales_for_activity = (
    v4_source_frame.loc[
        v4_source_frame[
            "has_observed_sales"
        ],
        [
            "store_nbr",
            "date",
            "target_sales",
        ],
    ]
)


store_daily_sales = (
    observed_sales_for_activity
    .groupby(
        [
            "store_nbr",
            "date",
        ],
        observed=True,
    )[
        "target_sales"
    ]
    .sum(
        min_count=1
    )
    .reset_index(
        name="store_total_sales"
    )
)


store_first_positive_date = (
    store_daily_sales.loc[
        store_daily_sales[
            "store_total_sales"
        ]
        .gt(
            0
        )
    ]
    .groupby(
        "store_nbr",
        observed=True,
    )[
        "date"
    ]
    .min()
    .rename(
        "store_first_positive_date"
    )
    .reset_index()
)


v4_source_frame = (
    v4_source_frame
    .merge(
        store_first_positive_date,
        on="store_nbr",
        how="left",
        validate="many_to_one",
    )
)


v4_source_frame[
    "store_started"
] = (
    v4_source_frame[
        "date"
    ]
    .ge(
        v4_source_frame[
            "store_first_positive_date"
        ]
    )
)


# =====================================================================
# 5. FINAL MODEL-ELIGIBLE SOURCE
# =====================================================================
#
# Feature engineering has already used the regularized calendar.
#
# From this point onward, models operate only on REAL observed
# store-family-days from active stores.
# =====================================================================

v4_model_source_frame = (
    v4_source_frame.loc[
        v4_source_frame[
            "is_observed_row"
        ]
        &
        v4_source_frame[
            "store_started"
        ]
    ]
    .copy()
)


v4_model_source_frame[
    "treatment"
] = (
    v4_model_source_frame[
        "treatment"
    ]
    .astype(
        "int8"
    )
)


preactivity_observed_rows_excluded = int(
    (
        v4_source_frame[
            "is_observed_row"
        ]
        &
        ~v4_source_frame[
            "store_started"
        ]
    )
    .sum()
)


# =====================================================================
# 6. AUDIT EVERY OUTER V4 WINDOW
# =====================================================================

v4_window_audit_rows = []


for protocol_row in (
    v4_protocol
    .itertuples(
        index=False
    )
):

    forecast_origin = (
        protocol_row.forecast_origin
    )


    training_start = (
        protocol_row.causal_training_start
    )


    forecast_start = (
        protocol_row.forecast_start
    )


    forecast_end = (
        protocol_row.forecast_end
    )


    historical_frame = (
        v4_model_source_frame.loc[
            v4_model_source_frame[
                "date"
            ]
            .between(
                training_start,
                forecast_origin,
                inclusive="both",
            )
        ]
    )


    future_frame = (
        v4_model_source_frame.loc[
            v4_model_source_frame[
                "date"
            ]
            .between(
                forecast_start,
                forecast_end,
                inclusive="both",
            )
        ]
    )


    assert not historical_frame.empty

    assert not future_frame.empty


    # Strict outer point-in-time separation.

    assert (
        historical_frame[
            "date"
        ]
        .max()
        <= forecast_origin
    )


    assert (
        future_frame[
            "date"
        ]
        .min()
        > forecast_origin
    )


    assert (
        future_frame[
            "date"
        ]
        .nunique()
        ==
        V4_HORIZON_DAYS
    )


    v4_window_audit_rows.append(
        {
            "forecast_origin":
                forecast_origin,

            "training_start":
                training_start,

            "historical_rows":
                len(
                    historical_frame
                ),

            "historical_dates":
                historical_frame[
                    "date"
                ]
                .nunique(),

            "historical_stores":
                historical_frame[
                    "store_nbr"
                ]
                .nunique(),

            "historical_families":
                historical_frame[
                    "family"
                ]
                .nunique(),

            "historical_treatment_rate":
                historical_frame[
                    "treatment"
                ]
                .mean(),

            "future_rows":
                len(
                    future_frame
                ),

            "future_dates":
                future_frame[
                    "date"
                ]
                .nunique(),

            "future_stores":
                future_frame[
                    "store_nbr"
                ]
                .nunique(),

            "future_families":
                future_frame[
                    "family"
                ]
                .nunique(),

            "future_treatment_rate":
                future_frame[
                    "treatment"
                ]
                .mean(),

            "latest_historical_date":
                historical_frame[
                    "date"
                ]
                .max(),

            "earliest_future_date":
                future_frame[
                    "date"
                ]
                .min(),
        }
    )


v4_window_audit = pd.DataFrame(
    v4_window_audit_rows
)


# =====================================================================
# 7. DISPLAY FINAL SOURCE AUDIT
# =====================================================================

print()
print(
    "=" * 80
)

print(
    "V4 POINT-IN-TIME SOURCE AND WINDOW AUDIT"
)

print(
    "=" * 80
)


print(
    "Observed pre-activity rows excluded:",
    f"{preactivity_observed_rows_excluded:,}",
)


print(
    "Final model-eligible source rows:",
    f"{len(v4_model_source_frame):,}",
)


print(
    "Final source date range:",
    (
        f"{v4_model_source_frame['date'].min().date()} "
        f"to "
        f"{v4_model_source_frame['date'].max().date()}"
    ),
)


print()


display(
    v4_window_audit
    .style.hide(
        axis="index"
    )
    .format(
        {
            "historical_treatment_rate":
                "{:.2%}",

            "future_treatment_rate":
                "{:.2%}",
        }
    )
)


# =====================================================================
# 8. FINAL POINT-IN-TIME CONTRACT ASSERTIONS
# =====================================================================

assert (
    v4_window_audit[
        "latest_historical_date"
    ]
    .eq(
        v4_window_audit[
            "forecast_origin"
        ]
    )
    .all()
)


assert (
    v4_window_audit[
        "earliest_future_date"
    ]
    >
    v4_window_audit[
        "forecast_origin"
    ]
).all()


assert (
    v4_window_audit[
        "future_dates"
    ]
    .eq(
        V4_HORIZON_DAYS
    )
    .all()
)


print()
print(
    "V4 CAUSAL SOURCE + OUTER WINDOWS: PASSED"
)

V4 OBSERVED / SYNTHETIC ROW AUDIT
Complete regularized rows: 3,008,016
Missing-sales rows: 7,128
Missing-treatment rows: 7,128
Fully unobserved rows: 7,128
Partially missing rows: 0
Unobserved calendar dates: [datetime.date(2013, 12, 25), datetime.date(2014, 12, 25), datetime.date(2015, 12, 25), datetime.date(2016, 12, 25)]

V4 POINT-IN-TIME SOURCE AND WINDOW AUDIT
Observed pre-activity rows excluded: 222,057
Final model-eligible source rows: 2,778,831
Final source date range: 2013-01-01 to 2017-08-15



forecast_origin,training_start,historical_rows,historical_dates,historical_stores,historical_families,historical_treatment_rate,future_rows,future_dates,future_stores,future_families,future_treatment_rate,latest_historical_date,earliest_future_date
2017-05-09 00:00:00,2015-05-09 00:00:00,1266441,730,54,33,33.82%,24948,14,54,33,50.14%,2017-05-09 00:00:00,2017-05-10 00:00:00
2017-05-23 00:00:00,2015-05-23 00:00:00,1268289,730,54,33,34.31%,24948,14,54,33,45.32%,2017-05-23 00:00:00,2017-05-24 00:00:00
2017-06-06 00:00:00,2015-06-06 00:00:00,1270137,730,54,33,34.68%,24948,14,54,33,45.55%,2017-06-06 00:00:00,2017-06-07 00:00:00
2017-06-20 00:00:00,2015-06-20 00:00:00,1271985,730,54,33,35.03%,24948,14,54,33,44.68%,2017-06-20 00:00:00,2017-06-21 00:00:00
2017-07-04 00:00:00,2015-07-04 00:00:00,1273833,730,54,33,35.38%,24948,14,54,33,42.60%,2017-07-04 00:00:00,2017-07-05 00:00:00
2017-07-18 00:00:00,2015-07-18 00:00:00,1275681,730,54,33,35.79%,24948,14,54,33,40.80%,2017-07-18 00:00:00,2017-07-19 00:00:00



V4 CAUSAL SOURCE + OUTER WINDOWS: PASSED


#### Inner Cross-Fitting and Past-Only Causal Calibration

At each outer forecast origin $T$, V4 reconstructs the causal pipeline using
only the corresponding two-year historical window.

Five contiguous inner date folds are created inside that past-only window.

Because every inner observation already occurred before the outer forecast
origin, these folds may be used for standard cross-fitting without violating
the outer point-in-time forecasting boundary.

For every historical observation, the pipeline produces out-of-fold nuisance
predictions.

---

First, a propensity model estimates:

$$
\hat e_T(X)
=
P(D=1\mid X).
$$

The practical causal-support population is:

$$
0.01
\leq
\hat e_T(X)
\leq
0.99.
$$

---

Second, separate outcome nuisance models estimate:

$$
\hat m_{0,T}(X)
$$

and:

$$
\hat m_{1,T}(X).
$$

These models are used to construct out-of-fold augmented potential-outcome
scores:

$$
\phi_{0,i}
=
\hat m_0(X_i)
+
\frac{1-D_i}{1-\hat e_i}
\left[
Y_i-\hat m_0(X_i)
\right],
$$

and:

$$
\phi_{1,i}
=
\hat m_1(X_i)
+
\frac{D_i}{\hat e_i}
\left[
Y_i-\hat m_1(X_i)
\right].
$$

For any sufficiently large historical group $G$, its doubly robust relative
promotion effect is estimated as:

$$
\hat\rho_G^{DR}
=
\frac{
\overline{\phi_1}_G
}{
\overline{\phi_0}_G
}
-1.
$$

---

A separate Tweedie T-learner is used for heterogeneous-response ranking.

It estimates positive conditional means:

$$
\hat\mu_{0,T}^{rank}(X)
$$

and:

$$
\hat\mu_{1,T}^{rank}(X),
$$

from which the raw relative-response score is:

$$
s_T(X)
=
\frac{
\hat\mu_{1,T}^{rank}(X)
}{
\hat\mu_{0,T}^{rank}(X)
}
-1.
$$

The raw score is used for **ranking**, not interpreted directly as a causal
percentage.

Keeping the ranking models distinct from the outcome nuisance models prevents
the calibration step from simply reproducing the same fitted response
surface.

---

The origin-specific low-baseline threshold is estimated from historical
out-of-fold no-promotion ranking-model predictions inside practical propensity
support:

$$
c_T
=
Q_{0.25}
\left(
\hat\mu_{0,T}^{rank,OOF}(X)
\right).
$$

Only historical observations satisfying:

$$
\hat\mu_{0,T}^{rank,OOF}(X)
\geq
c_T
$$

enter relative-uplift calibration.

This means that both the numerical baseline threshold and the causal
calibration are reconstructed independently at every historical forecast
origin.

---

Raw relative-response scores are converted into percentile ranks within each
inner held-out fold.

The ranks are divided into five groups:

$$
Q_1,\ldots,Q_5.
$$

For each group, V4 stores the corresponding past-only doubly robust relative
effect:

$$
\hat\rho_{T,Q_j}^{DR}.
$$

The resulting table is the causal calibration that will later be applied to
future target rows.

No outcome after the outer forecast origin $T$ contributes to this procedure.

In [35]:
# ---------------------------------------------------------------------
# V4 historical causal estimation and relative-uplift calibration.
#
# For EACH outer forecast origin T:
#
#   1. use only the two-year historical window ending at T;
#   2. create five contiguous inner date folds;
#   3. generate OOF propensity estimates;
#   4. define practical propensity overlap;
#   5. generate separate OOF AIPW outcome nuisance predictions;
#   6. construct phi0 / phi1 doubly robust pseudo-outcomes;
#   7. generate separate OOF Tweedie relative-uplift ranking scores;
#   8. estimate the point-in-time 25th-percentile baseline floor;
#   9. calibrate five ranking groups using group-level DR relative effects.
#
# The ranking T-learner and AIPW outcome nuisance models are deliberately
# separate.
#
# No observation after forecast origin T enters any calculation.
# ---------------------------------------------------------------------

import gc
import time

import numpy as np
import pandas as pd

from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

from xgboost import (
    XGBClassifier,
    XGBRegressor,
)


# =====================================================================
# 1. FREEZE V4 MODEL PARAMETERS
# =====================================================================

V4_PROPENSITY_MODEL_PARAMS = {
    "objective":
        "binary:logistic",

    "eval_metric":
        "logloss",

    "n_estimators":
        800,

    "max_depth":
        6,

    "learning_rate":
        0.05,

    "min_child_weight":
        10,

    "subsample":
        0.80,

    "colsample_bytree":
        0.80,

    "reg_lambda":
        5.0,

    "tree_method":
        "hist",

    "enable_categorical":
        True,

    "random_state":
        42,

    "n_jobs":
        -1,

    "verbosity":
        0,
}


# Outcome nuisance models used for AIPW estimation.
#
# These target conditional MEAN sales in each treatment state.

V4_AIPW_OUTCOME_MODEL_PARAMS = {
    "objective":
        "reg:squarederror",

    "n_estimators":
        1000,

    "max_depth":
        8,

    "learning_rate":
        0.05,

    "min_child_weight":
        10,

    "subsample":
        0.80,

    "colsample_bytree":
        0.80,

    "reg_lambda":
        5.0,

    "tree_method":
        "hist",

    "enable_categorical":
        True,

    "random_state":
        42,

    "n_jobs":
        -1,

    "verbosity":
        0,
}


# Separate outcome models used ONLY to rank contexts by relative response.
#
# Tweedie keeps the conditional mean positive and is naturally compatible
# with the multiplicative relative-response formulation.

V4_RELATIVE_RANK_MODEL_PARAMS = {
    "objective":
        "reg:tweedie",

    "tweedie_variance_power":
        1.5,

    "n_estimators":
        1000,

    "max_depth":
        6,

    "learning_rate":
        0.05,

    "min_child_weight":
        20,

    "subsample":
        0.80,

    "colsample_bytree":
        0.80,

    "reg_lambda":
        10.0,

    "reg_alpha":
        1.0,

    "tree_method":
        "hist",

    "enable_categorical":
        True,

    "random_state":
        42,

    "n_jobs":
        -1,

    "verbosity":
        0,
}


V4_MODEL_FEATURES = (
    CAUSAL_PROPENSITY_FEATURES.copy()
)


# =====================================================================
# 2. FEATURE PREPARATION
# =====================================================================

def prepare_v4_X(
    frame,
):
    """
    Prepare the frozen pre-treatment feature matrix used by all V4
    causal learners.

    Category definitions are inherited from the already-frozen causal
    feature contract.
    """

    X = (
        frame[
            V4_MODEL_FEATURES
        ]
        .copy()
    )


    for (
        feature,
        categorical_dtype,
    ) in CAUSAL_CATEGORY_DTYPES.items():

        if feature in X.columns:

            X[
                feature
            ] = (
                X[
                    feature
                ]
                .astype(
                    categorical_dtype
                )
            )


    return X


# =====================================================================
# 3. INNER FOLD CONSTRUCTION
# =====================================================================

def assign_v4_inner_folds(
    frame,
    n_folds=V4_INNER_FOLDS,
):
    """
    Split historical dates into contiguous blocks.

    The outer window already guarantees date <= T, so inner cross-fitting
    may use the remaining historical blocks irrespective of their
    chronological order relative to the held-out inner block.
    """

    unique_dates = np.array(
        sorted(
            frame[
                "date"
            ]
            .unique()
        )
    )


    date_blocks = np.array_split(
        unique_dates,
        n_folds,
    )


    date_to_fold = {}


    for fold_id, date_block in enumerate(
        date_blocks,
        start=1,
    ):

        for date_value in date_block:

            date_to_fold[
                pd.Timestamp(
                    date_value
                )
            ] = fold_id


    fold_series = (
        frame[
            "date"
        ]
        .map(
            date_to_fold
        )
        .astype(
            "int8"
        )
    )


    assert fold_series.notna().all()

    assert (
        fold_series.nunique()
        ==
        n_folds
    )


    return fold_series


# =====================================================================
# 4. STORAGE FOR ORIGIN-LEVEL RESULTS
# =====================================================================

v4_origin_summary_rows = []

v4_calibration_rows = []

v4_origin_causal_specs = {}


# =====================================================================
# 5. OUTER ORIGIN LOOP
# =====================================================================

for protocol_row in (
    v4_protocol
    .itertuples(
        index=False
    )
):

    origin_start_time = time.time()


    forecast_origin = (
        protocol_row.forecast_origin
    )


    training_start = (
        protocol_row.causal_training_start
    )


    print(
        "\n"
        + "=" * 80
    )

    print(
        "V4 HISTORICAL CAUSAL ESTIMATION"
    )

    print(
        "Forecast origin:",
        forecast_origin.date(),
    )

    print(
        "Historical window:",
        (
            f"{training_start.date()} "
            f"to "
            f"{forecast_origin.date()}"
        ),
    )

    print(
        "=" * 80
    )


    # -------------------------------------------------------------
    # Historical point-in-time population.
    # -------------------------------------------------------------

    historical_frame = (
        v4_model_source_frame.loc[
            v4_model_source_frame[
                "date"
            ]
            .between(
                training_start,
                forecast_origin,
                inclusive="both",
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    assert not historical_frame.empty


    assert (
        historical_frame[
            "date"
        ]
        .max()
        <= forecast_origin
    )


    historical_frame[
        "inner_fold"
    ] = assign_v4_inner_folds(
        historical_frame,
    )


    n_rows = len(
        historical_frame
    )


    treatment = (
        historical_frame[
            "treatment"
        ]
        .to_numpy(
            dtype="int8"
        )
    )


    outcome = (
        historical_frame[
            "target_sales"
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    fold_array = (
        historical_frame[
            "inner_fold"
        ]
        .to_numpy(
            dtype="int8"
        )
    )


    # =============================================================
    # 5A. OOF PROPENSITY ESTIMATION
    # =============================================================

    propensity_oof = np.full(
        n_rows,
        np.nan,
        dtype="float64",
    )


    for fold_id in range(
        1,
        V4_INNER_FOLDS + 1,
    ):

        training_mask = (
            fold_array
            != fold_id
        )


        holdout_mask = (
            fold_array
            == fold_id
        )


        X_train = prepare_v4_X(
            historical_frame.loc[
                training_mask
            ]
        )


        X_holdout = prepare_v4_X(
            historical_frame.loc[
                holdout_mask
            ]
        )


        y_train = (
            treatment[
                training_mask
            ]
        )


        propensity_model = XGBClassifier(
            **V4_PROPENSITY_MODEL_PARAMS
        )


        propensity_model.fit(
            X_train,
            y_train,
        )


        propensity_oof[
            holdout_mask
        ] = (
            propensity_model
            .predict_proba(
                X_holdout
            )[
                :,
                1,
            ]
        )


        del (
            X_train,
            X_holdout,
            y_train,
            propensity_model,
        )


        gc.collect()


    assert np.isfinite(
        propensity_oof
    ).all()


    historical_frame[
        "v4_propensity_oof"
    ] = propensity_oof


    propensity_auc = roc_auc_score(
        treatment,
        propensity_oof,
    )


    propensity_logloss = log_loss(
        treatment,
        propensity_oof,
    )


    propensity_brier = brier_score_loss(
        treatment,
        propensity_oof,
    )


    overlap_mask = (
        (
            propensity_oof
            >= V4_PROPENSITY_SUPPORT_LOW
        )
        &
        (
            propensity_oof
            <= V4_PROPENSITY_SUPPORT_HIGH
        )
    )


    overlap_rows = int(
        overlap_mask.sum()
    )


    overlap_share = (
        overlap_mask.mean()
    )


    print(
        "Historical rows:",
        f"{n_rows:,}",
    )


    print(
        "Treatment rate:",
        f"{treatment.mean():.2%}",
    )


    print(
        "OOF propensity AUC:",
        f"{propensity_auc:.4f}",
    )


    print(
        "Practical-overlap rows:",
        f"{overlap_rows:,}",
        f"({overlap_share:.2%})",
    )


    # =============================================================
    # 5B. OOF AIPW OUTCOME NUISANCE MODELS
    # =============================================================
    #
    # These models are distinct from the Tweedie ranking models.
    # =============================================================

    aipw_m0_oof = np.full(
        n_rows,
        np.nan,
        dtype="float64",
    )


    aipw_m1_oof = np.full(
        n_rows,
        np.nan,
        dtype="float64",
    )


    for fold_id in range(
        1,
        V4_INNER_FOLDS + 1,
    ):

        training_overlap_mask = (
            overlap_mask
            &
            (
                fold_array
                != fold_id
            )
        )


        holdout_overlap_mask = (
            overlap_mask
            &
            (
                fold_array
                == fold_id
            )
        )


        control_training_mask = (
            training_overlap_mask
            &
            (
                treatment
                == 0
            )
        )


        treated_training_mask = (
            training_overlap_mask
            &
            (
                treatment
                == 1
            )
        )


        assert (
            control_training_mask.sum()
            > 0
        )


        assert (
            treated_training_mask.sum()
            > 0
        )


        X_control_train = prepare_v4_X(
            historical_frame.loc[
                control_training_mask
            ]
        )


        X_treated_train = prepare_v4_X(
            historical_frame.loc[
                treated_training_mask
            ]
        )


        X_holdout = prepare_v4_X(
            historical_frame.loc[
                holdout_overlap_mask
            ]
        )


        y_control_train = (
            outcome[
                control_training_mask
            ]
        )


        y_treated_train = (
            outcome[
                treated_training_mask
            ]
        )


        aipw_m0_model = XGBRegressor(
            **V4_AIPW_OUTCOME_MODEL_PARAMS
        )


        aipw_m1_model = XGBRegressor(
            **V4_AIPW_OUTCOME_MODEL_PARAMS
        )


        aipw_m0_model.fit(
            X_control_train,
            y_control_train,
        )


        aipw_m1_model.fit(
            X_treated_train,
            y_treated_train,
        )


        aipw_m0_oof[
            holdout_overlap_mask
        ] = np.clip(
            aipw_m0_model.predict(
                X_holdout
            ),
            a_min=0.0,
            a_max=None,
        )


        aipw_m1_oof[
            holdout_overlap_mask
        ] = np.clip(
            aipw_m1_model.predict(
                X_holdout
            ),
            a_min=0.0,
            a_max=None,
        )


        del (
            X_control_train,
            X_treated_train,
            X_holdout,
            y_control_train,
            y_treated_train,
            aipw_m0_model,
            aipw_m1_model,
        )


        gc.collect()


    assert np.isfinite(
        aipw_m0_oof[
            overlap_mask
        ]
    ).all()


    assert np.isfinite(
        aipw_m1_oof[
            overlap_mask
        ]
    ).all()


    # =============================================================
    # 5C. DOUBLY ROBUST POTENTIAL-OUTCOME SCORES
    # =============================================================

    phi0 = np.full(
        n_rows,
        np.nan,
        dtype="float64",
    )


    phi1 = np.full(
        n_rows,
        np.nan,
        dtype="float64",
    )


    e_overlap = (
        propensity_oof[
            overlap_mask
        ]
    )


    d_overlap = (
        treatment[
            overlap_mask
        ]
    )


    y_overlap = (
        outcome[
            overlap_mask
        ]
    )


    m0_overlap = (
        aipw_m0_oof[
            overlap_mask
        ]
    )


    m1_overlap = (
        aipw_m1_oof[
            overlap_mask
        ]
    )


    phi0[
        overlap_mask
    ] = (
        m0_overlap
        +
        (
            1
            - d_overlap
        )
        /
        (
            1.0
            - e_overlap
        )
        *
        (
            y_overlap
            - m0_overlap
        )
    )


    phi1[
        overlap_mask
    ] = (
        m1_overlap
        +
        d_overlap
        /
        e_overlap
        *
        (
            y_overlap
            - m1_overlap
        )
    )


    # =============================================================
    # 5D. SEPARATE OOF TWEEEDIE RELATIVE-RANKING MODELS
    # =============================================================

    rank_mu0_oof = np.full(
        n_rows,
        np.nan,
        dtype="float64",
    )


    rank_mu1_oof = np.full(
        n_rows,
        np.nan,
        dtype="float64",
    )


    for fold_id in range(
        1,
        V4_INNER_FOLDS + 1,
    ):

        training_overlap_mask = (
            overlap_mask
            &
            (
                fold_array
                != fold_id
            )
        )


        holdout_overlap_mask = (
            overlap_mask
            &
            (
                fold_array
                == fold_id
            )
        )


        control_training_mask = (
            training_overlap_mask
            &
            (
                treatment
                == 0
            )
        )


        treated_training_mask = (
            training_overlap_mask
            &
            (
                treatment
                == 1
            )
        )


        X_control_train = prepare_v4_X(
            historical_frame.loc[
                control_training_mask
            ]
        )


        X_treated_train = prepare_v4_X(
            historical_frame.loc[
                treated_training_mask
            ]
        )


        X_holdout = prepare_v4_X(
            historical_frame.loc[
                holdout_overlap_mask
            ]
        )


        y_control_train = (
            outcome[
                control_training_mask
            ]
        )


        y_treated_train = (
            outcome[
                treated_training_mask
            ]
        )


        rank_m0_model = XGBRegressor(
            **V4_RELATIVE_RANK_MODEL_PARAMS
        )


        rank_m1_model = XGBRegressor(
            **V4_RELATIVE_RANK_MODEL_PARAMS
        )


        rank_m0_model.fit(
            X_control_train,
            y_control_train,
        )


        rank_m1_model.fit(
            X_treated_train,
            y_treated_train,
        )


        rank_mu0_oof[
            holdout_overlap_mask
        ] = (
            rank_m0_model.predict(
                X_holdout
            )
        )


        rank_mu1_oof[
            holdout_overlap_mask
        ] = (
            rank_m1_model.predict(
                X_holdout
            )
        )


        del (
            X_control_train,
            X_treated_train,
            X_holdout,
            y_control_train,
            y_treated_train,
            rank_m0_model,
            rank_m1_model,
        )


        gc.collect()


    assert (
        rank_mu0_oof[
            overlap_mask
        ]
        > 0
    ).all()


    assert (
        rank_mu1_oof[
            overlap_mask
        ]
        > 0
    ).all()


    # =============================================================
    # 5E. ORIGIN-SPECIFIC BASELINE FLOOR
    # =============================================================

    baseline_floor = np.quantile(
        rank_mu0_oof[
            overlap_mask
        ],
        V4_BASELINE_FLOOR_QUANTILE,
    )


    relative_domain_mask = (
        overlap_mask
        &
        (
            rank_mu0_oof
            >= baseline_floor
        )
    )


    relative_domain_rows = int(
        relative_domain_mask.sum()
    )


    relative_domain_share_of_overlap = (
        relative_domain_rows
        /
        overlap_rows
    )


    print(
        "Point-in-time baseline floor:",
        f"{baseline_floor:,.3f}",
    )


    print(
        "Relative calibration rows:",
        f"{relative_domain_rows:,}",
        (
            f"("
            f"{relative_domain_share_of_overlap:.2%} "
            f"of overlap)"
        ),
    )


    # =============================================================
    # 5F. OOF RAW RELATIVE RANKING SCORE
    # =============================================================

    relative_score_oof = np.full(
        n_rows,
        np.nan,
        dtype="float64",
    )


    relative_score_oof[
        relative_domain_mask
    ] = (
        rank_mu1_oof[
            relative_domain_mask
        ]
        /
        rank_mu0_oof[
            relative_domain_mask
        ]
        - 1.0
    )


    # =============================================================
    # 5G. WITHIN-INNER-FOLD PERCENTILE RANKS
    # =============================================================

    relative_rank = np.full(
        n_rows,
        np.nan,
        dtype="float64",
    )


    for fold_id in range(
        1,
        V4_INNER_FOLDS + 1,
    ):

        fold_relative_mask = (
            relative_domain_mask
            &
            (
                fold_array
                == fold_id
            )
        )


        fold_scores = pd.Series(
            relative_score_oof[
                fold_relative_mask
            ]
        )


        relative_rank[
            fold_relative_mask
        ] = (
            fold_scores
            .rank(
                method="average",
                pct=True,
            )
            .to_numpy()
        )


    assert np.isfinite(
        relative_rank[
            relative_domain_mask
        ]
    ).all()


    # Convert percentile ranks into five ordered groups.
    #
    # rank in (0, 0.2] -> 1
    # ...
    # rank in (0.8, 1.0] -> 5

    calibration_bin = np.full(
        n_rows,
        np.nan,
        dtype="float64",
    )


    calibration_bin[
        relative_domain_mask
    ] = np.ceil(
        relative_rank[
            relative_domain_mask
        ]
        *
        V4_UPLIFT_CALIBRATION_BINS
    )


    calibration_bin[
        relative_domain_mask
    ] = np.clip(
        calibration_bin[
            relative_domain_mask
        ],
        1,
        V4_UPLIFT_CALIBRATION_BINS,
    )


    # =============================================================
    # 5H. GROUP-LEVEL DOUBLY ROBUST CALIBRATION
    # =============================================================

    calibration_frame = pd.DataFrame(
        {
            "calibration_bin":
                calibration_bin[
                    relative_domain_mask
                ]
                .astype(
                    "int8"
                ),

            "raw_relative_score":
                relative_score_oof[
                    relative_domain_mask
                ],

            "phi0":
                phi0[
                    relative_domain_mask
                ],

            "phi1":
                phi1[
                    relative_domain_mask
                ],

            "treatment":
                treatment[
                    relative_domain_mask
                ],

            "baseline":
                rank_mu0_oof[
                    relative_domain_mask
                ],
        }
    )


    origin_calibration = (
        calibration_frame
        .groupby(
            "calibration_bin",
            observed=True,
        )
        .agg(
            rows=(
                "raw_relative_score",
                "size",
            ),

            treatment_rate=(
                "treatment",
                "mean",
            ),

            mean_baseline=(
                "baseline",
                "mean",
            ),

            median_raw_relative_score=(
                "raw_relative_score",
                "median",
            ),

            mean_phi0=(
                "phi0",
                "mean",
            ),

            mean_phi1=(
                "phi1",
                "mean",
            ),
        )
        .reset_index()
    )


    assert (
        origin_calibration[
            "mean_phi0"
        ]
        > 0
    ).all(), (
        "A calibration bin has a non-positive DR no-promotion mean."
    )


    assert (
        origin_calibration[
            "mean_phi1"
        ]
        > 0
    ).all(), (
        "A calibration bin has a non-positive DR promotion mean."
    )


    origin_calibration[
        "dr_relative_uplift"
    ] = (
        origin_calibration[
            "mean_phi1"
        ]
        /
        origin_calibration[
            "mean_phi0"
        ]
        - 1.0
    )


    origin_calibration[
        "forecast_origin"
    ] = (
        forecast_origin
    )


    origin_calibration = (
        origin_calibration[
            [
                "forecast_origin",
                "calibration_bin",
                "rows",
                "treatment_rate",
                "mean_baseline",
                "median_raw_relative_score",
                "mean_phi0",
                "mean_phi1",
                "dr_relative_uplift",
            ]
        ]
    )


    # =============================================================
    # 5I. ORIGIN-LEVEL DIAGNOSTICS
    # =============================================================

    overall_phi0 = (
        calibration_frame[
            "phi0"
        ]
        .mean()
    )


    overall_phi1 = (
        calibration_frame[
            "phi1"
        ]
        .mean()
    )


    overall_dr_relative_uplift = (
        overall_phi1
        /
        overall_phi0
        - 1.0
    )


    bin_effects = (
        origin_calibration[
            "dr_relative_uplift"
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    adjacent_increases = int(
        (
            np.diff(
                bin_effects
            )
            > 0
        )
        .sum()
    )


    top_bottom_spread = (
        bin_effects[
            -1
        ]
        -
        bin_effects[
            0
        ]
    )


    origin_minutes = (
        time.time()
        - origin_start_time
    ) / 60.0


    v4_origin_summary_rows.append(
        {
            "forecast_origin":
                forecast_origin,

            "historical_rows":
                n_rows,

            "treatment_rate":
                treatment.mean(),

            "propensity_auc":
                propensity_auc,

            "propensity_logloss":
                propensity_logloss,

            "propensity_brier":
                propensity_brier,

            "overlap_rows":
                overlap_rows,

            "overlap_share":
                overlap_share,

            "baseline_floor":
                baseline_floor,

            "relative_domain_rows":
                relative_domain_rows,

            "relative_domain_share_of_overlap":
                relative_domain_share_of_overlap,

            "overall_dr_relative_uplift":
                overall_dr_relative_uplift,

            "adjacent_bin_increases":
                adjacent_increases,

            "top_bottom_relative_spread":
                top_bottom_spread,

            "fit_minutes":
                origin_minutes,
        }
    )


    v4_calibration_rows.append(
        origin_calibration
    )


    # Store only the compact information required by Section 9.3.
    #
    # Full million-row OOF arrays are deliberately NOT retained for all
    # six origins.

    v4_origin_causal_specs[
        forecast_origin
    ] = {
        "training_start":
            training_start,

        "baseline_floor":
            float(
                baseline_floor
            ),

        "calibration":
            origin_calibration.copy(),
    }


    print()
    print(
        "Past-only causal calibration:"
    )


    display(
        origin_calibration
        .style.hide(
            axis="index"
        )
        .format(
            {
                "treatment_rate":
                    "{:.2%}",

                "mean_baseline":
                    "{:,.2f}",

                "median_raw_relative_score":
                    "{:+.2%}",

                "mean_phi0":
                    "{:,.2f}",

                "mean_phi1":
                    "{:,.2f}",

                "dr_relative_uplift":
                    "{:+.2%}",
            }
        )
    )


    print(
        "Overall historical DR relative uplift:",
        f"{overall_dr_relative_uplift:+.2%}",
    )


    print(
        "Adjacent calibration-bin increases:",
        f"{adjacent_increases}/4",
    )


    print(
        "Top-minus-bottom relative spread:",
        f"{top_bottom_spread:+.2%}",
    )


    print(
        "Origin fit minutes:",
        f"{origin_minutes:.2f}",
    )


    # -------------------------------------------------------------
    # Release large origin-level arrays before moving to the next T.
    # -------------------------------------------------------------

    del (
        historical_frame,
        treatment,
        outcome,
        fold_array,
        propensity_oof,
        aipw_m0_oof,
        aipw_m1_oof,
        phi0,
        phi1,
        rank_mu0_oof,
        rank_mu1_oof,
        relative_score_oof,
        relative_rank,
        calibration_bin,
        calibration_frame,
    )


    gc.collect()


# =====================================================================
# 6. COMBINE ORIGIN-LEVEL RESULTS
# =====================================================================

v4_origin_summary = pd.DataFrame(
    v4_origin_summary_rows
)


v4_causal_calibration = pd.concat(
    v4_calibration_rows,
    ignore_index=True,
)


# =====================================================================
# 7. DISPLAY GLOBAL V4 HISTORICAL CAUSAL SUMMARY
# =====================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "V4 PAST-ONLY CAUSAL ESTIMATION SUMMARY"
)

print(
    "=" * 80
)


display(
    v4_origin_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "treatment_rate":
                "{:.2%}",

            "propensity_auc":
                "{:.4f}",

            "propensity_logloss":
                "{:.4f}",

            "propensity_brier":
                "{:.4f}",

            "overlap_share":
                "{:.2%}",

            "baseline_floor":
                "{:,.3f}",

            "relative_domain_share_of_overlap":
                "{:.2%}",

            "overall_dr_relative_uplift":
                "{:+.2%}",

            "top_bottom_relative_spread":
                "{:+.2%}",

            "fit_minutes":
                "{:.2f}",
        }
    )
)


# =====================================================================
# 8. FINAL CONTRACT CHECKS
# =====================================================================

assert len(
    v4_origin_summary
) == len(
    V4_FORECAST_ORIGINS
)


assert (
    v4_causal_calibration
    .groupby(
        "forecast_origin"
    )[
        "calibration_bin"
    ]
    .nunique()
    .eq(
        V4_UPLIFT_CALIBRATION_BINS
    )
    .all()
)


assert np.isfinite(
    v4_origin_summary[
        "baseline_floor"
    ]
).all()


print()
print(
    "V4 HISTORICAL CAUSAL MODELS + CALIBRATION: PASSED"
)


V4 HISTORICAL CAUSAL ESTIMATION
Forecast origin: 2017-05-09
Historical window: 2015-05-09 to 2017-05-09
Historical rows: 1,266,441
Treatment rate: 33.82%
OOF propensity AUC: 0.9433
Practical-overlap rows: 772,350 (60.99%)
Point-in-time baseline floor: 14.972
Relative calibration rows: 579,262 (75.00% of overlap)

Past-only causal calibration:


forecast_origin,calibration_bin,rows,treatment_rate,mean_baseline,median_raw_relative_score,mean_phi0,mean_phi1,dr_relative_uplift
2017-05-09 00:00:00,1,115851,47.45%,525.24,-7.40%,386.83,484.72,+25.31%
2017-05-09 00:00:00,2,115852,53.02%,498.62,+1.10%,443.01,493.11,+11.31%
2017-05-09 00:00:00,3,115853,55.69%,465.97,+7.31%,459.81,487.52,+6.03%
2017-05-09 00:00:00,4,115852,57.05%,466.64,+17.41%,496.91,537.57,+8.18%
2017-05-09 00:00:00,5,115854,57.63%,435.30,+53.81%,720.98,862.94,+19.69%


Overall historical DR relative uplift: +14.29%
Adjacent calibration-bin increases: 2/4
Top-minus-bottom relative spread: -5.62%
Origin fit minutes: 4.99

V4 HISTORICAL CAUSAL ESTIMATION
Forecast origin: 2017-05-23
Historical window: 2015-05-23 to 2017-05-23
Historical rows: 1,268,289
Treatment rate: 34.31%
OOF propensity AUC: 0.9437
Practical-overlap rows: 763,999 (60.24%)
Point-in-time baseline floor: 15.757
Relative calibration rows: 572,999 (75.00% of overlap)

Past-only causal calibration:


forecast_origin,calibration_bin,rows,treatment_rate,mean_baseline,median_raw_relative_score,mean_phi0,mean_phi1,dr_relative_uplift
2017-05-23 00:00:00,1,114598,45.61%,562.56,-5.22%,527.69,541.25,+2.57%
2017-05-23 00:00:00,2,114600,52.71%,515.68,+2.76%,510.09,531.05,+4.11%
2017-05-23 00:00:00,3,114599,56.84%,498.33,+9.30%,517.66,540.47,+4.41%
2017-05-23 00:00:00,4,114600,59.25%,509.00,+20.21%,587.14,598.62,+1.96%
2017-05-23 00:00:00,5,114602,59.64%,411.62,+59.11%,555.21,828.82,+49.28%


Overall historical DR relative uplift: +12.69%
Adjacent calibration-bin increases: 3/4
Top-minus-bottom relative spread: +46.71%
Origin fit minutes: 4.94

V4 HISTORICAL CAUSAL ESTIMATION
Forecast origin: 2017-06-06
Historical window: 2015-06-06 to 2017-06-06
Historical rows: 1,270,137
Treatment rate: 34.68%
OOF propensity AUC: 0.9435
Practical-overlap rows: 744,571 (58.62%)
Point-in-time baseline floor: 14.235
Relative calibration rows: 558,428 (75.00% of overlap)

Past-only causal calibration:


forecast_origin,calibration_bin,rows,treatment_rate,mean_baseline,median_raw_relative_score,mean_phi0,mean_phi1,dr_relative_uplift
2017-06-06 00:00:00,1,111684,45.81%,509.03,-4.55%,495.07,520.28,+5.09%
2017-06-06 00:00:00,2,111685,50.65%,458.67,+4.98%,476.94,504.87,+5.85%
2017-06-06 00:00:00,3,111686,53.83%,430.43,+14.01%,474.99,503.51,+6.00%
2017-06-06 00:00:00,4,111685,56.48%,430.68,+30.88%,486.53,571.40,+17.44%
2017-06-06 00:00:00,5,111688,60.60%,400.47,+84.83%,690.27,"1,009.70",+46.28%


... (5 outputs left out)

forecast_origin,calibration_bin,rows,treatment_rate,mean_baseline,median_raw_relative_score,mean_phi0,mean_phi1,dr_relative_uplift
2017-07-18 00:00:00,1,113789,45.32%,428.03,-2.26%,414.01,445.17,+7.53%
2017-07-18 00:00:00,2,113791,49.03%,338.31,+6.68%,386.28,433.92,+12.33%
2017-07-18 00:00:00,3,113793,54.12%,298.44,+17.17%,419.02,464.15,+10.77%
2017-07-18 00:00:00,4,113791,56.96%,257.06,+36.33%,478.55,520.87,+8.84%
2017-07-18 00:00:00,5,113794,60.50%,215.54,+104.96%,964.68,861.02,-10.75%


Overall historical DR relative uplift: +2.35%
Adjacent calibration-bin increases: 1/4
Top-minus-bottom relative spread: -18.27%
Origin fit minutes: 4.72

V4 PAST-ONLY CAUSAL ESTIMATION SUMMARY


forecast_origin,historical_rows,treatment_rate,propensity_auc,propensity_logloss,propensity_brier,overlap_rows,overlap_share,baseline_floor,relative_domain_rows,relative_domain_share_of_overlap,overall_dr_relative_uplift,adjacent_bin_increases,top_bottom_relative_spread,fit_minutes
2017-05-09 00:00:00,1266441,33.82%,0.9433,0.2960,0.0898,772350,60.99%,14.972,579262,75.00%,+14.29%,2,-5.62%,4.99
2017-05-23 00:00:00,1268289,34.31%,0.9437,0.2966,0.0895,763999,60.24%,15.757,572999,75.00%,+12.69%,3,+46.71%,4.94
2017-06-06 00:00:00,1270137,34.68%,0.9435,0.3057,0.0913,744571,58.62%,14.235,558428,75.00%,+18.52%,4,+41.18%,4.69
2017-06-20 00:00:00,1271985,35.03%,0.9428,0.3057,0.0931,737339,57.97%,12.517,553004,75.00%,+2.40%,3,+0.46%,4.65
2017-07-04 00:00:00,1273833,35.38%,0.9433,0.3043,0.0928,767720,60.27%,11.842,575790,75.00%,-5.19%,1,-18.88%,4.72
2017-07-18 00:00:00,1275681,35.79%,0.9464,0.2966,0.0894,758611,59.47%,9.594,568958,75.00%,+2.35%,1,-18.27%,4.72



V4 HISTORICAL CAUSAL MODELS + CALIBRATION: PASSED


### 9.3 V4 Forecast Construction

The past-only causal analysis in Section 9.2 reveals substantial temporal
variation in heterogeneous promotion response.

At some forecast origins, historical out-of-fold uplift ranks clearly separate
higher-response from lower-response contexts.

At other origins, the ranking is weak or even inverted.

A production forecasting system should not force heterogeneous causal
adjustments when the historical evidence supporting that heterogeneity is weak.

V4 therefore introduces a **past-only heterogeneity reliability guardrail**
before any future forecast performance is inspected.

---

#### Heterogeneity reliability guardrail

For each forecast origin $T$, segmented uplift calibration is considered
reliable when:

1. at least three of the four transitions between the five historical
   calibration bins show an increasing doubly robust relative effect;

and

2. the highest predicted-response bin has a larger estimated causal effect
   than the overall supported historical population.

Formally, segmented calibration is used only if:

$$
\text{AdjacentIncreases}_T
\geq
3
$$

and:

$$
\hat\rho^{DR}_{T,Q_5}
>
\hat\rho^{DR}_{T,\text{overall}}.
$$

This rule uses only historical out-of-fold causal diagnostics available by the
forecast origin.

It is defined before inspecting any V4 future forecast errors.

---

If the historical heterogeneous ranking passes the reliability guardrail, V4
uses the five rank-specific causal effects:

$$
\hat\rho^{DR}_{T,Q_1},
\ldots,
\hat\rho^{DR}_{T,Q_5}.
$$

If the ranking does not pass, V4 falls back to the pooled historical relative
effect:

$$
\hat\rho^{DR}_{T,\text{overall}}.
$$

The fallback preserves the historical causal level while refusing to use
unsupported heterogeneous structure.

This creates two possible operational modes:

- **segmented calibration** when past heterogeneity is sufficiently supported;
- **pooled calibration** when only the average causal effect is considered
  reliable.

---

#### Final point-in-time models

After historical cross-fitting and calibration are complete, V4 refits the
models required for future prediction on the complete historical window
available at origin $T$.

A final propensity model estimates:

$$
\hat e_T(X)
$$

for future target contexts.

Separate Tweedie outcome models estimate:

$$
\hat\mu_{0,T}(X)
$$

and:

$$
\hat\mu_{1,T}(X).
$$

Their ratio produces the future ranking score:

$$
s_T(X)
=
\frac{
\hat\mu_{1,T}(X)
}{
\hat\mu_{0,T}(X)
}
-1.
$$

The raw score is not interpreted directly as a causal percentage.

Instead, it determines the target context's position in the historical
response ranking.

---

#### Historical percentile transfer

The final relative T-learner is also applied to the historical practical-
overlap population available at $T$.

Historical score percentiles define the reference ranking distribution.

A future supported context is mapped to the percentile it would occupy in this
historical score distribution and then to one of the five calibration bins.

Therefore, the future mapping answers:

> How high is this future context's predicted promotion response relative to
> comparable historical contexts known at $T$?

It does not force exactly 20% of future observations into every uplift bin.

---

#### Final V4 forecast

For every future target context, the no-promotion baseline forecast is:

$$
\hat\mu_{0,T}(X).
$$

For a non-promoted target:

$$
D=0
\quad\Rightarrow\quad
\hat Y_{V4}
=
\hat\mu_{0,T}(X).
$$

For a promoted target satisfying both:

$$
0.01
\leq
\hat e_T(X)
\leq
0.99
$$

and:

$$
\hat\mu_{0,T}(X)
\geq
c_T,
$$

V4 applies the historical causal relative adjustment:

$$
\hat Y_{V4}
=
\hat\mu_{0,T}(X)
\left[
1+\hat\rho_T(X)
\right].
$$

For a promoted target outside this support domain:

$$
\hat\rho_T(X)=0,
$$

and the forecast falls back to:

$$
\hat Y_{V4}
=
\hat\mu_{0,T}(X).
$$

All V4 predictions are finally constrained to be non-negative.

The next section compares these forecasts against the previously developed
forecasting architectures on exactly the same six 14-day evaluation windows.

In [36]:
# ---------------------------------------------------------------------
# V4 final point-in-time forecast construction.
#
# At each outer origin T:
#
#   1. refit a propensity model on ALL historical data available by T;
#   2. define practical historical support using that model;
#   3. fit final Tweedie mu0 / mu1 models on historical supported rows;
#   4. build a historical reference distribution of relative-response
#      scores;
#   5. score the future 14-day target rows;
#   6. apply the origin-specific baseline floor from Section 9.2;
#   7. use segmented causal calibration only when historical ranking
#      diagnostics are reliable;
#   8. otherwise fall back to the pooled historical DR relative effect;
#   9. evaluate future forecasts only AFTER predictions are constructed.
#
# No future outcome is used anywhere in model fitting, support
# determination, calibration selection or prediction construction.
# ---------------------------------------------------------------------

import gc
import time

import numpy as np
import pandas as pd

from xgboost import (
    XGBClassifier,
    XGBRegressor,
)


# =====================================================================
# 1. FREEZE THE HETEROGENEITY-RELIABILITY GUARDRAIL
# =====================================================================
#
# This rule uses ONLY historical causal diagnostics from Section 9.2.
#
# It is defined before inspecting V4 future forecast errors.
# =====================================================================

V4_MIN_ADJACENT_BIN_INCREASES = 3


# =====================================================================
# 2. STORAGE
# =====================================================================

v4_forecast_rows = []

v4_forecast_summary_rows = []


# =====================================================================
# 3. OUTER FORECAST-ORIGIN LOOP
# =====================================================================

for protocol_row in (
    v4_protocol
    .itertuples(
        index=False
    )
):

    origin_start_time = time.time()


    forecast_origin = (
        protocol_row.forecast_origin
    )


    training_start = (
        protocol_row.causal_training_start
    )


    forecast_start = (
        protocol_row.forecast_start
    )


    forecast_end = (
        protocol_row.forecast_end
    )


    print(
        "\n"
        + "=" * 80
    )

    print(
        "V4 FINAL FORECAST"
    )

    print(
        "Forecast origin:",
        forecast_origin.date(),
    )

    print(
        "=" * 80
    )


    # -------------------------------------------------------------
    # Historical and future point-in-time samples.
    # -------------------------------------------------------------

    historical_frame = (
        v4_model_source_frame.loc[
            v4_model_source_frame[
                "date"
            ]
            .between(
                training_start,
                forecast_origin,
                inclusive="both",
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    future_frame = (
        v4_model_source_frame.loc[
            v4_model_source_frame[
                "date"
            ]
            .between(
                forecast_start,
                forecast_end,
                inclusive="both",
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    assert not historical_frame.empty

    assert not future_frame.empty


    assert (
        historical_frame[
            "date"
        ]
        .max()
        <= forecast_origin
    )


    assert (
        future_frame[
            "date"
        ]
        .min()
        > forecast_origin
    )


    historical_treatment = (
        historical_frame[
            "treatment"
        ]
        .to_numpy(
            dtype="int8"
        )
    )


    future_treatment = (
        future_frame[
            "treatment"
        ]
        .to_numpy(
            dtype="int8"
        )
    )


    future_actual_sales = (
        future_frame[
            "target_sales"
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    # =============================================================
    # 3A. FINAL PROPENSITY MODEL
    # =============================================================
    #
    # This model is fit on the complete historical window because all
    # of those observations are available at forecast origin T.
    # =============================================================

    X_historical = prepare_v4_X(
        historical_frame
    )


    X_future = prepare_v4_X(
        future_frame
    )


    final_propensity_model = XGBClassifier(
        **V4_PROPENSITY_MODEL_PARAMS
    )


    final_propensity_model.fit(
        X_historical,
        historical_treatment,
    )


    historical_propensity_full = (
        final_propensity_model
        .predict_proba(
            X_historical
        )[
            :,
            1,
        ]
    )


    future_propensity = (
        final_propensity_model
        .predict_proba(
            X_future
        )[
            :,
            1,
        ]
    )


    historical_overlap_full = (
        (
            historical_propensity_full
            >= V4_PROPENSITY_SUPPORT_LOW
        )
        &
        (
            historical_propensity_full
            <= V4_PROPENSITY_SUPPORT_HIGH
        )
    )


    future_overlap = (
        (
            future_propensity
            >= V4_PROPENSITY_SUPPORT_LOW
        )
        &
        (
            future_propensity
            <= V4_PROPENSITY_SUPPORT_HIGH
        )
    )


    # =============================================================
    # 3B. FINAL RELATIVE T-LEARNER
    # =============================================================
    #
    # Fit separate no-promotion and promotion outcome models using the
    # practical historical-support population.
    # =============================================================

    historical_control_mask = (
        historical_overlap_full
        &
        (
            historical_treatment
            == 0
        )
    )


    historical_treated_mask = (
        historical_overlap_full
        &
        (
            historical_treatment
            == 1
        )
    )


    assert (
        historical_control_mask.sum()
        > 0
    )


    assert (
        historical_treated_mask.sum()
        > 0
    )


    X_control_train = prepare_v4_X(
        historical_frame.loc[
            historical_control_mask
        ]
    )


    X_treated_train = prepare_v4_X(
        historical_frame.loc[
            historical_treated_mask
        ]
    )


    y_control_train = (
        historical_frame.loc[
            historical_control_mask,
            "target_sales",
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    y_treated_train = (
        historical_frame.loc[
            historical_treated_mask,
            "target_sales",
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    final_mu0_model = XGBRegressor(
        **V4_RELATIVE_RANK_MODEL_PARAMS
    )


    final_mu1_model = XGBRegressor(
        **V4_RELATIVE_RANK_MODEL_PARAMS
    )


    final_mu0_model.fit(
        X_control_train,
        y_control_train,
    )


    final_mu1_model.fit(
        X_treated_train,
        y_treated_train,
    )


    # =============================================================
    # 3C. HISTORICAL REFERENCE SCORE DISTRIBUTION
    # =============================================================
    #
    # The final T-learner is applied to historical practical-overlap
    # rows in order to construct the score distribution against which
    # future contexts will be ranked.
    #
    # This step uses no future outcome.
    # =============================================================

    historical_overlap_frame = (
        historical_frame.loc[
            historical_overlap_full
        ]
    )


    X_historical_overlap = prepare_v4_X(
        historical_overlap_frame
    )


    historical_mu0_final = (
        final_mu0_model
        .predict(
            X_historical_overlap
        )
        .astype(
            "float64"
        )
    )


    historical_mu1_final = (
        final_mu1_model
        .predict(
            X_historical_overlap
        )
        .astype(
            "float64"
        )
    )


    assert (
        historical_mu0_final
        > 0
    ).all()


    assert (
        historical_mu1_final
        > 0
    ).all()


    # Origin-specific numerical floor previously estimated entirely
    # from historical OOF predictions in Section 9.2.

    origin_summary_row = (
        v4_origin_summary.loc[
            v4_origin_summary[
                "forecast_origin"
            ]
            .eq(
                forecast_origin
            )
        ]
    )


    assert (
        len(
            origin_summary_row
        )
        == 1
    )


    baseline_floor = float(
        origin_summary_row[
            "baseline_floor"
        ]
        .iloc[
            0
        ]
    )


    overall_dr_relative_uplift = float(
        origin_summary_row[
            "overall_dr_relative_uplift"
        ]
        .iloc[
            0
        ]
    )


    adjacent_bin_increases = int(
        origin_summary_row[
            "adjacent_bin_increases"
        ]
        .iloc[
            0
        ]
    )


    historical_relative_domain_full = (
        historical_mu0_final
        >= baseline_floor
    )


    historical_relative_score = (
        historical_mu1_final[
            historical_relative_domain_full
        ]
        /
        historical_mu0_final[
            historical_relative_domain_full
        ]
        - 1.0
    )


    assert (
        len(
            historical_relative_score
        )
        > 0
    )


    historical_relative_score_sorted = np.sort(
        historical_relative_score
    )


    # =============================================================
    # 3D. FUTURE BASELINE AND RELATIVE RESPONSE SCORE
    # =============================================================

    future_mu0 = (
        final_mu0_model
        .predict(
            X_future
        )
        .astype(
            "float64"
        )
    )


    future_mu1 = (
        final_mu1_model
        .predict(
            X_future
        )
        .astype(
            "float64"
        )
    )


    assert (
        future_mu0
        > 0
    ).all()


    assert (
        future_mu1
        > 0
    ).all()


    future_raw_relative_score = (
        future_mu1
        /
        future_mu0
        - 1.0
    )


    future_baseline_supported = (
        future_mu0
        >= baseline_floor
    )


    future_causal_support = (
        future_overlap
        &
        future_baseline_supported
    )


    # =============================================================
    # 3E. MAP FUTURE SCORES TO HISTORICAL PERCENTILES
    # =============================================================
    #
    # For every causally supported future context:
    #
    # percentile =
    #     fraction of historical reference scores <= future score
    #
    # This transfers the HISTORICAL ranking scale to the future batch.
    # =============================================================

    future_score_percentile = np.full(
        len(
            future_frame
        ),
        np.nan,
        dtype="float64",
    )


    supported_future_scores = (
        future_raw_relative_score[
            future_causal_support
        ]
    )


    future_score_percentile[
        future_causal_support
    ] = (
        np.searchsorted(
            historical_relative_score_sorted,
            supported_future_scores,
            side="right",
        )
        /
        len(
            historical_relative_score_sorted
        )
    )


    # Convert percentile into calibration bins 1,...,5.

    future_calibration_bin = np.full(
        len(
            future_frame
        ),
        np.nan,
        dtype="float64",
    )


    future_calibration_bin[
        future_causal_support
    ] = np.ceil(
        future_score_percentile[
            future_causal_support
        ]
        *
        V4_UPLIFT_CALIBRATION_BINS
    )


    future_calibration_bin[
        future_causal_support
    ] = np.clip(
        future_calibration_bin[
            future_causal_support
        ],
        1,
        V4_UPLIFT_CALIBRATION_BINS,
    )


    # =============================================================
    # 3F. HISTORICAL CAUSAL CALIBRATION
    # =============================================================

    origin_calibration = (
        v4_causal_calibration.loc[
            v4_causal_calibration[
                "forecast_origin"
            ]
            .eq(
                forecast_origin
            )
        ]
        .sort_values(
            "calibration_bin"
        )
        .reset_index(
            drop=True
        )
    )


    assert (
        len(
            origin_calibration
        )
        ==
        V4_UPLIFT_CALIBRATION_BINS
    )


    top_bin_dr_relative_uplift = float(
        origin_calibration.loc[
            origin_calibration[
                "calibration_bin"
            ]
            .eq(
                V4_UPLIFT_CALIBRATION_BINS
            ),
            "dr_relative_uplift",
        ]
        .iloc[
            0
        ]
    )


    # -------------------------------------------------------------
    # Reliability gate.
    #
    # Segmented calibration is deployed only when:
    #
    #   - at least 3/4 historical transitions increase;
    #   - the historical top-response bin has a causal effect above
    #     the pooled historical effect.
    #
    # Otherwise the pooled historical causal effect is used.
    # -------------------------------------------------------------

    segmented_calibration_reliable = (
        (
            adjacent_bin_increases
            >= V4_MIN_ADJACENT_BIN_INCREASES
        )
        and
        (
            top_bin_dr_relative_uplift
            >
            overall_dr_relative_uplift
        )
    )


    if segmented_calibration_reliable:

        calibration_mode = (
            "segmented"
        )


        bin_to_relative_uplift = dict(
            zip(
                origin_calibration[
                    "calibration_bin"
                ]
                .astype(
                    "int8"
                ),

                origin_calibration[
                    "dr_relative_uplift"
                ]
                .astype(
                    "float64"
                ),
            )
        )

    else:

        calibration_mode = (
            "pooled"
        )


        bin_to_relative_uplift = {
            calibration_bin:
                overall_dr_relative_uplift

            for calibration_bin in range(
                1,
                V4_UPLIFT_CALIBRATION_BINS + 1,
            )
        }


    # =============================================================
    # 3G. BUILD FUTURE CAUSAL RELATIVE ADJUSTMENT
    # =============================================================

    future_calibrated_relative_uplift = np.zeros(
        len(
            future_frame
        ),
        dtype="float64",
    )


    for calibration_bin in range(
        1,
        V4_UPLIFT_CALIBRATION_BINS + 1,
    ):

        bin_mask = (
            future_causal_support
            &
            (
                future_calibration_bin
                == calibration_bin
            )
        )


        future_calibrated_relative_uplift[
            bin_mask
        ] = (
            bin_to_relative_uplift[
                calibration_bin
            ]
        )


    # -------------------------------------------------------------
    # Promotion adjustment is relevant only when D=1.
    #
    # Non-promoted rows use the no-promotion baseline.
    #
    # Promoted rows without causal support also use the baseline.
    # -------------------------------------------------------------

    v4_prediction = (
        future_mu0
        *
        (
            1.0
            +
            future_treatment
            *
            future_calibrated_relative_uplift
        )
    )


    v4_prediction = np.clip(
        v4_prediction,
        a_min=0.0,
        a_max=None,
    )


    # =============================================================
    # 3H. DIAGNOSTIC ALTERNATIVES
    # =============================================================
    #
    # These are NOT additional selected champions.
    #
    # They help isolate the value of the causal adjustment:
    #
    # baseline-only:
    #     mu0 for every target row
    #
    # raw T-learner:
    #     mu0 if no promotion, mu1 if promotion
    # =============================================================

    baseline_only_prediction = (
        future_mu0.copy()
    )


    raw_t_learner_prediction = np.where(
        future_treatment
        == 1,
        future_mu1,
        future_mu0,
    )


    raw_t_learner_prediction = np.clip(
        raw_t_learner_prediction,
        a_min=0.0,
        a_max=None,
    )


    # =============================================================
    # 3I. FUTURE EVALUATION
    # =============================================================
    #
    # Actual future sales enter the pipeline ONLY here, after all
    # predictions and deployment decisions have already been frozen.
    # =============================================================

    actual_total_sales = (
        future_actual_sales.sum()
    )


    v4_absolute_error = np.abs(
        v4_prediction
        - future_actual_sales
    )


    baseline_absolute_error = np.abs(
        baseline_only_prediction
        - future_actual_sales
    )


    raw_t_absolute_error = np.abs(
        raw_t_learner_prediction
        - future_actual_sales
    )


    v4_wape = (
        v4_absolute_error.sum()
        /
        actual_total_sales
    )


    baseline_only_wape = (
        baseline_absolute_error.sum()
        /
        actual_total_sales
    )


    raw_t_learner_wape = (
        raw_t_absolute_error.sum()
        /
        actual_total_sales
    )


    v4_bias = (
        (
            v4_prediction
            - future_actual_sales
        )
        .sum()
        /
        actual_total_sales
    )


    promoted_mask = (
        future_treatment
        == 1
    )


    nonpromoted_mask = (
        future_treatment
        == 0
    )


    promoted_actual_sales = (
        future_actual_sales[
            promoted_mask
        ]
        .sum()
    )


    nonpromoted_actual_sales = (
        future_actual_sales[
            nonpromoted_mask
        ]
        .sum()
    )


    promoted_v4_wape = (
        np.abs(
            v4_prediction[
                promoted_mask
            ]
            -
            future_actual_sales[
                promoted_mask
            ]
        )
        .sum()
        /
        promoted_actual_sales
    )


    nonpromoted_v4_wape = (
        np.abs(
            v4_prediction[
                nonpromoted_mask
            ]
            -
            future_actual_sales[
                nonpromoted_mask
            ]
        )
        .sum()
        /
        nonpromoted_actual_sales
    )


    supported_promoted_mask = (
        promoted_mask
        &
        future_causal_support
    )


    supported_promoted_rows = int(
        supported_promoted_mask.sum()
    )


    promoted_rows = int(
        promoted_mask.sum()
    )


    promoted_support_share = (
        supported_promoted_rows
        /
        promoted_rows
    )


    # =============================================================
    # 3J. STORE FORECAST ROWS
    # =============================================================

    origin_forecast_frame = (
        future_frame[
            [
                "date",
                "store_nbr",
                "family",
                "target_sales",
                "treatment",
            ]
        ]
        .copy()
    )


    origin_forecast_frame[
        "forecast_origin"
    ] = (
        forecast_origin
    )


    origin_forecast_frame[
        "v4_propensity"
    ] = (
        future_propensity
    )


    origin_forecast_frame[
        "v4_mu0"
    ] = (
        future_mu0
    )


    origin_forecast_frame[
        "v4_mu1"
    ] = (
        future_mu1
    )


    origin_forecast_frame[
        "v4_raw_relative_score"
    ] = (
        future_raw_relative_score
    )


    origin_forecast_frame[
        "v4_causal_support"
    ] = (
        future_causal_support
    )


    origin_forecast_frame[
        "v4_score_percentile"
    ] = (
        future_score_percentile
    )


    origin_forecast_frame[
        "v4_calibration_bin"
    ] = (
        future_calibration_bin
    )


    origin_forecast_frame[
        "v4_calibrated_relative_uplift"
    ] = (
        future_calibrated_relative_uplift
    )


    origin_forecast_frame[
        "v4_baseline_only_prediction"
    ] = (
        baseline_only_prediction
    )


    origin_forecast_frame[
        "v4_raw_t_learner_prediction"
    ] = (
        raw_t_learner_prediction
    )


    origin_forecast_frame[
        "v4_prediction"
    ] = (
        v4_prediction
    )


    origin_forecast_frame[
        "v4_calibration_mode"
    ] = (
        calibration_mode
    )


    v4_forecast_rows.append(
        origin_forecast_frame
    )


    # =============================================================
    # 3K. ORIGIN SUMMARY
    # =============================================================

    origin_minutes = (
        time.time()
        - origin_start_time
    ) / 60.0


    v4_forecast_summary_rows.append(
        {
            "forecast_origin":
                forecast_origin,

            "calibration_mode":
                calibration_mode,

            "historical_adjacent_increases":
                adjacent_bin_increases,

            "historical_overall_dr_uplift":
                overall_dr_relative_uplift,

            "historical_top_bin_dr_uplift":
                top_bin_dr_relative_uplift,

            "baseline_floor":
                baseline_floor,

            "future_rows":
                len(
                    future_frame
                ),

            "future_treatment_rate":
                future_treatment.mean(),

            "future_overlap_share":
                future_overlap.mean(),

            "future_causal_support_share":
                future_causal_support.mean(),

            "promoted_support_share":
                promoted_support_share,

            "baseline_only_wape":
                baseline_only_wape,

            "raw_t_learner_wape":
                raw_t_learner_wape,

            "v4_wape":
                v4_wape,

            "v4_bias":
                v4_bias,

            "promoted_v4_wape":
                promoted_v4_wape,

            "nonpromoted_v4_wape":
                nonpromoted_v4_wape,

            "fit_minutes":
                origin_minutes,
        }
    )


    print(
        "Calibration mode:",
        calibration_mode,
    )


    print(
        "Historical overall DR uplift:",
        f"{overall_dr_relative_uplift:+.2%}",
    )


    print(
        "Historical top-bin DR uplift:",
        f"{top_bin_dr_relative_uplift:+.2%}",
    )


    print(
        "Baseline floor:",
        f"{baseline_floor:,.3f}",
    )


    print(
        "Future causal-support share:",
        f"{future_causal_support.mean():.2%}",
    )


    print(
        "Promoted rows with causal support:",
        f"{promoted_support_share:.2%}",
    )


    print(
        "Baseline-only WAPE:",
        f"{baseline_only_wape:.4f}",
    )


    print(
        "Raw T-learner WAPE:",
        f"{raw_t_learner_wape:.4f}",
    )


    print(
        "V4 calibrated WAPE:",
        f"{v4_wape:.4f}",
    )


    print(
        "V4 normalized bias:",
        f"{v4_bias:+.2%}",
    )


    print(
        "Origin fit minutes:",
        f"{origin_minutes:.2f}",
    )


    # -------------------------------------------------------------
    # Release origin-level model objects and large temporary frames.
    # -------------------------------------------------------------

    del (
        historical_frame,
        future_frame,
        X_historical,
        X_future,
        final_propensity_model,
        historical_propensity_full,
        future_propensity,
        X_control_train,
        X_treated_train,
        y_control_train,
        y_treated_train,
        final_mu0_model,
        final_mu1_model,
        historical_overlap_frame,
        X_historical_overlap,
        historical_mu0_final,
        historical_mu1_final,
        historical_relative_score,
        historical_relative_score_sorted,
        future_mu0,
        future_mu1,
        future_raw_relative_score,
        future_score_percentile,
        future_calibration_bin,
        future_calibrated_relative_uplift,
        v4_prediction,
        baseline_only_prediction,
        raw_t_learner_prediction,
    )


    gc.collect()


# =====================================================================
# 4. COMBINE ALL SIX FORECAST ORIGINS
# =====================================================================

v4_forecasts = pd.concat(
    v4_forecast_rows,
    ignore_index=True,
)


v4_forecast_summary = pd.DataFrame(
    v4_forecast_summary_rows
)


# =====================================================================
# 5. DISPLAY SIX-FOLD V4 SUMMARY
# =====================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "V4 SIX-ORIGIN FORECAST SUMMARY"
)

print(
    "=" * 80
)


display(
    v4_forecast_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "historical_overall_dr_uplift":
                "{:+.2%}",

            "historical_top_bin_dr_uplift":
                "{:+.2%}",

            "baseline_floor":
                "{:,.3f}",

            "future_treatment_rate":
                "{:.2%}",

            "future_overlap_share":
                "{:.2%}",

            "future_causal_support_share":
                "{:.2%}",

            "promoted_support_share":
                "{:.2%}",

            "baseline_only_wape":
                "{:.4f}",

            "raw_t_learner_wape":
                "{:.4f}",

            "v4_wape":
                "{:.4f}",

            "v4_bias":
                "{:+.2%}",

            "promoted_v4_wape":
                "{:.4f}",

            "nonpromoted_v4_wape":
                "{:.4f}",

            "fit_minutes":
                "{:.2f}",
        }
    )
)


# =====================================================================
# 6. POOLED SIX-ORIGIN METRICS
# =====================================================================

pooled_actual_sales = (
    v4_forecasts[
        "target_sales"
    ]
    .sum()
)


pooled_baseline_wape = (
    np.abs(
        v4_forecasts[
            "v4_baseline_only_prediction"
        ]
        -
        v4_forecasts[
            "target_sales"
        ]
    )
    .sum()
    /
    pooled_actual_sales
)


pooled_raw_t_wape = (
    np.abs(
        v4_forecasts[
            "v4_raw_t_learner_prediction"
        ]
        -
        v4_forecasts[
            "target_sales"
        ]
    )
    .sum()
    /
    pooled_actual_sales
)


pooled_v4_wape = (
    np.abs(
        v4_forecasts[
            "v4_prediction"
        ]
        -
        v4_forecasts[
            "target_sales"
        ]
    )
    .sum()
    /
    pooled_actual_sales
)


pooled_v4_bias = (
    (
        v4_forecasts[
            "v4_prediction"
        ]
        -
        v4_forecasts[
            "target_sales"
        ]
    )
    .sum()
    /
    pooled_actual_sales
)


print()
print(
    "Pooled baseline-only WAPE:",
    f"{pooled_baseline_wape:.4f}",
)


print(
    "Pooled raw T-learner WAPE:",
    f"{pooled_raw_t_wape:.4f}",
)


print(
    "Pooled V4 calibrated WAPE:",
    f"{pooled_v4_wape:.4f}",
)


print(
    "Pooled V4 normalized bias:",
    f"{pooled_v4_bias:+.2%}",
)


print()
print(
    "V4 FINAL FORECAST CONSTRUCTION: PASSED"
)


V4 FINAL FORECAST
Forecast origin: 2017-05-09
Calibration mode: pooled
Historical overall DR uplift: +14.29%
Historical top-bin DR uplift: +19.69%
Baseline floor: 14.972
Future causal-support share: 31.83%
Promoted rows with causal support: 37.05%
Baseline-only WAPE: 0.2779
Raw T-learner WAPE: 0.1675
V4 calibrated WAPE: 0.2779
V4 normalized bias: -14.97%
Origin fit minutes: 0.88

V4 FINAL FORECAST
Forecast origin: 2017-05-23
Calibration mode: segmented
Historical overall DR uplift: +12.69%
Historical top-bin DR uplift: +49.28%
Baseline floor: 15.757
Future causal-support share: 27.43%
Promoted rows with causal support: 30.61%
Baseline-only WAPE: 0.4128
Raw T-learner WAPE: 0.2063
V4 calibrated WAPE: 0.4099
V4 normalized bias: -38.10%
Origin fit minutes: 0.90

V4 FINAL FORECAST
Forecast origin: 2017-06-06
Calibration mode: segmented
Historical overall DR uplift: +18.52%
Historical top-bin DR uplift: +46.28%
Baseline floor: 14.235
Future causal-support share: 41.59%
Promoted rows with ca

forecast_origin,calibration_mode,historical_adjacent_increases,historical_overall_dr_uplift,historical_top_bin_dr_uplift,baseline_floor,future_rows,future_treatment_rate,future_overlap_share,future_causal_support_share,promoted_support_share,baseline_only_wape,raw_t_learner_wape,v4_wape,v4_bias,promoted_v4_wape,nonpromoted_v4_wape,fit_minutes
2017-05-09 00:00:00,pooled,2,+14.29%,+19.69%,14.972,24948,50.14%,56.40%,31.83%,37.05%,0.2779,0.1675,0.2779,-14.97%,0.2807,0.2196,0.88
2017-05-23 00:00:00,segmented,3,+12.69%,+49.28%,15.757,24948,45.32%,50.35%,27.43%,30.61%,0.4128,0.2063,0.4099,-38.10%,0.4197,0.3299,0.90
2017-06-06 00:00:00,segmented,4,+18.52%,+46.28%,14.235,24948,45.55%,59.80%,41.59%,60.77%,0.2337,0.1396,0.2437,-7.68%,0.2505,0.1600,0.90
2017-06-20 00:00:00,pooled,3,+2.40%,-0.82%,12.517,24948,44.68%,52.81%,35.75%,43.98%,0.2887,0.1471,0.2882,-20.08%,0.2950,0.1786,0.94
2017-07-04 00:00:00,pooled,1,-5.19%,-18.90%,11.842,24948,42.60%,54.16%,39.61%,45.65%,0.3203,0.1447,0.3234,-27.67%,0.3344,0.1750,0.89
2017-07-18 00:00:00,pooled,1,+2.35%,-10.75%,9.594,24948,40.80%,50.42%,40.73%,48.32%,0.4273,0.1391,0.4254,-40.27%,0.4501,0.1626,0.92



Pooled baseline-only WAPE: 0.3278
Pooled raw T-learner WAPE: 0.1579
Pooled V4 calibrated WAPE: 0.3290
Pooled V4 normalized bias: -24.95%

V4 FINAL FORECAST CONSTRUCTION: PASSED


### 9.4 Rolling Temporal Evaluation

The pure causal V4 architecture reveals an important distinction between
**causal conservatism** and **forecasting accuracy**.

When a promoted future context lies outside the practical causal-support
domain, the pre-specified V4 rule applies no promotion adjustment:

$$
\hat\rho_T(X)=0.
$$

The resulting forecast is therefore:

$$
\hat Y_{V4}
=
\hat\mu_{0,T}(X).
$$

This is conservative from a causal-inference perspective because the model
does not extrapolate an unsupported treatment effect.

However, it can be highly conservative from a forecasting perspective.

For a target row whose promotion is known in advance, forecasting baseline
no-promotion demand simply because its causal treatment effect cannot be
identified may discard useful predictive information about the known
promotion.

The six-origin V4 experiment shows substantial negative forecast bias,
suggesting that this distinction matters empirically.

---

#### Isolating the source of the V4 forecasting error

Two diagnostics are therefore evaluated.

First, inside promoted rows where causal support is available, the calibrated
causal prediction is compared directly with the raw promotion-aware
T-learner prediction.

This asks:

> Does causal calibration improve forecasting where the treatment effect is
> considered sufficiently supported?

Second, an exploratory hybrid prediction is constructed:

$$
\hat Y_{hybrid}
=
\begin{cases}
\hat Y_{V4},
&
D=1
\text{ and causal support is available},
\\[0.4em]
\hat Y_{raw\ T},
&
\text{otherwise}.
\end{cases}
$$

For non-promoted rows, both architectures use the no-promotion baseline.

For promoted rows outside causal support, the hybrid therefore uses the
predictive promotion-aware estimate instead of pretending that the promotion
has zero effect.

---

This hybrid is a **post-hoc diagnostic**, motivated by the observed failure
mode of the pre-specified causal-only V4 architecture.

It is not treated as an independently validated production champion and is not
used to retroactively modify the original V4 experiment.

Its purpose is to distinguish:

1. failure caused by the zero-uplift fallback outside causal support;
2. failure caused by the causal calibration itself inside supported regions.

---

The final comparison uses exactly the same six 14-day forecast windows as the
earlier forecasting experiments.

The models compared are:

- frozen V1 global forecasting champion;
- promotion-count predictive model from Section 3;
- raw relative T-learner;
- pre-specified causal-only V4;
- exploratory causal/predictive hybrid.

The primary metric remains WAPE.

This provides the final test of whether explicit causal structure improves
short-horizon forecasting accuracy beyond the previously developed predictive
models.

In [37]:
# ---------------------------------------------------------------------
# V4 rolling temporal evaluation.
#
# This cell performs NO model refitting.
#
# It evaluates:
#
#   1. the pre-specified causal-only V4;
#   2. the raw predictive relative T-learner;
#   3. causal calibration versus raw prediction inside supported
#      promoted rows;
#   4. a POST-HOC hybrid diagnostic using:
#
#        causal prediction   -> supported promoted rows
#        raw T prediction    -> all other rows
#
# The hybrid is diagnostic only because it was defined after observing
# the failure mode of causal-only V4.
#
# Finally, all architectures are compared with the frozen V1 and
# promotion-count benchmarks from Section 3 on the same six origins.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


# =====================================================================
# 1. BUILD POST-HOC HYBRID DIAGNOSTIC
# =====================================================================

supported_promoted_mask = (
    v4_forecasts[
        "treatment"
    ]
    .eq(
        1
    )
    &
    v4_forecasts[
        "v4_causal_support"
    ]
)


v4_forecasts[
    "v4_hybrid_prediction"
] = np.where(
    supported_promoted_mask,
    v4_forecasts[
        "v4_prediction"
    ],
    v4_forecasts[
        "v4_raw_t_learner_prediction"
    ],
)


v4_forecasts[
    "v4_hybrid_prediction"
] = (
    v4_forecasts[
        "v4_hybrid_prediction"
    ]
    .clip(
        lower=0.0
    )
)


# =====================================================================
# 2. SUPPORTED-PROMOTION DIAGNOSTIC
# =====================================================================
#
# Compare causal calibration and the raw predictive T-learner ONLY on
# promoted rows for which V4 considers causal adjustment support
# sufficient.
#
# This isolates the forecasting contribution of causal calibration from
# the separate zero-uplift fallback problem.
# =====================================================================

supported_promotion_rows = []


for forecast_origin, origin_frame in (
    v4_forecasts
    .groupby(
        "forecast_origin",
        observed=True,
    )
):

    supported_frame = (
        origin_frame.loc[
            origin_frame[
                "treatment"
            ]
            .eq(
                1
            )
            &
            origin_frame[
                "v4_causal_support"
            ]
        ]
    )


    assert not supported_frame.empty


    actual_sales = (
        supported_frame[
            "target_sales"
        ]
        .sum()
    )


    raw_t_wape = (
        np.abs(
            supported_frame[
                "v4_raw_t_learner_prediction"
            ]
            -
            supported_frame[
                "target_sales"
            ]
        )
        .sum()
        /
        actual_sales
    )


    causal_wape = (
        np.abs(
            supported_frame[
                "v4_prediction"
            ]
            -
            supported_frame[
                "target_sales"
            ]
        )
        .sum()
        /
        actual_sales
    )


    raw_t_bias = (
        (
            supported_frame[
                "v4_raw_t_learner_prediction"
            ]
            -
            supported_frame[
                "target_sales"
            ]
        )
        .sum()
        /
        actual_sales
    )


    causal_bias = (
        (
            supported_frame[
                "v4_prediction"
            ]
            -
            supported_frame[
                "target_sales"
            ]
        )
        .sum()
        /
        actual_sales
    )


    supported_promotion_rows.append(
        {
            "forecast_origin":
                forecast_origin,

            "supported_promoted_rows":
                len(
                    supported_frame
                ),

            "raw_t_supported_wape":
                raw_t_wape,

            "causal_supported_wape":
                causal_wape,

            "causal_minus_raw_wape":
                causal_wape
                -
                raw_t_wape,

            "raw_t_supported_bias":
                raw_t_bias,

            "causal_supported_bias":
                causal_bias,
        }
    )


supported_promotion_diagnostic = pd.DataFrame(
    supported_promotion_rows
)


print(
    "=" * 80
)

print(
    "SUPPORTED PROMOTED ROWS — CAUSAL CALIBRATION VS RAW T-LEARNER"
)

print(
    "=" * 80
)


display(
    supported_promotion_diagnostic
    .style.hide(
        axis="index"
    )
    .format(
        {
            "raw_t_supported_wape":
                "{:.4f}",

            "causal_supported_wape":
                "{:.4f}",

            "causal_minus_raw_wape":
                "{:+.4f}",

            "raw_t_supported_bias":
                "{:+.2%}",

            "causal_supported_bias":
                "{:+.2%}",
        }
    )
)


# =====================================================================
# 3. SIX-ORIGIN HYBRID DIAGNOSTIC
# =====================================================================

hybrid_summary_rows = []


for forecast_origin, origin_frame in (
    v4_forecasts
    .groupby(
        "forecast_origin",
        observed=True,
    )
):

    actual_sales = (
        origin_frame[
            "target_sales"
        ]
        .sum()
    )


    raw_t_wape = (
        np.abs(
            origin_frame[
                "v4_raw_t_learner_prediction"
            ]
            -
            origin_frame[
                "target_sales"
            ]
        )
        .sum()
        /
        actual_sales
    )


    causal_v4_wape = (
        np.abs(
            origin_frame[
                "v4_prediction"
            ]
            -
            origin_frame[
                "target_sales"
            ]
        )
        .sum()
        /
        actual_sales
    )


    hybrid_wape = (
        np.abs(
            origin_frame[
                "v4_hybrid_prediction"
            ]
            -
            origin_frame[
                "target_sales"
            ]
        )
        .sum()
        /
        actual_sales
    )


    raw_t_bias = (
        (
            origin_frame[
                "v4_raw_t_learner_prediction"
            ]
            -
            origin_frame[
                "target_sales"
            ]
        )
        .sum()
        /
        actual_sales
    )


    causal_v4_bias = (
        (
            origin_frame[
                "v4_prediction"
            ]
            -
            origin_frame[
                "target_sales"
            ]
        )
        .sum()
        /
        actual_sales
    )


    hybrid_bias = (
        (
            origin_frame[
                "v4_hybrid_prediction"
            ]
            -
            origin_frame[
                "target_sales"
            ]
        )
        .sum()
        /
        actual_sales
    )


    hybrid_summary_rows.append(
        {
            "forecast_origin":
                forecast_origin,

            "raw_t_learner_wape":
                raw_t_wape,

            "causal_only_v4_wape":
                causal_v4_wape,

            "hybrid_wape":
                hybrid_wape,

            "hybrid_minus_raw_wape":
                hybrid_wape
                -
                raw_t_wape,

            "raw_t_bias":
                raw_t_bias,

            "causal_only_v4_bias":
                causal_v4_bias,

            "hybrid_bias":
                hybrid_bias,
        }
    )


v4_hybrid_summary = pd.DataFrame(
    hybrid_summary_rows
)


print()
print(
    "=" * 80
)

print(
    "POST-HOC V4 HYBRID DIAGNOSTIC"
)

print(
    "=" * 80
)


display(
    v4_hybrid_summary
    .style.hide(
        axis="index"
    )
    .format(
        {
            "raw_t_learner_wape":
                "{:.4f}",

            "causal_only_v4_wape":
                "{:.4f}",

            "hybrid_wape":
                "{:.4f}",

            "hybrid_minus_raw_wape":
                "{:+.4f}",

            "raw_t_bias":
                "{:+.2%}",

            "causal_only_v4_bias":
                "{:+.2%}",

            "hybrid_bias":
                "{:+.2%}",
        }
    )
)


# =====================================================================
# 4. FROZEN EARLIER FORECASTING BENCHMARKS
# =====================================================================
#
# These are the previously computed six-fold WAPE results from Section 3.
#
# They are copied here only to assemble the final comparison table.
# No V4 result is used to alter these benchmark values.
# =====================================================================

frozen_forecasting_benchmarks = pd.DataFrame(
    {
        "forecast_origin": pd.to_datetime(
            [
                "2017-05-09",
                "2017-05-23",
                "2017-06-06",
                "2017-06-20",
                "2017-07-04",
                "2017-07-18",
            ]
        ),

        "v1_wape": [
            0.147075,
            0.163806,
            0.127854,
            0.127460,
            0.113343,
            0.122800,
        ],

        "promotion_count_wape": [
            0.149590,
            0.164729,
            0.129620,
            0.128959,
            0.113120,
            0.120036,
        ],
    }
)


# =====================================================================
# 5. MERGE ALL FORECASTING ARCHITECTURES
# =====================================================================

final_v4_comparison = (
    frozen_forecasting_benchmarks
    .merge(
        v4_hybrid_summary[
            [
                "forecast_origin",
                "raw_t_learner_wape",
                "causal_only_v4_wape",
                "hybrid_wape",
            ]
        ],
        on="forecast_origin",
        how="inner",
        validate="one_to_one",
    )
)


assert (
    len(
        final_v4_comparison
    )
    ==
    len(
        V4_FORECAST_ORIGINS
    )
)


# =====================================================================
# 6. FOLD-LEVEL COMPARISON
# =====================================================================

print()
print(
    "=" * 80
)

print(
    "FINAL SIX-ORIGIN FORECAST COMPARISON"
)

print(
    "=" * 80
)


display(
    final_v4_comparison
    .style.hide(
        axis="index"
    )
    .format(
        {
            "v1_wape":
                "{:.4f}",

            "promotion_count_wape":
                "{:.4f}",

            "raw_t_learner_wape":
                "{:.4f}",

            "causal_only_v4_wape":
                "{:.4f}",

            "hybrid_wape":
                "{:.4f}",
        }
    )
)


# =====================================================================
# 7. POOLED WAPE USING THE ACTUAL SALES VOLUME OF EACH FOLD
# =====================================================================
#
# Because all architectures are evaluated on the exact same rows,
# fold absolute error can be reconstructed as:
#
#   WAPE_fold * actual_sales_fold
#
# for the previously frozen V1 / promotion-count benchmarks.
# =====================================================================

actual_sales_by_origin = (
    v4_forecasts
    .groupby(
        "forecast_origin",
        observed=True,
    )[
        "target_sales"
    ]
    .sum()
    .rename(
        "actual_sales"
    )
    .reset_index()
)


pooled_comparison_frame = (
    final_v4_comparison
    .merge(
        actual_sales_by_origin,
        on="forecast_origin",
        how="inner",
        validate="one_to_one",
    )
)


model_wape_columns = {
    "V1 frozen champion":
        "v1_wape",

    "Promotion-count predictive":
        "promotion_count_wape",

    "Raw relative T-learner":
        "raw_t_learner_wape",

    "V4 causal-only":
        "causal_only_v4_wape",

    "V4 hybrid diagnostic":
        "hybrid_wape",
}


pooled_rows = []


total_actual_sales = (
    pooled_comparison_frame[
        "actual_sales"
    ]
    .sum()
)


for (
    model_name,
    wape_column,
) in model_wape_columns.items():

    pooled_wape = (
        (
            pooled_comparison_frame[
                wape_column
            ]
            *
            pooled_comparison_frame[
                "actual_sales"
            ]
        )
        .sum()
        /
        total_actual_sales
    )


    mean_fold_wape = (
        pooled_comparison_frame[
            wape_column
        ]
        .mean()
    )


    folds_beating_v1 = int(
        (
            pooled_comparison_frame[
                wape_column
            ]
            <
            pooled_comparison_frame[
                "v1_wape"
            ]
        )
        .sum()
    )


    pooled_rows.append(
        {
            "model":
                model_name,

            "pooled_wape":
                pooled_wape,

            "mean_fold_wape":
                mean_fold_wape,

            "folds_beating_v1":
                folds_beating_v1,
        }
    )


final_pooled_forecast_comparison = pd.DataFrame(
    pooled_rows
)


v1_pooled_wape = float(
    final_pooled_forecast_comparison.loc[
        final_pooled_forecast_comparison[
            "model"
        ]
        .eq(
            "V1 frozen champion"
        ),
        "pooled_wape",
    ]
    .iloc[
        0
    ]
)


final_pooled_forecast_comparison[
    "relative_wape_change_vs_v1"
] = (
    (
        final_pooled_forecast_comparison[
            "pooled_wape"
        ]
        /
        v1_pooled_wape
    )
    - 1.0
)


final_pooled_forecast_comparison = (
    final_pooled_forecast_comparison
    .sort_values(
        "pooled_wape"
    )
    .reset_index(
        drop=True
    )
)


print()
print(
    "=" * 80
)

print(
    "POOLED FORECAST ARCHITECTURE COMPARISON"
)

print(
    "=" * 80
)


display(
    final_pooled_forecast_comparison
    .style.hide(
        axis="index"
    )
    .format(
        {
            "pooled_wape":
                "{:.4f}",

            "mean_fold_wape":
                "{:.4f}",

            "relative_wape_change_vs_v1":
                "{:+.2%}",
        }
    )
)


# =====================================================================
# 8. FINAL DIAGNOSTIC COUNTS
# =====================================================================

supported_folds_causal_better = int(
    (
        supported_promotion_diagnostic[
            "causal_supported_wape"
        ]
        <
        supported_promotion_diagnostic[
            "raw_t_supported_wape"
        ]
    )
    .sum()
)


hybrid_folds_better_than_raw = int(
    (
        v4_hybrid_summary[
            "hybrid_wape"
        ]
        <
        v4_hybrid_summary[
            "raw_t_learner_wape"
        ]
    )
    .sum()
)


print()
print(
    "Causal calibration beats raw T-learner on supported promoted "
    "rows in:",
    f"{supported_folds_causal_better}/6 folds",
)


print(
    "Post-hoc hybrid beats raw T-learner overall in:",
    f"{hybrid_folds_better_than_raw}/6 folds",
)


print()
print(
    "V4 ROLLING TEMPORAL EVALUATION: PASSED"
)

SUPPORTED PROMOTED ROWS — CAUSAL CALIBRATION VS RAW T-LEARNER


forecast_origin,supported_promoted_rows,raw_t_supported_wape,causal_supported_wape,causal_minus_raw_wape,raw_t_supported_bias,causal_supported_bias
2017-05-09 00:00:00,4635,0.1519,0.2022,+0.0502,+4.68%,+2.38%
2017-05-23 00:00:00,3461,0.1936,0.2042,+0.0106,-8.43%,-6.58%
2017-06-06 00:00:00,6905,0.1274,0.1912,+0.0638,+6.22%,+13.63%
2017-06-20 00:00:00,4902,0.1207,0.1488,+0.0281,-2.77%,-5.78%
2017-07-04 00:00:00,4852,0.1157,0.1419,+0.0262,+5.67%,-9.37%
2017-07-18 00:00:00,4919,0.1127,0.2580,+0.1453,-0.21%,-21.25%



POST-HOC V4 HYBRID DIAGNOSTIC


forecast_origin,raw_t_learner_wape,causal_only_v4_wape,hybrid_wape,hybrid_minus_raw_wape,raw_t_bias,causal_only_v4_bias,hybrid_bias
2017-05-09 00:00:00,0.1675,0.2779,0.1738,+0.0064,+5.30%,-14.97%,+5.01%
2017-05-23 00:00:00,0.2063,0.4099,0.2069,+0.0006,-13.89%,-38.10%,-13.79%
2017-06-06 00:00:00,0.1396,0.2437,0.1587,+0.0191,+5.87%,-7.68%,+8.09%
2017-06-20 00:00:00,0.1471,0.2882,0.1528,+0.0057,-3.75%,-20.08%,-4.36%
2017-07-04 00:00:00,0.1447,0.3234,0.1496,+0.0048,+7.63%,-27.67%,+4.86%
2017-07-18 00:00:00,0.1391,0.4254,0.1660,+0.0269,-5.72%,-40.27%,-9.61%



FINAL SIX-ORIGIN FORECAST COMPARISON


forecast_origin,v1_wape,promotion_count_wape,raw_t_learner_wape,causal_only_v4_wape,hybrid_wape
2017-05-09 00:00:00,0.1471,0.1496,0.1675,0.2779,0.1738
2017-05-23 00:00:00,0.1638,0.1647,0.2063,0.4099,0.2069
2017-06-06 00:00:00,0.1279,0.1296,0.1396,0.2437,0.1587
2017-06-20 00:00:00,0.1275,0.1290,0.1471,0.2882,0.1528
2017-07-04 00:00:00,0.1133,0.1131,0.1447,0.3234,0.1496
2017-07-18 00:00:00,0.1228,0.1200,0.1391,0.4254,0.1660



POOLED FORECAST ARCHITECTURE COMPARISON


model,pooled_wape,mean_fold_wape,folds_beating_v1,relative_wape_change_vs_v1
V1 frozen champion,0.1340,0.1337,0,+0.00%
Promotion-count predictive,0.1347,0.1343,2,+0.46%
Raw relative T-learner,0.1579,0.1574,0,+17.78%
V4 hybrid diagnostic,0.1684,0.1680,0,+25.62%
V4 causal-only,0.3290,0.3281,0,+145.48%



Causal calibration beats raw T-learner on supported promoted rows in: 0/6 folds
Post-hoc hybrid beats raw T-learner overall in: 0/6 folds

V4 ROLLING TEMPORAL EVALUATION: PASSED


### 9.5 V4 Conclusion

The V4 experiment provides a clear empirical result:

> explicit causal promotion modeling did not improve short-horizon demand
> forecasting in this dataset.

The frozen V1 forecasting champion remains substantially more accurate.

Across the six common rolling origins:

$$
WAPE_{V1}
=
13.40\%,
$$

while:

$$
WAPE_{raw\ relative\ T}
=
15.79\%,
$$

$$
WAPE_{hybrid}
=
16.84\%,
$$

and:

$$
WAPE_{causal\text{-}only\ V4}
=
32.90\%.
$$

Relative to V1, pooled WAPE therefore deteriorates by approximately:

$$
+17.8\%
$$

for the raw relative T-learner,

$$
+25.6\%
$$

for the exploratory hybrid architecture, and:

$$
+145.5\%
$$

for causal-only V4.

V1 remains the preferred production forecasting architecture.

---

#### Why did causal-only V4 fail?

The first failure mechanism is the conservative causal-support rule.

For promoted future observations outside practical causal support, V4 applies:

$$
\hat\rho(X)=0
$$

and therefore predicts:

$$
\hat Y
=
\hat\mu_0(X).
$$

This is defensible for causal inference because it avoids extrapolating an
unsupported treatment effect.

For forecasting, however, it can be highly conservative because the future
promotion is already known.

Ignoring its predictive association can therefore create substantial
underprediction.

This effect is visible in the strong negative bias of causal-only V4,
including a pooled normalized bias of approximately:

$$
-25\%.
$$

---

However, the support fallback is not the complete explanation.

The causal calibration was compared directly with the raw promotion-aware
T-learner **only on promoted target rows that satisfied the causal-support
rules**.

The raw T-learner achieved lower WAPE in all six forecast windows:

$$
0/6
$$

supported-region folds were improved by causal calibration.

The post-hoc hybrid diagnostic reached the same conclusion.

The hybrid retained predictive T-learner forecasts outside causal support and
used causal calibration only inside supported promoted regions.

Nevertheless, it also underperformed the raw T-learner in all six origins.

Therefore:

$$
\boxed{
\text{the causal calibration itself did not improve forecast accuracy}
}
$$

in this experiment.

---

#### Why can a causal model be useful without improving forecasting?

Causal inference and forecasting optimize different questions.

Forecasting asks:

> Given all information known today, what sales value is most likely tomorrow?

Causal inference asks:

> How would expected sales change if the promotion state were changed while
> holding the relevant context fixed?

A predictive model may exploit stable associations that improve forecast
accuracy even when those associations should not be interpreted causally.

Conversely, a causal estimator deliberately removes or restricts some of those
associations in order to estimate treatment effects.

There is therefore no requirement that a better causal estimator must also
produce a better point forecast.

The current results provide a concrete example of this distinction.

---

#### Additional reasons for the forecasting gap

Several structural differences also favor the original forecasting model.

V1 is optimized directly for the forecasting objective and includes the richer
forecasting feature architecture developed earlier in the project.

The causal models use a more restrictive pre-treatment feature set designed
for treatment-effect identification rather than maximum predictive accuracy.

V4 also imposes:

- practical propensity-support restrictions;
- a low-baseline stability rule;
- causal calibration rather than direct forecast-error minimization;
- a coarse binary promotion treatment.

In addition, the dataset does not contain important promotion information such
as discount depth, prices, campaign mechanics or stock availability.

These missing variables can matter both for treatment-effect estimation and
for predicting the magnitude of promotion-driven demand changes.

---

#### Final role of the causal layer

The causal analysis is therefore not retained as a replacement for the demand
forecasting model.

Instead, the project separates two complementary systems:

**Forecasting**

$$
\boxed{
V1
}
$$

is retained for short-horizon demand prediction and operational planning.

**Causal promotion analysis**

The causal pipeline is retained for questions such as:

- whether observed promotional sales differences appear incremental;
- the estimated average effect of entering the historical promotion state;
- how promotion response varies across supported populations;
- which commercial contexts appear more promotion-responsive;
- how incremental sales could later feed a profitability or MROI framework.

The final architecture therefore avoids forcing a single model to solve two
different statistical problems.

The principal lesson from V4 is:

> causal structure can improve business interpretation without necessarily
> improving predictive accuracy.

This distinction is retained explicitly in the final system design.

## 10. Final Conclusions and Limitations

This project developed a point-in-time retail demand forecasting system and
then extended it to distinguish three related but fundamentally different
business questions:

1. **Forecasting:** how much will be sold over the next 14 days?
2. **Causal inference:** how much of the observed demand response is
   incremental because of promotion?
3. **Economic optimization:** is that incremental response profitable?

The central conclusion is that these questions should not be forced into a
single modeling objective.

---

### 10.1 Forecasting result

The final forecasting champion remains the global direct
horizon-conditioned V1 XGBoost model.

Across the six recent rolling 14-day validation origins:

$$
WAPE_{V1}
\approx
13.40\%.
$$

The predictive promotion-count extension achieved approximately:

$$
13.47\%
$$

WAPE and improved only two of the six origins.

Therefore, explicitly adding planned promotion information did not provide a
stable aggregate forecasting improvement over the frozen V1 architecture.

On the separate final August holdout, V1 achieved approximately:

$$
18.77\%
$$

WAPE versus approximately:

$$
23.21\%
$$

for the seasonal-naive benchmark.

The production forecasting recommendation therefore remains:

$$
\boxed{
\text{use V1 for short-horizon operational demand forecasting}
}
$$

---

### 10.2 Why predictive promotion response is not causal uplift

The promotion-aware predictive model learned a strong positive association
between promotion intensity and sales.

In particular, larger promotion counts were associated with larger
model-implied sales responses, with a Spearman correlation of approximately:

$$
0.80.
$$

However, this model estimates relationships of the form:

$$
E[Y\mid X,P],
$$

where $P$ is observed promotion information.

It does not estimate what would have happened to the same commercial context
under a different promotion decision.

Therefore, the predictive counterfactual experiment was treated as evidence of
a strong promotion association, not as evidence of incremental causal demand.

This distinction motivated the causal analysis.

---

### 10.3 Causal promotion problem

The causal unit is one:

$$
(store,\ family,\ date)
$$

observation.

The binary treatment is:

$$
D
=
\mathbb{1}
[
\text{onpromotion}>0
].
$$

This means that the treatment compares the historical state:

> at least one SKU in the store-family is promoted

with:

> no SKU in the store-family is promoted.

It does **not** estimate the effect of a specific discount percentage,
promotion mechanism or number of promoted products.

Only pre-treatment covariates are included in the causal models, with a
conservative 14-day gap for dynamic sales and promotion-history features.

The identification strategy relies on conditional exchangeability,
consistency, positivity and limited interference.

Because this is observational retail data, those assumptions cannot be proven
from the dataset.

---

### 10.4 Treatment assignment and overlap

Promotion assignment is highly predictable from historical commercial
context.

The retrospective propensity model achieved an out-of-fold AUC of
approximately:

$$
0.94.
$$

This is useful diagnostically but also shows that promotion assignment is far
from random.

Only approximately:

$$
63\%
$$

of the retrospective causal population lies inside the practical propensity
region:

$$
0.01
\leq
\hat e(X)
\leq
0.99.
$$

Raw inverse-probability weights were extremely unstable.

A 1% propensity clipping rule substantially improved weight stability and
covariate balance, while more aggressive clipping and trimming produced
materially different effect estimates.

The treatment-effect results must therefore be interpreted as
assumption-dependent observational estimates rather than experimental ground
truth.

---

### 10.5 Relative causal promotion response

Absolute incremental sales effects were estimated first, but they are difficult
to compare across store-family contexts with very different baseline sales
levels.

The policy-oriented analysis therefore focuses primarily on **relative causal
uplift**.

Define:

$$
\mu_0(X)
=
E[Y(0)\mid X]
$$

and:

$$
\mu_1(X)
=
E[Y(1)\mid X].
$$

The relative treatment effect is:

$$
\rho(X)
=
\frac{
\mu_1(X)
}{
\mu_0(X)
}
-1.
$$

A relative effect of:

$$
20\%
$$

means that expected sales under promotion are approximately 20% higher than
the counterfactual expected sales without promotion for the relevant
commercial population.

This representation is more interpretable across heterogeneous demand scales
and connects naturally to business decision-making.

---

Relative effects become numerically unstable when expected baseline demand is
close to zero.

A retrospective denominator audit therefore identified a primary
interpretation population combining:

- practical propensity overlap;
- a low-baseline stability guardrail.

Inside this eligible interpretation population, the group-level doubly robust
potential-outcome estimates imply an average relative promotion uplift of
approximately:

$$
\boxed{
27.1\%
}
$$

under the causal identification assumptions.

This should be interpreted as an estimated average effect for this supported
population, not as a universal effect of every promotion.

---

### 10.6 Heterogeneous relative uplift

A separate relative T-learner was developed to investigate whether some
commercial contexts appear systematically more responsive to promotion.

The model estimates:

$$
\hat\mu_0(X)
$$

and:

$$
\hat\mu_1(X),
$$

and constructs the ranking score:

$$
\hat\rho(X)
=
\frac{
\hat\mu_1(X)
}{
\hat\mu_0(X)
}
-1.
$$

The raw percentage predictions were **not well calibrated**.

They should therefore not be interpreted literally as individual causal
effects.

However, the ranking contained useful retrospective information.

When contexts were ordered out-of-fold by predicted relative response, the
doubly robust group-level relative effects across quintiles were approximately:

$$
21.5\%,
\quad
18.3\%,
\quad
25.8\%,
\quad
27.9\%,
\quad
42.5\%.
$$

The sequence is not perfectly monotonic, but the highest-ranked group exhibits
a materially larger estimated causal response than the lowest-ranked group.

The top 10% of contexts selected by the relative-response model had an
estimated group-level doubly robust uplift of approximately:

$$
\boxed{
44.3\%
}
$$

compared with approximately:

$$
27.1\%
$$

across the complete eligible interpretation population.

The top 20% had an estimated relative uplift of approximately:

$$
42.5\%.
$$

Therefore, the retrospective model provides evidence of useful
**response ranking**, even though its raw percentage predictions are not
sufficiently calibrated.

The operational interpretation is:

> machine learning can help identify contexts that appear more
> promotion-responsive, while doubly robust estimation is used to quantify
> the average causal response of the selected population.

---

### 10.7 Relative uplift and business economics

Relative causal uplift provides a natural bridge from statistical estimation
to commercial decision-making.

If:

$$
\mu_0(X)
$$

is expected sales without promotion and:

$$
\rho(X)
$$

is relative causal uplift, then expected incremental units are:

$$
\Delta Units(X)
=
\mu_0(X)\rho(X).
$$

For example, a context with expected baseline demand of 500 units and an
estimated causal relative uplift of 20% would imply:

$$
500\times0.20
=
100
$$

incremental units.

This creates the business chain:

$$
Baseline\ Demand
\rightarrow
Relative\ Causal\ Uplift
\rightarrow
Incremental\ Units.
$$

However, incremental units are not equivalent to incremental profit.

A complete promotion decision system would additionally require prices,
discount depth, unit margins and promotion costs:

$$
Incremental\ Units
\rightarrow
Incremental\ Profit
\rightarrow
MROI.
$$

The Favorita dataset does not provide enough information to estimate this
profitability layer credibly.

In particular, it does not identify:

- discount depth;
- regular and promotional prices;
- contribution margins;
- campaign and display costs;
- supplier funding;
- cannibalization across products;
- stock availability and lost sales.

The causal analysis therefore supports **incremental demand analysis**, not
promotion-profit optimization.

---

### 10.8 Does retrospective relative uplift transport through time?

The retrospective heterogeneity results are not automatically deployable.

V4 therefore reconstructed the complete causal pipeline independently at each
historical forecast origin using only information available by that date.

This experiment revealed substantial temporal instability.

The point-in-time historical doubly robust relative uplift for the eligible
populations varied considerably across the six origins, approximately from:

$$
-5\%
$$

to:

$$
+19\%.
$$

The quality of the heterogeneous ranking also changed over time.

Some historical origins showed strong separation between low- and
high-response groups, while later origins showed weak or inverted ranking.

Therefore:

$$
\boxed{
\text{retrospective heterogeneous causal structure did not transport stably}
}
$$

through the historical forecasting experiment.

This is an important limitation of the retrospective top-response results.

The approximate 44% uplift observed for the retrospectively selected top 10%
should therefore be interpreted as evidence of historical heterogeneity, not
as a stable production rule that can automatically be carried into future
periods.

---

### 10.9 Causal forecasting integration

V4 also tested whether explicit causal relative-uplift modeling could improve
future demand forecasts.

At every forecast origin, propensity models, outcome models, baseline support
thresholds and relative-effect calibration were reconstructed using only past
information.

The results were clear.

Pooled WAPE was approximately:

$$
13.40\%
$$

for V1,

$$
15.79\%
$$

for the raw relative T-learner,

$$
16.84\%
$$

for the exploratory causal/predictive hybrid, and:

$$
32.90\%
$$

for causal-only V4.

Causal calibration also failed to improve forecasting even when evaluation was
restricted to promoted observations satisfying the causal-support rules:

$$
0/6
$$

forecast origins improved relative to the raw T-learner.

The hybrid diagnostic, which retained predictive promotion estimates outside
causal support and introduced causal calibration only in supported regions,
also underperformed the raw T-learner in:

$$
6/6
$$

origins.

The conclusion is therefore stronger than simply saying that causal support
was too restrictive.

In this experiment:

$$
\boxed{
\text{causal calibration itself did not improve point forecasting}
}
$$

---

### 10.10 Why causal usefulness does not imply forecasting usefulness

Forecasting and causal inference optimize different objectives.

Forecasting asks:

> Given everything known today, what sales value should I predict?

Causal inference asks:

> How would expected sales change if I intervened on the promotion decision?

A predictive model may exploit associations that are highly useful for
forecasting even when they cannot be interpreted as causal.

A causal model deliberately restricts those relationships in order to estimate
incremental treatment response.

Therefore:

$$
\boxed{
\text{better causal interpretation}
\not\Rightarrow
\text{lower forecast error}
}
$$

The V4 results provide an empirical example of this distinction.

---

### 10.11 Main limitations

**Observed sales are only a proxy for demand.**

Low sales can reflect low demand, stockouts or operational constraints that
are not observed directly.

**Promotion treatment is coarse.**

The binary treatment combines very different promotion intensities and
unknown discount depths.

**Unobserved confounding may remain.**

Promotion managers may use commercial information that is absent from the
dataset.

**Treatment overlap is incomplete.**

Some store-family contexts are almost always promoted or almost never
promoted, limiting credible counterfactual comparison.

**Relative effects are unstable near zero baseline demand.**

This required an explicit low-baseline guardrail.

**Heterogeneous effects are temporally unstable.**

The strong retrospective ranking signal did not consistently reproduce in the
point-in-time V4 windows.

**Raw individual uplift percentages are not calibrated causal estimates.**

The relative T-learner is primarily useful for ranking populations, with
group-level doubly robust estimates used for causal magnitude.

**The V4 experiment is a development backtest rather than untouched external
validation.**

The V4 architecture was informed by the earlier retrospective causal
diagnostics, although every numerical quantity used at each forecast origin
was re-estimated using only historical information available at that date.

---

### 10.12 Final system design

The project therefore ends with three deliberately separated layers.

#### Demand forecasting

Use the frozen V1 model for operational short-horizon demand prediction:

$$
\boxed{
\text{V1 forecasting model}
}
$$

#### Promotion causal analysis

Use the support-aware causal pipeline to estimate relative incremental demand
and investigate which populations appear more promotion-responsive:

$$
\boxed{
\text{relative causal uplift analysis}
}
$$

#### Promotion optimization

With richer commercial data, extend:

$$
Baseline\ Demand
\times
Relative\ Uplift
=
Incremental\ Units
$$

into:

$$
Incremental\ Profit
$$

and ultimately:

$$
MROI.
$$

The broader methodological conclusion is therefore:

$$
\boxed{
\text{Prediction}
\neq
\text{Causal Inference}
\neq
\text{Economic Optimization}
}
$$

A robust retail decision system should connect these three layers while
evaluating each with the objective and assumptions appropriate to its role.